# Fused row-wise softmax

PyTorch's eager softmax is several passes over memory (max, subtract, exp, sum, divide). When a row fits in one block, a fused kernel loads it **once**, does everything in registers, and stores it once.

Write `softmax_rows(x)`.

## Inputs
- `x`: `float32` CUDA tensor of shape `(M, N)` with $1 \le N \le 8192$. Rows may be non-contiguous in memory: use `x.stride(0)` (the column stride is 1).
- Entries may be `-inf` (masked positions), but every row has at least one finite entry.

## Output
- `float32` tensor of shape `(M, N)`, contiguous: the softmax of each row, $y_{ij} = e^{x_{ij} - m_i} / \sum_k e^{x_{ik} - m_i}$ with $m_i = \max_k x_{ik}$.

## Requirements
- One program per row. `BLOCK = triton.next_power_of_2(N)` so the whole row is one block.
- Padding lanes must not change the max or the sum.
- Subtract the row max before `exp`: rows contain values up to about 1000.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from an equivalent PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def softmax_rows(x):
    """x: (M, N) float32 CUDA tensor (row stride may differ from N). Return the row softmax, (M, N)."""
    raise NotImplementedError


@triton.jit
def _softmax_kernel(x_ptr, y_ptr, N, stride_x, stride_y, BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]], "dtype": "float32"}], "e": {"__nd__": [[0.09003057330846786, 0.2447284758090973, 0.6652409434318542], [0.3333333432674408, 0.3333333432674408, 0.3333333432674408]], "dtype": "float32"}}, {"a": [{"__nd__": [[1000.0, 1001.0, 999.0, 1000.0]], "dtype": "float32"}], "e": {"__nd__": [[0.1966119259595871, 0.534446656703949, 0.07232949137687683, 0.1966119259595871]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.1012624502182007, 0.3384312689304352, -0.5399715304374695, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}], [-2.276782751083374, 0.9251465797424316, -2.026845693588257, 1.8596241474151611, 0.5905681252479553, -0.47184452414512634, 1.3429964780807495, 0.020786462351679802, 0.6909209489822388, 0.10722707957029343], [{"__f__": "-inf"}, 1.0614224672317505, -0.9073341488838196, -0.61249178647995, {"__f__": "-inf"}, -0.21001015603542328, -2.284914255142212, 2.0261175632476807, -2.1749773025512695, {"__f__": "-inf"}]], "dtype": "float32"}], "e": {"__nd__": [[0.602378785610199, 0.28091588616371155, 0.11670535802841187, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0052442182786762714, 0.1289023905992508, 0.00673328572884202, 0.32817041873931885, 0.09224767237901688, 0.03188272565603256, 0.19576312601566315, 0.05217979475855827, 0.10198541730642319, 0.05689093470573425], [0.0, 0.23222991824150085, 0.03242634981870651, 0.04354581609368324, 0.0, 0.06512413918972015, 0.008177538402378559, 0.6093683838844299, 0.00912783108651638, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[5.0], [-7.0]], "dtype": "float32"}], "e": {"__nd__": [[1.0], [1.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-6.3805623054504395, 2.824256181716919, 8.980062484741211, -1.1770752668380737, -1.9553866386413574, 0.8695073127746582, -1.8342986106872559, 0.41182610392570496, -5.699857711791992, 2.191287040710449, 1.9628970623016357, -12.17708683013916, -0.5511296987533569, 0.8145790696144104, 2.806123733520508, -4.080448627471924, -6.640171527862549, -4.7536702156066895, 2.7402048110961914, -4.259177207946777, -3.8207192420959473, 5.718827724456787, -3.930788040161133, -4.493163108825684, -0.45618778467178345, 0.99305260181427, 5.188148021697998, -4.3966383934021, -7.84077262878418, -0.6124832630157471, 3.218183994293213, -7.407632350921631, -2.4363150596618652, 1.5309789180755615, 1.1551214456558228, 0.41679495573043823, -1.2844476699829102], [-3.996732234954834, -4.640689849853516, 5.781311511993408, -5.586048126220703, 6.140000343322754, -6.575292587280273, 14.517909049987793, -8.69055461883545, 8.021371841430664, -4.485360622406006, -0.5780773758888245, -0.3277628421783447, -3.261113405227661, 11.141963005065918, 0.4745767116546631, -6.605368614196777, 3.2328014373779297, 3.562008857727051, -5.737414836883545, 2.678065299987793, 0.3447789251804352, 4.101678371429443, 3.5609772205352783, -0.04221203178167343, -2.041414260864258, 4.551863193511963, 1.5659959316253662, -7.446386814117432, -2.7243459224700928, -2.416562080383301, -1.3107726573944092, 11.421460151672363, -1.8864364624023438, 2.386683464050293, 8.276666641235352, 2.5182676315307617, 2.1593878269195557], [6.48979377746582, 0.986553430557251, 0.9299118518829346, 3.443692684173584, 2.060384511947632, -3.401787519454956, -5.829453945159912, -0.5668624639511108, 6.990060329437256, -1.132078766822815, 5.742147445678711, 2.0629935264587402, -1.544485092163086, -13.166975021362305, 1.1106683015823364, -1.3847256898880005, -8.002596855163574, 2.572524070739746, -0.12711258232593536, 1.87526535987854, -3.0220794677734375, 1.0101128816604614, -2.366159677505493, 5.324733257293701, 0.6216282844543457, -7.01262092590332, 1.071580171585083, 0.33820557594299316, -1.0172220468521118, -6.748311519622803, 5.297688961029053, -3.657984733581543, 7.913177013397217, -3.299720048904419, 0.1623843014240265, -0.22406460344791412, 0.7558721303939819], [13.21707534790039, -2.491438865661621, 10.347286224365234, 3.11729097366333, -3.4750378131866455, 6.277802467346191, 4.204122066497803, 4.772184371948242, -6.669921875, 2.5140581130981445, -2.0125794410705566, -0.7108280062675476, -0.6572914123535156, 0.20334172248840332, 7.9522175788879395, 1.8922009468078613, -9.22380542755127, 6.234338283538818, 0.43821555376052856, 0.6576368808746338, -9.133299827575684, 1.9987666606903076, -2.4948272705078125, 4.778099536895752, 2.2811410427093506, 4.138596534729004, -1.8652591705322266, -9.146743774414062, 4.843963623046875, 2.1017496585845947, -1.6530989408493042, -2.7853221893310547, -4.428056716918945, -8.81268310546875, -5.8860182762146, -0.3505612015724182, 8.047320365905762], [2.2658257484436035, 8.023893356323242, -5.049116134643555, 6.283408164978027, -2.0806655883789062, 7.331645965576172, 6.141976356506348, 0.1768140345811844, 0.6724849939346313, 6.087173938751221, -3.9788219928741455, -10.814554214477539, 4.084648132324219, 10.895608901977539, -9.480916976928711, 1.0482125282287598, -5.910771369934082, -1.4501004219055176, -5.000121116638184, 4.374605178833008, -2.7259202003479004, -0.964238166809082, -5.418849468231201, 3.6644628047943115, -2.549368381500244, -1.8918726444244385, -0.37799134850502014, -7.258813858032227, -0.10421276092529297, -8.059478759765625, -1.0905159711837769, -1.0987707376480103, -2.0018210411071777, -0.9991480112075806, -3.3721230030059814, 2.553816795349121, -0.017083927989006042], [-2.357511281967163, -1.1461318731307983, -3.385303497314453, -0.6375234723091125, -1.5668411254882812, -9.008167266845703, 8.237719535827637, -2.2249093055725098, -3.357191324234009, 1.502158761024475, -10.446245193481445, -1.9891939163208008, 1.5316667556762695, -1.3236110210418701, 0.31900978088378906, -2.2224326133728027, -1.8428738117218018, -0.20432713627815247, 0.7577710151672363, -5.476387023925781, 1.3942902088165283, 1.486557126045227, -3.094327688217163, -2.8651552200317383, 0.584106981754303, 5.814137935638428, 5.437135696411133, -4.160998821258545, -2.710212469100952, -2.8128342628479004, 1.9021613597869873, -0.8078786134719849, 1.5855607986450195, 6.215267181396484, 1.0473581552505493, 1.9624494314193726, 1.6596442461013794], [-3.9147517681121826, -0.8849633932113647, -1.3015860319137573, -2.698202133178711, 7.395681381225586, 2.639791488647461, -13.617536544799805, -4.274394512176514, -0.6905125379562378, -0.22413906455039978, -4.332582950592041, -1.8552000522613525, -2.185300827026367, -2.57157301902771, 2.9374756813049316, -0.47468289732933044, -3.0571982860565186, -1.6917835474014282, -0.23501954972743988, -5.712746620178223, -2.963489532470703, 2.8188319206237793, 4.787574291229248, -11.952916145324707, 12.100770950317383, -3.2516651153564453, 3.9266297817230225, -6.890009880065918, 3.454446315765381, 0.1279885470867157, -0.4004718065261841, -1.500507116317749, -1.2645092010498047, 3.683793783187866, -3.7735044956207275, -1.275113821029663, -1.9741590023040771], [-8.176451683044434, 0.46761757135391235, -0.19451187551021576, -2.4054291248321533, -6.839879989624023, -1.747335433959961, -0.8156474828720093, -0.05100807547569275, 0.25725656747817993, -5.998701095581055, 5.10971736907959, 3.9665660858154297, 3.948681116104126, 2.336538791656494, 6.587038040161133, 3.6610360145568848, 5.642842769622803, 1.636254906654358, 0.73001629114151, 3.171260118484497, -3.511873483657837, -1.1299093961715698, 1.0379841327667236, -2.9402551651000977, -12.224688529968262, -2.9183592796325684, -3.714277982711792, 4.301339149475098, -9.54255199432373, -3.8505938053131104, 7.953648090362549, -11.83401107788086, -1.384145736694336, -3.236823558807373, -8.615216255187988, 0.004569089971482754, -4.46788215637207]], "dtype": "float32"}], "e": {"__nd__": [[1.984328150683723e-07, 0.001973401289433241, 0.9303537011146545, 3.609601117204875e-05, 1.657459324633237e-05, 0.0002794336760416627, 1.8708145944401622e-05, 0.00017681141616776586, 3.919581956779439e-07, 0.0010479013435542583, 0.0008339343476109207, 6.02857819220759e-10, 6.750002648914233e-05, 0.00026449881261214614, 0.0019379410659894347, 1.9794301806541625e-06, 1.5306174816487328e-07, 1.0096334790432593e-06, 0.0018143134657293558, 1.6554624835407594e-06, 2.5664878648967715e-06, 0.03567071631550789, 2.298989329574397e-06, 1.3100882370054023e-06, 7.422268390655518e-05, 0.0003161795611958951, 0.020981695502996445, 1.4428484291784116e-06, 4.6073608928054455e-08, 6.348313763737679e-05, 0.0029261463787406683, 7.10497687350653e-08, 1.0246565580018796e-05, 0.0005414423067122698, 0.00037180856452323496, 0.00017769215628504753, 3.2421117794001475e-05], [8.402466278312204e-09, 4.413057030916434e-09, 0.0001482372172176838, 1.7146507547138867e-09, 0.00021219435438979417, 6.376058014900821e-10, 0.9230247735977173, 7.689633652052663e-11, 0.001392525271512568, 5.154638049731375e-09, 2.5651320356701035e-07, 3.294730959169101e-07, 1.753403999771308e-08, 0.03155430406332016, 7.349733550654491e-07, 6.187146905922702e-10, 1.1591866496019065e-05, 1.611114203114994e-05, 1.4737979725509831e-09, 6.656325240328442e-06, 6.45507270746748e-07, 2.7637695893645287e-05, 1.6094531019916758e-05, 4.383622069781268e-07, 5.937322100635356e-08, 4.335254925535992e-05, 2.1891169126320165e-06, 2.668338050515473e-10, 2.9991429784104184e-08, 4.080054694099999e-08, 1.232832858022448e-07, 0.04172945022583008, 6.932607732323959e-08, 4.973807790520368e-06, 0.001797530218027532, 5.673293799191015e-06, 3.962559276260436e-06], [0.12457747012376785, 0.0005074726068414748, 0.0004795274289790541, 0.005922902841120958, 0.0014851540327072144, 6.303479494818021e-06, 5.562348519561056e-07, 0.00010734273382695392, 0.20544828474521637, 6.0996135289315134e-05, 0.058984898030757904, 0.0014890339225530624, 4.038280894747004e-05, 3.61920576841257e-10, 0.0005745330126956105, 4.7378263843711466e-05, 6.331028146178141e-08, 0.002478511305525899, 0.00016663015412632376, 0.0012341708643361926, 9.2147902250872e-06, 0.0005195703124627471, 1.7756108718458563e-05, 0.03885618597269058, 0.0003523121413309127, 1.7037869781688642e-07, 0.00055250886362046, 0.00026536197401583195, 6.842013681307435e-05, 2.2192357107542193e-07, 0.037819430232048035, 4.878825620835414e-06, 0.517138659954071, 6.980839771131286e-06, 0.00022257710224948823, 0.00015123345656320453, 0.00040292940684594214], [0.9341458082199097, 1.406998535458115e-07, 0.05297623574733734, 3.838256816379726e-05, 5.261651025989522e-08, 0.000905162887647748, 0.00011379909119568765, 0.00020083764684386551, 2.155765566058676e-09, 2.0996811144868843e-05, 2.2712222857990128e-07, 8.348397386725992e-07, 8.807522249298927e-07, 2.0826771560678026e-06, 0.004829653538763523, 1.1274159987806343e-05, 1.6767333355094394e-10, 0.000866663467604667, 2.634064230733202e-06, 3.2803475278342376e-06, 1.8355662823044128e-10, 1.2541951036837418e-05, 1.4022390359968995e-07, 0.0002020291576627642, 1.6634077837807126e-05, 0.0001065813994500786, 2.63172211134588e-07, 1.8110540844773482e-10, 0.0002157836133847013, 1.3902408682042733e-05, 3.2537190008952166e-07, 1.0487252666280256e-07, 2.028762402517259e-08, 2.5293686589655806e-10, 4.721135926644138e-09, 1.1969195838901214e-06, 0.005311517510563135], [0.00016019238682929426, 0.05073864758014679, 1.0661123894806224e-07, 0.008901347406208515, 2.0748402675963007e-06, 0.025392159819602966, 0.007727387826889753, 1.9833329133689404e-05, 3.2558378734393045e-05, 0.007315302733331919, 3.109036583737179e-07, 3.3412198008342386e-10, 0.0009875211399048567, 0.8964149355888367, 1.267933091853024e-09, 4.740664007840678e-05, 4.5039222129616974e-08, 3.897944679920329e-06, 1.1196473082009106e-07, 0.0013196937507018447, 1.0883138656936353e-06, 6.336405476758955e-06, 7.36596845740678e-08, 0.0006487274076789618, 1.2984631894141785e-06, 2.5059730432985816e-06, 1.138798052124912e-05, 1.1698857882436187e-08, 1.4974324585637078e-05, 5.253141477368217e-09, 5.584717200690648e-06, 5.538806362892501e-06, 2.245052201033104e-06, 6.1190189626358915e-06, 5.70311158298864e-07, 0.00021365586144383997, 1.633754618524108e-05], [1.932056329678744e-05, 6.488065264420584e-05, 6.9128209361224435e-06, 0.00010789492080220953, 4.259940033080056e-05, 2.4984842639241833e-08, 0.7717383503913879, 2.2060132323531434e-05, 7.109912985470146e-06, 0.0009167548269033432, 5.9309912536775755e-09, 2.792402301565744e-05, 0.0009442094597034156, 5.4329659178620204e-05, 0.00028081360505893826, 2.2114836610853672e-05, 3.23239182762336e-05, 0.0001663932780502364, 0.0004354814882390201, 8.541010743101651e-07, 0.0008230126113630831, 0.0009025629260577261, 9.24750384001527e-06, 1.1629280379565898e-05, 0.00036605671630240977, 0.06837887316942215, 0.046902064234018326, 3.1825491078052437e-06, 1.3578244761447422e-05, 1.225393407366937e-05, 0.0013676410308107734, 9.099482849705964e-05, 0.000996493035927415, 0.10212455689907074, 0.0005817491910420358, 0.0014526296872645617, 0.00107311992906034], [1.0962783392187703e-07, 2.268512616865337e-06, 1.4955614915379556e-06, 3.7005102626608277e-07, 0.008953225798904896, 7.700568676227704e-05, 6.6996937511665244e-12, 7.651207312164843e-08, 2.755434707069071e-06, 4.392720256873872e-06, 7.218700659450406e-08, 8.597511396146729e-07, 6.180332547955913e-07, 4.200064438464324e-07, 0.00010370636300649494, 3.419194172238349e-06, 2.5843513640211313e-07, 1.0123802667294513e-06, 4.345184606791008e-06, 1.8157731318524384e-08, 2.838237662672327e-07, 9.210412099491805e-05, 0.0006596187595278025, 3.539895113147118e-11, 0.9894046783447266, 2.1276279937865183e-07, 0.00027886213501915336, 5.594775309702982e-09, 0.00017390932771377265, 6.246843895496568e-06, 3.6825888400926488e-06, 1.2257839898666134e-06, 1.5520530496360152e-06, 0.000218739514821209, 1.2625940826183069e-07, 1.5356810081357253e-06, 7.633251470906544e-07], [6.557441167842626e-08, 0.00037222536047920585, 0.0001919759379234165, 2.104057784890756e-05, 2.4957444111350924e-07, 4.0631617594044656e-05, 0.00010315518011339009, 0.00022159998479764909, 0.0003016111149918288, 5.78787933136482e-07, 0.03862284868955612, 0.01231345720589161, 0.012095188722014427, 0.002412504516541958, 0.16921408474445343, 0.00907172728329897, 0.0658230111002922, 0.001197674311697483, 0.0004839096509385854, 0.005558825097978115, 6.9588099904649425e-06, 7.533709140261635e-05, 0.000658435164950788, 1.2324963790888432e-05, 1.144478067871546e-09, 1.2597805834957398e-05, 5.6837088777683675e-06, 0.017209572717547417, 1.6728021634548895e-08, 4.959416855854215e-06, 0.6636631488800049, 1.6915177036835871e-09, 5.842455357196741e-05, 9.161944035440683e-06, 4.228451544463496e-08, 0.0002342645457247272, 2.675134737728513e-06]], "dtype": "float32"}}, {"a": [{"__nd__": [[-2.712899684906006, -0.617705225944519, -1.3580076694488525, 1.0709892511367798, 0.004066287074238062, -1.1174064874649048, 0.1938997209072113, 1.8959461450576782, -1.3006846904754639, -1.0197237730026245, -0.11523996293544769, 0.03449530526995659, 0.15315444767475128, -1.5213578939437866, 0.09028220176696777, -1.1879606246948242, -0.7971932888031006, 0.891850471496582, -0.7463492155075073, 0.578991711139679, -0.3523772954940796, 0.14236752688884735, 0.13130082190036774, 0.6171860694885254, 0.7108292579650879, -0.7590239644050598, 1.6389232873916626, 0.5864888429641724, 0.9383177161216736, 0.9797776937484741, -0.4837293028831482, -0.06992749869823456, 0.0771215632557869, 0.2802640497684479, 1.6931449174880981, 1.8525131940841675, 3.4546384811401367, 1.8364390134811401, 1.6878736019134521, -0.06433764100074768, -0.8311781883239746, 0.025647541508078575, -0.2744070887565613, -1.0003403425216675, -0.23563411831855774, 0.4401979148387909, 0.05740518495440483, 0.9108039140701294, 0.6271004676818848, 0.44251179695129395, 1.2754004001617432, -0.09445314109325409, 0.09075042605400085, 1.4748387336730957, 0.9747516512870789, -1.488274335861206, 0.14021627604961395, 1.5594043731689453, 0.4988020360469818, 1.2492283582687378, -1.4870405197143555, 0.4726766347885132, -1.9848406314849854, 0.7406110167503357, -1.1881482601165771, 1.447251796722412, -0.6468294262886047, 0.1604408323764801, 0.31411877274513245, 0.021568702533841133, -1.501475214958191, 0.7629859447479248, -0.462655633687973, 2.486783504486084, -0.22669552266597748, 0.2588516175746918, -0.29060086607933044, 1.763598084449768, 1.5402147769927979, 0.6544760465621948, 1.0815593004226685, -0.841153621673584, -0.9094827175140381, 1.3376961946487427, -0.822608232498169, -0.26563531160354614, -1.516539216041565, -0.3544527590274811, 0.44749483466148376, -0.0474802665412426, -1.4932805299758911, 0.8252791166305542, 0.19819268584251404, -2.5262417793273926, -1.0744565725326538, -0.32925519347190857, -0.5190133452415466, 0.1748136281967163, 1.0911855697631836, -0.0686315968632698, -0.08056388795375824, 1.3691643476486206, 1.6155253648757935, -0.30130741000175476, 1.631047010421753, -1.7450374364852905, 1.576757550239563, 0.4914526045322418, -0.8366335034370422, -0.49188435077667236, 0.1401863545179367, 1.342147707939148, 0.8810374140739441, 0.3061032295227051, -0.37620416283607483, 0.10031502693891525, -1.204565167427063, -0.40074822306632996, -0.8540200591087341, -0.36138826608657837, 0.44020721316337585, -0.32921794056892395, -1.7816236019134521, 1.0735528469085693, -0.028293510898947716, 0.4118833541870117, -0.7259706258773804, -0.27448907494544983, -0.3141723871231079, 1.4005481004714966, -1.2557907104492188, 0.8570351004600525, -0.11186674237251282, 0.6725757122039795, -0.6501891613006592, -1.6936484575271606, 0.8798738121986389, -0.4135172963142395, 0.4376605749130249, 0.6646175980567932, -1.772765040397644, -0.5414279699325562, -0.24104003608226776, 0.5347626209259033, 0.5671712160110474, 0.7473989725112915, -0.25030726194381714, 1.3253724575042725, 0.014216348528862, -0.08403080701828003, -0.5068209171295166, -1.1470558643341064, -0.03460926190018654, 0.24827995896339417, 0.2641370892524719, 2.1634232997894287, 0.29220935702323914, 0.3616483807563782, -1.31157386302948, -0.2628839612007141, 0.017545267939567566, 0.4623390734195709, -0.21425652503967285, 0.33030471205711365, -1.0041943788528442, 1.1145994663238525, 1.0124289989471436, -1.1544294357299805, -1.5254281759262085, -0.6918083429336548, 0.13486027717590332, -0.9079824090003967, 0.13566268980503082, 0.7225608229637146, 0.9445456266403198, 1.07985520362854, -1.4939899444580078, -2.020508289337158, 0.4616221487522125, -0.8071101307868958, -0.8000057339668274, -1.1210898160934448, -0.688048779964447, 0.4904581308364868, -1.507954478263855, -1.4203639030456543, 0.9426200985908508, 0.5578656196594238, -1.7671607732772827, -1.9262397289276123, 0.2673841416835785, 0.7396389245986938, 1.9406548738479614, 0.5452965497970581, 0.5255514979362488, -0.3737039268016815, -0.06873572617769241, -0.5477381348609924, 1.0371642112731934, 0.24482007324695587, 0.9271412491798401, 0.1761413812637329, -1.0695956945419312, -0.15751969814300537, 0.9333482980728149, 0.23229217529296875, -1.8631068468093872, 0.9185047745704651, -0.5800548195838928, 1.231311321258545, -0.459295392036438, -0.9882007837295532, 0.6913492679595947, -1.2966737747192383, 1.049200177192688, -0.6180459260940552, 0.4867170751094818, -1.5426133871078491, -0.08740663528442383, -0.6421816349029541, 1.4832717180252075, -0.8777981400489807, -0.28004589676856995, -0.9160393476486206, -0.09685338288545609, 0.050303615629673004, 1.0476925373077393, 0.6113992929458618, -1.6015394926071167, 1.784841537475586, 0.058411359786987305, 1.063283920288086, -0.3955858647823334, 0.6994031071662903, -0.16077971458435059, -0.3552517890930176, -0.16999131441116333, -0.9042177200317383, 0.4116048514842987, 0.3295886516571045, 0.6249212622642517, 1.2027095556259155, 0.9799156188964844, 0.4041544795036316, 1.0076812505722046, 0.1571345329284668, 0.9318086504936218, -0.0976504534482956, -1.7889283895492554, -0.716145932674408, -0.9550093412399292, -0.4689790904521942, -0.14893905818462372, 0.10439617931842804, 0.5301694869995117, -1.0346829891204834, -0.4424845576286316, 0.6812386512756348, 0.6124381422996521, 1.2468270063400269, -0.23438908159732819, 2.2310304641723633, -0.6471105813980103, -1.495080590248108, -0.3429053723812103, -1.333650827407837, 0.6275152564048767, -0.11910369247198105, -1.5156625509262085, 0.1492796540260315, 0.9590493440628052, -0.616042971611023, -0.4251432716846466, -0.7881735563278198, -0.6216292977333069, -1.394457221031189, -0.28829699754714966, -1.0966145992279053, 0.09335648268461227, 1.0566164255142212, 0.6891688108444214, 0.2110764980316162, 1.0370559692382812, 0.56704181432724, -0.6472184062004089, 0.3360808789730072, 0.5667043328285217, -0.3010125458240509, 0.5263882875442505, -0.8938995599746704, 0.32508766651153564, 0.5283083319664001, 0.11512940376996994, -0.3321060240268707, 0.22036536037921906, -0.8847764730453491, -0.6056326627731323, 0.4990556240081787, 0.9441207051277161, 0.7904790043830872, -0.07906392961740494, 0.41994011402130127, -0.6344720125198364, 0.20018182694911957, -0.6371357440948486, -1.203263759613037, 0.29426589608192444, -0.3451850116252899, -0.21388591825962067, -0.836739718914032, 1.488804578781128, 0.43819767236709595, -0.8122380375862122, 0.6187402009963989, 0.03566944599151611, -0.7481402158737183, -0.780459463596344, 0.9260759949684143, -0.3226335048675537, 0.3843432664871216, -0.6241826415061951, 0.4740607738494873, 0.4399504065513611, 0.46869510412216187, 0.395734041929245, 0.9948501586914062, 0.754725456237793, -0.5620949864387512, 1.6494990587234497, -0.45044276118278503, -0.5835374593734741, 0.019037220627069473, 0.826495885848999, 0.4775983989238739, -1.060370683670044, -0.45198285579681396, 0.9495616555213928, 0.25828322768211365, 0.24207541346549988, -1.531169056892395, 0.029383685439825058, 0.2713140845298767, -1.0429753065109253, -1.8242313861846924, 0.21899384260177612, 0.7885713577270508, -1.2994213104248047, -0.3215066194534302, 1.3324531316757202, 1.5768933296203613, 0.10983754694461823, -2.224862813949585, 0.17171451449394226, 0.9163401126861572, 2.518503427505493, 1.024078607559204, 0.771682858467102, -0.24172206223011017, -1.4089876413345337, -0.758944034576416, -0.8343595266342163, -0.5582348704338074, 0.7556449174880981, 0.12389646470546722, -1.9716651439666748, -0.3106238543987274, 1.1957429647445679, 0.8336483836174011, 1.5131338834762573, -1.0505566596984863, 0.3970963656902313, -0.32162338495254517, -1.0485124588012695, -0.6802975535392761, -1.7510918378829956, -1.6313860416412354, 0.16583934426307678, -0.1830812394618988, -0.4407960772514343, 1.25017511844635, -0.3594096302986145, 0.569607675075531, -1.070680856704712, -1.1762676239013672, -0.4479520320892334, -0.5532076358795166, -0.3075468838214874, 0.035897646099328995, 1.664878487586975, -0.906947135925293, -1.257534384727478, 0.49448490142822266, 1.2229582071304321, 1.737944483757019, -1.0596107244491577, 0.5099948048591614, -1.78354811668396, -0.06254146993160248, -1.0687874555587769, -0.5446218848228455, -1.462468147277832, 1.2466704845428467, 0.9413481950759888, 0.4693935215473175, -0.9356147646903992, 1.6694142818450928, 1.0587315559387207, -0.2425508201122284, 1.0919418334960938, 0.5062869191169739, 0.6284530758857727, 0.8053941130638123, 0.7811728715896606, 0.02197653241455555, -1.284294605255127, -1.0142771005630493, 0.4540611207485199, -0.40750810503959656, 0.3462398052215576, -1.8191875219345093, -0.40977156162261963, -0.3968091905117035, 1.5216752290725708, -0.27730828523635864, -0.6056787371635437, 0.11931116878986359, 1.1994701623916626, 2.5495996475219727, -1.3348884582519531, 1.7506275177001953, -1.9238853454589844, 1.3605189323425293, 1.4587873220443726, 1.1296741962432861, -2.229581594467163, -0.9365898370742798, 1.7957565784454346, 0.46767324209213257, 1.0651692152023315, 0.06862567365169525, 0.31791648268699646, 0.3528158366680145, -0.6039773225784302, 0.986548662185669, 2.3244054317474365, -1.0626482963562012, -0.954186201095581, -0.9498336911201477, -0.8065987825393677, 0.1624760776758194, 0.4817349314689636, 0.5520080924034119, 0.2539042830467224, -1.202752947807312, -0.1495099514722824, -1.0242431163787842, -0.12407568097114563, 0.5000200867652893, 0.0675613135099411, 1.3124994039535522, 0.1365402191877365, 1.0625452995300293, 2.887129545211792, 0.8283283114433289, -1.5313770771026611, -1.1878430843353271, -0.1332913041114807, 0.788564145565033, -0.36019235849380493, 0.6236777305603027, -0.44114041328430176, -0.25126466155052185, -0.2470417022705078, 1.9755868911743164, 0.9532443284988403, -1.350246548652649, -0.10628993064165115, 1.658305048942566, 0.139583021402359, 2.321028232574463, 0.29023468494415283, 0.2647891044616699, -0.26790136098861694, -0.15310193598270416, -0.14312925934791565, 0.6249590516090393, 1.2163385152816772, -0.292403906583786, 0.6114271283149719, -1.245302677154541, 0.1271517127752304, 0.626497745513916, -0.04928182438015938, -0.7493194937705994, 0.6487183570861816, 1.050462245941162, 1.2501376867294312, 0.5979557633399963, 1.8218854665756226, -0.24064242839813232, 0.5980180501937866, -1.528274416923523, 0.021823404356837273, 0.10801000148057938, 2.1669921875, 1.3190991878509521, 1.2473883628845215, -0.9288393259048462, -0.25161120295524597, 0.18749113380908966, -1.232893466949463, -0.3859810531139374, 0.7671307325363159, -0.03106674551963806, -0.8706114888191223, 1.9287492036819458, 0.4887879490852356, 0.012230290099978447, -1.2512844800949097, 0.9151436686515808, -0.2417575865983963, -0.4346827566623688, -0.6357727646827698, 1.2691559791564941, -1.8457436561584473, -1.0167653560638428, 0.8067200183868408, 0.3579193353652954, 0.5069186687469482, -0.30012908577919006, 0.47080132365226746, -0.7038435935974121, -0.6395004987716675, -0.19606661796569824, -0.5057998895645142, 0.4537142515182495, 0.02013843134045601, -0.669681191444397, -1.1873319149017334, 1.767228126525879, -0.9865624308586121, 0.1263323724269867, -1.418365716934204, -1.2465254068374634, 0.2589513659477234, 1.6492153406143188, 1.5698482990264893, -0.4410865902900696, -0.8937116861343384, 0.7718194127082825, 1.3606408834457397, -0.7563948035240173, -0.6894090175628662, 1.5125553607940674, -0.8224709630012512, 0.850003182888031, -0.7107597589492798, 0.435931921005249, -1.3219642639160156, 0.4895605444908142, -1.033728003501892, -1.1935480833053589, -0.5128194689750671, 1.260526418685913, 0.3461116850376129, 0.06030762195587158, 0.7933574914932251, -0.6044780611991882, 1.140871524810791, -1.3061810731887817, 2.0449962615966797, 0.5018421411514282, 0.22489331662654877, -2.310586452484131, 0.048424817621707916, -0.0029336833395063877, 0.7391248941421509, -1.0846836566925049, 1.7207071781158447, -0.14964039623737335, 0.36149337887763977, -0.5482578873634338, 2.5214459896087646, -0.5394930243492126, 0.16224484145641327, 1.0261240005493164, 2.410665988922119, 0.09641078859567642, -0.44830840826034546, -0.22277620434761047, 0.14995801448822021, 0.6994128823280334, 0.12894411385059357, -1.0118380784988403, 0.13327264785766602, 0.24495942890644073, -0.4725310206413269, 1.011426329612732, 1.3606200218200684, -2.291294813156128, 0.6804770231246948, 0.7475550174713135, 1.6281384229660034, 0.8993785977363586, 1.7288683652877808, 1.2337415218353271, -0.8623285293579102, -0.4486355781555176, -0.6169214844703674, 0.6591103672981262, -0.310668408870697, -0.8663884997367859, 0.5566091537475586, 1.5213943719863892, 0.13527381420135498, -0.501440167427063, -0.7020705342292786, -0.1349455863237381, 0.08732881397008896, 0.28613507747650146, 0.7963420748710632, 0.21882736682891846, -1.8866022825241089, 1.0659092664718628, 3.169635057449341, -0.7635179162025452, -1.2757093906402588, 1.4125834703445435, 1.808849573135376, 0.34608399868011475, 2.151397228240967, 1.9193003177642822, 1.0195903778076172, 2.6039774417877197, -1.7837255001068115, 1.1715044975280762, -1.102477788925171, -0.7048530578613281, -0.19932429492473602, -0.33353862166404724, -0.019130922853946686, 2.3215742111206055, 0.4032445549964905, -0.4143518805503845, -0.5240257978439331, -0.17550930380821228, -1.6878297328948975, 2.062448263168335, -0.7339510321617126, 1.9227190017700195, -1.386663556098938, -0.8435102701187134, -0.7347551584243774, 0.4960346817970276, 0.811703622341156, -1.2243424654006958, -1.5235605239868164, -0.13185380399227142, 0.28871917724609375, 0.48236083984375, -2.0211849212646484, 1.487839698791504, -0.07326330989599228, 0.41389018297195435, 0.5421633720397949, -0.922919511795044, 1.0108003616333008, 1.7811484336853027, 0.4659050405025482, -0.7544505596160889, -0.4936780631542206, 0.22988329827785492, 0.19660551846027374, 1.1065163612365723, -0.8014124035835266, -0.21556107699871063, 0.3964647948741913, 0.1128901019692421, -0.1619042307138443, -0.8825975060462952, -0.44521230459213257, 0.002261578105390072, 1.1772397756576538, -0.09080682694911957, 1.473362922668457, 0.7838230133056641, 0.31692788004875183, 0.44055041670799255, 0.2002800554037094, -0.3448520600795746, -0.6221181750297546, 1.369968056678772, -0.6755138039588928, 1.0938405990600586, -0.6890007853507996, 0.22440136969089508, 0.7652025818824768, 1.3493988513946533, 0.4052034616470337, -0.8386005163192749, -1.1536139249801636, -0.7078203558921814, 0.8240869045257568, -1.0317997932434082, 0.9333612322807312, -0.016237106174230576, -1.708678126335144, -0.667375385761261, 0.4517819285392761, -0.8995386958122253, 0.24431513249874115, 0.4585883319377899, -2.258347511291504, -2.1864209175109863, -0.9334760308265686, -1.625911831855774, 0.3637346923351288, -0.1316964477300644, -0.783598005771637, 0.17221729457378387, 0.4160146415233612, -2.0409162044525146, -1.0022350549697876, -0.4059177339076996, 1.5709937810897827, -0.25099462270736694, 1.2838635444641113, -1.3609403371810913, -1.2233372926712036, 0.5460708737373352, -0.5326452255249023, 1.4027737379074097, 0.8289719820022583, -1.0514241456985474, -1.3545647859573364, 0.9297327399253845, 0.3715551495552063, 0.3992440402507782, -1.1592119932174683, 0.31170710921287537, -1.4579070806503296, 0.37525489926338196, 0.5318344235420227, -0.7634512782096863, -0.6726502776145935, 3.318483829498291, 0.25733062624931335, 0.0874304473400116, -1.0930159091949463, 0.6263253092765808, 0.19150175154209137, 0.3090207576751709, 1.2423269748687744, 0.5986336469650269, 2.63387393951416, -0.5056797862052917, -0.7244845032691956, 0.944688081741333, -0.8365076780319214, 0.8987284302711487, -1.1886855363845825, 0.11070813238620758, 2.158181667327881, 0.12118431180715561, 0.4670651853084564, 2.0063581466674805, 0.609605610370636, -0.4199410676956177, -0.11656314879655838, 1.7505367994308472, 1.6551339626312256, 1.529373049736023, 0.14627258479595184, -0.6231218576431274, 0.39262357354164124, -1.5560981035232544, -1.418063759803772, -0.7127766013145447, -0.04756529629230499, 0.5771815776824951, -2.302910327911377, 1.8389060497283936, -0.09475699812173843, 0.8396637439727783, 1.7494146823883057, 0.573621392250061, 1.9473991394042969, -0.32823866605758667, -1.062883734703064, 0.06305088102817535, -0.8376298546791077, 1.6071356534957886, -1.9693002700805664, 0.5493793487548828, 0.3372539281845093, 0.4571116268634796, -0.37135428190231323, -0.8161616921424866, -0.1457548886537552, -1.739216685295105, -1.1813910007476807, -0.4998130798339844, -1.0596606731414795, -0.6839905381202698, 0.4763261675834656, -2.497640371322632, -0.8943396210670471, -1.6902570724487305, 0.29878947138786316, 0.8925885558128357, 0.097377248108387, -1.630159854888916, -0.6702303886413574, -0.7740591168403625, 0.06950987130403519, 1.1283212900161743, -0.5373690724372864, 1.7600610256195068, 0.4974062442779541, 1.4093478918075562, -0.33416610956192017, -1.4540830850601196, 2.8784615993499756, 0.12807896733283997, -0.7253988981246948, -1.456998348236084, -2.2409443855285645, 1.5098541975021362, 0.2687980830669403, 0.1113462746143341, -0.7228425145149231, 0.46649661660194397, 1.1863117218017578, -0.9204370975494385, -0.4259212613105774, -1.6790117025375366, -1.4303760528564453, -0.5999467968940735, 0.19349361956119537, 2.2207040786743164, 0.5665575861930847, 0.37597978115081787, 0.105125792324543, 1.0587631464004517, -0.230625182390213, -1.0189875364303589, -0.6944846510887146, 0.06533337384462357, 0.657419741153717, 0.6545246839523315, 0.31675466895103455, -0.08966410160064697, 0.7863269448280334, 0.4219902455806732, 1.606469988822937, -0.4835177958011627, 0.3579826354980469, -0.19547104835510254, -0.4560677707195282, 1.1411367654800415, 0.5078539252281189, -0.28973153233528137, -0.14297404885292053, -0.5265905857086182, 0.8943100571632385, 0.34363794326782227, 0.6359137892723083, 0.18180368840694427, 1.125241756439209, 1.5489987134933472, -0.5745829939842224, -0.5175478458404541, -0.015058929100632668, -0.652653157711029, -0.1150086298584938, 1.0635381937026978, -0.5594760775566101, -0.9249100089073181, 0.12778763473033905, 0.9130098819732666, -0.02094283513724804, -0.9790565371513367, 0.4502406716346741, -0.0843680128455162, -0.4051867127418518, 0.1555148810148239, 0.6090808510780334, -0.5328921675682068, 0.16832783818244934, 0.5210671424865723, -0.8647277355194092, 0.02406858652830124, -0.3628687262535095, -0.14407898485660553, 0.8196674585342407, 1.038399338722229, 0.2949858605861664, 1.3406016826629639, -0.9358614683151245, 0.797885537147522, -0.5014855265617371, -0.45569154620170593, 1.5683790445327759, 0.989896833896637, 1.4458023309707642, 1.6265360116958618, -2.192202091217041, 0.9869728684425354, -0.972379207611084, 0.990738570690155, 2.581589698791504, 0.04630807042121887, -1.946228265762329, 0.4464446008205414, 0.01778446137905121, 1.6510154008865356, 2.3252475261688232, -0.001378179294988513, -1.1777063608169556, -0.9580428004264832, -1.3572410345077515, -1.4094529151916504, -0.4245623052120209, 0.45364412665367126, -1.095906376838684, -0.15543018281459808, 2.9861886501312256, -1.7437593936920166, 0.6917325854301453, 0.23382139205932617, -0.18204616010189056, 0.6181694269180298, -0.3572638928890228, -0.9348795413970947, 0.2659928500652313, -0.07688630372285843, 0.5918949246406555, -1.0707728862762451, -1.125077724456787, 1.686989665031433, -0.01993975043296814, 0.6408283710479736, -0.5683964490890503, 0.09220179170370102, 0.05052397772669792, -0.8520601987838745, 0.662026584148407, 1.2988957166671753, 0.36843591928482056, 1.127429485321045, 0.790042519569397, -0.10837516188621521, 0.1611206978559494, -1.0438790321350098, 0.25056520104408264, 0.07498342543840408, 1.3830082416534424, -1.0964629650115967, -1.2351824045181274, 0.6129905581474304, -1.784654974937439, 0.12895049154758453, 0.8402915000915527, -1.5238374471664429, 0.5033131241798401, 0.24691471457481384, 1.1608699560165405, -0.8810552954673767, -0.5600551962852478, -0.7747516632080078, -0.9226455092430115, 0.8937289118766785, 0.8333778381347656, 1.86252760887146, 1.3452732563018799, -1.3148822784423828, 0.07305008172988892, 0.6686342358589172, 1.2620549201965332, -0.2897713780403137, -1.5826722383499146, -0.5726642608642578, -0.9494264721870422, 0.18804366886615753, 0.9755818843841553, 1.303582787513733, 0.19279855489730835, 0.3405610918998718, 0.5599579811096191, -0.33643844723701477, 0.7569509148597717, 1.933692216873169, -0.9507129788398743, 1.264723777770996, 0.958740770816803, 1.1182327270507812, 1.2833278179168701, 3.644256830215454, -0.042416445910930634, 0.2328624725341797], [-1.1214793920516968, 0.03922227397561073, 0.8966138362884521, -0.45934149622917175, -0.8062304854393005, -0.6704558730125427, -1.4689815044403076, 2.5268402099609375, 0.4654465615749359, -1.126442313194275, -0.04906603693962097, 0.44270896911621094, -0.21828491985797882, -0.3051646649837494, 0.43329888582229614, -0.9507322907447815, 0.11110348999500275, -0.8341395854949951, -1.0123258829116821, 0.5109030604362488, -0.8915020227432251, -0.5174732208251953, 0.060967762023210526, 1.099790334701538, -0.21749337017536163, 0.6854559779167175, -1.415682315826416, -1.0197579860687256, 1.7151097059249878, 0.16964657604694366, -0.28504037857055664, -1.0763837099075317, -0.38153204321861267, 0.3434501886367798, -0.17027446627616882, 0.48915764689445496, -0.2430727481842041, -0.8244973421096802, 2.662334680557251, -0.12041172385215759, -0.018238477408885956, 0.41814497113227844, -0.013912844471633434, 2.4586033821105957, 0.3907458186149597, 0.9268844723701477, 0.2907157838344574, 1.6894421577453613, 1.6782965660095215, 0.09084982424974442, -1.1313616037368774, 0.5905042290687561, -1.9032719135284424, 0.15159475803375244, 1.1228526830673218, 0.2435469627380371, -0.8369035124778748, 2.2263307571411133, 0.8408105969429016, -0.9082372188568115, -1.6364392042160034, -1.0775634050369263, 1.0958199501037598, -0.1941605806350708, 0.6874262094497681, 0.08862975984811783, 0.5822814106941223, -0.6138620972633362, 1.6126114130020142, -0.2976871728897095, -0.08697286248207092, 0.39325034618377686, 0.798532247543335, -0.1194382905960083, -0.44953426718711853, -0.07520128786563873, 2.397124767303467, -1.939947247505188, 0.6005778908729553, 0.2525029480457306, -1.063174843788147, -1.1779630184173584, -0.6584945321083069, -1.2509982585906982, -0.26703301072120667, 0.2927798330783844, 0.20671729743480682, -1.7224055528640747, -1.3591889142990112, 1.8075443506240845, -1.2182869911193848, -0.7140867710113525, 1.4856168031692505, -0.7177452445030212, -0.0601385161280632, 1.1003538370132446, 1.414108157157898, 0.6332772970199585, -0.526843786239624, -2.193899154663086, -0.5520085096359253, 0.3188117444515228, -0.3445015847682953, -0.32442134618759155, 0.5482346415519714, -0.5941275954246521, -1.4632352590560913, -0.35467803478240967, 0.1092020720243454, 0.6267597675323486, -0.17841745913028717, 0.03162184730172157, -1.0225337743759155, -0.4817771315574646, -0.592937707901001, -0.8491414785385132, -0.33152836561203003, 1.4486958980560303, -0.4545358419418335, -0.012751145288348198, -1.88225257396698, 0.3102662265300751, 1.3040764331817627, 0.09941376000642776, 0.4380315840244293, 0.5191600322723389, 1.1933189630508423, -0.1045001968741417, -0.08500939607620239, 0.9131899476051331, -0.47525909543037415, -0.3900115191936493, -0.3683456480503082, 0.2051142156124115, -0.2916574478149414, 1.3885552883148193, -0.13618367910385132, -0.7037561535835266, 1.3744069337844849, 1.1062407493591309, 1.8665379285812378, 0.8460217118263245, -1.0453002452850342, 1.6995302438735962, 0.9836179614067078, -0.9636607766151428, -0.3667208254337311, -1.4144024848937988, 0.20225805044174194, -1.084993600845337, 0.4753301441669464, -2.3275508880615234, 1.071422815322876, 1.5381373167037964, 0.24324947595596313, -1.3374123573303223, -0.9655756950378418, 0.6714019179344177, -2.7262966632843018, 0.6289912462234497, -1.584567904472351, -0.5759565234184265, -0.6653369069099426, -0.11546844244003296, 0.4120863080024719, -0.6136730313301086, 0.0909491628408432, -1.5589274168014526, -0.07447061687707901, -1.3732044696807861, -0.1419631391763687, -0.21911801397800446, -0.004384876694530249, -0.029940811917185783, -0.2872530519962311, 0.6416730284690857, -0.8050668835639954, -0.1159093827009201, -0.21556031703948975, 0.6951669454574585, -0.25561362504959106, -0.5911481380462646, 1.4989159107208252, -0.5357758402824402, -1.5044056177139282, 1.6110469102859497, -0.43273332715034485, -0.058638762682676315, -0.18606410920619965, -0.17320747673511505, -0.4181044101715088, -1.7914929389953613, 0.0941181406378746, 2.1783759593963623, 0.7967287302017212, 0.2483626902103424, -0.45931434631347656, -1.1875793933868408, -0.9017089605331421, -0.8880459666252136, -1.2763478755950928, 1.409253478050232, -0.020707596093416214, -0.7620664834976196, -0.1544545441865921, 1.5515869855880737, 0.678198516368866, 1.130927562713623, -0.7372671961784363, -1.381450891494751, 0.15370824933052063, 0.34351474046707153, 0.8817346692085266, -1.334373116493225, -0.2998257279396057, 1.662491798400879, -1.4604177474975586, -0.3926818370819092, -0.28585752844810486, -1.1178884506225586, 0.9653327465057373, -0.18028689920902252, -1.353928804397583, -0.8556584715843201, -0.7612094283103943, 1.7428061962127686, -0.3541225492954254, -0.6677992939949036, 1.3940050601959229, 0.31676432490348816, 2.3310391902923584, -0.4967324733734131, -1.815561294555664, 1.4573373794555664, -0.4731673300266266, 0.025118308141827583, -0.13670527935028076, -0.6260886192321777, 0.7691690921783447, -0.7306214570999146, 0.46012696623802185, -0.25949203968048096, 0.6414136290550232, 1.9013258218765259, 0.20240916311740875, -0.3086218237876892, 2.7664706707000732, 1.9169268608093262, -0.10919094830751419, 0.9710210561752319, 0.4785977900028229, -0.16451936960220337, -0.8193662166595459, 0.17922614514827728, -1.119040846824646, 1.5988253355026245, 1.7334991693496704, -0.7510777115821838, -0.6091015934944153, -1.1945773363113403, 1.3501653671264648, 0.12832163274288177, -1.0642670392990112, -0.30413511395454407, 1.0439729690551758, 0.3665013313293457, 1.1953250169754028, -2.170559883117676, 0.3840162754058838, 0.5038681626319885, 0.4100458323955536, 0.3336833119392395, -0.4826456308364868, -0.16221272945404053, -0.7450023293495178, -1.3185412883758545, 1.6237813234329224, -0.772051990032196, -0.5501456260681152, -3.0493054389953613, 0.6848044395446777, -0.225920170545578, -0.5509783029556274, -0.24533484876155853, -0.29963502287864685, -0.7515255212783813, -0.7110674977302551, 0.46655720472335815, 0.9740777015686035, 1.1129554510116577, 0.08159659057855606, 1.2592393159866333, 1.1477131843566895, 1.350131630897522, 1.02129065990448, -1.4158295392990112, 0.12193474173545837, 0.4624376595020294, 1.6015136241912842, -0.4077972173690796, 0.5376775860786438, 0.24142935872077942, -1.505656123161316, -0.4820628762245178, -1.0326788425445557, 2.1318416595458984, 0.5075845122337341, 1.2703547477722168, 0.5010296106338501, 0.6477439403533936, 1.2647192478179932, -0.4435780346393585, -0.981626033782959, -0.7848291397094727, 1.1284712553024292, 0.11894924193620682, -0.7636616230010986, -1.2416527271270752, -0.5413167476654053, 0.42745545506477356, 0.043516069650650024, -0.5497681498527527, -0.9843940734863281, -0.4290333390235901, 1.5554417371749878, 2.472681760787964, 0.2909768521785736, 1.645338773727417, 0.18747444450855255, -1.2248420715332031, -0.2766432464122772, -1.1978557109832764, 0.3906036615371704, 0.12179630249738693, -2.0626766681671143, -0.9837815761566162, 1.7979799509048462, -0.5561622977256775, 0.16162113845348358, -0.3715311586856842, -0.8316750526428223, 0.27713876962661743, -0.5161901712417603, -0.6806589961051941, -0.13918207585811615, -2.0063931941986084, -1.8096423149108887, -1.9443333148956299, 1.1035383939743042, 0.4489825963973999, -0.031825702637434006, 0.4179689884185791, 0.7106702923774719, -2.51725697517395, -1.9212045669555664, 0.10570133477449417, 0.9663575291633606, 0.6354088187217712, -1.493332028388977, -0.0892583504319191, 0.8388515114784241, -0.10858271270990372, -0.7473948001861572, -0.6459293365478516, 1.0323576927185059, -1.4410569667816162, -0.6913454532623291, 0.9125707149505615, -0.8224635720252991, -2.8983097076416016, 0.33168503642082214, -0.40903162956237793, -1.6125043630599976, -1.4255309104919434, 0.5048878788948059, 0.7902734279632568, -1.1984047889709473, 0.43272972106933594, -0.374538391828537, -0.01435118354856968, 0.7757927775382996, -1.6113163232803345, -0.6291418075561523, 3.0798146724700928, -1.324005126953125, 0.16907672584056854, -1.1610922813415527, 0.8383819460868835, 0.4282180964946747, -0.7843997478485107, -0.05312514305114746, 0.5158632397651672, 1.2447571754455566, 0.4668731093406677, 1.3465148210525513, 0.2699861228466034, 1.1645594835281372, 0.4481567442417145, 0.19332492351531982, 0.5681055188179016, 0.5728773474693298, 0.13249152898788452, -1.0025713443756104, -0.25122779607772827, 1.2139359712600708, 0.09365572780370712, -0.7965144515037537, -0.4119413197040558, 1.2326297760009766, -1.4107065200805664, -1.587450385093689, -0.7306625843048096, -0.9297811388969421, 0.9281891584396362, 0.2380237877368927, 0.6231802701950073, -0.37565508484840393, 0.7325043082237244, -0.16325223445892334, -0.9940526485443115, -0.8595595955848694, 0.6000074744224548, 0.5263962745666504, -0.6394684910774231, 0.9635207056999207, 0.4443279206752777, -0.3405940532684326, -1.2180838584899902, 1.3841689825057983, 0.3250211477279663, 0.8262199759483337, -0.9073838591575623, 0.2709020972251892, 0.8719857335090637, 0.21141135692596436, -0.01988566294312477, -1.20238196849823, 1.262367606163025, 0.7617283463478088, -0.667220413684845, -1.7953808307647705, 1.0703861713409424, -1.0687850713729858, -0.2885541021823883, -0.16359484195709229, 2.2014718055725098, 0.4390014708042145, 1.6959114074707031, 0.35837453603744507, -0.7618736624717712, 1.2035746574401855, 2.07765531539917, -0.44446682929992676, -0.44815653562545776, 1.1988862752914429, 1.9565902948379517, -0.3317403495311737, -0.6132606863975525, -0.16788195073604584, -0.296845018863678, -0.6659209728240967, 0.5025239586830139, -0.12401539832353592, -0.46045011281967163, 0.7070888876914978, -1.6571617126464844, 1.5262593030929565, 1.682063102722168, 0.6218653321266174, -0.48257017135620117, -0.9111251831054688, -0.8266360759735107, 0.5395205020904541, -0.4202984571456909, 0.7988381385803223, -0.09328091144561768, 0.414524644613266, 1.3398935794830322, 0.34357109665870667, -1.0160993337631226, -0.23225736618041992, -2.10512638092041, 0.24465002119541168, -2.281589984893799, -0.5370669960975647, -0.8056778311729431, -0.3780078887939453, 1.062345027923584, -0.22023455798625946, -0.05682217329740524, -1.581229567527771, 0.09057479351758957, 0.7555246353149414, -1.2925530672073364, -0.6740325093269348, -2.6421544551849365, 0.948612630367279, 0.20644213259220123, -1.8292337656021118, 0.643696665763855, -0.6668880581855774, -1.091821312904358, 1.8771557807922363, 0.038809582591056824, -0.9301914572715759, -1.9626189470291138, 0.6201300024986267, -0.37089744210243225, -1.4833922386169434, -0.10681598633527756, -1.0244296789169312, 0.4256701171398163, 0.8936594724655151, -0.16456924378871918, 0.460354745388031, 0.5545913577079773, 1.4048359394073486, 0.3753493130207062, -1.1966062784194946, 1.2120860815048218, -0.408008337020874, -0.2553510069847107, 0.2433326691389084, 0.42127537727355957, -0.49166542291641235, -0.211411252617836, 0.1344243288040161, -1.0831382274627686, -1.7134119272232056, -0.03225748613476753, 0.08353042602539062, -0.5962693095207214, -0.700786292552948, -0.2734336256980896, -0.16586926579475403, -0.42700719833374023, -0.22593463957309723, -0.17022927105426788, -0.9378567934036255, -2.304352283477783, -0.7247488498687744, 0.03570103645324707, -2.7007460594177246, -1.2137550115585327, -2.15165114402771, -0.580592930316925, 0.3678135573863983, 0.3336394429206848, 0.5234019160270691, 1.4512261152267456, -0.13626261055469513, 0.7186192870140076, 0.9661610126495361, -1.3888705968856812, 0.9689055681228638, 0.7738891243934631, -0.08011060953140259, 0.890362024307251, 0.6600370407104492, 0.07778751105070114, 0.31948986649513245, 0.831100583076477, 0.655013382434845, -1.3845014572143555, 0.8501046895980835, 1.0054608583450317, 0.8049554228782654, -0.11335843056440353, 0.4462859332561493, -0.7041276693344116, -0.0939669981598854, 0.13429418206214905, 0.8425203561782837, -1.0268547534942627, -0.8603340983390808, -0.3682417571544647, 0.7337885499000549, 0.2508762776851654, 1.201294183731079, 0.7301092147827148, -0.6151062846183777, -1.4188038110733032, -1.4339154958724976, -0.7555866837501526, 1.2631900310516357, 0.020204510539770126, 0.6525996327400208, -1.3508275747299194, -0.771766722202301, -0.9015883803367615, -0.8410693407058716, -0.0430513396859169, -0.22613105177879333, -0.8961687684059143, -0.7643694877624512, -0.12943410873413086, -1.3538600206375122, 1.4050482511520386, -0.4722440242767334, -0.612188994884491, 0.3085724413394928, 1.0145461559295654, 0.20179076492786407, 0.17562825977802277, 0.7213725447654724, -0.507543683052063, -0.6716867089271545, -0.4483776092529297, -0.6618067622184753, -1.1389106512069702, 1.6521389484405518, 1.0831996202468872, 0.021492857486009598, 1.1809322834014893, -1.5409878492355347, -0.4667600989341736, 1.4600608348846436, 0.7802481651306152, -0.24809613823890686, -1.4311881065368652, 1.3216047286987305, -1.141679286956787, -0.7645581960678101, -0.04966096207499504, -0.021198276430368423, 0.19016169011592865, 0.3572293221950531, 0.4004839062690735, 2.5462710857391357, 0.4627092778682709, 0.01614787057042122, -2.0494186878204346, 1.0014060735702515, -0.5132911205291748, 0.8465656638145447, 0.11537229269742966, 0.9506508708000183, 1.2116191387176514, 1.8641507625579834, -0.14789170026779175, -1.1364742517471313, 0.04955589398741722, -1.925650954246521, -0.3228864073753357, -0.006777097936719656, -2.487461566925049, 1.0672290325164795, -1.0238161087036133, -0.41409337520599365, 0.22975173592567444, -0.06328175961971283, 0.42369499802589417, -0.047695647925138474, 0.267026424407959, -1.044913411140442, 0.8155245184898376, -0.7696679830551147, -1.248397707939148, -0.242167130112648, -1.1747421026229858, -3.142880916595459, 1.6713218688964844, -0.8685963749885559, -0.13172292709350586, 2.1025190353393555, -1.167565107345581, -0.20991776883602142, 0.4042944312095642, 1.0343843698501587, -1.2816460132598877, -1.3953745365142822, 1.0321120023727417, 0.13797003030776978, -1.2792847156524658, -1.5726598501205444, 0.9237741827964783, 1.3264374732971191, -0.17613619565963745, -0.19948256015777588, -0.5276448726654053, -2.648298501968384, 1.3187822103500366, 0.028661321848630905, 1.262062668800354, -0.642854630947113, -0.3367542028427124, -0.07675487548112869, -0.0634840801358223, -0.3774130940437317, 1.346087098121643, 1.318474292755127, 1.1144829988479614, -0.5181586146354675, -0.11361701041460037, -1.0057740211486816, 2.639012098312378, -0.451785147190094, 0.2526623010635376, -0.8680679202079773, 0.6189295053482056, 0.9303823709487915, -0.9145233631134033, 0.6552199125289917, -0.7873601317405701, -0.5004743337631226, -0.08202241361141205, -0.6942928433418274, 1.0128117799758911, -0.3142062723636627, 0.10459426790475845, 0.5554931163787842, -0.809230625629425, -0.30384302139282227, -0.21661405265331268, -1.8289453983306885, 1.7488614320755005, 0.7311331629753113, 0.4496159255504608, 0.7246503233909607, -1.2398731708526611, -0.2039986103773117, 1.268604040145874, -0.5021433234214783, 0.2325456142425537, 0.38770297169685364, -1.5879307985305786, -0.5804014205932617, 0.5794149041175842, 0.12918256223201752, 2.1541695594787598, 0.45855891704559326, -2.2157866954803467, 1.6190338134765625, -1.4085277318954468, 0.4166150391101837, 0.3178146481513977, 0.29044029116630554, 0.2223052829504013, -0.8559183478355408, 0.6945271492004395, 0.2555784583091736, -0.4640139043331146, 0.7166336178779602, 2.408888816833496, -1.1728802919387817, -0.604435920715332, -0.1219019666314125, -0.9740276336669922, -0.038140080869197845, -0.9572827816009521, 0.035352855920791626, 0.9739869832992554, 0.2553965747356415, -0.31177079677581787, 0.4400399923324585, 0.22837908565998077, 0.721035897731781, -0.14119502902030945, -1.3981139659881592, 0.7349870800971985, -0.34464380145072937, -0.28105428814888, 0.9024535417556763, 0.35414257645606995, -0.43953248858451843, 2.398378372192383, 1.7303603887557983, -0.4639172852039337, -1.7593859434127808, 1.0899332761764526, 0.4312821924686432, 0.007508752401918173, 0.3232138156890869, -0.11407645791769028, 0.6410816311836243, 0.12337297201156616, 0.9704440236091614, 1.1323930025100708, -0.024259822443127632, 0.49421775341033936, 0.6037214398384094, 0.021506095305085182, 0.6513585448265076, -0.07549602538347244, 0.9764223098754883, -0.5382068753242493, -0.1830563247203827, -1.6460586786270142, -1.4351640939712524, 0.3663240373134613, -0.09372112900018692, 0.02742595411837101, 1.5185474157333374, 0.03540878742933273, 2.305332899093628, -1.76673424243927, 0.17582525312900543, 1.2210644483566284, 0.4310585558414459, 0.6739506125450134, -0.5924132466316223, -1.6799211502075195, -0.10030506551265717, 0.8762355446815491, 0.4238983988761902, -1.044247031211853, 0.72537761926651, -0.08362477272748947, -0.12504716217517853, 0.5037374496459961, -0.9518633484840393, 1.1035025119781494, 2.0024571418762207, -1.1821810007095337, 0.14000220596790314, -1.2061262130737305, 0.3103082478046417, 0.6406600475311279, 1.7602241039276123, -0.8733630776405334, -0.479227751493454, 0.71866774559021, -0.089366614818573, 0.48682042956352234, 1.7660082578659058, 0.7422913312911987, -1.2304608821868896, 1.7470574378967285, -1.303819179534912, 0.8052682280540466, 0.16488564014434814, 0.49744415283203125, 0.13881291449069977, 0.46905717253685, -1.950268030166626, 0.3539240062236786, -0.8854379057884216, -0.8183109760284424, 0.03307873010635376, 0.09860165417194366, -0.35041677951812744, 1.8436388969421387, 0.8011624217033386, -2.0051634311676025, -0.525266170501709, -1.0585441589355469, -0.631777822971344, 1.7637020349502563, 0.04128340631723404, 0.8915242552757263, 0.6574690937995911, -1.2709673643112183, -1.0988976955413818, -0.23227053880691528, 0.26393839716911316, -0.8136240839958191, 0.40992897748947144, 0.04107998311519623, 0.31606364250183105, 1.2810523509979248, 0.648442804813385, 1.859679102897644, -0.40722671151161194, 0.13247455656528473, -2.0173544883728027, -0.15387079119682312, 0.8624300360679626, -1.6210417747497559, -0.9110972881317139, 0.6123599410057068, -0.7605147361755371, 0.6048470735549927, 1.8322346210479736, 1.1087614297866821, -1.3817849159240723, -0.1413375586271286, 0.25038647651672363, -0.43947744369506836, -0.14772146940231323, -0.17121224105358124, -1.899011254310608, 0.6892682313919067, 0.8582718968391418, 1.4675021171569824, -1.8035708665847778, 0.316249281167984, 0.9919372200965881, 0.062407948076725006, -0.9401029944419861, -1.4388874769210815, 1.1967449188232422, -1.2607239484786987, -1.0209996700286865, 1.64901602268219, -1.7612197399139404, 0.6007413864135742, 0.41359567642211914, 0.033444784581661224, 0.4114905893802643, -1.406421184539795, -0.010781618766486645, 0.8182429075241089, 0.35141271352767944, -0.04398205131292343, -1.5771719217300415, -0.41923171281814575, 1.4769164323806763, 1.033905029296875, 0.5803134441375732, 0.14836817979812622, 0.9556900858879089, 0.5444757342338562, -0.9384940266609192, 0.5756096243858337, 0.7844091057777405, -1.257948637008667, 1.0141770839691162, -0.8833622336387634, 0.8651139736175537, 0.01573457568883896, 0.30709394812583923, -1.181169033050537, 1.0160623788833618, -0.19905011355876923, -0.4699890911579132, 1.1997812986373901, -0.550139844417572, 0.24663424491882324, 1.5372804403305054, -0.5677654147148132, -1.0209225416183472, 0.07601058483123779, -0.513470470905304, -0.20599402487277985, -0.7736221551895142, -0.7736648917198181, -2.0839755535125732, -1.4115666151046753, 0.15955013036727905, 0.04659377038478851, 0.4568742513656616, 1.2083959579467773, -0.9188737273216248, -0.06297965347766876, -0.5670714974403381, 0.49376389384269714, -0.7696893811225891, 0.5429316163063049, -1.2521387338638306, 0.23049089312553406, 0.2918374836444855, 1.1389163732528687, 0.5551743507385254, 0.9166436195373535, 0.2478964626789093, -0.6756529808044434, -0.8472994565963745, 0.20685742795467377, -0.716707170009613, -1.0687698125839233, -2.151654005050659, -1.0061002969741821, -0.9955487847328186, -1.4847781658172607, 0.7771985530853271, -1.6231752634048462, 0.1048969104886055, 0.16503340005874634, -0.573668897151947, -0.8354873657226562, -0.3066733181476593, 0.8237692713737488, 0.7586875557899475, -0.36828914284706116, -0.2750825583934784, 1.213037133216858, -0.9267081022262573, 0.03550276532769203, 0.8787186741828918, 1.2310221195220947, -1.0042948722839355, 0.47007519006729126, 0.033445362001657486, -1.7289685010910034, -0.9364240169525146, 0.5910446643829346, -0.028977176174521446, 0.015124048106372356, -1.1638798713684082, -0.23797619342803955, 0.5974433422088623, 0.994260847568512, 0.586102306842804, 0.3421415388584137, -0.5340971350669861, 2.4606761932373047, 0.8117091059684753, -1.9561940431594849, -0.8957533836364746, 0.18663033843040466], [-1.0021932125091553, 1.0161199569702148, 0.29418084025382996, 0.3038736879825592, -0.35771119594573975, 1.8299604654312134, 0.11290547996759415, -0.9235162734985352, -1.6348761320114136, 0.3143741190433502, -0.21789772808551788, -1.1351075172424316, 0.3740195333957672, -0.770061731338501, -1.0524500608444214, -0.09486322849988937, -1.0780495405197144, -0.03172951564192772, -1.2416173219680786, -0.802412211894989, 2.147369861602783, -0.9090840816497803, 0.15595655143260956, 2.5954456329345703, -0.044791508466005325, -0.34024882316589355, -1.522149682044983, 0.7414063811302185, -2.2916181087493896, -1.4569288492202759, -0.1907767951488495, 1.5020873546600342, 1.9111459255218506, 1.1614586114883423, 0.17545701563358307, -0.5987309217453003, 0.42235201597213745, 1.305936336517334, -0.22157403826713562, 1.0611891746520996, -1.2911629676818848, -0.002483874559402466, -0.338164359331131, 0.39756566286087036, -0.47920656204223633, -0.6454831957817078, -0.06649496406316757, -0.455529123544693, -0.9852549433708191, -2.285346269607544, 1.6125473976135254, -1.8907936811447144, -0.3382214903831482, -0.04414362460374832, 1.399170160293579, -0.32451438903808594, 0.07738538831472397, 0.023533258587121964, 0.5461252927780151, 1.1899467706680298, -0.7084337472915649, -0.14038948714733124, -1.6980491876602173, -1.6099371910095215, -1.33029043674469, -1.656649112701416, 0.629723846912384, 0.07208201289176941, -0.9386572241783142, 2.2653138637542725, 0.19750039279460907, 1.044440507888794, 0.540622353553772, -0.47715529799461365, -0.43636372685432434, 0.6307611465454102, 0.09814571589231491, 1.712689757347107, 0.741640567779541, 0.6232444047927856, -0.7257211208343506, -0.9507867097854614, -0.8365922570228577, 2.0871238708496094, -1.009127140045166, -1.1252055168151855, 0.6385465860366821, -0.6413957476615906, -0.12183798849582672, -2.3212203979492188, -0.03393140435218811, -0.04792512208223343, -0.5507066249847412, -0.3705484867095947, -0.1263083517551422, 0.486544132232666, -0.6032289862632751, 0.7185364961624146, 0.9188337922096252, -0.6091372966766357, -0.028664764016866684, 0.4981497526168823, -1.965896487236023, -0.5637993216514587, 1.9458835124969482, -1.34817636013031, -0.5744948983192444, 0.38967061042785645, 0.37875935435295105, 0.10128078609704971, -0.30248117446899414, -0.2338079810142517, 0.2146432250738144, 0.769395112991333, -0.5031321048736572, -1.336560845375061, -0.2812498211860657, 1.9249684810638428, 0.3161390423774719, 0.07539132982492447, -0.9365785717964172, -1.23197603225708, 0.22338493168354034, 0.21731583774089813, 0.28652423620224, -0.9030116200447083, -0.423228919506073, -0.3633658289909363, -0.6296714544296265, -0.6466562747955322, -0.5865364074707031, -1.7132076025009155, -0.7012191414833069, -1.8161664009094238, -0.5700082182884216, 1.4069483280181885, -1.3056589365005493, 0.9788195490837097, -2.114901065826416, 0.3779223561286926, -1.6271159648895264, -0.9503604769706726, 0.10671114921569824, 0.3804016709327698, -0.5254232883453369, 1.465803623199463, -2.1654932498931885, -0.04569537565112114, 0.5955315828323364, -0.24196456372737885, 0.397612601518631, 0.014472123235464096, 0.6327641010284424, -0.035845838487148285, 0.9966140389442444, 0.4650164246559143, -0.2891243100166321, -0.12865190207958221, 0.0016077349428087473, -0.09084621071815491, 2.2466814517974854, -0.5859490036964417, -0.22295589745044708, 0.7229182124137878, 0.7862223982810974, 0.8367885947227478, 0.2075098156929016, 0.6238767504692078, 1.9047987461090088, 1.1051042079925537, -0.1289123296737671, 1.0508126020431519, 1.1899381875991821, 1.7576355934143066, 1.1026991605758667, -0.7399037480354309, 0.5115659832954407, 0.029496055096387863, 0.02253195270895958, -0.1975063979625702, 0.18325874209403992, 0.5712107419967651, -0.05369679257273674, 0.3803691267967224, -0.9971290826797485, -0.49778181314468384, -0.5715189576148987, 0.5157078504562378, 0.5551089644432068, -0.5667065978050232, 0.19803310930728912, 0.10470959544181824, -2.1022255420684814, -0.8697978258132935, 0.4302367568016052, 0.9777799248695374, -0.2624550759792328, 0.2556520700454712, 1.3106651306152344, 0.5767583847045898, -0.019335400313138962, 1.151646614074707, 0.06103643774986267, 1.2641377449035645, 0.4498682916164398, -0.27227669954299927, 0.14437393844127655, 0.458838552236557, -0.6714610457420349, -0.4981028437614441, 0.38266295194625854, 0.1676977574825287, -2.146700620651245, 3.0337555408477783, 0.2471204698085785, 0.6333215236663818, 1.3204941749572754, 1.3567856550216675, 0.44554656744003296, 1.930889368057251, -0.8452062010765076, -0.31858769059181213, -0.9675549864768982, 1.3750598430633545, -0.5945308208465576, -0.2060537040233612, -0.20139528810977936, 3.4895434379577637, -1.2026121616363525, 1.4553824663162231, 0.8390299081802368, -0.5812327265739441, 0.49869227409362793, 0.3155769109725952, 0.17761796712875366, -1.2035927772521973, 0.745533287525177, -0.15222999453544617, -0.8383653163909912, 1.1586618423461914, 0.6858641505241394, 0.913226306438446, 0.5547184348106384, -1.4757658243179321, -0.4796583354473114, -0.1566169559955597, -1.0212136507034302, -1.448595643043518, -0.6200069785118103, 1.2508935928344727, -1.1843352317810059, 0.21778520941734314, 1.3306394815444946, 0.09613621979951859, -0.3036731779575348, -1.0825490951538086, -0.6205639839172363, -0.0714416429400444, -2.024364948272705, 0.12304941564798355, 2.03745698928833, -1.031321406364441, 1.4928221702575684, -0.5331178307533264, 1.037750005722046, 1.3017975091934204, 0.11107659339904785, -0.859958291053772, -0.3669600486755371, -0.8390836119651794, -0.41382110118865967, 1.14621102809906, -0.9673979878425598, 0.6699169874191284, -0.538261353969574, -1.0209612846374512, -1.0773794651031494, 1.2055453062057495, 0.9486619234085083, -0.23898525536060333, -0.4785955846309662, -2.0881097316741943, -1.3545795679092407, -0.488240510225296, 0.27377331256866455, -1.1237376928329468, -0.35917890071868896, -0.9863767623901367, -0.2656635046005249, 0.6355765461921692, 1.2824029922485352, -0.8673721551895142, 0.2609286308288574, 0.17394809424877167, 2.035621166229248, -0.22353753447532654, 0.6908199191093445, -0.6500848531723022, -0.9758613109588623, -0.1291184425354004, -0.40618425607681274, -0.2003900110721588, 0.6617311239242554, 0.6704478859901428, -0.6794804334640503, -0.6909661293029785, -0.8539170026779175, 0.6456151008605957, 0.2249104231595993, 0.09014953672885895, -0.8584358096122742, 0.18298037350177765, -0.5756979584693909, 0.47181347012519836, -0.6399527192115784, -0.04725583270192146, -0.07058047503232956, 1.2923948764801025, 0.8664869666099548, 0.6591524481773376, 0.824602484703064, 2.457510471343994, 0.008444635197520256, 0.36396142840385437, -0.366411954164505, 0.447477787733078, -0.7410596013069153, 0.7450657486915588, -1.5939342975616455, 0.9103870391845703, -2.954216718673706, 0.5600651502609253, -1.5484784841537476, -1.996465802192688, 0.7024366855621338, 1.1119616031646729, 0.8444595336914062, -0.7583529353141785, 1.504731297492981, -0.0010515390895307064, 0.3131386339664459, -2.725080966949463, -0.1919017732143402, 0.39770910143852234, -0.32467561960220337, 0.8097267746925354, 0.9864976406097412, 1.7833545207977295, -0.9058365821838379, -0.556154191493988, 0.29466545581817627, 1.056382656097412, -1.8939656019210815, 0.3287670910358429, -0.0063780490309000015, 1.1624937057495117, -0.9434173107147217, 0.8354892134666443, -0.24273377656936646, 1.6651946306228638, 0.2890273332595825, 0.5316323637962341, 0.47258538007736206, -0.6849929094314575, -0.6830459833145142, -0.6579973697662354, -2.3825507164001465, -1.9077372550964355, -0.37686488032341003, 1.5624799728393555, 0.1401379257440567, -0.6612042188644409, 0.7255409955978394, 0.5428038835525513, -0.9720131158828735, -0.4212544858455658, -1.6218491792678833, -1.299943447113037, -0.3681049644947052, 1.1749427318572998, 0.5855398774147034, 1.474826693534851, -0.3757052719593048, 0.8694232702255249, -1.2580969333648682, 0.47634202241897583, -1.348505973815918, 0.9681412577629089, -1.1626111268997192, 1.0089308023452759, -0.9468487501144409, 0.019645415246486664, -0.34426775574684143, -1.2888684272766113, 1.0963807106018066, 0.7938494086265564, -0.12224934250116348, -0.053256794810295105, 0.010377672500908375, 0.7611605525016785, 0.2316410094499588, 0.038728147745132446, -0.15808679163455963, 1.3223652839660645, 1.182902455329895, -0.027054226025938988, 0.9914777278900146, -1.4352507591247559, -1.0648629665374756, 0.4416116178035736, -0.8356594443321228, -1.1287190914154053, -0.018759362399578094, -0.0750703513622284, 0.5573917031288147, 0.7404997944831848, -0.46352171897888184, 0.6251226663589478, -0.7494770884513855, -0.5968028903007507, -2.168252944946289, 2.0943899154663086, 0.4348764419555664, -1.296950101852417, 1.042799472808838, -0.5940635204315186, 0.05480054393410683, -0.409878134727478, -1.2012567520141602, 0.4734925925731659, -0.09020866453647614, -1.2556729316711426, -0.4227549135684967, 0.0753418356180191, 0.8498224020004272, 0.01832103729248047, 0.17197680473327637, 0.5237056016921997, -0.6631273627281189, 0.6253742575645447, -0.7995818853378296, 1.6051234006881714, -0.323621541261673, 0.17784333229064941, 0.4634929299354553, -1.5333229303359985, -0.7704483270645142, 1.1230899095535278, 1.160399079322815, -1.5919253826141357, -1.0281094312667847, 0.9229275584220886, 1.2037910223007202, -1.1159672737121582, -0.46718308329582214, -0.5522409677505493, 0.4893425703048706, -2.1181836128234863, -0.4397728145122528, -2.0330398082733154, -0.6371892094612122, 0.6730086207389832, -0.7045314908027649, 0.4012955129146576, 2.2461845874786377, -1.5744354724884033, 0.4118957221508026, 1.5945160388946533, -0.19762428104877472, -0.9608320593833923, -1.1972373723983765, 0.12835198640823364, 0.20608338713645935, -0.05044622719287872, 0.050818074494600296, -0.3738763928413391, 1.702913522720337, -1.7543950080871582, 1.9374504089355469, 1.3646212816238403, 0.11883323639631271, 0.9061317443847656, 1.4468201398849487, -0.5075013637542725, -1.5511071681976318, -0.8527554273605347, -0.051839668303728104, 1.798072099685669, -0.8142013549804688, -0.6097147464752197, -1.5133377313613892, -0.11700516939163208, 0.7897853851318359, -0.4224269688129425, -1.7064156532287598, -0.15604379773139954, 0.30498167872428894, 0.8088240623474121, -2.5151021480560303, -0.9801839590072632, 0.5886459350585938, 0.3224598169326782, 1.4648922681808472, -0.23042438924312592, 0.04258403182029724, 0.1603555679321289, 0.27371272444725037, 0.9070536494255066, -0.06484141945838928, -0.2601359486579895, 0.9531537890434265, 0.997313916683197, 0.9714062809944153, 0.5366647243499756, -0.5016357898712158, 0.9204501509666443, 1.9802137613296509, 2.3781166076660156, -0.01645427942276001, -0.1895936280488968, -0.38895395398139954, 0.5858803391456604, 0.5264963507652283, 0.9246646761894226, 0.4232184588909149, 0.023754572495818138, 1.2780901193618774, -0.35103896260261536, -0.6294639706611633, 0.22596290707588196, 0.5743129849433899, 0.8134393095970154, 1.587020993232727, 0.4996318221092224, -0.8975927233695984, 0.08221527189016342, 1.836573839187622, -0.919725775718689, -0.4139474332332611, 0.9293082356452942, -3.4595751762390137, 0.39859989285469055, 0.8418046236038208, 1.0047383308410645, 0.4029605984687805, 0.5199682116508484, -0.45887359976768494, -0.042456191033124924, 0.07509708404541016, 1.0144343376159668, 0.5413101315498352, -0.17661546170711517, 0.8699724078178406, -0.44207707047462463, -1.574441909790039, -0.18826140463352203, 1.231632113456726, 0.6345394253730774, -0.2571147084236145, 1.1158612966537476, 0.7464960217475891, 0.2406732141971588, 0.45706605911254883, -0.020032677799463272, 0.4017323851585388, -1.2465835809707642, -0.31564316153526306, -0.6520457863807678, 0.522689163684845, -0.0752052366733551, -1.2793811559677124, 0.5013330578804016, -0.008351112715899944, 0.15578678250312805, -0.6979120969772339, -0.5840893983840942, 2.121312141418457, -1.4187071323394775, 0.16888990998268127, -0.8748643398284912, 1.7541139125823975, 1.6459611654281616, 2.382655143737793, 1.7015215158462524, -1.4145787954330444, -0.8370639085769653, 0.10881581902503967, -0.3474732041358948, -1.5188497304916382, -0.26604926586151123, -0.11849978566169739, -0.5424447059631348, 0.9355841875076294, 0.2480890452861786, 0.2464994341135025, -0.9247418642044067, 0.5904077291488647, -1.3589272499084473, -0.5521580576896667, -0.9979634881019592, -0.48539939522743225, -0.6540524959564209, 0.056874267756938934, -0.3445375859737396, -0.25653594732284546, -0.4298669397830963, 0.9273888468742371, 0.3907029330730438, 0.9236783981323242, -0.49129170179367065, 0.3445889949798584, -1.0904417037963867, 2.0884857177734375, 0.22969721257686615, 0.05787608027458191, 0.47982257604599, -1.1452840566635132, 0.6748460531234741, -0.8580005168914795, 1.0818712711334229, 1.3379930257797241, 1.371690273284912, -1.149537444114685, 1.8884822130203247, -1.2422373294830322, -0.32550421357154846, -0.7392093539237976, -2.0204598903656006, -0.9531819820404053, 0.3915420472621918, 0.17472602427005768, -0.17295952141284943, -0.45259153842926025, -0.8559048175811768, 1.514540433883667, -0.12282487004995346, -0.9626993536949158, -0.20236122608184814, 1.7255278825759888, -0.6161870360374451, -1.5733355283737183, 1.864754319190979, -0.4628094434738159, 0.3338008224964142, 1.1554213762283325, -0.5931960344314575, 0.7026064991950989, 2.5737180709838867, 0.989414393901825, -0.3221900761127472, 1.2469635009765625, 2.01027512550354, -1.3111835718154907, 0.2508803606033325, 1.0274431705474854, -0.3877239227294922, -0.23637992143630981, 0.18032298982143402, 0.9243380427360535, -0.2925782799720764, -0.8937737345695496, -2.566343307495117, -0.040590204298496246, 1.1477843523025513, -0.7664951682090759, 1.6204911470413208, -1.1695020198822021, -0.9053565263748169, 0.2503301799297333, -0.2919825315475464, 1.2602546215057373, -0.7017147541046143, -1.0577583312988281, -1.5212997198104858, 1.0710471868515015, 1.774025321006775, -0.46772539615631104, 0.8106869459152222, 0.20558276772499084, 1.6323976516723633, 0.5023481845855713, 0.18263103067874908, -2.1336748600006104, 0.6171572804450989, -1.9575241804122925, -0.7783775329589844, 0.1222035363316536, 0.962537944316864, -0.18355995416641235, 1.503221035003662, 0.38704097270965576, 0.06921933591365814, 0.2860116958618164, 1.0243287086486816, 0.6574423313140869, 0.5520114302635193, -0.14879575371742249, -1.3192235231399536, -0.5921804308891296, 0.2841533124446869, -0.8004947900772095, -0.4679909646511078, -0.16292385756969452, 1.7798916101455688, 0.15353521704673767, 1.2043544054031372, 0.23038740456104279, 0.34633320569992065, -1.059378981590271, 1.6178288459777832, -1.3254543542861938, -0.24285465478897095, -1.3206965923309326, -0.09375583380460739, 0.1387285441160202, 0.20914819836616516, 2.0675954818725586, 1.1860301494598389, 0.6585251092910767, -0.882820725440979, -1.7926311492919922, -0.3678446412086487, 1.2355676889419556, -0.0030698957853019238, 0.14721399545669556, -0.3990747630596161, -0.32485899329185486, -0.9131712913513184, 0.5205841064453125, 2.0464205741882324, 0.14067161083221436, -0.3348409831523895, 0.22105547785758972, 1.6148178577423096, -0.3972092568874359, 1.0181608200073242, 2.0738120079040527, -0.48712360858917236, 0.26486921310424805, 0.7633187174797058, 1.0427591800689697, 1.4473224878311157, -0.5305062532424927, 1.4680427312850952, -1.3827811479568481, -1.3156509399414062, 1.439936876296997, 0.71592116355896, 0.174049973487854, -1.909576177597046, -0.970358669757843, 0.5226048827171326, 0.4205683171749115, 0.7659631371498108, 0.20109911262989044, 0.9328667521476746, 0.5736297369003296, 1.2811490297317505, -0.7223787307739258, 1.7977516651153564, -1.3402515649795532, -0.5902504920959473, -1.0113627910614014, 0.06929931789636612, 0.5595657229423523, -0.47382664680480957, 1.4748529195785522, -1.7212481498718262, 2.3024117946624756, 0.09174023568630219, 1.2861331701278687, -0.17834913730621338, -0.7754903435707092, 0.7958094477653503, -0.26557835936546326, -0.7992343306541443, 0.9216225743293762, 1.7963062524795532, -0.21053467690944672, 2.6827776432037354, -0.704190731048584, -1.8724961280822754, -1.5446091890335083, 2.9503798484802246, -0.5270668268203735, 0.5908823609352112, -2.4532575607299805, 1.0399601459503174, -0.4780605435371399, -0.9498339891433716, 0.3169325292110443, -2.742453098297119, -0.0006446485058404505, 0.2378913164138794, 0.07598675042390823, -0.31544920802116394, -2.2853012084960938, -0.21747203171253204, 0.7943652272224426, -0.40846943855285645, 0.5480843186378479, -1.2746844291687012, -0.15280477702617645, -0.3505997955799103, 0.36218175292015076, 0.4564121961593628, 0.9138505458831787, 0.7821449637413025, 0.8582268953323364, 2.0629847049713135, 0.009001210331916809, -0.20860326290130615, 0.8406399488449097, 2.0281498432159424, 0.7505439519882202, -0.8900011777877808, 1.0293104648590088, 2.308743476867676, 0.2653762698173523, -2.04323148727417, -1.629311203956604, -1.5529687404632568, -0.09930190443992615, -0.7652444839477539, 0.16785447299480438, 0.8785224556922913, -1.4224237203598022, -2.055454730987549, 0.561155378818512, 0.9826148748397827, -0.17797529697418213, 0.15418758988380432, -1.3185945749282837, -0.47790318727493286, -0.6920074224472046, 0.06289782375097275, -1.4016276597976685, -1.1928768157958984, -0.42252370715141296, -0.709947407245636, 0.9985870122909546, -1.2759144306182861, -1.7068456411361694, -0.44880157709121704, 0.33917519450187683, -0.4913642108440399, -1.4847949743270874, 0.8121827244758606, 0.2690487802028656, 2.422581911087036, 1.3561066389083862, -0.3477615714073181, 0.5370165109634399, 1.1841282844543457, -1.0877858400344849, 0.6809642910957336, 0.3190729320049286, 0.1505453884601593, 1.4112287759780884, -0.3840813934803009, -0.641639769077301, -0.13536851108074188, 0.3669070601463318, -0.28794583678245544, -0.19826863706111908, 1.4039980173110962, 0.8273167014122009, 0.8670207262039185, -0.8870106339454651, -0.44266313314437866, 1.0890984535217285, 0.5921432971954346, -0.6521175503730774, -0.6555547118186951, 0.6733279824256897, -0.11430861055850983, 0.16634203493595123, -0.8456107974052429, -1.0490057468414307, 1.3568881750106812, 0.031797267496585846, -0.7281210422515869, -0.24572300910949707, -1.2281267642974854, 0.6310008764266968, -0.47467362880706787, 0.982408881187439, -0.2763242721557617, -0.17570547759532928, -1.2018909454345703, 1.309261441230774, -0.07606986910104752, 1.5258153676986694, 0.22373530268669128, -0.058975253254175186, 0.26650822162628174, -0.7357122898101807, 1.259721279144287, -0.3250844180583954, 0.09733279049396515, -0.26316994428634644, 0.48533153533935547, 1.4687129259109497, -0.2494441270828247, 0.75180584192276, 0.9507396817207336, 0.17031230032444, 0.3397098481655121, -1.221269965171814, -0.3161107897758484, 0.9716405272483826, -0.7995457053184509, 0.01339742261916399, -2.612971305847168, 0.25273045897483826, 0.33645519614219666, 0.3641037046909332, -0.4519335627555847, 2.205098867416382, 1.1063406467437744, 0.9612886309623718, 0.016048304736614227, -1.762681484222412, -0.11892519146203995, -0.20206597447395325, 0.8564605116844177, -0.7269099950790405, 1.4776283502578735, -0.0541129894554615, -0.8000919818878174, 0.6885603070259094, 1.795296549797058, -0.8697536587715149, 0.22141574323177338, 1.7097843885421753, -0.27236679196357727, 1.3659424781799316, 0.8241191506385803, -0.7648513317108154, 0.07033338397741318, 0.2369651198387146, -1.3351263999938965, -0.1373819261789322, 0.6783545613288879, -1.5307576656341553, 1.1633622646331787, -1.457503080368042, 1.2393213510513306, -2.151771068572998, 1.5661938190460205, -1.1344977617263794, -0.09466543793678284, 1.1667461395263672, -0.39174172282218933, -1.4946123361587524, 0.13849051296710968, -1.1236289739608765, -1.7510319948196411, -1.5690234899520874, 0.6997811794281006, -1.457481026649475, -0.6465911269187927, 0.6686347126960754, 0.6531752943992615, 0.9879864454269409, 0.6200665831565857, 0.12681712210178375, 0.6448413133621216, 0.1770191490650177, -0.4827485978603363, -0.9543923735618591, -0.1652195006608963, 1.04979407787323, -1.1442815065383911, -0.43119537830352783, 0.08924102038145065, -0.22786568105220795, -0.2607358694076538, -0.904564619064331, 0.09789230674505234, 0.42139407992362976, 0.8638492822647095, 1.698514699935913, -0.8878600001335144, 0.021132443100214005, 1.1511691808700562, -1.3478506803512573, 0.46114128828048706, 0.614419162273407, 0.3927043080329895, -1.4075520038604736, 1.573041319847107, 0.13058078289031982, 1.113834261894226, 0.08905526250600815, 0.5599504709243774, -0.0932173952460289, 0.27396538853645325, 0.6193220019340515, 0.4431208670139313], [-0.6821291446685791, 1.3066645860671997, 1.3942002058029175, 2.131145715713501, 0.16659663617610931, -0.07815947383642197, -1.1100420951843262, 0.061714306473731995, -0.10957080125808716, -0.7724420428276062, 0.9218354821205139, -0.46994128823280334, -0.34058621525764465, 0.016561618074774742, -0.2509470283985138, 1.261359691619873, -0.09309139847755432, -0.6657054424285889, 0.6489136219024658, 0.38768482208251953, 2.8069536685943604, 0.9671762585639954, -0.35233384370803833, 1.4466296434402466, 1.0620594024658203, -0.7148979306221008, 0.4999665915966034, 0.7376247644424438, -1.2224233150482178, -0.22350585460662842, 2.5634641647338867, -0.30192506313323975, -0.7339383959770203, 0.7574535608291626, -0.9420870542526245, 0.039003558456897736, 1.2926249504089355, 0.8737499117851257, -2.976902723312378, -0.8795304298400879, -0.8907971382141113, -0.561127245426178, -1.8134359121322632, -0.5122841596603394, -0.8606390953063965, 1.0644704103469849, 0.8047879934310913, -1.2564035654067993, -1.149990200996399, -0.5556143522262573, 0.04004063084721565, -0.9945117235183716, 2.1347246170043945, -0.6075707674026489, 1.2911041975021362, 0.1415020078420639, -0.14007683098316193, 0.7140181660652161, 1.2240161895751953, -0.2552747428417206, -2.0594067573547363, -0.8522191643714905, -0.6758703589439392, -0.11299660056829453, -0.0676041692495346, 0.8630908727645874, -0.7063405513763428, -1.2905162572860718, 0.315416157245636, -0.980843722820282, -1.166776180267334, -0.043383680284023285, -1.1566721200942993, 1.6246891021728516, -0.3500381112098694, 0.8101686239242554, 0.03552261367440224, -0.587681770324707, -2.045154333114624, 0.8636487722396851, -1.3272762298583984, 0.7399781346321106, -0.550804853439331, -0.1503487378358841, -1.9093565940856934, 0.3729690909385681, -1.2639464139938354, 0.2909087538719177, 1.0106604099273682, 0.21454450488090515, -1.4440076351165771, -1.2025560140609741, -0.033500198274850845, -0.9504704475402832, -0.03202834352850914, 1.5611677169799805, -0.522973895072937, -0.10806038230657578, 0.08871674537658691, -0.4722137451171875, 1.1682419776916504, -0.8873820900917053, 1.239209771156311, -0.8999095559120178, -0.9356403946876526, -2.1455612182617188, 0.4049682319164276, -0.04637405648827553, 2.036003589630127, 0.5708135366439819, -1.0161824226379395, 1.4694664478302002, -1.941748023033142, 0.2377096265554428, 0.06155272573232651, 0.03313867747783661, 0.6889364719390869, 0.44613906741142273, 0.655146062374115, -0.03499802574515343, 1.0740307569503784, 0.842189371585846, 0.8238441944122314, 0.5254872441291809, 0.5131423473358154, -0.025116365402936935, 0.9948891997337341, 0.5250585675239563, -0.5794277191162109, -0.6711970567703247, 0.8588594198226929, -0.98512202501297, -0.3376584053039551, 1.072163701057434, -0.48335981369018555, 0.17676638066768646, -1.45182466506958, -1.1746230125427246, -0.11339814960956573, 0.7209113240242004, 0.31430456042289734, -1.1488956212997437, 0.3445008397102356, 1.3544737100601196, 1.1272966861724854, 0.9230945706367493, 0.6484031081199646, 1.2743839025497437, 0.18688631057739258, -1.327962040901184, 0.6542686223983765, 0.9602063894271851, 0.3317528963088989, 0.41879770159721375, -1.4105836153030396, 1.4504024982452393, -2.026724100112915, 0.7321247458457947, 1.2898164987564087, -0.3494798243045807, 1.7509300708770752, -1.020272135734558, 0.8363679051399231, -1.4387739896774292, -0.7007798552513123, -0.22849076986312866, -0.5994221568107605, -0.353669673204422, 0.6171534061431885, -0.34589657187461853, -0.39937707781791687, 0.036152906715869904, 0.5613044500350952, -1.0384408235549927, -0.053155139088630676, -0.1308777928352356, 0.7260698676109314, -0.47560444474220276, 2.65482234954834, 0.48831871151924133, 1.7574670314788818, -0.7597905993461609, -0.2912237048149109, 0.47855791449546814, -1.788361668586731, -1.498356580734253, -0.3278377652168274, -0.8600121140480042, 0.757281482219696, 0.01642163284122944, 1.0847562551498413, -0.08612094074487686, -1.423111915588379, -0.6414913535118103, -0.13802015781402588, 1.1414166688919067, -0.7571492791175842, -0.669366717338562, 1.7052421569824219, 0.2376948446035385, 0.09324128925800323, 1.6315813064575195, 1.034508466720581, 0.15834707021713257, 0.0002691904082894325, 0.44916126132011414, 1.3679195642471313, 0.798713743686676, -1.1473692655563354, 0.00024887651670724154, 0.4652113616466522, 1.4240145683288574, 0.14978237450122833, 0.278137743473053, 0.605580747127533, -0.9268826246261597, 0.11535431444644928, 1.9352401494979858, -0.3635759651660919, -1.311287760734558, 0.4776690602302551, 0.2690153121948242, -0.8667095899581909, -0.7033829092979431, -0.25417470932006836, 0.4495078921318054, -0.15739652514457703, 0.3056536912918091, 1.3827269077301025, 1.4362430572509766, 0.21862228214740753, -0.3579752445220947, 0.6573969125747681, 1.1759002208709717, -0.14951114356517792, 0.9628463983535767, -1.3880627155303955, -0.4064287841320038, 0.28289058804512024, 0.7209615707397461, -1.558828592300415, -0.9823485612869263, -1.2097344398498535, 0.5634089708328247, 1.0302900075912476, -0.3215178847312927, 0.6250673532485962, -0.36152416467666626, 0.36385467648506165, -1.576742172241211, 1.315493106842041, 0.10382774472236633, -1.6396735906600952, 0.8312438726425171, -1.6846901178359985, 0.2267332673072815, -0.19616015255451202, 1.0306611061096191, -0.8381853699684143, -0.8993389010429382, -1.7280879020690918, -0.02759408950805664, -0.42929279804229736, -0.7905338406562805, -1.4044796228408813, 0.6806823015213013, -0.6260355114936829, 1.5071985721588135, -0.6546034812927246, 1.911666989326477, 0.3766140341758728, -0.4677184522151947, -0.041665975004434586, 0.11209297180175781, 2.017392873764038, -0.8563032746315002, -1.1755247116088867, -0.6063608527183533, 0.05407362058758736, 1.4048655033111572, 1.6904042959213257, -2.4779469966888428, 1.476876139640808, 1.6018214225769043, -1.6649140119552612, 0.5390201807022095, 1.3954366445541382, 0.22567681968212128, 0.3359467089176178, -0.8456838130950928, 0.673262357711792, 0.09405986964702606, -0.6276773810386658, 0.2279144525527954, 0.8664845824241638, -1.2111294269561768, 0.6707448363304138, -1.0562214851379395, -2.203051805496216, 0.6024131178855896, -0.49232515692710876, 2.18098783493042, 1.5816701650619507, 1.8810971975326538, 0.7808173298835754, -0.5254446268081665, -0.11197734624147415, 0.5382829308509827, 0.8268280029296875, 1.6488620042800903, 1.4302867650985718, -0.10414452850818634, 1.7581647634506226, -1.2378220558166504, -1.0181517601013184, 0.4794057011604309, -0.15426108241081238, -0.12544317543506622, 0.9068446755409241, -1.0799956321716309, 1.0615870952606201, -0.5398868918418884, -0.647160530090332, 1.878426432609558, 1.413552165031433, -0.8848287463188171, 1.445257306098938, -0.07549291849136353, -1.3304991722106934, 0.24288661777973175, -0.2884520888328552, 1.2257099151611328, 0.9761711955070496, 0.8839240074157715, -0.880493700504303, 0.9415029287338257, 0.7421268224716187, 1.2447952032089233, 1.7525461912155151, -0.21061141788959503, -0.5039494633674622, -0.0567881241440773, -0.5472521781921387, 0.10404428094625473, 0.01672172173857689, 0.7842536568641663, 0.41741976141929626, 0.4421655535697937, 1.3116934299468994, 1.1977922916412354, 0.4732501804828644, -0.43487223982810974, -0.9659824967384338, -0.9795088768005371, -0.3545694053173065, -0.31748300790786743, -0.281075656414032, 0.39874982833862305, 0.25982847809791565, -0.3469032049179077, -0.8925725221633911, -0.20882543921470642, 0.365579217672348, -0.08737309277057648, 1.015842080116272, 0.29946720600128174, 0.06022383272647858, -0.5233633518218994, 0.10191140323877335, 0.1848379522562027, -1.0000978708267212, -1.6521049737930298, -0.3332712948322296, 1.445430874824524, 1.493272066116333, 0.4905143082141876, 0.6219684481620789, 0.27595698833465576, -1.0794421434402466, -0.827646017074585, 1.2994940280914307, -0.7275723814964294, -0.4075455963611603, -1.2966464757919312, -0.7168689966201782, -0.08217304199934006, -0.5389823317527771, -0.3376874029636383, 0.4123605787754059, -1.2436785697937012, 0.7888098955154419, -0.8981189727783203, -0.33707234263420105, 1.5042023658752441, 0.6609732508659363, -0.1332089900970459, 0.2545079290866852, -0.732071042060852, -1.9032344818115234, 1.2662873268127441, 0.029556049033999443, -1.2107439041137695, -1.0590693950653076, 0.2056054323911667, -0.1002703532576561, 1.3212908506393433, 0.6720204949378967, -0.08982642740011215, -0.012358445674180984, 1.0310697555541992, -2.063797950744629, 0.695049524307251, 1.6220327615737915, -1.9638159275054932, -0.7137842774391174, -0.2504517734050751, 0.5381594896316528, -1.953341007232666, -0.13531553745269775, -0.134311243891716, 0.48004937171936035, -0.7413476705551147, -0.36539769172668457, 0.11913971602916718, 0.5554842352867126, -0.4538971483707428, 0.9854496121406555, -1.473527431488037, -0.40130141377449036, 1.8027386665344238, 1.194892406463623, -1.2180421352386475, 0.7420541644096375, -0.7369170188903809, -0.5014349222183228, 1.2038263082504272, -0.8122280240058899, -0.4429294466972351, 0.5277984142303467, 0.6193714141845703, 1.4595367908477783, -0.5460712313652039, -0.23060710728168488, 0.49444523453712463, 0.35183149576187134, -1.8558106422424316, -0.17836833000183105, 0.14367349445819855, -0.5762653946876526, 1.7689640522003174, 0.1484885811805725, 0.9837749004364014, 0.5242814421653748, -0.10349372029304504, -0.2978944778442383, -2.303997755050659, -1.0168511867523193, -0.24559415876865387, 1.0592848062515259, 0.992196798324585, 0.7765293717384338, 0.8489582538604736, 1.0090572834014893, 0.8088016510009766, 0.2466980665922165, 0.2171168327331543, 0.030692262575030327, 0.28702178597450256, 1.407518982887268, -0.9646862745285034, -0.8411213159561157, 0.42040348052978516, 0.8407496809959412, -0.23512764275074005, -0.7644227147102356, -0.7668859958648682, -1.128769040107727, -0.9240131378173828, 0.2599128782749176, 0.036949217319488525, -1.9969234466552734, 0.8519455194473267, 0.8699632287025452, -0.44200146198272705, 0.13102012872695923, -0.18769553303718567, 0.620516300201416, -0.9803490042686462, -0.9689494967460632, 0.02239244617521763, 1.9887751340866089, 0.5831705927848816, 0.7705103158950806, -0.9960594773292542, 0.6431064009666443, -0.6415506601333618, -0.828403651714325, -0.8206668496131897, -0.16682885587215424, -0.1899794191122055, 1.8117107152938843, -0.3490597605705261, -0.761746346950531, 1.717814326286316, 0.29774168133735657, -0.6941776275634766, 1.013675332069397, -1.4535374641418457, 0.5415376424789429, 1.3476524353027344, -0.8490683436393738, 0.1659800112247467, 1.00513756275177, -0.3921438753604889, 0.9320511817932129, -2.429610013961792, 0.6897432804107666, -1.5836364030838013, 1.8545531034469604, 0.9030617475509644, -1.5071640014648438, -0.2536337673664093, -0.20393002033233643, 2.0080058574676514, 0.605165421962738, -0.38207098841667175, 1.5859895944595337, 0.8387559056282043, 0.5743621587753296, 0.541092574596405, -1.130275011062622, -0.19145065546035767, -0.5164578557014465, -0.29242101311683655, -0.1661524623632431, 0.9322307109832764, 0.4985184371471405, -0.6744000315666199, 1.4751157760620117, 0.7774288654327393, -1.3242825269699097, -1.7643632888793945, -1.2655292749404907, 1.5453643798828125, 0.7484257221221924, -0.5135685801506042, -1.816648006439209, 0.0001269440836040303, -1.210974931716919, 1.312307357788086, -0.5722556710243225, -0.4935574531555176, -0.10312486439943314, 0.7945647239685059, 0.08354990929365158, 0.7893020510673523, -1.9006534814834595, 1.119368553161621, -0.8380031585693359, -0.3695993423461914, -0.9827022552490234, -2.4002022743225098, -1.684911847114563, 0.212310791015625, -1.0977747440338135, 0.9638365507125854, 0.31269752979278564, 0.8738961219787598, 0.5441843271255493, -0.12483981251716614, 1.8840140104293823, 0.3889886736869812, 0.15675972402095795, -0.911848783493042, -0.15608349442481995, -0.04564899951219559, 0.6431841850280762, 0.39007410407066345, -1.5500613451004028, -1.1568456888198853, 1.0102455615997314, -0.06497612595558167, -0.2717915177345276, 1.0785900354385376, 0.8453016877174377, 0.564379870891571, -1.742960810661316, -1.3073170185089111, 0.11150616407394409, 0.6062952280044556, 0.7480079531669617, -0.5785722136497498, 1.048351526260376, -0.18255937099456787, 0.384224534034729, -0.08447118103504181, -1.5497349500656128, 0.15010228753089905, 0.29334786534309387, -0.14450959861278534, -1.2858498096466064, 0.16407491266727448, 0.3975323438644409, 1.6324660778045654, -0.42665913701057434, -0.9271538853645325, 0.7090689539909363, 0.2082669734954834, 0.03712988644838333, 1.1595299243927002, -0.8331551551818848, 0.8652620315551758, 0.4917891323566437, 0.23580484092235565, -0.005308005027472973, -1.0291837453842163, -2.8502931594848633, -1.165280818939209, 0.14540636539459229, 1.5276530981063843, 2.299722909927368, -0.9681147336959839, 1.0116543769836426, 2.2248523235321045, -0.09209287911653519, 1.2756377458572388, 0.06219956651329994, -0.4235980212688446, -0.5394800901412964, -0.8590461015701294, -0.8619032502174377, -1.002198338508606, 0.8266000747680664, 0.6503158807754517, -0.7227157354354858, -0.26081469655036926, 0.9238904714584351, -0.5572329759597778, -0.2034909874200821, -0.008134466595947742, 0.5637457370758057, -0.23505131900310516, -1.181807518005371, -2.169196844100952, -1.5260515213012695, -1.6864105463027954, 0.8501124978065491, 0.5159547328948975, -0.47810128331184387, -0.056998640298843384, -0.9132258296012878, 0.3560534417629242, -0.15121649205684662, 0.8314914703369141, -0.938493013381958, 0.2164732664823532, -0.0964777022600174, -0.3823569118976593, -1.6018738746643066, 0.597612202167511, -0.36650165915489197, 0.804894745349884, -1.0681219100952148, -1.4920881986618042, -2.612229824066162, 0.531851589679718, -0.7702443599700928, 0.7228894829750061, -1.4433162212371826, 0.698544442653656, -0.2338027060031891, -0.8707549571990967, -0.8089333176612854, 0.9042605757713318, -0.46960705518722534, -1.993863582611084, 0.6096426844596863, 0.5377011895179749, 1.6680978536605835, 0.6939175128936768, 2.222959041595459, 0.6167980432510376, -1.323620080947876, -0.9142046570777893, 0.21841523051261902, -1.5154670476913452, 0.5451770424842834, 0.26581355929374695, 0.6564127802848816, 0.4718520939350128, 0.41146939992904663, 0.8256988525390625, 0.05656455457210541, -0.002001111628487706, -1.4434349536895752, -0.37946081161499023, 1.1259409189224243, -0.8803730607032776, -1.6481229066848755, 0.22317545115947723, -0.9141290187835693, 1.7631198167800903, 0.3724859654903412, -0.7956557273864746, 1.2836313247680664, 1.37107253074646, 1.0458197593688965, 1.3518919944763184, 0.01827145554125309, -0.7914846539497375, 0.9096699953079224, 0.42338815331459045, -0.526994526386261, -0.09338817000389099, 0.7967283129692078, -0.371435284614563, -0.48089084029197693, 1.979087233543396, -0.37290337681770325, -0.979567289352417, 0.7047576904296875, -0.9477912783622742, 1.696934461593628, 0.6805931925773621, 0.6164063215255737, 1.1368461847305298, 1.2997959852218628, -0.17814792692661285, -1.0485155582427979, -1.3331677913665771, 1.0993883609771729, 2.0254404544830322, -1.6266587972640991, 0.3162611126899719, -0.5256789922714233, 1.2436635494232178, -0.24922913312911987, -0.8715735077857971, -0.19328147172927856, -1.1454252004623413, 0.6953615546226501, 1.4269970655441284, 0.9459555149078369, -0.7000585198402405, 3.0439555644989014, 1.328637719154358, 1.500605821609497, -1.252639651298523, -0.5600854754447937, 1.5886765718460083, 2.181488037109375, 0.22381512820720673, -0.5889931917190552, -0.7324166893959045, 0.8610796332359314, 0.9173641800880432, 1.2142887115478516, -2.0835516452789307, 1.4032549858093262, -1.0437428951263428, 0.8975051641464233, 0.38189759850502014, 0.24485719203948975, 0.06797914952039719, -0.07605437189340591, 0.8052687048912048, 1.1846791505813599, 2.132864475250244, -0.11480866372585297, -0.08694511651992798, 0.8847343921661377, -0.3588041365146637, -1.0434296131134033, -0.6081706881523132, 0.1252993494272232, 0.8685130476951599, 1.3354750871658325, -2.0500094890594482, 0.9416500926017761, -1.6064492464065552, -1.0396965742111206, 0.5013787150382996, -0.32751983404159546, 0.7003710269927979, -1.9732272624969482, 0.06872326135635376, 2.4964678287506104, -1.6652052402496338, 1.497189998626709, -0.6906421184539795, 0.8712279796600342, -0.7777092456817627, -3.7036328315734863, -0.7715247273445129, 1.9608447551727295, -0.8763235807418823, 1.682092547416687, -1.1758453845977783, 0.29965537786483765, 1.0441983938217163, -2.4378180503845215, 0.12980353832244873, 0.09726065397262573, 0.33297449350357056, -0.42941635847091675, -0.4562351107597351, 0.5776116847991943, -0.6083095073699951, 1.8597832918167114, 0.2569921910762787, -0.636502206325531, 0.32093873620033264, -0.20718403160572052, 0.37135395407676697, -1.5781794786453247, -0.2635071873664856, 1.4138075113296509, -1.590351939201355, 0.23344837129116058, -1.0300507545471191, 2.1995086669921875, -0.9493889808654785, 0.26258692145347595, 0.3530423045158386, 0.1544463187456131, 0.646898627281189, 0.051291704177856445, 0.4387161135673523, 0.23490062355995178, -0.43862128257751465, -1.3056596517562866, -0.8663191795349121, 0.8482260704040527, 1.1972229480743408, 1.1170910596847534, 0.5814272165298462, -0.8556495904922485, 0.19623900949954987, 0.034241050481796265, -1.8681972026824951, 0.30258676409721375, 1.4544519186019897, -0.42558079957962036, 0.007722186855971813, 1.4340711832046509, 1.1666158437728882, 1.1046804189682007, 0.24043405055999756, 0.20699664950370789, 0.0909198522567749, -0.3129296600818634, 1.0496495962142944, 1.926450490951538, 0.6658695936203003, -0.4708003103733063, -0.8689492344856262, 0.8566774129867554, 0.08510128408670425, -0.37263253331184387, -1.3924040794372559, -1.1146513223648071, -1.0195014476776123, -0.20912234485149384, -0.997753918170929, 1.1770836114883423, -0.8658270239830017, -0.6851133108139038, 0.1154099702835083, 0.5215838551521301, 0.03693300113081932, -1.4554845094680786, 1.195415735244751, 2.1741750240325928, 0.9778653383255005, -1.1489646434783936, 0.7245725989341736, 1.470494270324707, -2.9671247005462646, -0.9850830435752869, 0.7462592720985413, -0.7702884078025818, -1.2277610301971436, 0.9832817316055298, 0.10297510027885437, -0.010553816333413124, 1.8251320123672485, 0.20937705039978027, 1.7580891847610474, 0.07929400354623795, -0.6039980053901672, -0.9355377554893494, -1.1035603284835815, 1.4396350383758545, 0.08640021085739136, 0.6328657865524292, 0.30190956592559814, 0.5809398293495178, -1.7447609901428223, 1.746008276939392, 0.42460528016090393, 1.9179579019546509, 0.554783284664154, -1.6386692523956299, -0.24274109303951263, -0.9879858493804932, -0.3892478942871094, 0.9131482839584351, 0.23707064986228943, -0.8121995329856873, -1.4660346508026123, 0.7426691055297852, 0.4235442280769348, 1.4925353527069092, 0.900469958782196, -0.2249804437160492, -0.8886356949806213, 0.7080678939819336, 0.14573362469673157, 0.8838571310043335, 1.29728102684021, -0.3851786255836487, -0.045616522431373596, -0.47236913442611694, 0.3045411705970764, -0.21520072221755981, 1.928634762763977, -1.826371431350708, 0.8121553659439087, 2.369985342025757, -0.12114976346492767, -0.8103693723678589, 1.040098786354065, -0.462424099445343, -0.13180680572986603, 0.7718585133552551, -0.057275909930467606, -0.24491897225379944, 0.384360134601593, -0.12773649394512177, -0.06281739473342896, 0.4422764480113983, 0.8249086737632751, 0.9163568019866943, 2.5267014503479004, -0.957374632358551, -1.0227612257003784, 0.0019323222804814577, 0.7955586910247803, 1.2843103408813477, -0.11176416277885437, 0.22100234031677246, 0.7098227143287659, 0.9946733713150024, -0.9563496112823486, 2.2364025115966797, 2.1305012702941895, 1.0270743370056152, 0.5866344571113586, 1.1953082084655762, 0.7070015668869019, 0.9746297597885132, -1.490451455116272, 2.188416004180908, 0.4941955506801605, 2.374467372894287, -0.5700263977050781, 1.4967440366744995, 1.562744379043579, -0.23190146684646606, 0.2756885588169098, 0.8967772722244263, 0.3265626132488251, 0.6036157608032227, 0.02808949537575245, 0.7801888585090637, -1.4312031269073486, 0.9419256448745728, 0.46304112672805786, -0.9457587599754333, -0.9561039805412292, 0.33622676134109497, -0.07519987225532532, 0.3242783844470978, 0.9659749865531921, 0.06281071901321411, -0.610395073890686, 1.1067783832550049, 0.5482755899429321, 1.0744930505752563, -0.2108365148305893, -0.3823716342449188, 0.3579151928424835, 0.08415362983942032, -0.04188195988535881, -1.8977268934249878, 0.844147264957428, 1.8711187839508057, 0.7177984714508057, 0.3556407392024994, -0.020862162113189697]], "dtype": "float32"}], "e": {"__nd__": [[3.587777973734774e-05, 0.00029157949029468, 0.00013907456013839692, 0.0015781477559357882, 0.0005429863231256604, 0.00017690473760012537, 0.0006564966752193868, 0.0036009897012263536, 0.00014727965753991157, 0.00019505745149217546, 0.0004819199093617499, 0.0005597628187388182, 0.0006302851252257824, 0.00011811515287263319, 0.0005918777314946055, 0.00016485351079609245, 0.0002436723734717816, 0.001319315517321229, 0.0002563820453360677, 0.000964885635767132, 0.0003801783896051347, 0.0006235228502191603, 0.0006166605744510889, 0.0010024516377598047, 0.0011008601868525147, 0.00025315297534689307, 0.0027848274912685156, 0.0009721466922201216, 0.0013820672174915671, 0.0014405720867216587, 0.0003333818749524653, 0.0005042591947130859, 0.0005841392558068037, 0.0007157149375416338, 0.002939993981271982, 0.003447936149314046, 0.017114073038101196, 0.0033929564524441957, 0.0029245370533317327, 0.0005070858169347048, 0.00023553032951895148, 0.0005548320477828383, 0.0004110072331968695, 0.00019887520465999842, 0.00042725616367533803, 0.0008398437057621777, 0.0005727349780499935, 0.0013445595977827907, 0.001012439839541912, 0.0008417892386205494, 0.001936075510457158, 0.0004920423380099237, 0.0005921549163758755, 0.002363400300964713, 0.0014333499129861593, 0.00012208818225190043, 0.0006221829680725932, 0.0025719567202031612, 0.0008905327995307744, 0.0018860618583858013, 0.00012223891098983586, 0.0008675685385242105, 7.430492405546829e-05, 0.0011341387871652842, 0.00016482258797623217, 0.0022990922443568707, 0.00028320992714725435, 0.0006348944152705371, 0.0007403600611723959, 0.0005525735905393958, 0.00012048710050294176, 0.00115980114787817, 0.00034048198722302914, 0.006501585710793734, 0.00043109236867167056, 0.0007005526567809284, 0.0004044050583615899, 0.0031545967794954777, 0.0025230725295841694, 0.0010405387729406357, 0.0015949173830449581, 0.0002331924915779382, 0.00021779084636364132, 0.002060520928353071, 0.00023755748406983912, 0.00041462836088612676, 0.00011868568253703415, 0.000379390170564875, 0.000845994392875582, 0.0005157064297236502, 0.00012147850793553516, 0.0012343465350568295, 0.0006593210273422301, 4.324043766246177e-05, 0.00018466831534169614, 0.0003890713269356638, 0.0003218238998670131, 0.0006440855213440955, 0.0016103446250781417, 0.0005049130995757878, 0.0004989241133444011, 0.002126392675563693, 0.002720424672588706, 0.0004000983899459243, 0.002762979594990611, 9.44414859986864e-05, 0.002616977784782648, 0.0008840118534862995, 0.00023424893151968718, 0.0003306741709820926, 0.0006221643416211009, 0.002069713780656457, 0.0013051265850663185, 0.0007344494224525988, 0.00037122698267921805, 0.0005978458211757243, 0.0001621387928025797, 0.00036222647759132087, 0.0002302113571204245, 0.00037676800275221467, 0.0008398515055887401, 0.00038908582064323127, 9.104867785936221e-05, 0.0015821986598894, 0.0005256966687738895, 0.0008163974271155894, 0.00026166034513153136, 0.0004109735309612006, 0.0003949840902350843, 0.0021941850427538157, 0.00015404229634441435, 0.001274173497222364, 0.00048354826867580414, 0.0010595436906442046, 0.0002822600363288075, 9.942160249920562e-05, 0.001303608762100339, 0.0003576306044124067, 0.0008377154590561986, 0.001051145140081644, 9.185881935991347e-05, 0.0003146906092297286, 0.00042495268280617893, 0.0009231396252289414, 0.0009535473654977977, 0.0011418635258451104, 0.0004210327460896224, 0.0020352834835648537, 0.0005485257715918124, 0.0004971973830834031, 0.00032577174715697765, 0.00017173662490677088, 0.0005223869811743498, 0.0006931856623850763, 0.0007042651996016502, 0.004705282859504223, 0.0007243156433105469, 0.0007763987523503602, 0.00014568460755981505, 0.000415770715335384, 0.0005503547727130353, 0.0008586461772210896, 0.000436488218838349, 0.0007524410029873252, 0.00019811021047644317, 0.0016484939260408282, 0.0014883849071338773, 0.00017047495930455625, 0.0001176353616756387, 0.0002707536914385855, 0.000618859427049756, 0.00021811784245073795, 0.0006193562294356525, 0.0011138510890305042, 0.0013907013926655054, 0.0015922017628327012, 0.00012139236059738323, 7.170135359046981e-05, 0.0008580308640375733, 0.00024126785865519196, 0.00024298802600242198, 0.00017625433974899352, 0.0002717735187616199, 0.0008831332088448107, 0.00011970895138802007, 0.00013066724932286888, 0.0013880261685699224, 0.0009447152260690928, 9.237507038051262e-05, 7.878937321947888e-05, 0.0007065556710585952, 0.0011330369161441922, 0.0037656386848539114, 0.0009329153108410537, 0.0009146755328401923, 0.0003721563261933625, 0.000504860479850322, 0.00031271111220121384, 0.001525659579783678, 0.0006907914648763835, 0.0013667065650224686, 0.0006449412903748453, 0.00018556814757175744, 0.000461969175375998, 0.001375216175802052, 0.0006821912829764187, 8.392395102418959e-05, 0.0013549538562074304, 0.00030276685720309615, 0.0018525701016187668, 0.0003416280378587544, 0.00020130418124608696, 0.0010796230053529143, 0.00014787157124374062, 0.0015441332943737507, 0.00029148015892133117, 0.0008798355120234191, 0.00011563104635570198, 0.0004955217591486871, 0.00028452929109334946, 0.002383414888754487, 0.00022480194456875324, 0.0004086961562279612, 0.00021636753808707, 0.0004908627015538514, 0.0005686820368282497, 0.0015418070834130049, 0.0009966674260795116, 0.00010901422501774505, 0.003222328145056963, 0.0005733115249313414, 0.00156603439245373, 0.0003641012590378523, 0.0010883532231673598, 0.00046046561328694224, 0.0003790871414821595, 0.00045624346239492297, 0.00021894053497817367, 0.0008161700679920614, 0.0007519024075008929, 0.0010102358646690845, 0.001800333964638412, 0.0014407708076760173, 0.0008101118728518486, 0.0014813352609053254, 0.0006327987648546696, 0.0013731004437431693, 0.0004904716042801738, 9.038601274369285e-05, 0.0002642437466420233, 0.00020809787383768708, 0.00033833575434982777, 0.0004659502301365137, 0.0006002907175570726, 0.0009189092670567334, 0.00019216125656384975, 0.00034741961280815303, 0.001068762270733714, 0.0009977034060284495, 0.0018815381918102503, 0.00042778844363056123, 0.005034393630921841, 0.0002831303281709552, 0.00012126003275625408, 0.00038379651959985495, 0.00014250357344280928, 0.0010128598660230637, 0.00048006148426793516, 0.00011878977238666266, 0.000627847679425031, 0.0014110187767073512, 0.0002920645638369024, 0.00035349687095731497, 0.00024588019005022943, 0.00029043754329904914, 0.00013409662642516196, 0.0004053378361277282, 0.00018062142771668732, 0.0005937001551501453, 0.001555627561174333, 0.0010772714158520103, 0.0006678705685772002, 0.0015254943864420056, 0.0009534239652566612, 0.00028309979825280607, 0.0007567998254671693, 0.0009531023097224534, 0.0004002163768745959, 0.000915441254619509, 0.00022121128858998418, 0.0007485257228836417, 0.0009172006393782794, 0.0006067684735171497, 0.0003879637224599719, 0.0006741032120771706, 0.00022323866141960025, 0.0002951209316961467, 0.0008907586452551186, 0.0013901105849072337, 0.0011921300319954753, 0.0004996730131097138, 0.0008230014936998487, 0.00028673140332102776, 0.0006606338429264724, 0.0002859686501324177, 0.0001623499410925433, 0.0007258067489601672, 0.00038292258977890015, 0.00043665000703185797, 0.0002342240622965619, 0.0023966387379914522, 0.0008381654624827206, 0.000240033827139996, 0.0010040107881650329, 0.0005604204488918185, 0.0002559232816565782, 0.0002477842499502003, 0.0013652513734996319, 0.0003916561836376786, 0.000794220541138202, 0.0002896969090215862, 0.000868770235683769, 0.0008396358462050557, 0.0008641211898066103, 0.000803319038823247, 0.0014624495524913073, 0.0011502600973472, 0.0003082536277361214, 0.00281443540006876, 0.000344665750162676, 0.00030171425896696746, 0.0005511764902621508, 0.0012358493404462934, 0.000871849013492465, 0.00018728793656919152, 0.00034413536195643246, 0.001397694693878293, 0.0007001545745879412, 0.0006888980860821903, 0.00011696197179844603, 0.00055690883891657, 0.0007093378808349371, 0.00019057438476011157, 8.725078077986836e-05, 0.0006731793400831521, 0.001189857954159379, 0.00014746584929525852, 0.00039209777605719864, 0.0020497457589954138, 0.0026173333171755075, 0.0006035660044290125, 5.844902989338152e-05, 0.0006420924910344183, 0.001352024031803012, 0.006711121182888746, 0.0015058254357427359, 0.0011699318420141935, 0.00042466295417398214, 0.00013216224033385515, 0.0002531732025090605, 0.00023478221555706114, 0.0003094458079431206, 0.001151318196207285, 0.0006121114129200578, 7.529040885856375e-05, 0.0003963882045354694, 0.0017878353828564286, 0.0012447205372154713, 0.002455662237480283, 0.00018913504027295858, 0.0008044141577556729, 0.0003920519957318902, 0.00018952206301037222, 0.00027388829039409757, 9.387142927153036e-05, 0.00010580860543996096, 0.000638331170193851, 0.0004503101808950305, 0.0003480067243799567, 0.0018878483679145575, 0.0003775142249651253, 0.0009558734600432217, 0.0001853668800322339, 0.00016679245163686574, 0.00034552530269138515, 0.0003110053949058056, 0.00039760975050739944, 0.0005605483893305063, 0.002858054358512163, 0.00021834377548657358, 0.0001537739299237728, 0.0008866965072229505, 0.00183715985622257, 0.0030746993143111467, 0.00018743032705970109, 0.0009005563333630562, 9.087362559512258e-05, 0.0005079974653199315, 0.00018571819236967713, 0.0003136871091555804, 0.00012527981016319245, 0.0018812436610460281, 0.0013862618943676353, 0.0008647249196656048, 0.0002121732395607978, 0.0028710472397506237, 0.0015589214162901044, 0.0004243111761752516, 0.0016115629114210606, 0.0008972233044914901, 0.001013810164295137, 0.001210044021718204, 0.0011810873402282596, 0.0005527989706024528, 0.00014971346536185592, 0.0001961227535502985, 0.0008515677182003856, 0.00035978612140752375, 0.0007645272416993976, 8.76919730217196e-05, 0.0003589726984500885, 0.00036365611595101655, 0.0024767268914729357, 0.0004098165372852236, 0.00029510734020732343, 0.0006093111587688327, 0.001794511335901916, 0.006923090200871229, 0.0001423273206455633, 0.0031139440834522247, 7.897509203758091e-05, 0.0021080884616822004, 0.002325767185539007, 0.001673532766290009, 5.817387136630714e-05, 0.00021196645684540272, 0.0032576927915215492, 0.0008632385870441794, 0.0015689895953983068, 0.0005791975418105721, 0.0007431770791299641, 0.0007695714011788368, 0.00029560987604781985, 0.001450359239242971, 0.005527126602828503, 0.00018686185649130493, 0.00020826925174333155, 0.00020917771325912327, 0.0002413912588963285, 0.0006361879059113562, 0.0008754629525355995, 0.0009391976636834443, 0.0006970953545533121, 0.000162432887009345, 0.00046568430843763053, 0.0001941779046319425, 0.0004776805581059307, 0.0008916181395761669, 0.0005785813555121422, 0.002009250922128558, 0.0006198999471962452, 0.0015648780390620232, 0.009702582843601704, 0.0012381160631775856, 0.0001169376409961842, 0.00016487289394717664, 0.0004732986562885344, 0.0011898494558408856, 0.00037721885018981993, 0.0010089804418385029, 0.00034788690390996635, 0.0004206298617646098, 0.00042240991024300456, 0.00389950443059206, 0.001402851426973939, 0.0001401581394020468, 0.00048625245108269155, 0.002839328721165657, 0.0006217890768311918, 0.0055084917694330215, 0.0007228867616504431, 0.0007047245162539184, 0.00041368984966538846, 0.0004640145634766668, 0.0004686651809606701, 0.0010102740488946438, 0.0018250385764986277, 0.0004036765603814274, 0.0009966952493414283, 0.0001556663919473067, 0.0006141072371974587, 0.001011829823255539, 0.0005147781921550632, 0.0002556216495577246, 0.0010345649207010865, 0.0015460833674296737, 0.0018877777038142085, 0.0009833583608269691, 0.003343934426084161, 0.0004251216887496412, 0.000983419711701572, 0.00011730102414730936, 0.0005527143366634846, 0.0006024639587849379, 0.0047221058048307896, 0.0020225555635988712, 0.0018825946608558297, 0.00021361569815780967, 0.0004204841097816825, 0.0006523028714582324, 0.0001576101203681901, 0.00036761522642336786, 0.0011646182974800467, 0.0005242408369667828, 0.0002264233335154131, 0.003721072105690837, 0.0008816594490781426, 0.0005474374629557133, 0.00015473800885956734, 0.0013504073722288013, 0.00042464787838980556, 0.00035014073364436626, 0.00028635867056436837, 0.0019240234978497028, 8.539386180927977e-05, 0.00019563536625355482, 0.0012116495054215193, 0.0007735089166089892, 0.0008977903053164482, 0.000400570104829967, 0.0008659430895932019, 0.00026751463883556426, 0.00028529317933134735, 0.00044450053246691823, 0.0003261045494582504, 0.0008512723725289106, 0.0005517838289961219, 0.0002768114791251719, 0.00016495719319209456, 0.003166069043800235, 0.000201634262339212, 0.0006136043230071664, 0.00013092860172037035, 0.00015547616931144148, 0.0007006225059740245, 0.002813637023791671, 0.0025989587884396315, 0.0003479056467767805, 0.00022125286341179162, 0.00117009156383574, 0.0021083455067127943, 0.0002538194239605218, 0.0002714041038416326, 0.002454241970553994, 0.0002375900949118659, 0.0012652450241148472, 0.00026567085296846926, 0.0008362685912288725, 0.00014417871716432273, 0.0008823408279567957, 0.00019234485807828605, 0.00016393496480304748, 0.00032382344943471253, 0.0019074914744123816, 0.0007644293364137411, 0.0005743997171521187, 0.0011955664958804846, 0.00029546188306994736, 0.0016923771472647786, 0.0001464723754907027, 0.004179781302809715, 0.0008932442287914455, 0.0006771624321117997, 5.3647316235583276e-05, 0.0005676146247424185, 0.0005391987506300211, 0.0011324546067044139, 0.0001827893138397485, 0.003022153861820698, 0.00046562356874346733, 0.000776278437115252, 0.00031254859641194344, 0.006730898283421993, 0.0003153001016471535, 0.0006360408151522279, 0.0015089085791260004, 0.006025067064911127, 0.0005955162341706455, 0.00034540219348855317, 0.00043278528028167784, 0.0006282737012952566, 0.001088363816961646, 0.0006152089918032289, 0.00019660168618429452, 0.0006178776966407895, 0.0006908877403475344, 0.0003371361526660621, 0.0014868933940306306, 0.0021083015017211437, 5.4692307458026335e-05, 0.0010679486440494657, 0.0011420417577028275, 0.002754954854026437, 0.0013292849762365222, 0.0030469191260635853, 0.0018570777028799057, 0.0002283065696246922, 0.00034528918331488967, 0.00029180810088291764, 0.0010453722206875682, 0.0003963705385103822, 0.00022738153347745538, 0.0009435289539396763, 0.002476031193509698, 0.0006191154243424535, 0.00032752935658209026, 0.0002679893805179745, 0.00047251631622202694, 0.0005901322583667934, 0.0007199292886070907, 0.001199140097014606, 0.0006730672321282327, 8.19751076051034e-05, 0.001570151187479496, 0.012869982980191708, 0.00025201786775141954, 0.00015100432210601866, 0.002220752416178584, 0.003300626063719392, 0.0007644081488251686, 0.004649035632610321, 0.0036860776599496603, 0.0014990820782259107, 0.007309976499527693, 9.085750207304955e-05, 0.001745021902024746, 0.00017956551164388657, 0.0002672447299119085, 0.0004430548578966409, 0.00038740833406336606, 0.000530535529833287, 0.0055114999413490295, 0.0008093750802800059, 0.00035733223194256425, 0.00032021483639255166, 0.0004537328495644033, 0.00010000179463531822, 0.0042533669620752335, 0.0002595804980956018, 0.0036987008061259985, 0.0001351458195131272, 0.0002326435933355242, 0.0002593718527350575, 0.000888071779627353, 0.0012177028693258762, 0.00015896362310741097, 0.00011785527021856979, 0.00047397951129823923, 0.0007217920501716435, 0.0008760110940784216, 7.165285205701366e-05, 0.0023943271953612566, 0.0005025798454880714, 0.0008180374279618263, 0.0009299968951381743, 0.00021488399943336844, 0.001485962886363268, 0.0032104498241096735, 0.0008617135463282466, 0.00025431340327486396, 0.00033008158789016306, 0.0006805499433539808, 0.0006582754431292415, 0.0016352226957678795, 0.00024264646344818175, 0.00043591915164142847, 0.0008039062959142029, 0.0006054112454876304, 0.0004599480889737606, 0.0002237256121588871, 0.0003464732435531914, 0.0005420072702690959, 0.0017550588818266988, 0.0004938397323712707, 0.0023599148262292147, 0.0011842215899378061, 0.0007424427312798798, 0.0008401398081332445, 0.0006606987444683909, 0.00038305012276396155, 0.00029029560391791165, 0.0021281023509800434, 0.0002752016589511186, 0.0016146257985383272, 0.00027151493122801185, 0.0006768294260837138, 0.0011623748578131199, 0.002084776060655713, 0.0008109621121548116, 0.00023378862533718348, 0.00017061404651030898, 0.00026645290199667215, 0.0012328758602961898, 0.00019271609198767692, 0.0013752338709309697, 0.0005320730269886553, 9.793850040296093e-05, 0.00027745048282667994, 0.0008496290538460016, 0.00021996736177243292, 0.000690442742779851, 0.0008554316591471434, 5.652428444591351e-05, 6.073966505937278e-05, 0.00021262750669848174, 0.00010638940875651315, 0.0007780203013680875, 0.00047405410441569984, 0.0002470077888574451, 0.0006424154271371663, 0.000819777138531208, 7.025290688034147e-05, 0.0001984987611649558, 0.0003603587974794209, 0.002601937623694539, 0.00042074345401488245, 0.001952530350536108, 0.00013866729568690062, 0.0001591234904481098, 0.0009336379589512944, 0.0003174666198901832, 0.0021990740206092596, 0.0012389132753014565, 0.00018897104018833488, 0.00013955420581623912, 0.0013702529249712825, 0.0007841286133043468, 0.0008061436237767339, 0.00016966160910669714, 0.00073857675306499, 0.00012585252989083529, 0.0007870350382290781, 0.000920440477784723, 0.0002520346606615931, 0.0002759908093139529, 0.01493558194488287, 0.0006994879222474992, 0.0005901922704651952, 0.00018127259681932628, 0.0010116553166881204, 0.0006549243116751313, 0.0007365953060798347, 0.0018730901647359133, 0.0009840251877903938, 0.007531818933784962, 0.0003261436941102147, 0.00026204949244856834, 0.0013908995315432549, 0.00023427841369993985, 0.0013284210581332445, 0.0001647340541239828, 0.000604091736022383, 0.0046806843020021915, 0.0006104535423219204, 0.0008627138449810445, 0.004021362401545048, 0.0009948813822120428, 0.0003553406277205795, 0.00048128265189006925, 0.0031136616598814726, 0.002830339130014181, 0.0024958658032119274, 0.0006259625079110265, 0.00029000439099036157, 0.0008008242002688348, 0.00011408226419007406, 0.00013096813927404583, 0.000265135575318709, 0.0005156625411473215, 0.0009631406865082681, 5.4060703405411914e-05, 0.003401337191462517, 0.0004918928607366979, 0.0012522304896265268, 0.003110169665887952, 0.0009597177850082517, 0.003791121067479253, 0.00038946702261455357, 0.00018681786605156958, 0.000575977610424161, 0.00023401566431857646, 0.0026976964436471462, 7.546866982011124e-05, 0.0009367319871671498, 0.0007576880743727088, 0.0008541693678125739, 0.000373031769413501, 0.00023909386072773486, 0.00046743627171963453, 9.499280713498592e-05, 0.0001659401023061946, 0.00032806271337904036, 0.00018742095562629402, 0.00027287870761938393, 0.0008707405650056899, 4.449503103387542e-05, 0.00022111396538093686, 9.975935245165601e-05, 0.0007290974026545882, 0.0013202896807342768, 0.0005960920825600624, 0.00010593842307571322, 0.00027665949892252684, 0.0002493752399459481, 0.0005797098856419325, 0.0016712701180949807, 0.0003159704792778939, 0.003143458627164364, 0.000889290648046881, 0.0022135786712169647, 0.00038716531707905233, 0.00012633470760192722, 0.009618844836950302, 0.0006146769737824798, 0.0002618099970277399, 0.00012596695160027593, 5.751659409725107e-05, 0.00244762166403234, 0.0007075554458424449, 0.0006044773035682738, 0.00026248014182783663, 0.0008622235036455095, 0.0017710530664771795, 0.0002154180983779952, 0.00035322195617482066, 0.00010088751878356561, 0.00012936550774611533, 0.0002968037442769855, 0.0006562300841324031, 0.004982674028724432, 0.0009529624367132783, 0.0007876057643443346, 0.0006007288466207683, 0.0015589706599712372, 0.0004294016398489475, 0.0001952011079993099, 0.0002700300537981093, 0.0005772937438450754, 0.0010436063166707754, 0.0010405894136056304, 0.000742314150556922, 0.0004944044048897922, 0.0011871904134750366, 0.0008246904471889138, 0.002695901319384575, 0.00033345239353366196, 0.0007735579274594784, 0.00044476534822024405, 0.00034273244091309607, 0.0016928260447457433, 0.0008986304164864123, 0.00040475677815265954, 0.00046873794053681195, 0.00031939460313878953, 0.001322564552538097, 0.0007625406724400818, 0.0010214021895080805, 0.000648603483568877, 0.0016661313129588962, 0.0025453325361013412, 0.00030442807474173605, 0.0003222959057893604, 0.0005327002727426589, 0.00028156538610346615, 0.00048203140613622963, 0.0015664326492697, 0.00030906195752322674, 0.0002144567115465179, 0.0006144979270175099, 0.0013475288869813085, 0.0005295751616358757, 0.00020315339497756213, 0.0008483205456286669, 0.0004970297450199723, 0.00036062230356037617, 0.000631774659268558, 0.0009943593759089708, 0.0003173882432747632, 0.0006399216363206506, 0.0009105829522013664, 0.00022775947581976652, 0.0005539566627703607, 0.00037621063529513776, 0.0004682202998083085, 0.001227439264766872, 0.001527545158751309, 0.000726329511962831, 0.0020665163174271584, 0.0002121209108736366, 0.0012009922647848725, 0.0003275145136285573, 0.0003428614290896803, 0.002595143159851432, 0.0014552234206348658, 0.002295762300491333, 0.0027505436446517706, 6.038953142706305e-05, 0.0014509746106341481, 0.00020451446471270174, 0.001456448808312416, 0.007148141041398048, 0.0005664143827743828, 7.723012822680175e-05, 0.0008451063768006861, 0.0005504864384420216, 0.002818706212565303, 0.005531782750040293, 0.0005400381051003933, 0.0001665526651777327, 0.00020746758673340082, 0.00013918122567702085, 0.0001321007584920153, 0.0003537022857926786, 0.0008512127096764743, 0.00018074939725920558, 0.0004629354807548225, 0.010712928138673306, 9.456226689508185e-05, 0.001080036861822009, 0.0006832352955825627, 0.0004507765406742692, 0.0010034379083663225, 0.0003783251449931413, 0.00021232929429970682, 0.0007055733585730195, 0.0005007623112760484, 0.0009774164063856006, 0.00018534982518758625, 0.00017555285012349486, 0.002921953098848462, 0.0005301065975800157, 0.0010264343582093716, 0.000306317291688174, 0.0005930149927735329, 0.0005688074161298573, 0.00023066297580953687, 0.0010484252125024796, 0.001982102869078517, 0.0007816865108907223, 0.0016697803512215614, 0.0011916097719222307, 0.0004852395795751363, 0.0006353261996991932, 0.00019040223560296, 0.0006947715301066637, 0.0005828916328027844, 0.0021560348104685545, 0.0001806488144211471, 0.00015724977129139006, 0.000998254748992622, 9.07730936887674e-05, 0.0006152128917165101, 0.0012530168751254678, 0.00011782263754867017, 0.0008945591398514807, 0.0006922399043105543, 0.0017265627393499017, 0.000224070914555341, 0.00030888302717357874, 0.0002492025960236788, 0.00021494289103429765, 0.0013217960949987173, 0.001244383747689426, 0.0034826386254280806, 0.002076192758977413, 0.00014520341937895864, 0.0005817657802253962, 0.0010553757892921567, 0.001910409308038652, 0.0004047406546305865, 0.00011109055049018934, 0.00030501277069561183, 0.00020926291472278535, 0.0006526634097099304, 0.0014345403760671616, 0.0019914149306714535, 0.0006557741435244679, 0.0007601980469189584, 0.0009466939372941852, 0.0003862865560222417, 0.0011528227478265762, 0.0037395108956843615, 0.00020899386436212808, 0.0019155147019773722, 0.0014105833834037185, 0.001654494204558432, 0.0019514845917001367, 0.020687289535999298, 0.0005183244938962162, 0.0006825804011896253], [0.00020421977387741208, 0.0006519047310575843, 0.0015365424333140254, 0.00039596849819645286, 0.00027990370290353894, 0.00032060834928415716, 0.00014427116548176855, 0.00784409511834383, 0.0009983687195926905, 0.00020320876501500607, 0.0005968167097307742, 0.0009759243694134057, 0.0005039067473262548, 0.0004619753162842244, 0.000966783904004842, 0.00024224360822699964, 0.00070048967609182, 0.0002721998607739806, 0.00022777316917199641, 0.0010447984095662832, 0.0002570251817815006, 0.00037360642454586923, 0.0006662359810434282, 0.0018827080493792892, 0.0005043057608418167, 0.0012440558057278395, 0.00015216931933537126, 0.0002260866021970287, 0.0034834756515920162, 0.0007427226519212127, 0.0004713664238806814, 0.00021364001440815628, 0.00042800893425010145, 0.0008837076602503657, 0.00052868970669806, 0.001022324082441628, 0.0004915695171803236, 0.00027483716257847846, 0.008982296101748943, 0.0005557198892347515, 0.0006155016599223018, 0.0009522438049316406, 0.0006181698408909142, 0.007326692808419466, 0.0009265073458664119, 0.0015837656101211905, 0.0008383133099414408, 0.0033952011726796627, 0.0033575696870684624, 0.0006864449242129922, 0.00020221156592015177, 0.0011313651921227574, 9.344791760668159e-05, 0.0007294354727491736, 0.0019266322487965226, 0.0007996891508810222, 0.00027144854539074004, 0.005808088928461075, 0.001453146804124117, 0.00025275960797443986, 0.00012202634388813749, 0.00021338813530746847, 0.0018752478063106537, 0.0005162109737284482, 0.0012465092586353421, 0.0006849226192571223, 0.0011221003951504827, 0.0003392760409042239, 0.003144114511087537, 0.00046544268843717873, 0.0005746167153120041, 0.000928830704651773, 0.001392990816384554, 0.0005562611040659249, 0.0003998709435109049, 0.000581420841626823, 0.006889824755489826, 9.008276538224891e-05, 0.0011428198777139187, 0.0008068833267316222, 0.00021648067922797054, 0.00019300442363601178, 0.0003244662657380104, 0.00017941075202543288, 0.0004799313610419631, 0.0008400453953072429, 0.000770772632677108, 0.00011197443382116035, 0.00016101333312690258, 0.003820820478722453, 0.00018537654250394553, 0.000306920672301203, 0.0027691423892974854, 0.0003057998837903142, 0.0005902449483983219, 0.0018837692914530635, 0.0025780389551073313, 0.0011808070121333003, 0.0003701218811329454, 6.98798248777166e-05, 0.00036092408117838204, 0.0008622005116194487, 0.0004441554192453623, 0.000453164306236431, 0.0010845395736396313, 0.0003460379666648805, 0.00014510257460642606, 0.0004396584117785096, 0.0006991589907556772, 0.00117313617374748, 0.0005244020721875131, 0.0006469687214121222, 0.00022545990941580385, 0.00038718359428457916, 0.00034644996048882604, 0.00026814680313691497, 0.0004499550850596279, 0.0026687674690037966, 0.0003978759632445872, 0.000618888414464891, 9.543291525915265e-05, 0.0008548639598302543, 0.0023094220086932182, 0.0006923488108441234, 0.0009713702602311969, 0.0010534609900787473, 0.0020672925747931004, 0.0005646329955197871, 0.0005757460603490472, 0.0015622245846316218, 0.00038971551111899316, 0.000424394995206967, 0.00043369020568206906, 0.0007695379899814725, 0.00046825764002278447, 0.0025129972491413355, 0.0005470238975249231, 0.00031010780367068946, 0.002477692672982812, 0.001894891494885087, 0.004053005948662758, 0.001460739178583026, 0.0002203849726356566, 0.0034296256490051746, 0.0016762159066274762, 0.00023913191398605704, 0.00043439544970169663, 0.00015236418403219432, 0.0007673432119190693, 0.0002118084958055988, 0.0010082850931212306, 6.113748531788588e-05, 0.0018300507217645645, 0.002918465994298458, 0.0007994513143785298, 0.0001645581069169566, 0.00023867443087510765, 0.001226693973876536, 4.103311584913172e-05, 0.0011757569154724479, 0.00012852303916588426, 0.0003523833293002099, 0.0003222537343390286, 0.0005584738100878894, 0.0009464918985031545, 0.00033934018574655056, 0.0006865130853839219, 0.00013186104479245842, 0.0005818458157591522, 0.000158772396389395, 0.0005438714870251715, 0.0005034871282987297, 0.0006240879301913083, 0.0006083408370614052, 0.0004703245940618217, 0.0011907625012099743, 0.00028022960759699345, 0.0005582275916822255, 0.00050528155406937, 0.0012561955954879522, 0.00048544330638833344, 0.0003470705123618245, 0.0028062155470252037, 0.00036683064536191523, 0.00013924995437264442, 0.00313919922336936, 0.0004066459077876061, 0.0005911308107897639, 0.0005204074550420046, 0.0005271413247101009, 0.00041263841558247805, 0.00010449959518155083, 0.0006886920891702175, 0.005536136217415333, 0.0013904807856306434, 0.0008035495411604643, 0.0003959792375098914, 0.00019115731993224472, 0.0002544150920584798, 0.0002579150313977152, 0.00017491991457063705, 0.002565553644672036, 0.0006139837787486613, 0.0002925424196291715, 0.0005371200386434793, 0.0029579836409538984, 0.001235059811733663, 0.001942252623848617, 0.00029988796450197697, 0.0001574684720253572, 0.0007309787324629724, 0.0008837647037580609, 0.0015138491289690137, 0.00016505899839103222, 0.0004644483851734549, 0.0033049213234335184, 0.00014551197818946093, 0.000423263234551996, 0.00047098140930756927, 0.00020495442731771618, 0.001645844429731369, 0.0005234226118773222, 0.00016186252469196916, 0.00026640499709174037, 0.0002927932364400476, 0.0035813043359667063, 0.00043990270933136344, 0.0003214612079318613, 0.0025267298333346844, 0.0008604370523244143, 0.0064492253586649895, 0.0003814362280536443, 0.00010201448458246887, 0.0026919294614344835, 0.0003905315534211695, 0.000642774801235646, 0.0005467386217787862, 0.0003351531340740621, 0.0013526829425245523, 0.0003018875722773373, 0.0009930719388648868, 0.00048356418847106397, 0.0011904536513611674, 0.004196482710540295, 0.0007674591615796089, 0.0004603809502441436, 0.009968115016818047, 0.004262465517967939, 0.0005619906587526202, 0.0016552332090213895, 0.0010115852346643806, 0.0005317411269061267, 0.0002762509975582361, 0.0007498718332499266, 0.00020471838070079684, 0.00310106691904366, 0.0035481275990605354, 0.00029577480745501816, 0.00034089499968104064, 0.00018982427718583494, 0.002418351825326681, 0.0007126552518457174, 0.0002162443706765771, 0.00046245119301602244, 0.0017804992385208607, 0.0009043147438205779, 0.002071443945169449, 7.152995385695249e-05, 0.0009202932706102729, 0.0010374741395935416, 0.0009445626055821776, 0.0008751185960136354, 0.00038684747414663434, 0.0005329690757207572, 0.00029757723677903414, 0.000167692982358858, 0.003179430728778243, 0.0002896357618737966, 0.0003615970490500331, 2.97066435450688e-05, 0.0012432454386726022, 0.0005000739474780858, 0.00036129608633928, 0.0004904587985947728, 0.0004645369481295347, 0.00029564238502644, 0.0003078487643506378, 0.0009994781576097012, 0.0016603004187345505, 0.0019076579483225942, 0.0006801223498769104, 0.0022081611678004265, 0.0019751295913010836, 0.0024182701017707586, 0.001740568084642291, 0.00015214690938591957, 0.0007081180810928345, 0.0009953692788258195, 0.0031094145961105824, 0.00041691356454975903, 0.001073150197044015, 0.000797997519839555, 0.00013907592801842839, 0.0003870729706250131, 0.00022318416449707001, 0.005284417886286974, 0.0010413369163870811, 0.0022328428458422422, 0.0010345333721488714, 0.0011980135459452868, 0.002220295136794448, 0.0004022597859147936, 0.00023487421276513487, 0.00028595858020707965, 0.0019374876283109188, 0.0007060071802698076, 0.00029207614716142416, 0.00018109529628418386, 0.00036480367998592556, 0.000961151032242924, 0.0006547098746523261, 0.00036173357511870563, 0.00023422496451530606, 0.00040815328247845173, 0.002969407942146063, 0.00743057020008564, 0.0008385322289541364, 0.0032487153075635433, 0.0007560825324617326, 0.0001841653574956581, 0.0004753412213176489, 0.00018920298316515982, 0.0009263756219297647, 0.0007080200593918562, 7.967847341205925e-05, 0.0002343684755032882, 0.0037844509351998568, 0.0003594279696699232, 0.0007367858197540045, 0.0004323108878452331, 0.0002728715189732611, 0.0008270084508694708, 0.00037408608477562666, 0.000317353755235672, 0.0005453861667774618, 8.42916633700952e-05, 0.00010262009891448542, 8.9688524894882e-05, 0.001889777835458517, 0.0009820661507546902, 0.0006071952520869672, 0.0009520762250758708, 0.0012758226366713643, 5.057306771050207e-05, 9.178707841783762e-05, 0.0006967157241888344, 0.0016475319862365723, 0.0011833267053589225, 0.00014080051914788783, 0.0005733049474656582, 0.0014503027778118849, 0.0005623325705528259, 0.00029686614288948476, 0.00032856897450983524, 0.0017599379643797874, 0.000148356644785963, 0.0003139804466627538, 0.0015612575225532055, 0.00027539668371900916, 3.454857869655825e-05, 0.0008733716676943004, 0.00041639924165792763, 0.0001249822526006028, 0.00015067800995893776, 0.0010385325876995921, 0.0013815336860716343, 0.00018909912614617497, 0.0009662338416092098, 0.0004310127696953714, 0.0006178989424370229, 0.0013616724172607064, 0.00012513082765508443, 0.00033413138589821756, 0.013636302202939987, 0.00016677923849783838, 0.0007422995404340327, 0.00019628817972261459, 0.001449621981009841, 0.0009618843323551118, 0.00028608139837160707, 0.0005943991127423942, 0.0010499936761334538, 0.002176412846893072, 0.0009997939923778176, 0.0024095396511256695, 0.0008211142267100513, 0.0020086851436644793, 0.0009812555508688092, 0.0007605189457535744, 0.001106305862776935, 0.0011115975212305784, 0.0007156331557780504, 0.0002300058549735695, 0.00048757705371826887, 0.0021103564649820328, 0.0006883736932650208, 0.0002826365234795958, 0.0004151893954258412, 0.0021501779556274414, 0.0001529283617855981, 0.00012815311492886394, 0.0003018751449417323, 0.00024737243074923754, 0.0015858332626521587, 0.0007952845189720392, 0.0011689444072544575, 0.0004305317415855825, 0.0013039852492511272, 0.0005324153462424874, 0.0002319735795026645, 0.0002653677365742624, 0.0011421681847423315, 0.0010611116886138916, 0.0003306986764073372, 0.0016428647795692086, 0.00097750558052212, 0.0004458943731151521, 0.00018541420286055654, 0.0025019985623657703, 0.0008675709250383079, 0.0014320984482765198, 0.00025297541287727654, 0.0008218667353503406, 0.0014991624047979712, 0.0007743992027826607, 0.0006144886719994247, 0.0001883485383586958, 0.002215079963207245, 0.0013426552759483457, 0.0003216473269276321, 0.00010409409878775477, 0.0018281545490026474, 0.00021526956697925925, 0.0004697130643762648, 0.0005322329816408455, 0.005665485747158527, 0.0009723128168843687, 0.0034172367304563522, 0.000896995363291353, 0.00029259882285259664, 0.0020886030979454517, 0.005005694460123777, 0.0004019024199806154, 0.00040042222826741636, 0.0020788339897990227, 0.004434926901012659, 0.0004498597118072212, 0.000339480146067217, 0.0005299561307765543, 0.00046583483344875276, 0.0003220655780751258, 0.0010360804153606296, 0.0005537208635360003, 0.0003955297579523176, 0.001271261484362185, 0.00011952366912737489, 0.0028840056620538235, 0.0033702400978654623, 0.001167408307082951, 0.00038687666528858244, 0.00025203070254065096, 0.00027424999279901385, 0.0010751297231763601, 0.00041173407225869596, 0.0013934170128777623, 0.0005710034165531397, 0.0009488026262260973, 0.002393638016656041, 0.0008838145295158029, 0.0002269152901135385, 0.00049691490130499, 7.636693044332787e-05, 0.0008005717536434531, 6.401299469871446e-05, 0.00036635732976719737, 0.0002800584479700774, 0.0004295199760235846, 0.0018135131103917956, 0.000502925249747932, 0.000592205673456192, 0.00012895281543023884, 0.0006862561567686498, 0.001334351603873074, 0.00017210814985446632, 0.0003194636956322938, 4.4635151425609365e-05, 0.0016185545828193426, 0.0007705605821684003, 0.00010062918590847403, 0.0011931746266782284, 0.00032175425440073013, 0.00021036724501755089, 0.004096269141882658, 0.0006516357534565032, 0.0002472709456924349, 8.806341793388128e-05, 0.0011653841938823462, 0.0004325849295128137, 0.0001422070199623704, 0.0005633269320242107, 0.0002250328689115122, 0.0009594365837983787, 0.00153200956992805, 0.0005317145842127502, 0.0009932981338351965, 0.0010914555750787258, 0.0025542452931404114, 0.0009123515919782221, 0.00018943952454719692, 0.002106456086039543, 0.00041682555456645787, 0.0004855708102695644, 0.000799517787527293, 0.0009552293922752142, 0.00038337387377396226, 0.0005073823849670589, 0.0007170176249928772, 0.00021220184862613678, 0.00011298603203613311, 0.0006069331429898739, 0.0006814388907514513, 0.0003452976525295526, 0.0003110301331616938, 0.0004768693179357797, 0.0005310238339006901, 0.0004089810827281326, 0.0005000667297281325, 0.0005287135718390346, 0.0002453827764838934, 6.257236964302137e-05, 0.00030366567079909146, 0.0006496132118627429, 4.2095045500900596e-05, 0.00018621857452671975, 7.289536006283015e-05, 0.00035075334017165005, 0.0009055021801032126, 0.0008750802371650934, 0.001057939138263464, 0.0026755286380648613, 0.0005469807074405253, 0.0012860045535489917, 0.0016472082352265716, 0.00015630442067049444, 0.0016517352778464556, 0.0013590826420113444, 0.000578573439270258, 0.0015269662253558636, 0.0012128317030146718, 0.0006775366491638124, 0.0008627853821963072, 0.0014391050208359957, 0.0012067541247233748, 0.000156988826347515, 0.0014667154755443335, 0.0017132321372628212, 0.0014019670197740197, 0.0005596534465439618, 0.0009794214274734259, 0.0003099926107097417, 0.0005706117954105139, 0.000716924318112433, 0.001455633551813662, 0.00022448779782280326, 0.0002651622926350683, 0.0004337352584116161, 0.0013056609313935041, 0.0008055718499235809, 0.0020838456694036722, 0.0013008657842874527, 0.00033885418088175356, 0.00015169505786616355, 0.0001494199241278693, 0.0002944441803265363, 0.0022169023286551237, 0.0006396240787580609, 0.0012038449058309197, 0.00016236526425927877, 0.00028971838764846325, 0.0002544457674957812, 0.0002703201025724411, 0.0006004172028042376, 0.0004999684751965106, 0.00025582851958461106, 0.00029186945175752044, 0.0005507285241037607, 0.00016187365690711886, 0.0025547875557094812, 0.0003908923245035112, 0.00033984414767473936, 0.000853417266625911, 0.0017288682283833623, 0.0007669847109355032, 0.0007471786811947823, 0.0012895500985905528, 0.0003773346543312073, 0.00032021396327763796, 0.00040033372351899743, 0.0003233933530282229, 0.0002006908180192113, 0.0032708824146538973, 0.0018517302814871073, 0.000640448706690222, 0.002041843719780445, 0.0001342479226877913, 0.0003930418170057237, 0.0026992708444595337, 0.0013677526731044054, 0.0004891063435934484, 0.0001498280034866184, 0.002350259106606245, 0.00020013593893963844, 0.0002918143873102963, 0.0005964617594145238, 0.000613682612311095, 0.000758117064833641, 0.0008959686965681612, 0.0009355738293379545, 0.007998003624379635, 0.0009956397116184235, 0.0006370345945470035, 8.074188372120261e-05, 0.0017062993720173836, 0.0003751721524167806, 0.0014615339459851384, 0.0007034863228909671, 0.0016218569362536073, 0.002105472609400749, 0.00404334207996726, 0.0005406566779129207, 0.000201180373551324, 0.0006586762028746307, 9.137986489804462e-05, 0.0004538604407571256, 0.000622596766334027, 5.210258677834645e-05, 0.0018223918741568923, 0.00022517098113894463, 0.00041429686825722456, 0.0007887330139055848, 0.0005883925477974117, 0.00095754349604249, 0.0005976351676508784, 0.0008186876075342298, 0.00022047024685889482, 0.0014168631751090288, 0.00029032709426246583, 0.00017987792671192437, 0.0004920149222016335, 0.00019362707098480314, 2.7052927180193365e-05, 0.003334233071655035, 0.00026298046577721834, 0.0005494694923982024, 0.005131714977324009, 0.00019502174109220505, 0.0005081407143734396, 0.0009391456260345876, 0.0017635084223002195, 0.00017399562057107687, 0.00015529112715739757, 0.0017595055978745222, 0.000719564501196146, 0.00017440695955883712, 0.0001300626463489607, 0.0015788472956046462, 0.002361644757911563, 0.0005255996948108077, 0.0005134710227139294, 0.00036982548772357404, 4.436175004229881e-05, 0.002343634841963649, 0.0006450561922974885, 0.002214404521510005, 0.00032958079827949405, 0.0004476098401937634, 0.0005805182736366987, 0.0005882735713385046, 0.0004297755367588252, 0.0024085091426968575, 0.0023429132997989655, 0.0019105742685496807, 0.00037335045635700226, 0.0005595087422989309, 0.00022927040117792785, 0.008775229565799236, 0.00039897189708426595, 0.0008070119074545801, 0.0002631194656714797, 0.0011639860458672047, 0.0015893152449280024, 0.0002511757193133235, 0.0012070033699274063, 0.00028523572837002575, 0.0003800115955527872, 0.0005774684250354767, 0.00031305637094192207, 0.001725872396491468, 0.000457817135611549, 0.0006959448219276965, 0.001092440215870738, 0.00027906522154808044, 0.0004625862929970026, 0.0005047494196332991, 0.000100658209703397, 0.0036030556075274944, 0.0013021985068917274, 0.0009826883906498551, 0.0012937838910147548, 0.00018141785403713584, 0.0005111573846079409, 0.0022289373446255922, 0.00037937788874842227, 0.0007909397245384753, 0.0009236923651769757, 0.00012809156032744795, 0.0003508205118123442, 0.0011188884964212775, 0.0007132690516300499, 0.005403735209256411, 0.000991515931673348, 6.836694228695706e-05, 0.0031643721740692854, 0.00015326192078646272, 0.0009507880313321948, 0.0008613412501290441, 0.0008380824001505971, 0.0007828815723769367, 0.0002663357590790838, 0.0012553922133520246, 0.0008093687356449664, 0.0003941226750612259, 0.0012834534281864762, 0.006971355527639389, 0.00019398791482672095, 0.00034248922020196915, 0.0005548923509195447, 0.00023666566994506866, 0.000603373278863728, 0.00024066197511274368, 0.0006493870751000941, 0.0016601497773081064, 0.0008092215284705162, 0.0004589335003402084, 0.0009733231272548437, 0.0007876511081121862, 0.001289116102270782, 0.0005442894180305302, 0.00015486629854422063, 0.0013072268338873982, 0.00044409226393327117, 0.0004732490924652666, 0.0015455415705218911, 0.0008932073251344264, 0.00040389044443145394, 0.0068984669633209705, 0.0035370083060115576, 0.00039416077197529376, 0.00010790920350700617, 0.0018642412032932043, 0.0009648361592553556, 0.0006315549253486097, 0.0008660043822601438, 0.000559251697268337, 0.0011900585377588868, 0.0007091372390277684, 0.001654278370551765, 0.0019451009575277567, 0.0006118066376075149, 0.001027510268613696, 0.0011464180424809456, 0.0006404571468010545, 0.0012023517629131675, 0.0005812494782730937, 0.0016641977708786726, 0.00036593995173461735, 0.0005219750455580652, 0.00012085813796147704, 0.00014923347043804824, 0.0009041544399224222, 0.0005707521340809762, 0.0006442597950808704, 0.0028618499636650085, 0.0006494233966805041, 0.006285552401095629, 0.0001071191654773429, 0.0007473258883692324, 0.002125453669577837, 0.0009646204416640103, 0.001229824498295784, 0.00034663171391002834, 0.00011683409684337676, 0.0005670066457241774, 0.0015055472031235695, 0.0009577382588759065, 0.0002206172066507861, 0.0012947252253070474, 0.0005765438545495272, 0.0005531498463824391, 0.0010373385157436132, 0.00024196977028623223, 0.001889709965325892, 0.004643080290406942, 0.00019219204841647297, 0.000721028249245137, 0.00018764463311526924, 0.0008548998739570379, 0.0011895569041371346, 0.0036442296113818884, 0.00026172990328632295, 0.0003881719312630594, 0.001286066835746169, 0.0005732428980991244, 0.001019937451928854, 0.0036653694696724415, 0.0013168100267648697, 0.00018313346663489938, 0.0035965617280453444, 0.00017018003563862294, 0.0014024056727066636, 0.0007391950348392129, 0.0010308307828381658, 0.000720171257853508, 0.0010019800392910838, 8.915782382246107e-05, 0.0008930120966397226, 0.00025858855224214494, 0.0002765426761470735, 0.000647911976557225, 0.0006917867576703429, 0.0004415358998812735, 0.003961250651627779, 0.001396659412421286, 8.439538214588538e-05, 0.0003707062569446862, 0.00021748545987065881, 0.0003332517808303237, 0.003656925866380334, 0.0006532497354783118, 0.0015287419082596898, 0.001209721202030778, 0.00017586360627319664, 0.00020888386643491685, 0.0004969083238393068, 0.0008161633741110563, 0.00027784184203483164, 0.0009444522438570857, 0.0006531169055961072, 0.0008598343702033162, 0.002256857231259346, 0.0011988510377705097, 0.004025301896035671, 0.00041715148836374283, 0.0007156209903769195, 8.337276085512713e-05, 0.0005374336615204811, 0.0014849051367491484, 0.00012391977361403406, 0.0002520377456676215, 0.0011563642183318734, 0.00029299670131877065, 0.0011477092048153281, 0.0039163320325315, 0.0018996739527210593, 0.00015741588140372187, 0.0005442118272185326, 0.0008051773766055703, 0.0004039126797579229, 0.0005407487042248249, 0.0005281941266730428, 9.384691657032818e-05, 0.0012488075299188495, 0.0014787435065954924, 0.002719431882724166, 0.00010324504546588287, 0.0008599939756095409, 0.001690218923613429, 0.0006671961746178567, 0.00024483221932314336, 0.0001486788532929495, 0.0020743871573358774, 0.00017767431563697755, 0.000225806055823341, 0.003260683733969927, 0.00010771150118671358, 0.001143006724305451, 0.0009479215950705111, 0.0006481491727754474, 0.0009459282737225294, 0.00015358511882368475, 0.0006201085052452981, 0.0014207200147211552, 0.0008907723240554333, 0.0005998587002977729, 0.0001294771209359169, 0.0004121735109947622, 0.0027451543137431145, 0.001762663247063756, 0.0011198943248018622, 0.0007270856876857579, 0.0016300503630191088, 0.0010804705088958144, 0.0002452264598105103, 0.001114638871513307, 0.0013734556268900633, 0.00017816810577642173, 0.0017282302724197507, 0.00025912586716003716, 0.0014888958539813757, 0.0006367713795043528, 0.0008521564304828644, 0.0001923866366269067, 0.0017314916476607323, 0.0005136930849403143, 0.00039177475264295936, 0.0020806954707950354, 0.000361599144525826, 0.0008021618123166263, 0.00291596632450819, 0.00035528160515241325, 0.00022582347446586937, 0.0006763337878510356, 0.0003751048643607646, 0.0005101384012959898, 0.00028918133466504514, 0.0002891689946409315, 7.799935701768845e-05, 0.0001527968852315098, 0.0007352615357376635, 0.0006567279924638569, 0.000989847001619637, 0.0020986972376704216, 0.0002500853734090924, 0.0005885703722015023, 0.0003555282310117036, 0.0010270440252497792, 0.0002903208660427481, 0.0010788034414872527, 0.00017920625396072865, 0.0007893161964602768, 0.0008392541785724461, 0.001957830972969532, 0.0010920921340584755, 0.0015676292823627591, 0.0008031749748624861, 0.0003189464332535863, 0.0002686411899048835, 0.0007708806660957634, 0.00030611749389208853, 0.00021527285571210086, 7.289514906005934e-05, 0.00022919560433365405, 0.00023162677825894207, 0.00014201007434166968, 0.0013635879149660468, 0.00012365567090455443, 0.0006961554754525423, 0.0007393042324110866, 0.0003531903785187751, 0.00027183323982171714, 0.0004612788907252252, 0.001428593066520989, 0.0013385787606239319, 0.00043371471110731363, 0.00047608366003260016, 0.0021084602922201157, 0.0002481337869539857, 0.0006494844565168023, 0.001509290304966271, 0.0021467239130288363, 0.00022960978094488382, 0.0010030005360022187, 0.0006481495802290738, 0.00011124195589218289, 0.00024573461269028485, 0.0011319768382236362, 0.0006089273374527693, 0.0006363827269524336, 0.00019574176985770464, 0.0004940812359564006, 0.0011392432497814298, 0.0016941509675234556, 0.0011263960041105747, 0.0008825519471429288, 0.0003674469771794975, 0.007341895252466202, 0.001411467557772994, 8.863103721523657e-05, 0.00025593480677343905, 0.0007554446347057819], [0.0002141506556654349, 0.0016116163460537791, 0.0007829382084310055, 0.0007905640522949398, 0.0004079570062458515, 0.00363670545630157, 0.0006531314575113356, 0.00023167990730144083, 0.00011374929454177618, 0.0007989090518094599, 0.0004691747308243066, 0.00018749748414848, 0.0008480100659653544, 0.0002701051125768572, 0.00020365408272482455, 0.0005306006642058492, 0.00019850680837407708, 0.0005651795654557645, 0.00016855391731951386, 0.00026150690973736346, 0.004995250608772039, 0.00023504780256189406, 0.000681863515637815, 0.00781905185431242, 0.000557845167350024, 0.0004151434695813805, 0.000127322506159544, 0.001224489533342421, 5.898332892684266e-05, 0.00013590336311608553, 0.0004820732865482569, 0.0026200804859399796, 0.00394426891580224, 0.001863723387941718, 0.0006952906842343509, 0.00032058326178230345, 0.0008900031098164618, 0.0021534129045903683, 0.00046745306462980807, 0.0016859123716130853, 0.00016040629998315126, 0.0005819526268169284, 0.0004160097159910947, 0.0008682143525220454, 0.0003612847358454019, 0.00030594022246077657, 0.0005458684172481298, 0.00036994111724197865, 0.0002178088907385245, 5.935442459303886e-05, 0.0029260842129588127, 8.806536061456427e-05, 0.00041598593816161156, 0.0005582066951319575, 0.00236384104937315, 0.00042172716348432004, 0.0006303393747657537, 0.0005972920916974545, 0.0010072692530229688, 0.0019175809575244784, 0.00028727480093948543, 0.0005069860490038991, 0.0001067856719600968, 0.00011662174802040681, 0.00015425121819134802, 0.00011129939957754686, 0.0010950954165309668, 0.0006270052981562912, 0.0002281984779983759, 0.0056205615401268005, 0.0007107873680070043, 0.0016579106450080872, 0.0010017415042966604, 0.000362026592483744, 0.0003770995535887778, 0.0010962319793179631, 0.0006435621762648225, 0.003234283532947302, 0.0012247763806954026, 0.001088022836484015, 0.0002823512477334589, 0.0002254472638014704, 0.00025271964841522276, 0.004703192505985498, 0.0002126708859577775, 0.00018936331616714597, 0.0011047999141737819, 0.0003071933169849217, 0.0005164791946299374, 5.726287781726569e-05, 0.0005639364244416356, 0.0005560998106375337, 0.0003363547148182988, 0.00040275341598317027, 0.0005141755100339651, 0.0009490078664384782, 0.0003191445139236748, 0.0011968034086748958, 0.0014622136950492859, 0.00031726446468383074, 0.0005669143283739686, 0.0009600858320482075, 8.169366628862917e-05, 0.00033197965240105987, 0.004083690699189901, 0.00015151681145653129, 0.0003284478443674743, 0.0008613867103122175, 0.0008520390256308019, 0.0006455829716287553, 0.0004311222874093801, 0.000461769086541608, 0.0007230773335322738, 0.001259245560504496, 0.0003527434018906206, 0.00015328702284023166, 0.0004403734637890011, 0.003999167121946812, 0.0008003202965483069, 0.0006290836608968675, 0.0002286733069922775, 0.00017018684593494982, 0.0007294259849004447, 0.000725012447219342, 0.0007769664516672492, 0.00023647946363780648, 0.0003820853598881513, 0.00040565666859038174, 0.00031081613269634545, 0.0003055815468542278, 0.00032451655715703964, 0.00010517917689867318, 0.0002893548517022282, 9.488888463238254e-05, 0.00032992480555549264, 0.0023822991643100977, 0.00015809782780706882, 0.0015526097267866135, 7.038441981421784e-05, 0.0008513261564075947, 0.0001146354406955652, 0.00022554337920155376, 0.0006490982486866415, 0.0008534395019523799, 0.00034496732405386865, 0.0025267181918025017, 6.691209273412824e-05, 0.0005573412054218352, 0.0010582845425233245, 0.0004580179520417005, 0.0008682550978846848, 0.0005919043906033039, 0.0010984299005940557, 0.0005628578364849091, 0.0015804849099367857, 0.0009287962457165122, 0.00043691936298273504, 0.0005129718920215964, 0.0005843386752530932, 0.0005327364197000861, 0.005516806151717901, 0.00032470724545419216, 0.0004668075416702777, 0.0012020589783787727, 0.0012806145241484046, 0.001347035518847406, 0.0007179377134889364, 0.001088710967451334, 0.003919313196092844, 0.0017615989781916142, 0.0005128383054398, 0.0016685087466612458, 0.001917564426548779, 0.0033829675521701574, 0.001757367281243205, 0.000278375024208799, 0.0009730533929541707, 0.000600864237640053, 0.0005966942990198731, 0.0004788400256074965, 0.0007007363601587713, 0.001032856642268598, 0.0005528994370251894, 0.0008534117368981242, 0.0002152378874598071, 0.0003546357329469174, 0.00032942675170488656, 0.000977091956883669, 0.0010163589613512158, 0.00033101587905548513, 0.0007111661252565682, 0.000647800334263593, 7.128225843189284e-05, 0.00024446574389003217, 0.0008970483322627842, 0.0015509964432567358, 0.00044872844591736794, 0.0007533463067375124, 0.002163620200008154, 0.00103860255330801, 0.0005722280475310981, 0.0018455259269103408, 0.0006201177602633834, 0.00206525856629014, 0.0009148327517323196, 0.0004443427606020123, 0.0006740113021805882, 0.0009230759460479021, 0.00029809490661136806, 0.00035452190786600113, 0.0008553715306334198, 0.0006899166037328541, 6.818143447162583e-05, 0.012120196595788002, 0.0007469464326277375, 0.0010990423616021872, 0.0021849912591278553, 0.002265744376927614, 0.0009108876110985875, 0.004022916313260794, 0.0002505520824342966, 0.0004242340219207108, 0.00022169841395225376, 0.0023075295612215996, 0.00032193257356993854, 0.000474764674436301, 0.00047698148409835994, 0.01911858841776848, 0.00017525829025544226, 0.002500523580238223, 0.0013500580098479986, 0.00032624226878397167, 0.000960606848821044, 0.0007998705259524286, 0.0006967947701923549, 0.0001750865048961714, 0.001229553367011249, 0.0005010184831917286, 0.00025227194419130683, 0.0018585182260721922, 0.0011583329178392887, 0.0014540372649207711, 0.0010159622179344296, 0.0001333673280896619, 0.000361121550668031, 0.000498825334943831, 0.00021011590433772653, 0.00013704060984309763, 0.00031383457826450467, 0.002038086298853159, 0.00017849092546384782, 0.000725352787412703, 0.002207271521911025, 0.0006422702572308481, 0.0004306086921133101, 0.00019761563453357667, 0.000313659809762612, 0.0005431748577393591, 7.705411553615704e-05, 0.0006597905303351581, 0.004475305788218975, 0.00020800280617550015, 0.0025959170889109373, 0.00034232312464155257, 0.001646855496801436, 0.0021445187740027905, 0.0006519380840472877, 0.0002468830207362771, 0.0004042012442369014, 0.0002520908019505441, 0.0003856969124171883, 0.0018355216598138213, 0.00022173322213348, 0.00114000728353858, 0.0003405669122003019, 0.00021016894606873393, 0.00019863987108692527, 0.0019477269379422069, 0.0015064857434481382, 0.0004593845806084573, 0.0003615055466070771, 7.22955956007354e-05, 0.00015054972027428448, 0.0003580356133170426, 0.0007671223138459027, 0.00018964146147482097, 0.000407358689699322, 0.00021756468049716204, 0.0004472910368349403, 0.0011015235213562846, 0.002103327540680766, 0.0002450594329275191, 0.0007573319016955793, 0.0006942423060536385, 0.0044670975767076015, 0.0004665361193474382, 0.0011640875600278378, 0.00030453564249910414, 0.0002198645524913445, 0.0005127326585352421, 0.0003886536869686097, 0.00047746123163960874, 0.0011307134991511703, 0.00114061264321208, 0.0002957139222417027, 0.00029233688837848604, 0.0002483790449332446, 0.0011126368772238493, 0.0007305395556613803, 0.0006384367006830871, 0.0002472591877449304, 0.0007005413062870502, 0.0003280529344920069, 0.0009351308108307421, 0.00030763691756874323, 0.0005564721650443971, 0.0005436428473331034, 0.002124449238181114, 0.0013876402517780662, 0.0011278014862909913, 0.0013307200279086828, 0.006811607629060745, 0.0005883473786525428, 0.0008395234472118318, 0.00040442286990582943, 0.0009126484510488808, 0.00027805345598608255, 0.0012289786245673895, 0.00011850304872496054, 0.0014499146491289139, 3.040649608010426e-05, 0.0010214088251814246, 0.00012401399726513773, 7.923413068056107e-05, 0.001177689409814775, 0.0017737204907462, 0.0013574083568528295, 0.0002732863067649305, 0.0026270169764757156, 0.0005827868008054793, 0.0007979226065799594, 3.8236554246395826e-05, 0.0004815312859136611, 0.0008683389169164002, 0.0004216591769363731, 0.0013110711006447673, 0.001564576756209135, 0.0034711023326963186, 0.00023581236018799245, 0.00033452740171924233, 0.0007833177223801613, 0.001677828375250101, 8.7786465883255e-05, 0.0008104908629320562, 0.0005796908517368138, 0.0018656534375622869, 0.00022711481142323464, 0.0013452863786369562, 0.000457665795693174, 0.003084261668846011, 0.0007789137307554483, 0.0009927762439474463, 0.0009358529350720346, 0.00029408829868771136, 0.00029466141131706536, 0.00030213550780899823, 5.385645272326656e-05, 8.658578735776246e-05, 0.00040021748282015324, 0.0027831897605210543, 0.0006711622118018568, 0.0003011681546922773, 0.0012052158126607537, 0.001003929297439754, 0.0002207122597610578, 0.00038284051697701216, 0.00011524079309310764, 0.00015900401922408491, 0.00040373875526711345, 0.0018890242790803313, 0.0010477631585672498, 0.0025496201124042273, 0.0004006818344350904, 0.0013917208416387439, 0.00016579896328039467, 0.0009393751970492303, 0.00015146688383538276, 0.0015361187979578972, 0.00018241090583615005, 0.0016000717878341675, 0.00022633680782746524, 0.000594974379055202, 0.00041347838123328984, 0.00016077478358056396, 0.0017462985124439, 0.0012904191389679909, 0.0005162667948752642, 0.0005531428032554686, 0.0005894858040846884, 0.001248918822966516, 0.000735473120585084, 0.0006064371555112302, 0.0004980926751159132, 0.0021890834905207157, 0.0019041203195229173, 0.0005678281304426491, 0.001572387875057757, 0.0001388816599501297, 0.00020114176732022315, 0.0009073103428818285, 0.0002529554767534137, 0.00018869913765229285, 0.000572557735722512, 0.0005412074388004839, 0.0010186817962676287, 0.0012233799789100885, 0.00036699610063806176, 0.0010900682536885142, 0.0002757227630354464, 0.0003212019510101527, 6.672769086435437e-05, 0.00473749078810215, 0.0009012200171127915, 0.00015948069631122053, 0.0016551922308281064, 0.00032208304037339985, 0.0006162627832964063, 0.0003872207016684115, 0.00017549599579069763, 0.0009367023594677448, 0.000533076177816838, 0.00016620135284028947, 0.000382266502128914, 0.0006290525780059397, 0.0013647074811160564, 0.0005941869458183646, 0.0006928751245141029, 0.000984937883913517, 0.00030058951233513653, 0.0010903425281867385, 0.00026224812609143555, 0.0029044414404779673, 0.000422103883465752, 0.0006969518144614995, 0.0009273823234252632, 0.00012590781261678785, 0.0002700007171370089, 0.0017935691867023706, 0.0018617496825754642, 0.00011874135088874027, 0.0002086719759972766, 0.0014682118780910969, 0.001944313058629632, 0.00019112079462502152, 0.0003656548506114632, 0.0003358390531502664, 0.0009516673162579536, 7.015375740593299e-05, 0.00037581619108095765, 7.638857641723007e-05, 0.0003084882628172636, 0.0011435372289270163, 0.00028839800506830215, 0.0008714587311260402, 0.005514065735042095, 0.00012083639012416825, 0.0008807454723864794, 0.0028737958054989576, 0.0004787835932802409, 0.0002231938997283578, 0.00017620279686525464, 0.0006632984150201082, 0.0007169143063947558, 0.0005546996253542602, 0.0006138134631328285, 0.000401415309170261, 0.0032028185669332743, 0.00010093512537423521, 0.00404939753934741, 0.002283567562699318, 0.0006570145487785339, 0.0014437580248340964, 0.0024792046751827, 0.0003512055263854563, 0.0001236884418176487, 0.00024866772582754493, 0.0005539272096939385, 0.0035225662868469954, 0.0002584420726634562, 0.0003170813142787665, 0.00012844942102674395, 0.0005189812509343028, 0.0012851854553446174, 0.00038239191053435206, 0.00010589598241494969, 0.0004991113091818988, 0.0007914404268376529, 0.001309888088144362, 4.7170600737445056e-05, 0.00021891620417591184, 0.0010510226711630821, 0.0008053949568420649, 0.0025244164280593395, 0.0004633341741282493, 0.0006087800138629973, 0.000684869650285691, 0.0007670758641324937, 0.001445089583285153, 0.0005467718001455069, 0.0004497703048400581, 0.0015132678672671318, 0.0015815914375707507, 0.001541142468340695, 0.0009977848967537284, 0.00035327160730957985, 0.0014645790215581656, 0.004226319026201963, 0.006291718687862158, 0.0005738790496252477, 0.0004826440126635134, 0.0003954083367716521, 0.0010481199715286493, 0.000987690407782793, 0.001470764516852796, 0.0008907745941542089, 0.0005974242812953889, 0.0020942757837474346, 0.00041068808059208095, 0.00031088062678463757, 0.0007313088281080127, 0.0010360658634454012, 0.0013159476220607758, 0.0028523369692265987, 0.0009615098242647946, 0.00023776439775247127, 0.0006333912024274468, 0.003660836024209857, 0.00023255974519997835, 0.0003856481926050037, 0.0014776099706068635, 1.8343913325225003e-05, 0.0008691127295605838, 0.001353809260763228, 0.0015933775575831532, 0.0008729109540581703, 0.0009812636999413371, 0.0003687059215735644, 0.000559149426408112, 0.0006288986187428236, 0.0016089021228253841, 0.0010024307994171977, 0.0004889486590400338, 0.0013924852246418595, 0.00037495119613595307, 0.00012083561159670353, 0.00048328741104342043, 0.001999205444008112, 0.001100381719879806, 0.0004511312290560454, 0.0017806509276852012, 0.0012307376600801945, 0.0007421461632475257, 0.0009214412420988083, 0.0005718291504308581, 0.0008718395256437361, 0.0001677188993198797, 0.0004254850500728935, 0.00030393904307857156, 0.0009839372942224145, 0.0005411344463936985, 0.0001623073621885851, 0.0009631469729356468, 0.0005785481771454215, 0.0006817477406002581, 0.00029031335725449026, 0.0003253116155974567, 0.00486676674336195, 0.0001411983830621466, 0.0006907396018505096, 0.0002432302717352286, 0.0033710747957229614, 0.003025507554411888, 0.006320338696241379, 0.0031983633525669575, 0.00014178249693941325, 0.00025260046822950244, 0.0006504658376798034, 0.0004121551173739135, 0.0001277433620998636, 0.0004471185093279928, 0.0005182061577215791, 0.00033914516097866, 0.001486912602558732, 0.0007476702448911965, 0.0007464826921932399, 0.00023139613040257245, 0.001052875886671245, 0.00014989660121500492, 0.00033586687641218305, 0.00021505837503354996, 0.0003590542764868587, 0.00030332975438795984, 0.0006175420712679625, 0.0004133668262511492, 0.00045139240683056414, 0.0003795574593823403, 0.0014747766545042396, 0.0008622764144092798, 0.0014693146804347634, 0.00035694483085535467, 0.0008234163396991789, 0.000196062057511881, 0.004709601867944002, 0.0007340448792092502, 0.0006181610515341163, 0.0009426504839211702, 0.00018559908494353294, 0.0011456403881311417, 0.00024736684281378984, 0.0017211436061188579, 0.0022235626820474863, 0.0022997672203928232, 0.00018481134611647576, 0.0038558824453502893, 0.00016844943456817418, 0.0004213099309708923, 0.00027856839005835354, 7.735560939181596e-05, 0.00022490789706353098, 0.0008630002848803997, 0.0006947825895622373, 0.0004907394759356976, 0.0003710294549819082, 0.00024788579321466386, 0.0026529126334935427, 0.0005159697029739618, 0.00022277752577792853, 0.00047652097418904305, 0.0032760733738541603, 0.00031503569334745407, 0.00012096938007744029, 0.003765467321500182, 0.0003672575985547155, 0.0008145809406414628, 0.0018525054911151528, 0.0003223625826649368, 0.0011778894113376737, 0.007650995627045631, 0.001569146872498095, 0.0004227085446473211, 0.0020300920587033033, 0.004355296958237886, 0.0001572268083691597, 0.0007497601327486336, 0.001629968755878508, 0.0003958950110245496, 0.00046058298903517425, 0.0006986821535974741, 0.0014702841872349381, 0.00043541286140680313, 0.0002386741543887183, 4.4814405555371195e-05, 0.0005601937882602215, 0.0018384117865934968, 0.0002710701955948025, 0.00294942082837224, 0.00018115824786946177, 0.00023592558864038438, 0.0007493477314710617, 0.0004356723220553249, 0.002057254547253251, 0.0002892114862333983, 0.0002025759022217244, 0.00012743077240884304, 0.001702614245004952, 0.0034388701897114515, 0.0003654566244222224, 0.001312330598011613, 0.0007165555143728852, 0.002984747989103198, 0.0009641252108849585, 0.00070029660128057, 6.907535862410441e-05, 0.001081419992260635, 8.238050213549286e-05, 0.0002678683085832745, 0.0006592326681129634, 0.0015275354962795973, 0.00048556493129581213, 0.002623052569106221, 0.0008591245859861374, 0.0006252129678614438, 0.000776568369474262, 0.0016249001491814852, 0.00112587446346879, 0.0010132156312465668, 0.0005027420702390373, 0.0001559677766636014, 0.0003226901462767273, 0.0007751265075057745, 0.0002620088343974203, 0.00036535956314764917, 0.0004956892225891352, 0.003459102939814329, 0.0006802144926041365, 0.001945408759638667, 0.0007345516933128238, 0.0008248538360930979, 0.00020224785839673132, 0.0029415790922939777, 0.0001549989974591881, 0.00045761046931147575, 0.00015573819109704345, 0.0005311886197887361, 0.000670216977596283, 0.0007191149052232504, 0.0046122376807034016, 0.0019100852077826858, 0.0011270941467955709, 0.00024130272504407912, 9.714860789244995e-05, 0.0004038438491988927, 0.0020070888567715883, 0.0005816117045469582, 0.0006759282550774515, 0.0003914266708306968, 0.0004215818480588496, 0.00023408906417898834, 0.0009818682447075844, 0.004515600856393576, 0.0006715205381624401, 0.00041739456355571747, 0.0007277287659235299, 0.002932735253125429, 0.0003921575553249568, 0.0016149088041856885, 0.004640999250113964, 0.00035843573277816176, 0.000760322087444365, 0.0012516170972958207, 0.0016551255248486996, 0.002480450551956892, 0.0003432183002587408, 0.0025323820300400257, 0.00014636329433415085, 0.00015652598813176155, 0.002462198259308934, 0.001193677424453199, 0.0006943130283616483, 8.642671309644356e-05, 0.000221077716560103, 0.0009838542900979519, 0.0008884170674718916, 0.0012549313250929117, 0.0007133499020710588, 0.0014828774146735668, 0.0010353581747040153, 0.0021006918977946043, 0.0002832965401466936, 0.003521437756717205, 0.00015272232121787965, 0.0003233135212212801, 0.0002121959551004693, 0.0006252629682421684, 0.0010208988096565008, 0.00036323367385193706, 0.002549686934798956, 0.00010433686838950962, 0.005832988303154707, 0.0006394530646502972, 0.0021111881360411644, 0.0004881017084699124, 0.0002686427906155586, 0.001292950939387083, 0.0004473291046451777, 0.00026233927928842604, 0.0014662970788776875, 0.0035163515713065863, 0.0004726420156657696, 0.008532610721886158, 0.00028849628870375454, 8.969157352112234e-05, 0.00012449477799236774, 0.011150646954774857, 0.00034440081799402833, 0.00105337577406317, 5.0179944082628936e-05, 0.001650499296374619, 0.0003616990288719535, 0.0002256621519336477, 0.0008009555749595165, 3.757804006454535e-05, 0.0005830239970237017, 0.0007400844478979707, 0.0006294583436101675, 0.0004255675885360688, 5.9357102145440876e-05, 0.00046937449951656163, 0.0012910849181935191, 0.000387766573112458, 0.0010092444717884064, 0.00016307146870531142, 0.0005007305881008506, 0.00041086849523708224, 0.0008380307117477059, 0.0009208389674313366, 0.0014549451880156994, 0.0012754035415127873, 0.0013762254966422915, 0.004591020755469799, 0.0005886749713681638, 0.00047355578863061965, 0.0013522334629669785, 0.004433846566826105, 0.0012357296654954553, 0.00023957625671755522, 0.0016330152284353971, 0.00587003817781806, 0.0007607077131979167, 7.561399979749694e-05, 0.00011438406363595277, 0.00012345839058980346, 0.0005282507627271116, 0.0002714094298426062, 0.0006900247535668314, 0.0014044421259313822, 0.0001406745723215863, 7.469538104487583e-05, 0.0010225229198113084, 0.0015585136134177446, 0.0004882842185907066, 0.0006806583842262626, 0.00015606590022798628, 0.0003617559268604964, 0.00029203263693489134, 0.0006212731241248548, 0.0001436306774849072, 0.00017697281145956367, 0.00038235491956584156, 0.0002868402807507664, 0.0015836062375456095, 0.00016287101607304066, 0.00010585045674815774, 0.00037243831320665777, 0.0008189705549739301, 0.00035691895755007863, 0.00013216854131314903, 0.0013142949901521206, 0.0007635065703652799, 0.006577794905751944, 0.0022642062976956367, 0.0004120362864341587, 0.0009981358889490366, 0.001906455960124731, 0.0001965834671864286, 0.0011526711750775576, 0.0008026717696338892, 0.0006781838019378483, 0.0023925183340907097, 0.0003973396960645914, 0.00030711834551766515, 0.0005095379892736673, 0.0008420000085607171, 0.00043743455898948014, 0.0004784751799888909, 0.002375280950218439, 0.001334336819127202, 0.0013883811188861728, 0.00024029379710555077, 0.00037473152042366564, 0.0017336276359856129, 0.001054704887792468, 0.00030391724430955946, 0.00030287442496046424, 0.0011439025402069092, 0.0005203826003707945, 0.0006889819051139057, 0.0002504507137928158, 0.0002043567510554567, 0.0022659767419099808, 0.0006022485322318971, 0.00028167443815618753, 0.0004562997492030263, 0.0001708432100713253, 0.0010964948451146483, 0.00036292613367550075, 0.0015581926563754678, 0.00044254789827391505, 0.0004893938312307, 0.00017538473184686154, 0.0021605852525681257, 0.0005406667478382587, 0.0026829931885004044, 0.0007296815747395158, 0.0005499887047335505, 0.0007615693029947579, 0.0002795442705973983, 0.002056157449260354, 0.00042148682405240834, 0.0006430392386391759, 0.00044840777991339564, 0.0009478577994741499, 0.0025340798310935497, 0.0004546049749478698, 0.0012372899800539017, 0.0015096190618351102, 0.000691722787450999, 0.000819408567622304, 0.0001720186701277271, 0.0004252861253917217, 0.0015415034722536802, 0.00026225761394016445, 0.0005912685883231461, 4.277276821085252e-05, 0.0007511485600844026, 0.0008167460327968001, 0.0008396428893320262, 0.00037127366522327065, 0.005292107351124287, 0.0017637783894315362, 0.0015256282640621066, 0.0005928380414843559, 0.00010010218102252111, 0.0005179857835173607, 0.00047666169120930135, 0.0013737967237830162, 0.0002820157678797841, 0.002556773368269205, 0.0005526694003492594, 0.00026211439399048686, 0.0011614601826295257, 0.003512802766636014, 0.0002444765414111316, 0.000727990991435945, 0.003224900458008051, 0.00044430274283513427, 0.0022865866776555777, 0.0013300769496709108, 0.00027151615358889103, 0.0006259098299778998, 0.0007393992855213583, 0.0001535070623503998, 0.0005085131269879639, 0.0011496668448671699, 0.00012623121438082308, 0.0018672746373340487, 0.00013582535029854625, 0.002014636993408203, 6.783660501241684e-05, 0.0027935453690588474, 0.00018761184765025973, 0.0005307056708261371, 0.0018736039055511355, 0.0003943075716961175, 0.00013087734987493604, 0.0006700574886053801, 0.00018966208153869957, 0.00010127513814950362, 0.00012149212852818891, 0.001174566219560802, 0.0001358283479930833, 0.0003056014538742602, 0.0011385463876649737, 0.0011210804805159569, 0.0015669078566133976, 0.0010845707729458809, 0.0006622811197303236, 0.0011117763351649046, 0.000696377654094249, 0.00036000731051899493, 0.00022463583445642143, 0.0004945526015944779, 0.0016668102471157908, 0.00018578524759504944, 0.0003790535847656429, 0.0006378568941727281, 0.0004645212320610881, 0.0004495005705393851, 0.00023611249343957752, 0.0006433991366066039, 0.0008891509496606886, 0.0013839849270880222, 0.003188760718330741, 0.0002400897938059643, 0.0005958597757853568, 0.0018446450121700764, 0.0001515661715529859, 0.0009252040181308985, 0.001078462926670909, 0.0008640039013698697, 0.0001427822862751782, 0.002812739694491029, 0.000664778403006494, 0.0017770450795069337, 0.0006377384415827692, 0.0010212917113676667, 0.0005314747104421258, 0.0007672696956433356, 0.001083763432689011, 0.0009086807840503752], [0.000289643561700359, 0.0021163427736610174, 0.002309948205947876, 0.004826736636459827, 0.0006768006714992225, 0.0005298641626723111, 0.0001888094120658934, 0.0006094119744375348, 0.0005134791135787964, 0.00026463146787136793, 0.001440312946215272, 0.0003581095370464027, 0.0004075623583048582, 0.0005825073458254337, 0.0004457834002096206, 0.0020226016640663147, 0.0005220110761001706, 0.00029443984385579824, 0.0010962974047288299, 0.0008442638209089637, 0.009487532079219818, 0.001507120905444026, 0.0004028024850413203, 0.002434288617223501, 0.0016571251908317208, 0.0002803061215672642, 0.0009445860632695258, 0.0011979953851550817, 0.00016873962886165828, 0.00045818559010513127, 0.007437160238623619, 0.0004236277600284666, 0.000275019439868629, 0.001221987302415073, 0.00022333952074404806, 0.000595727760810405, 0.002086837776005268, 0.0013726934557780623, 2.9191496651037596e-05, 0.00023775715089868754, 0.00023509343736805022, 0.00032689954969100654, 9.344231511931866e-05, 0.0003432626836001873, 0.00024229138216469437, 0.0016611253377050161, 0.0012812202330678701, 0.00016310214414261281, 0.00018141549662686884, 0.0003287066938355565, 0.0005963458679616451, 0.000211932638194412, 0.004844042006880045, 0.00031206433777697384, 0.0020836666226387024, 0.0006600279593840241, 0.0004980514058843255, 0.0011700460454449058, 0.0019484634976834059, 0.00044385832734405994, 7.306675252038985e-05, 0.00024434007355012, 0.0002914620563387871, 0.000511723046656698, 0.0005354866734705865, 0.0013581395614892244, 0.0002827150747179985, 0.00015763210831210017, 0.000785402487963438, 0.00021484922035597265, 0.00017839568317867815, 0.0005486147711053491, 0.00018020733841694891, 0.002908722497522831, 0.000403728277888149, 0.0012881325092166662, 0.0005936576635576785, 0.00031833312823437154, 7.411558908643201e-05, 0.0013588974252343178, 0.00015194277511909604, 0.0012008181074634194, 0.00033029139740392566, 0.000492961669806391, 8.48957133712247e-05, 0.0008319307817146182, 0.00016187650908250362, 0.0007663882570341229, 0.0015741025563329458, 0.0007100424263626337, 0.0001352023537037894, 0.00017212555394507945, 0.0005540638812817633, 0.00022147499839775264, 0.0005548799526877701, 0.0027297025080770254, 0.0003396128595340997, 0.0005142552545294166, 0.0006260917289182544, 0.0003572966670617461, 0.001842764439061284, 0.00023589766351506114, 0.001978293526917696, 0.00023296089784707874, 0.00022478395840153098, 6.703527469653636e-05, 0.0008589823846705258, 0.0005469766329042614, 0.004388679750263691, 0.00101393461227417, 0.00020738931198138744, 0.0024905195459723473, 8.218987932195887e-05, 0.0007266825996339321, 0.0006093134870752692, 0.0005922440905123949, 0.0011410642182454467, 0.0008950855117291212, 0.0011031513568013906, 0.0005532345967367291, 0.001677082502283156, 0.0013300470309332013, 0.001305869547650218, 0.0009690027218312025, 0.0009571140399202704, 0.0005587285850197077, 0.0015494717517867684, 0.0009685874101705849, 0.00032097153598442674, 0.0002928273461293429, 0.0013524048263207078, 0.00021393198403529823, 0.00040875739068724215, 0.0016739541897550225, 0.0003533363342285156, 0.000683718710206449, 0.0001341495953965932, 0.00017700131866149604, 0.0005115176318213344, 0.0011781391222029924, 0.0007845299551263452, 0.00018161418847739697, 0.0008085811277851462, 0.002219981048256159, 0.0017688358202576637, 0.0014421275118365884, 0.0010957379126921296, 0.002049116650596261, 0.0006906730122864246, 0.00015183859795797616, 0.001102183829061687, 0.0014966530725359917, 0.0007983387913554907, 0.0008709441754035652, 0.00013979773211758584, 0.002443490084260702, 7.549421570729464e-05, 0.001191424555145204, 0.0020809851121157408, 0.00040395374526269734, 0.003300107317045331, 0.00020654288528021425, 0.0013223267160356045, 0.00013591180322691798, 0.00028429157100617886, 0.0004559072549454868, 0.0003146176168229431, 0.0004022647626698017, 0.0010620258981361985, 0.00040540381451137364, 0.0003842921578325331, 0.0005940319388173521, 0.0010043387301266193, 0.00020282415789552033, 0.0005432800971902907, 0.0005026541766710579, 0.001184232416562736, 0.0003560872282832861, 0.008148608729243279, 0.0009336474468000233, 0.0033217505551874638, 0.00026800070190802217, 0.000428185478085652, 0.0009245786350220442, 9.581493213772774e-05, 0.00012805036385543644, 0.0004127914144191891, 0.0002424433478154242, 0.0012217770563438535, 0.000582425796892494, 0.0016951668076217175, 0.0005256624426692724, 0.00013805722119286656, 0.0003016564587596804, 0.0004990767920389771, 0.0017939888639375567, 0.00026870949659496546, 0.00029336378793232143, 0.003152724588289857, 0.000726671889424324, 0.0006289309239946306, 0.002928839297965169, 0.0016120930667966604, 0.0006712403264828026, 0.0005730937700718641, 0.0008977947290986776, 0.0022500320337712765, 0.0012734612682834268, 0.00018189160618931055, 0.0005730821285396814, 0.0009123206837102771, 0.0023798546753823757, 0.0006655158940702677, 0.0007566629792563617, 0.0010498062474653125, 0.00022676121443510056, 0.000642993429210037, 0.0039680115878582, 0.00039829948218539357, 0.00015439162962138653, 0.0009237571503035724, 0.0007497917395085096, 0.0002408250147709623, 0.00028355250833556056, 0.0004443468642421067, 0.0008981059654615819, 0.0004894995945505798, 0.0007777723367325962, 0.0022835968993604183, 0.0024091354571282864, 0.0007129437290132046, 0.00040053651900961995, 0.0011056371731683612, 0.0018569310195744038, 0.0004933747695758939, 0.00150060944724828, 0.00014298182213678956, 0.00038159178802743554, 0.0007602678379043937, 0.0011781983776018023, 0.00012053637328790501, 0.00021452613873407245, 0.00017089438915718347, 0.0010064545786008239, 0.001605306868441403, 0.00041540845995768905, 0.0010704640299081802, 0.0003991175617557019, 0.0008243827032856643, 0.00011839636135846376, 0.0021351096220314503, 0.0006356244557537138, 0.000111175118945539, 0.0013155683409422636, 0.00010628137533785775, 0.000718749884981662, 0.0004708878987003118, 0.0016059026820585132, 0.0002477932721376419, 0.00023309387324843556, 0.0001017676477204077, 0.0005573459202423692, 0.0003729660529643297, 0.0002598868450149894, 0.00014065366121940315, 0.0011316845193505287, 0.0003063550393562764, 0.0025862876791507006, 0.0002977269177790731, 0.0038755666464567184, 0.000834968697745353, 0.0003589064290281385, 0.00054955790983513, 0.0006408998160623014, 0.004307758994400501, 0.00024334419867955148, 0.00017684178601484746, 0.00031244216370396316, 0.0006047733477316797, 0.0023347162641584873, 0.003106290241703391, 4.8078407417051494e-05, 0.002509042154997587, 0.0028429615776985884, 0.00010840412869583815, 0.0009822052670642734, 0.0023128062020987272, 0.0007179909734986722, 0.0008016939391382039, 0.0002459421521052718, 0.0011233185650780797, 0.0006294460035860538, 0.00030585244530811906, 0.0007195993675850332, 0.0013627564767375588, 0.00017065615975297987, 0.0011204940965399146, 0.0001992496836464852, 6.329006282612681e-05, 0.0010464861989021301, 0.0003501827013678849, 0.005073407664895058, 0.0027862456627190113, 0.003758884035050869, 0.001250873669050634, 0.0003387747856322676, 0.0005122448783367872, 0.0009814813965931535, 0.0013097717892378569, 0.0029798916075378656, 0.002394828712567687, 0.000516272964887321, 0.003324069082736969, 0.00016616116045042872, 0.00020698129083029926, 0.0009253628086298704, 0.0004910368006676435, 0.0005053933127783239, 0.0014188825152814388, 0.00019456855079624802, 0.001656342763453722, 0.00033391726901754737, 0.00029995114891789854, 0.0037488583475351334, 0.0023550856858491898, 0.0002365007676417008, 0.002430950291454792, 0.0005312789580784738, 0.00015145385987125337, 0.0007304543978534639, 0.00042937390389852226, 0.0019517664331942797, 0.0015207385877147317, 0.0013867305824533105, 0.0002375282347202301, 0.0014689206145703793, 0.0012034010142087936, 0.0019893741700798273, 0.003305444959551096, 0.0004641318810172379, 0.0003461356391198933, 0.0005413099424913526, 0.0003314669302199036, 0.0006357621168717742, 0.0005826005944982171, 0.001255179406143725, 0.000869744922965765, 0.000891535950358957, 0.002127012237906456, 0.0018980313325300813, 0.0009196841856464744, 0.0003708908916451037, 0.00021806597942486405, 0.00021513619867619127, 0.0004019030020572245, 0.00041708798380568624, 0.00043255285709165037, 0.00085365743143484, 0.0007429350516758859, 0.0004049959243275225, 0.0002346764231333509, 0.00046496157301589847, 0.000825805589556694, 0.0005250046378932893, 0.0015822802670300007, 0.0007729755016043782, 0.0006085043423809111, 0.0003394806117285043, 0.0006344075663946569, 0.0006892597302794456, 0.0002107520413119346, 0.00010980161459883675, 0.0004105545813217759, 0.002431372180581093, 0.0025505193043500185, 0.0009356996160931885, 0.0010671518975868821, 0.0007550146547146142, 0.00019467627862468362, 0.00025041867047548294, 0.002101221587508917, 0.00027677579782903194, 0.00038116585346870124, 0.00015666875697206706, 0.0002797541383188218, 0.0005277417949400842, 0.00033421945408917964, 0.0004087455163244158, 0.0008653558325022459, 0.00016519086784683168, 0.0012609114637598395, 0.00023337840684689581, 0.00040899700252339244, 0.0025785500183701515, 0.0011095984373241663, 0.0005014837370254099, 0.0007389927632175386, 0.0002755334717221558, 8.541705028619617e-05, 0.0020325928926467896, 0.0005901260883547366, 0.00017072196351364255, 0.0001986830320674926, 0.0007037235773168504, 0.0005182769964449108, 0.0021475243847817183, 0.0011219243751838803, 0.0005237181903794408, 0.0005659024463966489, 0.0016065590316429734, 7.274660310940817e-05, 0.0011480610119178891, 0.002901006257161498, 8.039598469622433e-05, 0.0002806184347718954, 0.00044600421097129583, 0.0009813603246584535, 8.12425569165498e-05, 0.0005004284321330488, 0.0005009312881156802, 0.0009259586222469807, 0.00027298927307128906, 0.00039757456397637725, 0.0006454320391640067, 0.000998510280624032, 0.00036390143213793635, 0.001534914248622954, 0.00013126953854225576, 0.00038355335709638894, 0.0034755875822156668, 0.0018925352487713099, 0.00016948052507359535, 0.0012033135863021016, 0.0002742014767136425, 0.0003470070951152593, 0.001909518730826676, 0.0002543095324654132, 0.00036791455931961536, 0.0009712448227219284, 0.0010643841233104467, 0.0024659119080752134, 0.00033185858046635985, 0.00045494342339225113, 0.0009393850341439247, 0.000814530358184129, 8.956543751992285e-05, 0.0004793408152181655, 0.0006614627782255411, 0.0003219881618861109, 0.0033601613249629736, 0.0006646554102189839, 0.001532345893792808, 0.0009678350179456174, 0.0005166090559214354, 0.00042533865780569613, 5.7213073887396604e-05, 0.00020725066133309156, 0.0004481759970076382, 0.0016525337705388665, 0.001545305596664548, 0.0012455214746296406, 0.001339080510661006, 0.0015715811168774962, 0.001286372891627252, 0.0007332438253797591, 0.000711871194653213, 0.0005907969898544252, 0.0007634151261299849, 0.002340919803828001, 0.00021834882500115782, 0.00024706684052944183, 0.0008723438368178904, 0.001328133512288332, 0.0004528914869297296, 0.0002667621592991054, 0.00026610586792230606, 0.00018530648958403617, 0.0002274128346471116, 0.0007429977995343506, 0.0005945051671005785, 7.777785504003987e-05, 0.0013430865947157145, 0.0013675052905455232, 0.00036825615097768605, 0.0006531457765959203, 0.00047489069402217865, 0.0010656033409759402, 0.0002149555366486311, 0.00021741993259638548, 0.0005859137745574117, 0.004186227917671204, 0.0010265415767207742, 0.001238047145307064, 0.00021160487085580826, 0.001089949393644929, 0.0003016385599039495, 0.0002502290008123964, 0.00025217249640263617, 0.0004849041870329529, 0.0004738073330372572, 0.003506910987198353, 0.00040412344969809055, 0.0002674770657904446, 0.0031926115043461323, 0.0007716428954154253, 0.0002861747343558818, 0.001578855561092496, 0.00013392000983003527, 0.0009846810717135668, 0.0022048894315958023, 0.0002451111504342407, 0.0006763834971934557, 0.0015654329909011722, 0.00038708190550096333, 0.001455102115869522, 5.0459457270335406e-05, 0.0011419851798564196, 0.00011758291657315567, 0.003660420421510935, 0.0014135250821709633, 0.0001269275089725852, 0.00044458729098550975, 0.0004672433133237064, 0.004267510958015919, 0.0010493703885003924, 0.00039100064896047115, 0.002798306755721569, 0.0013254882069304585, 0.0010175390634685755, 0.0009842428844422102, 0.00018502763123251498, 0.00047311076195910573, 0.00034183301613666117, 0.00042767313425429165, 0.00048523227451369166, 0.0014553633518517017, 0.0009432191145606339, 0.00029189090128056705, 0.0025046290829777718, 0.0012466423213481903, 0.00015239832282532007, 9.81421471806243e-05, 0.00016162048268597573, 0.002686903113499284, 0.0012110050301998854, 0.0003428220807109028, 9.31426475290209e-05, 0.0005730122793465853, 0.00017068252782337368, 0.002128318650647998, 0.0003232818271499127, 0.000349751440808177, 0.000516799686010927, 0.001268188701942563, 0.000622865161858499, 0.001261532073840499, 8.563779556425288e-05, 0.001754867727868259, 0.0002478384121786803, 0.0003959075838793069, 0.00021445027959998697, 5.19653876835946e-05, 0.00010625781578710303, 0.000708458130247891, 0.0001911398721858859, 0.0015020959544926882, 0.000783270166721195, 0.001372894155792892, 0.0009872906375676394, 0.0005056983791291714, 0.003769863862544298, 0.0008453652844764292, 0.0006701757083646953, 0.0002301960630575195, 0.0004901427309960127, 0.0005473733763210475, 0.0010900342604145408, 0.0008462833939120173, 0.0001215977972606197, 0.00018017606635112315, 0.0015734496992081404, 0.0005368958227336407, 0.000436587433796376, 0.0016847462393343449, 0.001334192929789424, 0.0010074322344735265, 0.00010026527161244303, 0.00015500589506700635, 0.0006405238527804613, 0.001050556660629809, 0.0012104992056265473, 0.00032124624704010785, 0.00163456448353827, 0.0004773360851686448, 0.0008413474424742162, 0.0005265303771011531, 0.00012163748760940507, 0.0006657288759015501, 0.0007682598661631346, 0.0004958485951647162, 0.00015836941020097584, 0.0006750961183570325, 0.0008526187739335, 0.002931431867182255, 0.00037394961691461504, 0.00022669971804134548, 0.0011642695171758533, 0.0007055990281514823, 0.0005946125602349639, 0.001826779916882515, 0.0002490428742021322, 0.00136109150480479, 0.0009368932223878801, 0.0007252997602336109, 0.0005699064349755645, 0.00020471043535508215, 3.313158231321722e-05, 0.00017866263806354254, 0.0006626099930144846, 0.002639733487740159, 0.005713019985705614, 0.00021760151139460504, 0.0015756679931655526, 0.0053009032271802425, 0.0005225325585342944, 0.002051687566563487, 0.0006097077275626361, 0.0003750960750039667, 0.00033405315480194986, 0.00024267766275443137, 0.00024198528262786567, 0.00021030983771197498, 0.0013094733003526926, 0.0010978358332067728, 0.00027812327607534826, 0.00044140618410892785, 0.0014432758325710893, 0.00032817505416460335, 0.00046744849532842636, 0.0005682978662662208, 0.001006793580017984, 0.0004529260622803122, 0.000175734210642986, 6.546943041030318e-05, 0.0001245526655111462, 0.00010609868331812322, 0.001340626971796155, 0.0009598095784895122, 0.0003551992413122207, 0.0005411960300989449, 0.0002298792969668284, 0.0008179764845408499, 0.0004925340763293207, 0.0013158940710127354, 0.0002241436595795676, 0.0007114132167771459, 0.000520246394444257, 0.00039088886114768684, 0.00011545793677214533, 0.0010414740536361933, 0.0003971358819399029, 0.0012813570210710168, 0.00019689257896970958, 0.00012885555042885244, 4.203700882499106e-05, 0.0009751894394867122, 0.00026521366089582443, 0.0011804720852524042, 0.0001352958643110469, 0.0011520803673192859, 0.00045349192805588245, 0.00023985275765880942, 0.00025514879962429404, 0.0014152206713333726, 0.00035822924110107124, 7.80162081355229e-05, 0.0010540791554376483, 0.000980910612270236, 0.0030377672519534826, 0.001146762166172266, 0.005290876608341932, 0.0010616485960781574, 0.00015249931311700493, 0.00022965438256505877, 0.000712796114385128, 0.00012587799574248493, 0.0009882713202387094, 0.0007473949226550758, 0.001104549621231854, 0.0009183993097394705, 0.0008645849884487689, 0.0013082936638966203, 0.0006062816828489304, 0.0005717941676266491, 0.00013527979899663478, 0.000392022542655468, 0.0017664392944425344, 0.00023755688744131476, 0.00011023972183465958, 0.0007161972462199628, 0.00022967175755184144, 0.003340580966323614, 0.0008315289742313325, 0.0002585591282695532, 0.0020681535825133324, 0.0022571375593543053, 0.0016304313903674483, 0.0022142569068819284, 0.0005835042102262378, 0.00025963986990973353, 0.0014228969812393188, 0.000874951365403831, 0.00033825013088062406, 0.0005218561855144799, 0.0012709355214610696, 0.0003951813851017505, 0.00035420976928435266, 0.00414586765691638, 0.0003946016659028828, 0.00021512362582143396, 0.0011592608643695712, 0.0002220691676484421, 0.003126641269773245, 0.0011315835872665048, 0.0010612327605485916, 0.0017858082428574562, 0.0021018562838435173, 0.00047944646212272346, 0.0002007910079555586, 0.00015105021884664893, 0.0017201530281454325, 0.004342565778642893, 0.0001126314964494668, 0.000786066404543817, 0.0003386953903827816, 0.0019871240947395563, 0.0004465498495846987, 0.00023965651053003967, 0.00047224535956047475, 0.00018224555242341012, 0.0011484192218631506, 0.002386963227763772, 0.0014754756120964885, 0.0002844966948032379, 0.012024912051856518, 0.0021633601281791925, 0.0025692929048091173, 0.00016371719539165497, 0.0003272402682341635, 0.0028058358002454042, 0.005075945984572172, 0.0007166555151343346, 0.00031791592482477427, 0.0002754382439889014, 0.001355410786345601, 0.0014338871696963906, 0.0019296016544103622, 7.1323687734548e-05, 0.0023309593088924885, 0.00020175160898361355, 0.0014056925429031253, 0.0008393919561058283, 0.000731895212084055, 0.0006132418056949973, 0.0005309807602316141, 0.0012818361865356565, 0.0018733046017587185, 0.004835039842873812, 0.0005107966135255992, 0.0005252293776720762, 0.001387854921631515, 0.0004002046480309218, 0.00020181482250336558, 0.00031187720014713705, 0.0006494199042208493, 0.0013655235525220633, 0.0021782023832201958, 7.375661516562104e-05, 0.0014691367978230119, 0.00011493088095448911, 0.00020256961579434574, 0.0009459208231419325, 0.00041292267269454896, 0.0011541866697371006, 7.96429012552835e-05, 0.0006136983283795416, 0.006955221761018038, 0.00010837256559170783, 0.002560531720519066, 0.000287188304355368, 0.0013692359207198024, 0.00026324126520194113, 1.4113719771557953e-05, 0.0002648743102326989, 0.0040709227323532104, 0.000238520820857957, 0.0030805787537246943, 0.00017678509175311774, 0.0007731210207566619, 0.001627790043130517, 5.004697959520854e-05, 0.0006523516494780779, 0.0006314639467746019, 0.000799314642790705, 0.00037291995249688625, 0.0003630516293924302, 0.00102085096295923, 0.0003118338936474174, 0.0036796152126044035, 0.000740830902941525, 0.0003031652304343879, 0.0007897519390098751, 0.000465725373942405, 0.0008305882220156491, 0.00011822631495306268, 0.0004402193007990718, 0.0023556870874017477, 0.00011679593444569036, 0.0007235926459543407, 0.0002045330184046179, 0.005168247036635876, 0.00022171465388964862, 0.0007449872209690511, 0.000815517152659595, 0.0006686270935460925, 0.0010940906358882785, 0.0006030933000147343, 0.000888465903699398, 0.0007246442255564034, 0.00036950301728211343, 0.00015526301285717636, 0.000240919049247168, 0.001338100410066545, 0.0018969509983435273, 0.0017508754972368479, 0.0010247535537928343, 0.00024350332387257367, 0.0006971629918552935, 0.0005928972968831658, 8.846287528285757e-05, 0.0007753906538709998, 0.0024534049443900585, 0.0003743530542124063, 0.0005773809971287847, 0.0024039088748395443, 0.0018397703533992171, 0.0017292803386226296, 0.0007286650943569839, 0.000704703270457685, 0.0006274726474657655, 0.000418991461629048, 0.0016366876661777496, 0.003933286760002375, 0.0011150446953251958, 0.00035780202597379684, 0.00024028625921346247, 0.001349457073956728, 0.0006238322239369154, 0.0003947085642721504, 0.00014236241986509413, 0.00018794114293996245, 0.00020670212688855827, 0.0004648235335480422, 0.0002112466172548011, 0.001859129755757749, 0.0002410376473562792, 0.00028878048760816455, 0.0006430292269214988, 0.0009652277221903205, 0.0005944955046288669, 0.00013365951599553227, 0.0018935258267447352, 0.005038961302489042, 0.0015233170706778765, 0.0001816016447264701, 0.00118246057536453, 0.00249308068305254, 2.947833127109334e-05, 0.00021394032228272408, 0.0012083842884749174, 0.0002652019902598113, 0.0001678413391346112, 0.0015315903583541512, 0.0006350827170535922, 0.0005669246311299503, 0.0035542957484722137, 0.0007063827360980213, 0.0033238178584724665, 0.0006202199729159474, 0.00031318128458224237, 0.00022480703773908317, 0.0001900372008094564, 0.0024173210840672255, 0.00062464305665344, 0.0010788446525111794, 0.0007748657371848822, 0.0010242542484775186, 0.0001000849370029755, 0.003283904632553458, 0.0008760169730521739, 0.003900024341419339, 0.0009978106245398521, 0.00011128683399874717, 0.0004494565073400736, 0.00021332020696718246, 0.00038820449844934046, 0.0014278547605499625, 0.0007262184517458081, 0.00025431677931919694, 0.00013225679867900908, 0.0012040537549182773, 0.0008750879787839949, 0.0025486410595476627, 0.0014098662650212646, 0.0004575104685500264, 0.00023560212866868824, 0.0011631045490503311, 0.0006628268747590482, 0.0013866379158571362, 0.002096576848998666, 0.00038978742668405175, 0.0005473911878652871, 0.00035724113695323467, 0.0007769075455144048, 0.0004620067193172872, 0.003941887523978949, 9.2241374659352e-05, 0.001290694228373468, 0.006128869019448757, 0.0005075678345747292, 0.0002547826443333179, 0.0016211303882300854, 0.0003608116530813277, 0.0005021874094381928, 0.001239717355929315, 0.00054104597074911, 0.0004484787059482187, 0.0008414615294896066, 0.0005042356206104159, 0.0005380560760386288, 0.0008916347869671881, 0.0013072603615000844, 0.0014324435032904148, 0.007168714422732592, 0.00021995116549078375, 0.00020602942095138133, 0.000574047677218914, 0.0012694498291239142, 0.00206955848261714, 0.0005123541341163218, 0.000714642577804625, 0.0011651474051177502, 0.0015491374069824815, 0.00022017673472873867, 0.0053624846041202545, 0.004823627416044474, 0.0016001529293134809, 0.001030103536322713, 0.0018933223327621818, 0.001161864958703518, 0.001518396195024252, 0.0001290666259592399, 0.00511123426258564, 0.0009391505154781044, 0.006156400311738253, 0.0003240033402107656, 0.0025593899190425873, 0.0027340096421539783, 0.000454354943940416, 0.0007548120338469744, 0.0014046697178855538, 0.0007942059310153127, 0.001047745463438332, 0.0005892612971365452, 0.0012500877492129803, 0.00013694468361791223, 0.001469541690312326, 0.0009103428456000984, 0.00022252099006436765, 0.0002202308241976425, 0.0008019184460863471, 0.0005314346635714173, 0.0007923938683234155, 0.0015053115785121918, 0.000610080489423126, 0.0003111842379439622, 0.0017329121474176645, 0.00099133828189224, 0.0016778579447418451, 0.0004640274273697287, 0.0003908830985892564, 0.0008195008267648518, 0.0006232413579709828, 0.0005494392826221883, 8.588878699811175e-05, 0.0013326536864042282, 0.0037215629126876593, 0.0011744775110855699, 0.0008176389965228736, 0.0005611106171272695]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.9164896011352539, -1.8176401853561401, 0.7178133726119995, -1.5418223142623901, -1.8683485984802246, 1.1538199186325073, -1.8205891847610474, -1.7974598407745361, 1.9321192502975464, -2.4773106575012207, 2.074714183807373, 2.0966546535491943, 2.013404130935669, -2.292635440826416, 1.620017409324646, 2.105530261993408, 1.571191430091858, -1.2368251085281372, 1.975806713104248, -2.9468984603881836, -0.26776543259620667, -1.1092243194580078, 2.91276216506958, -0.5884366035461426, -1.322266936302185, 1.129616141319275, -2.862828016281128, -1.3041226863861084, -2.991215467453003, 0.4253300130367279, -2.4681236743927, -2.1329543590545654, -0.7066137790679932, -1.5136384963989258, 1.514733910560608, 3.427990674972534, 1.9022871255874634, 2.2660415172576904, -3.472092628479004, -0.6205256581306458, 1.7743600606918335, 1.2047674655914307, 0.9071511030197144, -3.1241981983184814, 1.7611268758773804, 4.557584762573242, -1.2402094602584839, -1.7774697542190552, 0.6792964339256287, 0.3556983768939972, 0.18573126196861267, 1.7166475057601929, -2.3392162322998047, 1.9699351787567139, -2.414778232574463, -3.417435646057129, 2.6034440994262695, 2.119994878768921, 1.0215272903442383, 1.4803920984268188, 2.594318389892578, 1.6265122890472412, -0.8541368246078491, 1.8023009300231934, -1.9064834117889404, 0.784261167049408, -3.6028501987457275, 0.4848494827747345, 0.5158585906028748, -0.2537170946598053, -2.693528652191162, -3.586068868637085, 1.5616050958633423, -1.276537537574768, 1.6897473335266113, -0.3596786558628082, -0.5008476972579956, -4.320972442626953, 1.472752571105957, 0.20198804140090942, -1.1408103704452515, -1.692830204963684, -3.457411050796509, -2.1516356468200684, 2.5905094146728516, 0.620837926864624, 2.9216887950897217, 1.4722602367401123, -0.9861342310905457, 0.7658509016036987, -3.619851589202881, 2.344111442565918, -0.3998197615146637, -0.011671938002109528, 0.426645427942276, 0.2968735694885254, 0.31453654170036316, -0.13018567860126495, -1.4515715837478638, -0.7644966244697571, -3.3717739582061768, 0.5838278532028198, -3.1866514682769775, 1.25108003616333, -1.267500400543213, -1.1003613471984863, -4.087126731872559, -0.03769225999712944, -1.854673147201538, 0.5041927099227905, -0.08027677237987518, -0.33943384885787964, 1.192795753479004, -1.057255744934082, -1.0014541149139404, -1.3996506929397583, -0.5946242213249207, -0.8862351179122925, 1.2543354034423828, 1.0883243083953857, -0.8489347100257874, -1.5246227979660034, -0.9378882646560669, 1.0332049131393433, -0.6285950541496277, -0.6224039793014526, -2.577343702316284, 2.113102436065674, -1.026087760925293, -0.05577174946665764, 2.0185697078704834, -3.6961476802825928, -0.6933322548866272, 1.7714530229568481, -2.5718822479248047, -1.6919329166412354, -0.6162810325622559, -0.39875084161758423, -0.7211874127388, -0.1991708129644394, -2.1800923347473145, -1.7399344444274902, -0.7477515935897827, -0.9712377786636353, 2.4114365577697754, -0.5929535031318665, 0.7662408947944641, 2.7102103233337402, -0.20336635410785675, 0.0014470366295427084, -1.1131592988967896, 0.02763167954981327, -1.7149616479873657, 0.05675836279988289, 1.8903000354766846, -1.9253296852111816, 0.9846677780151367, 1.6319419145584106, -1.8742395639419556, 2.573552131652832, -0.07247364521026611, 1.2497185468673706, 1.2329961061477661, 1.22136652469635, 0.42544978857040405, 0.7399805188179016, 0.7992980480194092, -1.7432537078857422, 2.5188210010528564, -1.087002158164978, 0.09121228754520416, -1.4452457427978516, 3.1088643074035645, 2.460818290710449, 0.6371802687644958, 2.042811155319214, 4.254215240478516, 0.3862398564815521, 2.5103423595428467, 0.8875643610954285, -3.0499038696289062, -0.6787194609642029, 3.302586078643799, 0.07862688601016998, 2.32895565032959, -0.5673652291297913, -0.616479218006134, 1.9469578266143799, 2.528353691101074, 0.08026178181171417, -2.498307943344116, -2.653480291366577, 2.0611116886138916, 0.415582537651062, -0.843921959400177, 1.1947523355484009, 1.2661056518554688, 2.2831528186798096, 0.4092273712158203, 3.077583074569702, 3.635043144226074, -0.300373375415802, -0.49125805497169495, 2.804703950881958, 2.1060192584991455, 0.056232500821352005, -4.833984851837158, 0.21182896196842194, 0.33287766575813293, 1.076042890548706, -0.9720305800437927, -0.4131700396537781, 1.194416880607605, -1.3576909303665161, -1.2288455963134766, 1.073353886604309, -0.6842003464698792, -0.764801025390625, -0.0740245059132576, -1.3546096086502075, -0.9492636919021606, -3.392824649810791, -1.9712270498275757, 1.8614453077316284, -2.072723627090454, 0.5575513243675232, -0.6639094352722168, -1.936145544052124, -0.04209012910723686, -1.0791116952896118, 0.20773054659366608, 0.7307026386260986, -0.31125447154045105, -0.1396273523569107, -0.6017388701438904, -1.767842173576355, -5.208847522735596, -1.4187530279159546, -1.9930022954940796, 1.8930437564849854, -1.1057310104370117, -1.6793497800827026, 0.7699043154716492, 2.1783838272094727, -3.5657176971435547, 1.7709561586380005, 2.8089137077331543, -1.819530963897705, -2.3695547580718994, -1.1045784950256348, 3.784585475921631, 3.3893344402313232, -0.5181686282157898, -1.6721210479736328, 1.8759974241256714, 0.3246387541294098, 1.3852239847183228, -0.3726674020290375, -2.690995931625366, 0.905719518661499, -3.9425604343414307, 0.32119908928871155, 1.4036481380462646, 1.8198976516723633, 1.03671395778656, -0.5423794984817505, 0.6303085684776306, -2.318418025970459, 2.8210859298706055, 0.35028859972953796, -1.196008563041687, 1.9339653253555298, -2.186877727508545, 0.05884730815887451, -1.7093127965927124, -3.9492955207824707, -0.1355077475309372, 0.5383511781692505, -1.8588775396347046, 1.9687989950180054, 0.05451101064682007, 1.585217833518982, -0.9258816242218018, 2.3775887489318848, -1.028468132019043, 0.03376653417944908, 0.6426129937171936, -1.0935629606246948, -1.0687814950942993, 1.2355854511260986, -1.2157518863677979, 2.208054304122925, 0.2115502506494522, -1.9176710844039917, 1.1338107585906982, 2.165595769882202, 1.2006621360778809, 1.5736804008483887, 0.7318323850631714, 0.6386168003082275, 3.0160059928894043, -1.338221549987793, -1.6553529500961304, 0.4164324104785919, -1.6292868852615356, -0.7204458117485046, -1.6408761739730835, -0.6282092332839966, -1.093501091003418, 0.2940542697906494, -0.31274116039276123, 1.025926113128662, -1.1480990648269653, -0.219048410654068, 3.6675124168395996, 1.7537200450897217, -2.3715269565582275, -3.7920634746551514, -1.5037866830825806, 1.555674433708191, -0.8820852041244507, -2.080030918121338, 0.8454229235649109, -1.582067608833313, 1.2586230039596558, -2.1039326190948486, 0.12871550023555756, 1.988082766532898, 4.296686172485352, -0.4874490797519684, 0.2014458030462265, 4.169161319732666, 2.0309364795684814, 2.153106689453125, -1.3795435428619385, 1.1758787631988525, 4.8039774894714355, -0.37298959493637085, -1.105708122253418, 0.3508199155330658, -0.23707282543182373, 1.179848074913025, 0.14354050159454346, 2.9935801029205322, 1.7686082124710083, 3.397118091583252, 0.7823817729949951, 1.8947834968566895, -1.4303454160690308, 1.8659347295761108, 3.2521963119506836, 0.2712748646736145, -1.4455435276031494, -2.4170422554016113, 0.7758896350860596, 0.8663745522499084, 1.7611855268478394, 0.0353044793009758, -2.598864793777466, -0.46517857909202576, -4.009631156921387, 2.7112371921539307, 0.08086078613996506, -0.2907460629940033, 1.5122625827789307, -1.585057258605957, -0.7358472943305969, 1.637913703918457, -1.6502636671066284, 0.9043941497802734, -0.27151545882225037, 1.8297394514083862, 4.270678997039795, 0.3121316432952881, 3.0858311653137207, -0.022592242807149887, 0.5515339374542236, 1.0431797504425049, -2.0852699279785156, 2.130496025085449, -0.9513320326805115, 1.0622786283493042, -2.9961843490600586, 0.15268748998641968, 1.037622332572937, 0.5826871395111084, 4.09599494934082, 3.0221827030181885, 1.4954363107681274, -1.0099302530288696, -0.6552444100379944, -0.6876901984214783, -2.4266295433044434, -0.41052454710006714, 2.537212610244751, 2.062835454940796, 2.3691511154174805, -0.8640080690383911, 3.541217565536499, 3.0374932289123535, 0.1048588827252388, 0.5455663204193115, 1.1309380531311035, 1.2880573272705078, 1.1920721530914307, 0.1772104650735855, -1.4261258840560913, 0.022023387253284454, 0.5501461029052734, -3.1921019554138184, 3.908764362335205, 0.5731385350227356, 1.322485327720642, -0.72562575340271, -0.12471593916416168, -1.9220868349075317, -1.083959937095642, -1.9521387815475464, 1.888479471206665, -4.361286163330078, 0.409373939037323, 1.6415373086929321, 1.746610164642334, -1.8457975387573242, -1.7177155017852783, 0.6708227396011353, -0.5504005551338196, -1.2819002866744995, 1.1245864629745483, 0.7003337740898132, -0.8668951392173767, -2.271024227142334, 3.0112626552581787, 1.6733475923538208, 0.22489511966705322, 0.0058305165730416775, -1.9550144672393799, -1.4139288663864136, 0.43317604064941406, -1.8790853023529053, 2.499127149581909, 3.48287034034729, 3.9123263359069824, 0.26071447134017944, -0.35362324118614197, -0.17929662764072418, -1.2887681722640991, -1.5672508478164673, 0.7302479147911072, 0.9275543093681335, -2.07281756401062, 0.9847521781921387, -0.47550109028816223, 1.7089216709136963, -1.4397950172424316, 1.4259780645370483, 0.04455270618200302, -2.910432815551758, -0.8065290451049805, -3.509901762008667, 0.818253219127655, -0.30819398164749146, -2.8756728172302246, -1.8127822875976562, -3.8176419734954834, 0.8977158665657043, 2.3341565132141113, 1.4955474138259888, 1.7311183214187622, 1.7964282035827637, 0.13514584302902222, 0.364632248878479, -2.6661863327026367, 2.8237051963806152, 1.9937947988510132, -0.2859146296977997, -1.017004132270813, 1.3586736917495728, -1.6498550176620483, 2.222120523452759, 0.2964187562465668, -2.189995288848877, 2.72847056388855, 0.8379773497581482, 1.1687073707580566, 3.2152981758117676, -0.6942080855369568, -2.1041693687438965, 2.3084776401519775, -2.97334623336792, 1.8859933614730835, -1.682663917541504, 2.659885883331299, -1.985985279083252, -3.0475738048553467, 2.251800537109375, 0.39531978964805603, 0.8157453536987305, -1.0831266641616821, -4.332518577575684, 1.4976015090942383, -1.7793058156967163, -0.05995733290910721, -3.8051939010620117, -0.3829626739025116, -1.4793164730072021, 1.312753438949585, -0.9511283040046692, 2.805022716522217, -3.3869926929473877, 2.95939040184021, 2.484644651412964, 3.260572671890259, 2.3526151180267334, -0.22073881328105927, 0.27567124366760254, -1.2078031301498413, 0.7955938577651978, -1.1163874864578247, -0.872367799282074, 0.1040923222899437, -3.1356751918792725, 3.7576472759246826, 0.3013884723186493, 3.214754581451416, -1.8113089799880981, 2.58074688911438, 2.602905035018921, -3.248833417892456, 0.15211732685565948, 0.9733196496963501, 0.8875871896743774, 1.923350214958191, 1.584067463874817, -0.4490215480327606, 2.519390344619751, 1.2636367082595825, 0.851564347743988, 2.971575975418091, -2.2229678630828857, 0.9829357266426086, -0.30621641874313354, 1.4935648441314697, -0.22557702660560608, 3.3338894844055176, 0.48726022243499756, 1.5993667840957642, 0.35834169387817383, -1.0320179462432861, -0.12324673682451248, -0.7896628379821777, -0.253328800201416, 0.4912806749343872, 0.0661836639046669, 3.4363820552825928, -3.180513620376587, 0.8215989470481873, -0.20787809789180756, 0.026013927534222603, 3.956681251525879, 1.1088560819625854, -0.2978897988796234, -0.11572224646806717, -0.910341739654541, -3.0677614212036133, 3.8066933155059814, 3.062437057495117, -1.257619023323059, -2.45269513130188, -1.3920338153839111, -0.47004181146621704, 2.7991461753845215, 2.202463150024414, 0.7200743556022644, 0.5923718810081482, -1.97691810131073, -0.24274206161499023, 1.276999831199646, -0.8481586575508118, -0.34072691202163696, 2.784161329269409, 1.6729931831359863, 0.4535963833332062, -2.6622610092163086, 0.38927677273750305, -3.8728866577148438, -3.004417657852173, -1.6957385540008545, 1.4299283027648926, -2.146446943283081, 0.15856507420539856, 1.9429856538772583, -1.5456500053405762, 1.5477240085601807, -1.0369552373886108, -1.7932827472686768, 0.6402965188026428, -1.281415581703186, 0.08933856338262558, 4.625874042510986, -2.678396701812744, 4.554299831390381, -2.8810646533966064, -5.734477519989014, 0.27300626039505005, 0.19131343066692352, -0.38771551847457886, 2.3655290603637695, 0.812213122844696, -1.8395376205444336, -0.04417843371629715, 2.4344325065612793, 1.4301390647888184, 0.07338219881057739, 1.567941427230835, 2.2809226512908936, -2.5951714515686035, -3.403599739074707, 0.8391072154045105, 2.95234751701355, 1.5345910787582397, 0.9141148924827576, -1.1149481534957886, -3.5865447521209717, 1.3070313930511475, -0.7845117449760437, -0.331728994846344, -1.2002307176589966, -4.285547256469727, -0.7009580135345459, 2.6005380153656006, -0.008223654702305794, 0.4301665723323822, 0.8357402086257935, -0.4539102613925934, -1.8873162269592285, 1.9999268054962158, 0.6129329800605774, 2.4517898559570312, 0.2910497784614563, -0.5613574981689453, 1.210005521774292, -4.123984336853027, -1.1278587579727173, 2.002091884613037, 0.7836944460868835, 0.24410030245780945, -4.013429641723633, -0.19398969411849976, -0.5011810660362244, -0.2171507030725479, 1.3300014734268188, 0.37189367413520813, 2.0280873775482178, -3.1213815212249756, -0.5391519665718079, 1.546836256980896, -1.393415927886963, 0.23366820812225342, 0.805469810962677, 3.9309229850769043, 0.7713987827301025, -2.443544626235962, -2.394904375076294, 0.09925039857625961, 1.6095607280731201, -0.4941161572933197, 2.493598222732544, 0.3135007619857788, -1.6522517204284668, 3.429959535598755, 0.2210579812526703, 1.8864951133728027, -0.948710560798645, 0.10287211090326309, 0.8707460761070251, -0.6940737366676331, -0.5480725169181824, 1.447083592414856, -0.8533015251159668, -1.380366325378418, -0.7915210723876953, 1.4010306596755981, 2.303723096847534, -0.29820746183395386, -4.421719551086426, -1.346867322921753, -0.07533106952905655, 0.9809383749961853, 2.8292534351348877, -3.484832763671875, -3.241978645324707, 0.12775267660617828, 2.1093451976776123, 2.3333580493927, -5.59750509262085, 6.167700290679932, -0.420869916677475, 0.24220526218414307, 2.2636070251464844, 3.13771653175354, -0.41220471262931824, -0.8214097619056702, 2.1157734394073486, 2.9135944843292236, 0.06981753557920456, 2.632241725921631, -0.13684417307376862, -1.504813551902771, -1.9111082553863525, -1.1079511642456055, -0.8410605192184448, 2.906252861022949, 1.0117849111557007, 0.2592909634113312, 1.3742873668670654, 5.326190948486328, -0.6114233732223511, 2.533663034439087, -3.539940118789673, -2.096212148666382, -0.6915865540504456, -1.0640865564346313, -0.8299697637557983, 0.24568407237529755, -1.816037654876709, -0.9905052781105042, -2.3167402744293213, 0.2971823811531067, 2.5012166500091553, 0.04849736765027046, 1.655293583869934, -1.7514632940292358, -0.7633121013641357, 0.719921350479126, -2.8493025302886963, -0.8886589407920837, -2.5076451301574707, -0.6206289529800415, 1.0250967741012573, -0.17768830060958862, -1.6695804595947266, 2.805025339126587, 0.2680715322494507, -0.4696641266345978, 0.24928100407123566, -1.4824470281600952, -0.1546388864517212, 1.1858652830123901, -3.1559736728668213, 0.32728832960128784, -0.9837035536766052, -1.293702244758606, 0.05576455593109131, 1.4194153547286987, 4.600143909454346, 0.651850163936615, 1.8611208200454712, -0.6257635354995728, 5.03689432144165, 3.239612340927124, -0.6115186214447021, 0.334889680147171, 1.1704565286636353, -1.3163425922393799, 1.068928837776184, 0.6585873365402222, 0.3050599694252014, 0.9536047577857971, 0.45668235421180725, -2.8145737648010254, 2.2133214473724365, -3.0935490131378174, 1.5910866260528564, -0.24686002731323242, 3.0529367923736572, 0.37439990043640137, -0.2203666865825653, 1.35486900806427, -0.8638914227485657, 1.3135828971862793, -1.890710473060608, -1.336375117301941, -0.15667924284934998, -1.251930832862854, -0.5839066505432129, -0.47836124897003174, -2.183807134628296, -2.3885931968688965, 1.2644225358963013, 0.8340526819229126, 1.7572429180145264, -0.26992133259773254, 2.1107358932495117, -1.2091537714004517, 1.2893333435058594, 3.4363744258880615, 1.046756386756897, -0.7692663669586182, -0.7607829570770264, 2.3792366981506348, 1.6873507499694824, -0.2143433839082718, -0.41507604718208313, -0.024223700165748596, -0.6159246563911438, -1.1675465106964111, 0.05283629149198532, 2.399386167526245, 1.340059518814087, -1.7603977918624878, -2.1089229583740234, 3.5133490562438965, 0.09048411250114441, -0.36649858951568604, 3.3503777980804443, -3.393571138381958, 1.4746954441070557, -1.726851463317871, 2.0837955474853516, -1.3053617477416992, 0.7614758610725403, 2.889461040496826, -0.9860802888870239, -2.791619062423706, 0.7359123229980469, 2.0827484130859375, 2.214799404144287, -1.0546445846557617, -1.7949923276901245, 1.1210988759994507, -1.6624900102615356, -2.2748396396636963, 2.675916910171509, 0.505626916885376, -1.4741287231445312, 4.484095096588135, 0.23800213634967804, 3.5717051029205322, 3.4073972702026367, 1.5259841680526733, 0.8469837307929993, -2.409407138824463, -1.5015184879302979, -3.1980652809143066, 0.2363974004983902, -1.7500214576721191, 1.0301622152328491, -0.7487051486968994, 0.7720778584480286, -0.674222469329834, 1.6479445695877075, -1.7394157648086548, -2.1437020301818848, 2.2215957641601562, -0.2670174539089203, 2.0748353004455566, -0.6617817282676697, 0.4898104667663574, 2.5608701705932617, -1.6995056867599487, -1.2993276119232178, -0.08527516573667526, 1.349979043006897, 2.031958818435669, 0.8061535358428955, 3.1748766899108887, -2.040217638015747, -0.4527778625488281, 1.7196776866912842, 2.984607696533203, 2.6305572986602783, 1.8155122995376587, 0.22628341615200043, 1.9376442432403564, 0.23204727470874786, 0.5573301315307617, 1.8003638982772827, -0.3177299499511719, -0.7903414368629456, 1.1502488851547241, -1.4860351085662842, 0.9933704733848572, 2.1774988174438477, -3.1796185970306396, 3.6130828857421875, -0.5909611582756042, 0.2686828672885895, 0.6229304075241089, -0.12126973271369934, -1.3236173391342163, -1.817318320274353, 1.0759367942810059, -0.07059349119663239, -0.16327334940433502, -0.12754687666893005, 0.2773814797401428, -1.6035422086715698, -2.393598794937134, -2.0419199466705322, 2.0115790367126465, -1.1731427907943726, -6.061432838439941, -0.26342469453811646, 2.4149630069732666, -1.2083762884140015, 2.499452590942383, 4.4846062660217285, -1.4108552932739258, -0.9265257120132446, -0.4488878548145294, -3.253032684326172, -1.1126095056533813, -0.39231452345848083, 0.8883585333824158, 0.6073391437530518, -1.3425793647766113, 0.5403509736061096, 0.47570812702178955, 0.8547991514205933, -1.4211052656173706, 2.4934706687927246, 0.05456826090812683, 1.467290997505188, 1.4009604454040527, -1.779773473739624, 1.9860167503356934, -2.0251030921936035, 1.4923192262649536, 0.5586156249046326, 1.9042977094650269, 0.2960917055606842, -1.5156376361846924, 2.7741878032684326, -2.7926981449127197, 1.3353681564331055, 0.8918058276176453, 4.195559978485107, -1.4070212841033936, 1.3068122863769531, -0.1495271921157837, 0.3503299057483673, 0.22580048441886902, -1.8852503299713135, 0.45282062888145447, -1.349928617477417, 0.4687412679195404, -5.672543048858643, 1.0135552883148193, 1.3498852252960205, 1.4894160032272339, -1.6540484428405762, -1.606695294380188, 4.693490028381348, 5.137939929962158, -1.745956301689148, 1.5199406147003174, 0.30985012650489807, 2.1390199661254883, -0.19992853701114655, -1.5248572826385498, 3.445472478866577, -1.274782419204712, -1.5259623527526855, -0.8240014910697937, 1.2038135528564453, -0.29635071754455566, -1.9581283330917358, 2.614945650100708, 2.305124282836914, 0.8447715640068054, -2.4874160289764404, 3.363957166671753, -1.6480108499526978, -0.27124541997909546, -2.608128786087036, -2.4356210231781006, 0.6929451823234558, 0.3115939199924469, 2.2354073524475098, -1.1098449230194092, -5.361657619476318, 3.7504677772521973, 1.1606121063232422, -2.705984115600586, -0.541584849357605, 0.436453253030777, -1.0902131795883179, -0.8362806439399719, -1.4882925748825073, -0.6107083559036255, -1.512535810470581, 1.2917402982711792, 2.996400833129883, -0.08872618526220322, -2.3363170623779297, -2.1656854152679443, -1.803416132926941, 3.5027031898498535, 1.8415908813476562, 2.1093716621398926, 0.8200433850288391, -0.6118625998497009, 0.9083874821662903, -2.8362812995910645, -0.07174577564001083, 0.6005957126617432, -1.3685938119888306, -1.8997151851654053, -0.46287405490875244, 1.2013880014419556, 0.3404143452644348, 2.026221752166748, -2.6879920959472656, -1.437059760093689, -0.27670806646347046, 2.6402018070220947, -2.285978317260742, -2.5295631885528564, -1.0564186573028564, 2.366480588912964, 1.844861388206482, -1.1303379535675049, 1.6332935094833374, 0.8989474177360535, -1.2164028882980347, 0.05492692440748215, 1.2399792671203613, 1.7134647369384766, 0.7310437560081482, -2.133531093597412, 0.7352532148361206, 1.4942876100540161, -1.4380793571472168, 0.9647623300552368, -0.5222168564796448, 5.3719682693481445, 3.8261990547180176, -3.660959482192993, 1.544547438621521, 1.3027513027191162, 0.30390027165412903, -3.7313737869262695, -2.181959867477417, 0.43933340907096863, -2.574100971221924, -3.77543044090271, 1.1735436916351318, 0.3020828366279602, -1.4831148386001587, 1.904601812362671, -0.5531185865402222, -1.7898712158203125, 2.0322439670562744, -2.2344672679901123, 2.510626792907715, -1.4041482210159302, 3.9452943801879883, 0.8969245553016663, 3.433852195739746, 1.6571831703186035, -4.041985034942627, -0.6531580686569214, 2.737837553024292, -0.3573976159095764, -1.9578948020935059, -4.368055820465088, -0.763356626033783, 0.9347107410430908, -2.1433699131011963, -1.3819400072097778, 2.9793739318847656, -3.2313661575317383, -1.6755832433700562, 0.9447400569915771, 0.7361454367637634, 0.7254566550254822, -0.05215417593717575, 2.933734178543091, 0.6573260426521301, 0.9469982981681824, 0.4426012635231018, 1.5828168392181396, 0.5096646547317505, 2.04899263381958, 1.0866299867630005, -0.3606111407279968, 2.1864044666290283, -2.8998918533325195, 0.9655370116233826, 1.7966443300247192, -0.9088612198829651, 3.549004554748535, -2.207960367202759, 0.7209519743919373, -0.673321008682251, -0.7664107084274292, -0.275541752576828, -1.6136282682418823, 2.8316445350646973, -0.08567633479833603, -4.393518447875977, 1.9735071659088135, 0.2897832691669464, -2.0146944522857666, -3.2594594955444336, -1.3880846500396729, 0.01465908344835043, 0.33481526374816895, 1.5948625802993774, -0.41496798396110535, 0.970142662525177, -2.223052501678467, -1.8180651664733887, -0.27434757351875305, -1.2272905111312866, -1.2747457027435303, 0.32398179173469543, -4.336933135986328, -0.6180527210235596, 2.25935435295105, 2.5372095108032227, -1.3373245000839233, -0.8233274817466736, 0.3716656267642975, -2.425844430923462, -0.22593048214912415, -0.38536426424980164, -1.7407015562057495, 2.5560288429260254, 0.8182188272476196, -1.207806944847107, -2.106940269470215, -2.185443878173828, 4.278007984161377, -3.5467193126678467, -0.6976535320281982, -2.4258694648742676, 1.442488670349121, -1.9104523658752441, -2.40006685256958, 3.333458662033081, -0.9475812315940857, -1.8569961786270142, -0.3225860595703125, 0.7757195830345154, -0.36280006170272827, 0.8979274034500122, -0.06419440358877182, -0.8605607151985168, 0.4068630635738373, -1.6924684047698975, 1.0783956050872803, 3.0323102474212646, 3.1918981075286865, 0.10562045127153397, -0.0071592857129871845, 0.06326956301927567, 0.011826088652014732, -0.4119122326374054, -0.927080512046814, 1.0760258436203003, -2.230440616607666, 1.1437065601348877, -1.4614317417144775, -0.23338249325752258, -2.1716020107269287, 1.681196928024292, 0.019500842317938805, 0.8869745135307312, 0.35326921939849854, 1.1417171955108643, 2.546999216079712, 0.7575335502624512, -4.157417297363281, 0.8963077664375305, -3.5758330821990967, -1.60689377784729, 2.2753446102142334, 2.8532299995422363, 4.226029872894287, -0.635944664478302, -1.7993813753128052, -2.0965638160705566, -0.03249051049351692, 3.0848207473754883, -2.368922472000122, -1.5556364059448242, -0.1623314917087555, 1.7270666360855103, -6.29459285736084, -0.8057893514633179, 0.41625311970710754, 0.6299555897712708, 1.2376072406768799, -0.7934725880622864, 4.679277420043945, -1.8292193412780762, -0.038125261664390564, 0.12707959115505219, -0.9082593321800232, -3.4134228229522705, -0.5907363891601562, 3.0795674324035645, -0.5594514608383179, 1.4501798152923584, 4.032487392425537, 1.8616411685943604, 0.7782298922538757, 1.5543949604034424, 1.0473662614822388, -0.6213353276252747, -2.053605318069458, 0.03163084015250206, 0.6298519372940063, -0.9099065065383911, 1.916398048400879, -0.8724021911621094, -2.781582832336426, -1.5866881608963013, 2.016146659851074, -2.4319851398468018, 0.5197693109512329, -2.295231580734253, -0.081305593252182, -0.9226530194282532, -1.1246776580810547, -0.02446177415549755, 1.188340663909912, 0.5793377757072449, -0.253900408744812, -2.6321043968200684, -1.9726630449295044, 1.122052788734436, 2.460465908050537, 0.2875475585460663, 1.6831870079040527, 0.26863181591033936, -4.6597747802734375, -2.466642379760742, 2.4107437133789062, 0.7430644035339355, 6.308063507080078, 0.5505121946334839, 2.975416421890259, -3.583979845046997, 2.3563570976257324, -0.37943029403686523, -0.35772374272346497, 2.830315351486206, -2.564265012741089, -0.37196803092956543, 0.9335581064224243, -0.10704991966485977, 1.213436245918274, -3.18715763092041, -0.08325418084859848, -3.0229976177215576, 2.1169657707214355, -2.2242696285247803, 2.7017831802368164, -0.5416592359542847, -1.427598476409912, 2.3508434295654297, -1.156610369682312, 1.7757916450500488, -0.6706582307815552, 0.7930933237075806, 1.5143543481826782, 1.0395766496658325, -2.1514034271240234, 2.2761662006378174, 1.7719318866729736, -1.388157606124878, -1.2071605920791626, 2.6383554935455322, -1.5572023391723633, 2.5964653491973877, 2.0520687103271484, 3.86727237701416, -0.6897448301315308, 0.40122362971305847, -0.9035011529922485, 4.017018795013428, 1.8589545488357544, 0.01628829538822174, 0.6568343639373779, -0.3557456135749817, 1.6873139142990112, 0.7805740833282471, 0.8982523083686829, 0.059530109167099, 3.706008195877075, -0.7782952785491943, 0.5625229477882385, 2.209625244140625, 0.8207019567489624, -0.8967496752738953, 2.5443687438964844, 0.654516339302063, 1.6882224082946777, -0.6921054720878601, 0.5880916714668274, 1.2878528833389282, -1.3743685483932495, -1.4895161390304565, -2.778085470199585, 0.8774648904800415, -2.739893913269043, -0.5633012056350708, -1.088291883468628, -1.8819737434387207, -2.845262050628662, 0.2987278401851654, 3.323287010192871, -2.6464593410491943, -0.28195151686668396, -1.7355899810791016, -0.7193583846092224, 0.6743059754371643, -2.4226512908935547, -2.7844841480255127, 2.092444658279419, -1.8042497634887695, 1.330493450164795, 0.19034256041049957, -0.3389440178871155, 1.9486535787582397, -2.7274835109710693, 1.6380407810211182, -1.082373023033142, -2.6755008697509766, 7.169071197509766, 4.765864849090576, 3.7340054512023926, 3.628260850906372, -2.0413272380828857, -4.8306684494018555, 0.34595513343811035, 1.7843267917633057, -1.543518304824829, 0.8729246258735657, 1.0968677997589111, 2.866683006286621, 2.064436435699463, -0.44811075925827026, -0.7236605286598206, 2.7697572708129883, 0.7428187131881714, 1.271776556968689, -3.926539897918701, -4.124978065490723, -3.0151243209838867, 2.5907392501831055, -0.7992098331451416, -0.3549085855484009, -1.594812035560608, -0.27883604168891907, -1.5369764566421509, 2.3214380741119385, 4.376938819885254, -1.6896692514419556, -1.890184998512268, -1.9222846031188965, 1.1692310571670532, -0.4772419035434723, -2.582190990447998, 2.140188694000244, -5.842554092407227, 0.5952316522598267, 1.9626842737197876, -2.020500659942627, 3.467703104019165, 0.2502140700817108, -0.5457676649093628, -0.5167706608772278, -0.9861226677894592, -0.5259634852409363, -0.8686724305152893, -4.03619384765625, 2.237835645675659, 1.2769430875778198, -1.527609944343567, 2.150688648223877, 2.568190336227417, 0.5698991417884827, -0.3097793459892273, -2.902578353881836, -2.976109504699707, 0.3611378073692322, -2.091773271560669, -0.3702484667301178, 2.7795464992523193, -2.460460662841797, 0.38421979546546936, 0.29317909479141235, 0.08947476744651794, -1.6264208555221558, 1.3581883907318115, -0.08625297993421555, -5.947286128997803, -1.335212230682373, -2.014068841934204, 0.09773038327693939, -3.468205451965332, 0.14326193928718567, 1.4019557237625122, -3.042146682739258, -0.5892568230628967, 0.2860735356807709, -1.4754934310913086, 3.00152850151062, 2.454681396484375, 1.4185196161270142, -2.318058967590332, -1.2657051086425781, 0.7258898615837097, -0.6937205195426941, -0.9168668389320374, 1.7703611850738525, -0.7703567147254944, -3.247490167617798, 0.03780922293663025, 0.14004701375961304, 0.1092657595872879, 0.10680412501096725, -2.9326305389404297, 0.4625071883201599, 0.5613532066345215, -1.4843647480010986, -2.4549477100372314, -1.9825719594955444, -0.12622101604938507, 0.6374584436416626, -1.453743815422058, 0.8085113167762756, 0.5221811532974243, -1.5006335973739624, -0.3835430145263672, -0.7214575409889221, -1.917791724205017, 1.5141054391860962, 0.4711659252643585, -1.8479115962982178, 1.3488065004348755, -1.6815320253372192, 0.39305561780929565, -2.087519884109497, 0.536134660243988, 0.3560226559638977, 1.9975848197937012, -1.070024847984314, 1.2316052913665771, 0.6519676446914673, -0.9261925220489502, 2.6756887435913086, 2.9487552642822266, -0.9277976751327515, -1.8337260484695435, 0.3892074525356293, 0.40540772676467896, -0.9337073564529419, -3.555447816848755, 0.6214359402656555, -1.2910380363464355, 0.502934992313385, -0.14762544631958008, -0.6886757016181946, 0.3623051941394806, -0.5457621812820435, 3.5618536472320557, 1.5105650424957275, -0.23742029070854187, -1.9305394887924194, 2.215494155883789, -0.7682992219924927, -0.716310977935791, -0.12465449422597885, -4.0050835609436035, 0.7954991459846497, 0.5067355632781982, 1.7899653911590576, 0.21870122849941254, -2.302217721939087, -0.13632333278656006, -5.5052170753479, 0.8638615608215332, 3.2418925762176514, -2.0413625240325928, -2.2565112113952637, -0.49342548847198486, 0.8044217228889465, -2.208193063735962, -3.7859582901000977, 0.29051706194877625, -1.2257155179977417, 1.0822073221206665, 1.9521962404251099, -0.33976680040359497, -3.0313236713409424, -1.439866304397583, 1.774640679359436, -0.7088772654533386, -0.3592222332954407, 3.3574087619781494, -0.6035401225090027, 1.4475606679916382, -3.9191739559173584, 0.8037136793136597, -1.5668532848358154, -0.9867598414421082, -2.099905014038086, 3.3080902099609375, -0.8398430347442627, -1.2104252576828003, -1.0853650569915771, -0.9243699908256531, 1.2724556922912598, -0.17775870859622955, 0.4245585501194, -3.8244850635528564, 2.2764217853546143, -0.6534552574157715, -2.985895872116089, -2.669806957244873, -0.9596015810966492, 3.5865814685821533, -1.4058350324630737, -0.4559873640537262, 3.965789556503296, 5.050622463226318, 1.5812054872512817, 0.7260397672653198, -0.314649373292923, 1.2232475280761719, 0.8256299495697021, -4.743621349334717, -1.7490195035934448, 3.2022783756256104, 2.0183908939361572, -0.13958124816417694, 1.0060334205627441, 1.3054800033569336, -1.277178406715393, 0.2847142517566681, -4.823479652404785, 3.6389708518981934, 5.196968078613281, 2.9515421390533447, -2.354478359222412, -3.542537212371826, 0.10228591412305832, 1.1820135116577148, 0.44059789180755615, 0.5416019558906555, -3.1700856685638428, -1.9801231622695923, 0.598579466342926, -2.587522506713867, 2.348137617111206, -2.2091763019561768, -1.542802095413208, 0.30699458718299866, -1.7498100996017456, 0.21849732100963593, 0.09605079889297485, 0.8064743280410767, -0.13018620014190674, -2.1064648628234863, 4.4952497482299805, 2.252272605895996, 0.19270123541355133, -1.62429678440094, -0.88744056224823, 4.20902681350708, -2.2781832218170166, 2.3918397426605225, -1.7969807386398315, 0.47724470496177673, 1.2364774942398071, 0.16582608222961426, -1.7818965911865234, 1.6511099338531494, -1.6950829029083252, 0.24849988520145416, 1.7726784944534302, 2.3859894275665283, 2.3095550537109375, -1.8909807205200195, -1.1210088729858398, 0.9855154156684875, -1.195863127708435, 1.8914005756378174, -1.0217971801757812, -1.0474846363067627, -1.0978580713272095, -0.13170723617076874, -1.7334723472595215, 0.8011696934700012, -0.42878156900405884, 3.2497596740722656, -1.8548338413238525, -1.7769951820373535, 1.7814921140670776, -0.1273377537727356, 0.17153313755989075, -1.322615385055542, 0.9530612230300903, 0.5383552312850952, -2.1998581886291504, 4.779215335845947, 3.721818685531616, -0.4041604697704315, 1.2459330558776855, -1.7512125968933105, 2.2487006187438965, -1.8991472721099854, 2.9977550506591797, -1.441109538078308, -0.9492528438568115, 0.11047884076833725, 0.5886216759681702, 1.2373523712158203, 3.9875569343566895, -4.658891201019287, -0.2880120277404785, -4.5096049308776855, 1.69180428981781, -0.016037536785006523, 3.138460874557495, -0.8605156540870667, 0.028408516198396683, -2.4828941822052, -3.1902072429656982, -2.32639741897583, 1.1587079763412476, -2.8755626678466797, 1.1466811895370483, 3.41162371635437, -0.4161533713340759, -1.074751377105713, -0.9081840515136719, 2.157133102416992, -1.6839170455932617, -0.6895241737365723, -3.896740674972534, -2.3483362197875977, -4.985591411590576, -2.8144378662109375, -1.2433655261993408, -5.081057071685791, -0.18768922984600067, -0.5077164173126221, 0.017870666459202766, -1.62114679813385, 4.687868595123291, 2.0102427005767822, -0.8954038619995117, -1.8711153268814087, 1.6940003633499146, -0.13730119168758392, 1.4011454582214355, 3.0458898544311523, -1.5484752655029297, 2.012331247329712, 0.6952781677246094, -3.4278321266174316, -1.8252190351486206, -0.10158319026231766, -2.0475680828094482, -4.092132091522217, 0.34556785225868225, 1.1913204193115234, -1.6952134370803833, -1.3316816091537476, -0.9251219034194946, -2.4889755249023438, -0.8030595779418945, -1.7057056427001953, -2.996467351913452, -0.09692367166280746, 2.596017837524414, -0.207414448261261, -1.2417067289352417, 0.6879963874816895, -2.7042510509490967, 2.8507115840911865, -0.7943495512008667, -1.7688508033752441, -1.7969388961791992, -2.2228102684020996, 1.196612000465393, -1.3095223903656006, -0.021278563886880875, -0.08919341117143631, 0.7762997150421143, 1.2568804025650024, 0.3637866675853729, 2.598695993423462, 0.2990059554576874, 0.23907163739204407, -0.17810018360614777, 0.9226089119911194, 1.9375638961791992, 3.0028488636016846, -1.707849383354187, -1.9036108255386353, -1.9875775575637817, -0.15006409585475922, 1.0857384204864502, 0.6956527233123779, -2.9416441917419434, -1.4535843133926392, 3.459906578063965, 0.010267321951687336, -1.186690092086792, 1.6229722499847412, -0.2826978266239166, -1.7011324167251587, 0.22192437946796417, -2.4191458225250244, -0.8225764632225037, 1.3184592723846436, -0.8545224666595459, 1.6379567384719849, -1.8332490921020508, -2.2094838619232178, -0.35723018646240234, -0.6359937787055969, 1.0172374248504639, -2.502246379852295, 3.813076972961426, 4.273043155670166, -0.7498911619186401, 4.196824073791504, 0.5490849614143372, -0.1515488475561142, 4.168750286102295, 1.0840723514556885, -0.9653957486152649, 3.8170969486236572, -3.9523091316223145, -3.328833818435669, -1.9212578535079956, -3.2351226806640625, -1.1788557767868042, 2.0327234268188477, 1.399994134902954, -2.0398313999176025, -3.208409070968628, 0.8560928702354431, -1.780686855316162, 2.7170956134796143, -1.1466749906539917, 0.4300045967102051, -1.3312983512878418, 0.18541976809501648, 2.4564201831817627, 1.5201613903045654, 0.05635983869433403, 2.8019025325775146, 0.9059835076332092, -2.9814329147338867, 2.352907419204712, -3.253497838973999, 0.2692948877811432, 0.2509567141532898, 0.1755237579345703, 3.1753389835357666, 0.9251197576522827, -0.28307870030403137, -0.699673593044281, 1.9906004667282104, -1.3348232507705688, 4.568938255310059, 2.8739631175994873, 0.6994369029998779, -1.421870231628418, 1.4752428531646729, -1.236997127532959, 1.0092027187347412, 1.5008201599121094, -0.17594827711582184, -0.5144208669662476, 2.017617702484131, 2.2862346172332764, -1.4593861103057861, -1.1867499351501465, 2.6342647075653076, -3.2177436351776123, 2.5736002922058105, -1.2749046087265015, -1.449498176574707, 0.2487197369337082, -1.6067874431610107, -0.6844108700752258, 0.04867532476782799, -0.6946987509727478, 1.1082472801208496, 2.5666120052337646, -3.196573257446289, -0.8674877285957336, -4.893265724182129, 1.164366602897644, -1.5676350593566895, 0.6333982348442078, -0.27836328744888306, 2.720982789993286, -0.8653272986412048, 1.789726734161377, 5.299925327301025, -0.08744105696678162, 1.8876913785934448, 2.2573578357696533, -0.7435451745986938, -2.7832720279693604, -0.381980836391449, 1.7990630865097046, 1.2698227167129517, -0.83939528465271, 1.6835654973983765, 1.4457833766937256, -0.4917364716529846, 1.673652172088623, -0.7767832279205322, 0.21467968821525574, 2.85015869140625, -1.0088508129119873, 3.0884342193603516, 1.1408363580703735, 2.858602285385132, -1.9378896951675415, -0.06381186842918396, 2.391664505004883, 0.041156429797410965, 2.2087345123291016, -3.138559341430664, 0.5430795550346375, 1.6184470653533936, -0.010574686340987682, -0.22982119023799896, 0.5791599154472351, -1.2639890909194946, 2.1123979091644287, 4.572627067565918, 2.96895694732666, 1.0710498094558716, -1.1070330142974854, 0.8154039978981018, 0.044604964554309845, 1.698681354522705, 3.007863759994507, 3.531550884246826, 0.48617640137672424, -2.8955636024475098, -0.0014321873895823956, 0.35944992303848267, 2.793184757232666, -0.5304690599441528, 2.509783983230591, 1.035576343536377, 3.0405609607696533, 1.699985384941101, 0.7553319334983826, 2.1057772636413574, 1.116235613822937, 0.3867892324924469, -2.2741732597351074, 2.8194000720977783, -1.6136131286621094, -1.7530220746994019, 0.2580048143863678, 3.6226966381073, 1.0241600275039673, -3.2608766555786133, -1.1554875373840332, -0.9820888042449951, 1.6290360689163208, 1.3431137800216675, -2.3582050800323486, -0.7577600479125977, 2.1912238597869873, -0.21212881803512573, -1.6309751272201538, -0.22057633101940155, -2.2150206565856934, 4.1478166580200195, 0.3697130084037781, -1.2039639949798584, -0.6469038724899292, -1.0786328315734863, 3.4574573040008545, -2.7438852787017822, 0.4567123055458069, 0.44491660594940186, -1.9968986511230469, 1.5540443658828735, 1.9340978860855103, -1.319822907447815, -6.083544731140137, 1.389575481414795, -0.6186736822128296, 0.20308595895767212, -1.0565369129180908, 0.022802770137786865, -1.2464020252227783, -3.4846649169921875, 0.016049249097704887, -1.1841821670532227, -1.0984625816345215, 0.058216918259859085, -0.2527581751346588, 1.9385079145431519, 7.785491466522217, -0.8025336861610413, -1.174702525138855, 1.038150429725647, 3.305800199508667, 0.6554773449897766, -1.322912335395813, 1.6667237281799316, 0.36766529083251953, -1.4919084310531616, -0.5756481885910034, 0.7842883467674255, 2.826205015182495, 3.465211868286133, -0.7988520860671997, -2.718944549560547, -0.22142575681209564, 4.435808181762695, 1.4170881509780884, 3.498366355895996, -0.02090579643845558, 0.5116199254989624, 1.2441130876541138, 3.0921597480773926, 1.971990704536438, 3.752699375152588, 2.3130176067352295, 3.4890315532684326, -0.9898967742919922, 3.147491931915283, 0.8982812166213989, -0.1811652034521103, -1.3585890531539917, 0.20687171816825867, 2.479254961013794, -1.0912846326828003, 2.6992311477661133, -1.0771938562393188, -0.1783033013343811, 1.5366551876068115, -4.608778476715088, 0.15775780379772186, -0.5411103367805481, -2.0588173866271973, -3.1093664169311523, 0.7221086621284485, -1.7702840566635132, 0.28899815678596497, -2.381701946258545, 3.2714619636535645, -1.141796350479126, 1.5877811908721924, 2.060462474822998, -3.5245931148529053, -1.0002305507659912, -0.08441358059644699, -0.6380161643028259, 1.6382102966308594, -4.481685161590576, -0.1743195652961731, 2.488621711730957, -1.1021637916564941, -2.7713799476623535, -3.433861255645752, -3.1067311763763428, 0.9397221803665161, -2.078666925430298, 1.4532735347747803, -1.550918698310852, -0.7458119988441467, 0.3046166002750397, -3.906151056289673, -1.3024637699127197, -0.7346261739730835, 0.9057007431983948, -1.1622817516326904, 1.1169382333755493, -0.41477900743484497, -1.7820416688919067, -0.39817100763320923, 3.4907119274139404, 1.2893353700637817, 0.9570196270942688, 0.32397255301475525, -0.3395172357559204, 1.7441942691802979, -0.8501401543617249, -1.5737336874008179, -2.4585726261138916, -0.974185049533844, 2.044861078262329, -0.6920035481452942, -0.2943791151046753, 0.9761092662811279, -0.8662943840026855, -0.1561565101146698, -1.7269785404205322, -2.07504940032959, 2.456392526626587, 1.0957993268966675, 4.205741882324219, -0.3419930934906006, -1.05805242061615, -0.8761507868766785, -1.5538547039031982, -4.936042308807373, 0.7277604341506958, 3.0012621879577637, 0.9723467230796814, -3.5650525093078613, 0.6558066606521606, -0.18771059811115265, -0.12546105682849884, 0.14499816298484802, -3.4043099880218506, -1.8588850498199463, -1.4730383157730103, -0.3249708116054535, -3.121138572692871, 0.07051761448383331, -2.4448230266571045, 3.2292375564575195, 0.8886036276817322, -2.403491735458374, 1.5541415214538574, 1.7925076484680176, -1.9358081817626953, -0.3190509080886841, -4.183474540710449, -0.5541988015174866, 0.7599139213562012, -2.5603342056274414, 1.5771843194961548, -1.749070644378662, -0.43485474586486816, -2.709916114807129, 0.04144246503710747, 1.8983418941497803, -1.8528294563293457, -0.551706075668335, 1.4993971586227417, 2.516151189804077, 2.6693453788757324, -1.9628701210021973, -2.7666175365448, 1.0484143495559692, -1.5333975553512573, 2.5707433223724365, 1.2645939588546753, -0.5411726236343384, 1.239871859550476, -1.080055832862854, 3.826216220855713, -2.427049398422241, 0.31260859966278076, 0.9421132206916809, 2.193382978439331, -2.1194987297058105, 3.5035922527313232, -2.158414363861084, 1.8434758186340332, 1.195927381515503, -1.4030940532684326, 1.7018520832061768, 0.49134963750839233, -0.26847004890441895, 2.288004159927368, 0.529945433139801, 1.1349892616271973, 0.8404348492622375, 1.4727023839950562, -0.03749948740005493, -1.8438022136688232, -2.842282772064209, 3.1921660900115967, -1.8972558975219727, 3.2622036933898926, 0.4219626486301422, 0.26467907428741455, 2.855930805206299, -2.0737130641937256, 1.687180519104004, 0.7058362364768982, -0.33577534556388855, -0.7292503714561462, -2.7329771518707275, -0.5418462157249451, -3.4389235973358154, -3.217662811279297, -2.032435417175293, -1.9581758975982666, -3.132458448410034, -2.1336257457733154, 2.6815011501312256, 3.387092351913452, 0.9975831508636475, 4.783031940460205, -1.5007890462875366, -0.8062779903411865, -0.18515846133232117, -0.6558318138122559, -1.4100507497787476, -2.642099380493164, -0.4475260078907013, -2.72849440574646, 3.124964714050293, -0.9886474609375, -1.0947948694229126, -3.1429150104522705, -0.808574378490448, 0.555134117603302, 0.32823416590690613, -0.6628643870353699, -2.9334774017333984, 1.809454321861267, -1.547922968864441, -2.2857494354248047, -1.1339837312698364, -0.1754380464553833, -5.482811450958252, 0.6289562582969666, 1.5812456607818604, -2.6896796226501465, -0.3260215222835541, 2.7998242378234863, 2.6350231170654297, -0.9638886451721191, -1.7253841161727905, -2.5153582096099854, 1.0017497539520264, -0.11424265056848526, 1.8107759952545166, 1.2536519765853882, 1.2738442420959473, -0.8688973784446716, 1.554060935974121, -6.515218257904053, 1.4799950122833252, 2.7878329753875732, -3.2037415504455566, -2.853421926498413, 1.4640008211135864, -4.455098628997803, -1.6019227504730225, -1.2260124683380127, -5.186467170715332, -1.259823203086853, 0.9242196679115295, 1.7866990566253662, 3.352128267288208, 3.6452224254608154, 0.3989788591861725, -2.2259128093719482, 0.5950454473495483, -2.4149558544158936, -1.6368728876113892, 1.327651858329773, -1.4310855865478516, -0.9838804006576538, 1.8118361234664917, -3.131498336791992, -0.8223369717597961, 2.466893196105957, -1.1770882606506348, -2.983349561691284, -0.27915892004966736, -0.2035098373889923, -1.4809452295303345, 0.4242272973060608, -2.861924409866333, -4.49428129196167, 0.08892898261547089, 0.5049505233764648, -0.9950356483459473, 0.07667341083288193, -2.2058522701263428, 0.22560831904411316, 1.452709436416626, -0.4066920280456543, -0.4142993688583374, 0.34315696358680725, -1.894482135772705, -1.0743356943130493, -3.736607551574707, 2.017054319381714, -0.19432032108306885, -0.6360583901405334, -1.9231455326080322, -0.3994889557361603, 2.97123122215271, 3.2497339248657227, 0.05688796192407608, 3.3098578453063965, -1.8124734163284302, 1.1970572471618652, 0.9140491485595703, -0.6950560212135315, 0.5504007935523987, -3.875699281692505, -0.33057019114494324, -1.8925248384475708, 2.0204358100891113, 2.0941317081451416, -1.8674458265304565, 0.7919841408729553, 2.141270160675049, -1.2743126153945923, -2.5827486515045166, 1.2148005962371826, 0.17049768567085266, -0.676609218120575, -1.7294843196868896, 0.645326554775238, 0.6803435683250427, 2.1646201610565186, -0.2516273260116577, -2.4262630939483643, -2.0083677768707275, -3.8358547687530518, -1.8885186910629272, 1.0201348066329956, 0.42967596650123596, 1.7922866344451904, -0.7662343382835388, 0.6467987298965454, 0.3024326264858246, -1.4103820323944092, -1.756178379058838, -0.9038957953453064, -1.3380907773971558, 1.8491069078445435, 0.4744338393211365, 0.44286248087882996, 0.14392276108264923, -2.5526585578918457, 0.7131603956222534, 2.857428550720215, -3.127901077270508, -3.7334024906158447, -1.3059375286102295, -1.706291913986206, -1.1331840753555298, 3.9143009185791016, -0.56040358543396, -1.6563817262649536, 1.3513306379318237, -0.05914996191859245, -0.22496996819972992, 0.3365364968776703, 0.47627225518226624, -3.7244038581848145, 0.1604030728340149, 0.6911413669586182, -1.5588608980178833, 2.4185025691986084, 0.5491864681243896, -0.4301759600639343, 0.5394797325134277, 1.4616812467575073, 2.4211435317993164, 0.7957942485809326, 0.21569523215293884, 4.873330116271973, -1.2956840991973877, -0.1764562875032425, 1.1761091947555542, 1.4597867727279663, -0.966076135635376, 2.740408182144165, -0.7066522836685181, -1.9349913597106934, -0.043023157864809036, 0.24077892303466797, 1.7866135835647583, -0.29158759117126465, -0.5633652806282043, -1.4634053707122803, -2.288661241531372, 2.297297477722168, 0.34934473037719727, 3.3516998291015625, 0.3374389111995697, -2.0410823822021484, -0.8572755455970764, 1.7236500978469849, 0.6434831023216248, 3.817681312561035, -0.31749507784843445, -0.0914452075958252, -2.3227078914642334, -2.690796375274658, 1.289968490600586, 2.46696400642395, -1.395330548286438, -2.3556032180786133, 1.4758107662200928, 1.3181978464126587, 0.3848593235015869, 1.5944312810897827, 3.621783971786499, -0.3984406292438507, -1.393368124961853, -1.5560493469238281, 2.743776559829712, -0.21363365650177002, -1.4157520532608032, 3.204031229019165, -3.3269262313842773, 2.0729081630706787, -1.0419933795928955, 0.3610305190086365, -2.901010751724243, -2.770723581314087, -0.547680139541626, 3.535132646560669, 3.9656550884246826, 2.376413583755493, 0.3713507056236267, -1.8987140655517578, 1.0035972595214844, 1.090449333190918, 3.8558413982391357, -0.8150877356529236, -0.9035982489585876, 0.735304594039917, -2.305875062942505, 0.7132360339164734, 2.4315125942230225, 0.07824566960334778, 1.707372784614563, 2.3634467124938965, 0.6194944977760315, -0.015405862592160702, 0.2433224767446518, -1.0741132497787476, 3.1424057483673096, 0.406621515750885, 2.135676383972168, 1.0417941808700562, -0.46310001611709595, -1.5585650205612183, -0.9684107899665833, -4.0524582862854, -0.20055332779884338, -0.3722492456436157, -0.20184223353862762, -0.02482975833117962, 3.1630396842956543, -1.0232648849487305, 0.6787888407707214, 2.6140244007110596, -0.8683627843856812, -0.7124741673469543, -3.8944625854492188, 0.26478201150894165, -0.43119102716445923, 0.5859655737876892, 1.0045024156570435, 2.552813768386841, 1.4284194707870483, -0.7066025733947754, 1.0946836471557617, -1.6690727472305298, -1.5341802835464478, 0.26859381794929504, 5.3284454345703125, 2.5664961338043213, 0.40211209654808044, -2.4716134071350098, 5.524807453155518, -2.6450445652008057, -0.3924151360988617, -0.24267742037773132, 1.064072608947754, -1.4678754806518555, -1.5633357763290405, -2.701979160308838, -0.6358980536460876, -1.87480628490448, -1.6785324811935425, -3.4270968437194824, 2.5523831844329834, -2.659288167953491, 0.10325821489095688, -0.17705371975898743, 0.18895408511161804, -0.32352760434150696, 5.185402870178223, 0.12489254772663116, -6.785148620605469, 1.629709243774414, -1.5995515584945679, -1.8071829080581665, 0.23991523683071136, 3.876772403717041, -0.038358449935913086, 0.17191489040851593, -1.4608259201049805, -2.849586248397827, -1.0022145509719849, 1.1620309352874756, 0.5400207042694092, -2.0984718799591064, -2.293308973312378, -0.644051194190979, 2.712580442428589, -1.3739402294158936, -1.3429458141326904, 0.16690775752067566, 4.193957805633545, 0.20711788535118103, -2.083293914794922, -1.8708243370056152, -0.5103449821472168, 1.6092708110809326, 2.803635597229004, -0.43395259976387024, 2.9853599071502686, -0.5284156203269958, 0.8078939318656921, 1.177383542060852, 3.340867519378662, -0.7220081090927124, 1.9508795738220215, 1.5014764070510864, -0.9635781645774841, 2.464655637741089, 2.8998517990112305, 0.033891111612319946, -1.6093426942825317, 0.5246331691741943, 0.21089798212051392, -1.7624667882919312, 0.8844114542007446, -0.4457653760910034, 1.0987935066223145, -1.3764065504074097, 1.4685176610946655, 3.9926552772521973, -2.8668129444122314, -3.8606202602386475, 0.23236580193042755, 1.749611496925354, 1.359836459159851, 0.6881343126296997, 0.7108142971992493, 1.3618299961090088, 1.1332839727401733, 3.765859842300415, -1.785341501235962, -0.1475527584552765, -1.8600255250930786, 0.5996326208114624, -0.0368640162050724, 1.294571042060852, -0.5569320321083069, 1.030885100364685, -3.740908145904541, -1.68346107006073, 1.5080095529556274, -0.5539973974227905, 1.9782248735427856, -0.9328319430351257, -0.7361378073692322, 2.9193642139434814, 2.542118549346924, 0.05082740634679794, -0.16859117150306702, 0.6495753526687622, 0.4541513919830322, -4.156275272369385, 0.7814953923225403, -3.4755780696868896, 1.8488397598266602, -0.7843091487884521, -4.747325420379639, 1.1532458066940308, 0.14821475744247437, -0.17520973086357117, 0.9188993573188782, -4.223567485809326, -2.035595655441284, -1.625988245010376, 1.2802079916000366, -2.7357590198516846, -0.46377524733543396, -0.8224634528160095, 3.1694176197052, -2.7583868503570557, -0.5178607702255249, 0.7841667532920837, -0.42542535066604614, 3.276125431060791, -0.5281810164451599, -1.3379956483840942, 0.7659902572631836, -0.9594833850860596, -2.011645555496216, 0.9081722497940063, -2.321686267852783, -0.7533859014511108, -1.360136866569519, 0.8933424353599548, -3.625725269317627, -0.7516266703605652, -1.5301507711410522, -6.238662242889404, -0.9408655762672424, -1.890693187713623, 3.6288342475891113, -2.0682835578918457, -0.2841646671295166, 1.131319284439087, -2.8576126098632812, 2.471238613128662, 1.2869161367416382, 0.004139468539506197, -0.7722935676574707, -0.4930642247200012, 0.38142722845077515, -1.0793120861053467, 1.1596823930740356, 1.6332602500915527, 1.1214200258255005, -0.6447210907936096, -0.3401593267917633, -0.198832169175148, 3.162662982940674, 3.704911231994629, 0.17887109518051147, -0.8364641666412354, -2.364968776702881, -0.04716654494404793, 4.038967609405518, -0.199406236410141, -2.362616777420044, -1.3038055896759033, 0.5758841037750244, -0.054638639092445374, 1.1667969226837158, 1.3068397045135498, -1.2693182229995728, 1.5861505270004272, -1.5994688272476196, -1.0509077310562134, 1.3661024570465088, -1.3753567934036255, -0.5468172430992126, 2.1197733879089355, 0.2551999092102051, 0.22525104880332947, 0.38698792457580566, 3.9209020137786865, 1.4412591457366943, 0.7802623510360718, 0.918048620223999, 0.5963148474693298, 1.3172948360443115, 0.9200593829154968, 1.1587952375411987, -1.0113980770111084, 0.20146919786930084, -0.02781171165406704, 2.3947596549987793, 3.452637195587158, -1.2648468017578125, -0.23471279442310333, -0.10145677626132965, 0.06279414147138596, -4.3099284172058105, -3.3476345539093018, -2.047234058380127, -0.759469747543335, -0.5486020445823669, 1.7631726264953613, -0.003947501536458731, -0.18646365404129028, -0.7565574645996094, -0.5898867845535278, 1.9563181400299072, -0.35367661714553833, -2.7476143836975098, -2.189652442932129, -0.014433902688324451, 0.4047934114933014, -1.8387833833694458, -2.9703636169433594, -3.4884278774261475, -2.917940139770508, 2.544541120529175, -0.9361138939857483, -0.4151894748210907, -0.7722427248954773, -3.1183321475982666, -2.652052402496338, -2.3885252475738525, -3.6403210163116455, -0.7807720303535461, -2.431685447692871, 0.3608129322528839, -0.034949205815792084, 2.7419023513793945, -2.265137195587158, 1.3890478610992432, -3.6200571060180664, -1.638641595840454, 0.46888336539268494, 0.39354032278060913, -3.326202154159546, -0.6516391634941101, 2.006129026412964, 1.3281770944595337, 1.4035694599151611, 0.005364848766475916, 1.4601324796676636, 0.945705235004425, -0.7953272461891174, 2.0392138957977295, -0.17285394668579102, 0.1690613031387329, -3.286463499069214, 1.4137316942214966, -2.057405710220337, 0.2654421627521515, 0.31008079648017883, -0.9674365520477295, 0.4375916123390198, 1.9011962413787842, -2.736945390701294, -0.8013559579849243, -2.2316572666168213, 2.2537403106689453, 0.6409158706665039, 1.612393856048584, -2.474236249923706, -0.39852333068847656, 0.4395223557949066, -3.161785364151001, 1.8565030097961426, -2.0373880863189697, -0.021789072081446648, 1.0578949451446533, 2.9801883697509766, -1.6939877271652222, -0.7896243333816528, -0.020260760560631752, -2.418595314025879, -0.30704155564308167, -3.097012996673584, 1.262333631515503, 2.3225629329681396, -1.5749279260635376, 2.774498462677002, 0.7513827085494995, -0.9936389923095703, 1.363633394241333, -0.14593470096588135, 0.792574405670166, -1.0267404317855835, 0.625184953212738, -2.705094814300537, 2.1584019660949707, 0.7013111114501953, 0.43111586570739746, -0.28227055072784424, -1.2465806007385254, 3.2422192096710205, 2.498027801513672, -1.0879383087158203, 0.5827075242996216, -1.282488226890564, -0.18664167821407318, -0.24704211950302124, -1.3593782186508179, -1.1937447786331177, -1.6724011898040771, -1.803571343421936, -2.9153003692626953, -0.33626532554626465, -1.4281891584396362, 0.5173403024673462, -2.1265923976898193, 0.0144949434325099, -1.44588303565979, -0.4443635046482086, -0.23243559896945953, 1.8634283542633057, 1.669113039970398, 2.4342763423919678, 0.16549299657344818, 1.372939109802246, 1.7190922498703003, -0.9778546094894409, -1.242769479751587, -3.4428577423095703, -0.27005311846733093, -3.156566858291626, -1.753209114074707, -1.811213731765747, -2.6221563816070557, 0.5868173241615295, 0.12932278215885162, 1.3647340536117554, 0.8730813264846802, 2.9515531063079834, -2.524509906768799, -0.4455606937408447, 2.419731855392456, -0.9600204229354858, 3.111997127532959, 1.7287877798080444, -0.45768049359321594, 2.9422683715820312, -0.41389089822769165, 2.382153034210205, 0.26244133710861206, -1.000853180885315, 1.4821518659591675, 1.3690606355667114, 1.0074801445007324, 0.9051161408424377, 1.5272213220596313, -0.1764458566904068, -0.3469410836696625, 0.016439735889434814, -1.830235481262207, 0.14827944338321686, 2.7629306316375732, 1.9507766962051392, -3.0671489238739014, -0.6742374897003174, -1.6070588827133179, -3.6228067874908447, -2.246293067932129, -1.147283911705017, -1.743138074874878, -2.091247320175171, 1.8119902610778809, -0.3517175018787384, 0.10445652157068253, -0.08212894946336746, 1.0118154287338257, 0.5126646161079407, -1.0917260646820068, 0.4341021776199341, -0.8441821932792664, 0.37379106879234314, -0.7701050639152527, 2.5155832767486572, 1.0011855363845825, -2.08831524848938, -1.2324928045272827, -0.3200594186782837, -1.6597325801849365, -0.048605695366859436, 0.8401752710342407, 1.1569240093231201, 0.9502015709877014, -3.812403678894043, -2.190805673599243, 1.376803994178772, -1.0102040767669678, -2.4545154571533203, 1.7474967241287231, -0.6708294749259949, 3.7589263916015625, 0.08748076111078262, -0.8134949803352356, 0.05459212511777878, 2.6607580184936523, -4.405036926269531, -1.7121003866195679, 0.7751581072807312, -2.247662305831909, -1.0665148496627808, -2.4464809894561768, -0.507194995880127, -1.8933664560317993, -1.8870381116867065, -2.3797366619110107, 0.9826566576957703, 1.70280921459198, 0.24045461416244507, -1.1134896278381348, 0.316087007522583, -2.453099012374878, -0.4927700459957123, 1.4137122631072998, -0.3379441797733307, -0.006658393424004316, 0.3582477271556854, -3.0528767108917236, 0.8412408232688904, 1.1924327611923218, 0.9143797159194946, 3.3439388275146484, -4.3588762283325195, 3.2521426677703857, -0.5575249791145325, -0.48097696900367737, 5.527586460113525, -0.4633450210094452, -2.3431396484375, 0.0941326916217804, 3.591041088104248, 0.8290177583694458, 3.9658632278442383, -0.13914543390274048, -1.531917929649353, -1.5145630836486816, -3.447626829147339, 3.3309199810028076, -0.18484050035476685, -0.7596165537834167, -0.3936922252178192, 2.191244602203369, -2.527438163757324, 1.633935570716858, 2.49792218208313, 0.5042136907577515, -3.0676956176757812, -0.7852597236633301, -0.9449858665466309, -3.2221782207489014, 1.3950501680374146, -0.7923742532730103, 0.1573513001203537, -3.156484365463257, 0.29776671528816223, -0.5496147871017456, -0.1660519242286682, 0.4944954812526703, -0.39614179730415344, 0.5715072154998779, -3.063534736633301, 1.5738450288772583, 3.2816874980926514, 0.516501247882843, 0.18835681676864624, 2.6356987953186035, -0.3801528513431549, -0.7114979028701782, 0.6928091645240784, -0.7754155397415161, -0.8334919810295105, 0.06445146352052689, -0.6452393531799316, 0.6886802315711975, 0.3258364498615265, -1.0461615324020386, 1.8407162427902222, 0.27948227524757385, 4.754266262054443, -6.7485246658325195, -2.7231907844543457, -1.3844823837280273, -1.944692611694336, 0.19140683114528656, 3.1119484901428223, 2.7354629039764404, -0.7837443351745605, 1.8874093294143677, 3.468632459640503, 1.560336709022522, -0.1497696042060852, 2.2751121520996094, -1.4581801891326904, -1.1153100728988647, -2.374633550643921, -0.6587360501289368, 2.403930187225342, 2.253154993057251, -1.926818609237671, -0.9019244909286499, -1.0762971639633179, -0.5847079157829285, -1.5744138956069946, -2.3919663429260254, 1.4090397357940674, 0.7116031050682068, 0.8771219849586487, 0.5872573256492615, -0.6627733111381531, 4.065080165863037, -0.02536545880138874, -0.0014478596858680248, 0.7073807716369629, -0.08538983017206192, 2.368992328643799, -3.653562068939209, -0.9084902405738831, -0.4173997938632965, -2.9483587741851807, -1.4489296674728394, -3.813326358795166, 1.9987421035766602, 0.20960703492164612, 1.3374372720718384, 2.537125587463379, 0.4814351499080658, -0.18046244978904724, 3.0766971111297607, -3.1400742530822754, 4.194031715393066, -2.8788812160491943, 0.06912200152873993, 0.6757147908210754, 1.270349144935608, -1.3729051351547241, -0.3907836377620697, 0.3870069682598114, -1.2858576774597168, -1.2931265830993652, 0.7683159112930298, 1.2120249271392822, -2.4854536056518555, 1.8651299476623535, 0.6547967791557312, -3.2830283641815186, 1.7614960670471191, -1.3837043046951294, 2.491722822189331, 0.5623372197151184, 1.1002013683319092, 2.9029388427734375, 0.15174610912799835, 0.980421781539917, -2.147737979888916, -1.4534363746643066, 4.377958297729492, -1.953115701675415, 0.04247698560357094, -1.9947365522384644, -0.8548725247383118, 0.5601123571395874, 1.6960530281066895, -1.301795482635498, 1.5510319471359253, 0.16765816509723663, 4.1826372146606445, -3.8710885047912598, -1.8253185749053955, 0.7102514505386353, -4.310102462768555, 0.31763458251953125, 1.640748143196106, 3.5477168560028076, -2.2725093364715576, -2.9187464714050293, -1.744239091873169, 0.8394595980644226, 1.4137212038040161, 0.9563763737678528, -0.375643789768219, -0.4649386405944824, -1.0194501876831055, -2.564469337463379, -1.766416072845459, -3.8538148403167725, 0.19338247179985046, 2.140793800354004, -1.483426570892334, -1.3709630966186523, -2.1177713871002197, 0.9367547035217285, -2.3105928897857666, 3.230067014694214, -3.97208309173584, 1.6741225719451904, -1.3162288665771484, -1.430786371231079, 3.4857919216156006, -2.4887590408325195, 0.9618706107139587, 0.6843332648277283, 2.239520311355591, 0.9771043658256531, 0.16152723133563995, -3.0983707904815674, -0.10289445519447327, 3.7710015773773193, 1.9258750677108765, -1.9982397556304932, 2.336369514465332, -0.9667556285858154, -1.1516627073287964, -1.083419680595398, 1.2642079591751099, -0.8719649910926819, 0.5483595132827759, -0.11937716603279114, -0.850780725479126, -0.41915565729141235, -2.129983425140381, 1.2601776123046875, -1.3276768922805786, 1.164223074913025, -1.9476721286773682, -0.8234297037124634, -3.9725635051727295, 0.7969207167625427, 2.9571666717529297, 0.05129547044634819, 0.5376916527748108, -1.7561256885528564, -1.597375750541687, 0.910142183303833, -2.431032657623291, -2.4756104946136475, 2.4301183223724365, 0.490885853767395, 2.0864920616149902, -0.5952171087265015, -2.9126830101013184, 0.19393734633922577, 0.32663899660110474, 2.4184653759002686, -3.7467195987701416, -1.8591820001602173, 4.15065860748291, 0.49257388710975647, 0.6083744168281555, 1.2602483034133911, -0.9629713892936707, -0.8322835564613342, -0.8325143456459045, -2.5962066650390625, -0.7525156736373901, -2.4559805393218994, -1.1789395809173584, -1.8059650659561157, -1.607489824295044, 1.708885669708252, 1.7784351110458374, 1.9181209802627563, 3.7570571899414062, 0.9852865934371948, -2.266399383544922, 0.3798956274986267, -0.8472710847854614, 0.39330652356147766, -1.3760457038879395, -0.00405155261978507, -2.1103873252868652, -2.4340460300445557, 2.2838237285614014, -0.8487495183944702, -1.7939178943634033, -0.4345969557762146, 1.1770597696304321, 0.18041260540485382, -1.9028311967849731, -0.9617824554443359, -1.433986783027649, -3.4917218685150146, -4.259814262390137, -4.213344097137451, 2.5214734077453613, -1.7251007556915283, -3.942263603210449, 2.1267223358154297, 0.31255799531936646, -1.4813933372497559, -2.5491936206817627, 0.37691593170166016, -0.4372653067111969, -1.320876121520996, 0.8426192402839661, -0.026417085900902748, -0.6994538903236389, -1.8677972555160522, 0.15966641902923584, 0.01752777397632599, -1.4284156560897827, 1.4019391536712646, -0.18098817765712738, -2.5101702213287354, 0.605491042137146, 0.2926305830478668, 0.7916393876075745, -4.759459972381592, -0.1252477616071701, 1.5880306959152222, -1.9213470220565796, 1.5902748107910156, 1.7640407085418701, 0.96120685338974, 4.977931499481201, 1.2266194820404053, -2.753350019454956, 0.4557036757469177, 0.1265455186367035, 3.2585978507995605, 1.2277686595916748, 2.0176780223846436, -1.1296237707138062, 1.2743619680404663, -2.182060956954956, -2.2183125019073486, 0.6201428174972534, -3.152292013168335, -1.936460256576538, -1.3332926034927368, -0.4628863036632538, -0.49161088466644287, -1.9139724969863892, -0.5538272261619568, 0.7588333487510681, 1.1165473461151123, -0.7834194898605347, -0.5440608859062195, -0.07454802840948105, -3.9385406970977783, -2.3225059509277344, -0.4480805993080139, 2.413093090057373, 1.8660106658935547, -1.4377079010009766, -1.2057331800460815, -1.8385083675384521, -0.5941051244735718, 0.7523191571235657, -2.315486431121826, 0.3435683250427246, 0.8078683614730835, 3.2847938537597656, 1.1698939800262451, 1.0408165454864502, -0.7034420371055603, -1.4572856426239014, -2.6304776668548584, 1.9042072296142578, 0.9385047554969788, 2.878915786743164, 2.414992332458496, -0.7159493565559387, -1.3978289365768433, -1.2673314809799194, 2.067805051803589, 3.690725564956665, 4.857728481292725, 1.9529844522476196, 2.2858572006225586, 2.751412868499756, 2.4552366733551025, 1.8898428678512573, 0.014027649536728859, 2.57061767578125, 0.9402596950531006, 1.127470850944519, -0.08011795580387115, 1.4713984727859497, -0.40378400683403015, -3.7219104766845703, 0.6235913634300232, -0.14250126481056213, 0.9419640302658081, -2.9499764442443848, 2.7175121307373047, -0.6400314569473267, -3.7591655254364014, 2.938676118850708, -4.440436840057373, -0.826309084892273, 1.3978502750396729, 0.31416261196136475, -1.2758244276046753, 1.964726448059082, 2.686933755874634, 1.3884716033935547, -0.300555944442749, 1.0558888912200928, 3.660707473754883, 0.1809929460287094, -2.855360507965088, -2.357870101928711, -0.9113531112670898, 1.5319764614105225, -4.648289203643799, 1.8319076299667358, -2.0143141746520996, 0.08551011979579926, -3.392454147338867, 2.4731569290161133, 0.23397693037986755, 1.983533263206482, 5.789851188659668, 2.196712017059326, 2.735731601715088, 1.920548915863037, 0.9014940857887268, -3.7676913738250732, -0.3646351993083954, -1.5070191621780396, -1.0215293169021606, -3.7689499855041504, 0.5302253365516663, -0.14113037288188934, -1.328522801399231, -0.6626675724983215, 0.6344512104988098, -3.866406202316284, -1.6855305433273315, -1.7397863864898682, -0.34596630930900574, -1.8375850915908813, -1.1836769580841064, 0.676666259765625, 3.4510416984558105, 0.13222011923789978, -0.7943164110183716, 0.9775066375732422, -0.5772544145584106, -0.8398630619049072, -3.723536729812622, -0.7826616764068604, -0.11733105778694153, -2.946965456008911, -0.7770774960517883, -2.3164565563201904, 1.8280531167984009, 2.4392974376678467, 5.093084812164307, 0.6389499306678772, 3.219014883041382, -2.5142722129821777, 1.1905633211135864, 1.8592076301574707, 1.9853535890579224, 0.05179600045084953, -2.541931629180908, -0.5835532546043396, -2.9218413829803467, 3.3587470054626465, 1.0114526748657227, 1.2694059610366821, -2.2342658042907715, 1.738218903541565, -1.732354760169983, 0.2743055820465088, -0.023565499112010002, -0.17973698675632477, 0.9248015284538269, -1.556292176246643, -2.015532970428467, 1.4427558183670044, 1.4474455118179321, 2.1866722106933594, 1.517971396446228, 4.428621292114258, 5.475288391113281, 1.0382012128829956, 0.3361678123474121, -2.211876392364502, 1.1311384439468384, -0.6672676205635071, -2.4917478561401367, 0.6437145471572876, -0.7286087870597839, -0.6143547296524048, 0.3958520293235779, -2.299506425857544, -0.41341862082481384, 1.5205321311950684, -0.4616694748401642, -1.1351783275604248, -2.2521708011627197, 0.3335251212120056, 1.0079742670059204, 3.6605191230773926, -2.4499337673187256, -0.5921433568000793, 2.487689971923828, -2.491990566253662, 2.8806498050689697, -2.179091691970825, 2.0316991806030273, 1.1805318593978882, 0.36782628297805786, -0.16035611927509308, -3.137150526046753, 2.7561614513397217, -0.9311216473579407, -1.91667902469635, 1.1652767658233643, 0.36975252628326416, 1.2458113431930542, -3.5254740715026855, -1.9461177587509155, -0.8200725317001343, -1.9095081090927124, 2.0787405967712402, -0.7015581130981445, 0.7040292620658875, 0.8866922855377197, 1.177978515625, 0.5469484329223633, -0.07620424032211304, -1.6755446195602417, 0.1135062649846077, 1.4441370964050293, 1.3226486444473267, 1.1858880519866943, 0.16782228648662567, 0.7369744777679443, -2.104325294494629, -0.253385990858078, 0.7551519274711609, -1.4786103963851929, 1.1132467985153198, -0.3591170608997345, -1.8233002424240112, 4.4276442527771, -1.345841646194458, -1.7619948387145996, 1.5526189804077148, -0.37342140078544617, 1.5515292882919312, -1.9291844367980957, -0.4238322079181671, 1.593624472618103, 1.8341577053070068, 0.8274646401405334, -0.8600817322731018, -1.6938813924789429, 0.5202839970588684, 0.9751532673835754, -2.7545392513275146, 2.7320189476013184, -2.4472649097442627, 1.3534501791000366, -1.3819094896316528, -1.6965876817703247, -0.004109658766537905, -1.7045300006866455, 1.4163780212402344, -1.107371211051941, 0.03380650281906128, 3.0776562690734863, -1.3541312217712402, -1.0107181072235107, 2.0923709869384766, 0.3175346553325653, 0.2773497402667999, -0.30730822682380676, 1.6658775806427002, 1.283774733543396, 2.1357312202453613, -5.129626750946045, 0.5486174821853638, 4.512635231018066, -2.445061683654785, 0.820724606513977, 1.506136178970337, 2.5656075477600098, -2.0087180137634277, 0.42917901277542114, 0.05685941129922867, -2.3833510875701904, -3.587705373764038, -0.38075006008148193, 2.0539567470550537, 1.071425437927246, 0.5295975804328918, -1.2341125011444092, 1.2670379877090454, 1.1448086500167847, 1.1104369163513184, 0.1826031357049942, -1.3346738815307617, -0.24755364656448364, -3.4158599376678467, 3.908688545227051, 2.3358612060546875, 2.7495598793029785, -0.5078471302986145, 0.6643435955047607, 2.6151833534240723, -1.1194885969161987, -1.2747433185577393, -1.6128864288330078, -2.970097064971924, -3.3231492042541504, 2.8596200942993164, -2.804091453552246, -0.8639134168624878, 1.5932371616363525, 0.6482923030853271, 0.5331540703773499, 1.8694850206375122, 2.240124464035034, -0.29011380672454834, -2.9495553970336914, -1.4167606830596924, 1.2704882621765137, 1.8228257894515991, -2.2610838413238525, 0.7695937752723694, 0.4613082706928253, -0.6477243900299072, -1.230165958404541, -0.6410579085350037, -3.4158873558044434, -0.014844812452793121, -0.4997023046016693, -0.6256206035614014, 0.3617987334728241, 4.489445209503174, 2.3432257175445557, -2.0364513397216797, 0.8984464406967163, -0.7386754155158997, 0.4607182741165161, 1.450709581375122, -0.46581172943115234, -0.858627200126648, -3.4503211975097656, 0.1442403793334961, -0.09792076051235199, -1.286878228187561, 3.8688714504241943, 0.21018372476100922, 0.4596056640148163, 0.49818605184555054, 3.9014065265655518, 0.9301960468292236, -1.1879479885101318, -2.1683096885681152, -2.1146817207336426, 2.0531795024871826, 1.5322835445404053, 1.0423177480697632, 2.7817342281341553, 4.272284984588623, -2.982621908187866, 0.33480018377304077, -4.44291353225708, -1.0208816528320312, 1.9401040077209473, 2.997192144393921, 0.6133401989936829, 2.0064473152160645, -1.0617656707763672, -0.059576213359832764, 1.8916350603103638, -0.44495436549186707, -0.3089229464530945, 2.6226227283477783, 3.7655410766601562, -2.267155885696411, 0.6664639711380005, 0.15257035195827484, -1.1509889364242554, 3.005809783935547, 1.361870527267456, -0.8743936419487, 1.3452028036117554, -1.6185276508331299, 1.068837285041809, 0.5351426005363464, 0.35041335225105286, 0.6699652671813965, -0.289909303188324, 0.7291497588157654, 1.9982430934906006, -2.5478029251098633, 3.8039329051971436, -0.19782711565494537, -0.7033220529556274, 0.24812187254428864, -0.23759880661964417, -1.8387045860290527, 0.6612992882728577, -1.02218496799469, -2.1253628730773926, 4.5316901206970215, 0.2609672546386719, 2.8598015308380127, 1.0948467254638672, -0.05675167217850685, 1.1361618041992188, -0.49738752841949463, 1.431650996208191, -3.622774124145508, 0.340682715177536, -2.0127334594726562, -0.016229506582021713, -2.7054882049560547, -1.0917460918426514, 2.475022315979004, 1.2864978313446045, -2.256483793258667, -2.365736722946167, 0.6530848741531372, -1.6539174318313599, 2.0859737396240234, -1.4415181875228882, -0.7923710942268372, -2.742534637451172, 2.826000452041626, 4.443478107452393, -0.44962242245674133, 0.5156185030937195, -1.3654093742370605, 1.5276826620101929, -1.2025359869003296, -2.092235565185547, 2.779919147491455, 2.2618558406829834, 1.0612660646438599, 0.8566949963569641, 0.23533202707767487, 0.06162825971841812, 1.343610405921936, 2.545886516571045, 1.514593482017517, 3.224929094314575, 0.22124598920345306, 3.2163422107696533, -2.768916606903076, -1.984432339668274, 0.2797083854675293, -1.7526084184646606, -1.5885589122772217, -0.9611585140228271, 3.270066261291504, 4.542919635772705, -2.530783176422119, -3.3149569034576416, 0.20448507368564606, 0.9925702214241028, -1.1304802894592285, -0.7972908020019531, -0.8398974537849426, -1.9471120834350586, 1.849449634552002, 1.939535140991211, -0.3386322259902954, -2.0960326194763184, 0.04324636608362198, -3.3697071075439453, -2.9905941486358643, -0.3605884611606598, -1.2498319149017334, 1.9440370798110962, -0.44505050778388977, -2.0517964363098145, -0.3086690604686737, 0.32543236017227173, 3.1102588176727295, 0.7800756096839905, 3.178541421890259, -3.0701396465301514, 3.93387770652771, -0.472218781709671, -0.6162614226341248, -2.0442254543304443, 1.087826132774353, 0.09711704403162003, 1.3960578441619873, -2.953601598739624, -1.133710265159607, 0.4123421907424927, -1.5303229093551636, -2.3882062435150146, -3.6064603328704834, -1.9096606969833374, -3.9939351081848145, 1.6349540948867798, 2.7436556816101074, -2.746093273162842, -1.7807507514953613, -2.0760769844055176, -1.0673786401748657, -2.386793613433838, -1.2311642169952393, -3.0569076538085938, 1.3115969896316528, -4.027857303619385, -0.16204717755317688, -0.9072303771972656, -1.339174509048462, -1.5455516576766968, 1.4810909032821655, -5.269399166107178, 0.12439676374197006, -1.8338669538497925, -0.3956542909145355, -1.1740163564682007, -1.9735562801361084, 0.7663599848747253, 0.35072317719459534, -1.8225245475769043, 0.22635667026042938, 1.8665432929992676, 0.30918172001838684, -2.6916565895080566, -1.3822863101959229, 1.2090038061141968, 0.025227956473827362, 0.4951174557209015, 1.4210668802261353, 2.5229618549346924, 0.5891157388687134, 0.58163982629776, 0.01810467801988125, 3.2654080390930176, 3.5346596240997314, 0.4048735499382019, -0.9909356832504272, -0.028454836457967758, -1.9028915166854858, 0.7298458814620972, -1.563185214996338, -2.985661745071411, -1.9730063676834106, 0.8180460929870605, -0.37062641978263855, -0.7286446690559387, -1.7486622333526611, -0.06456878781318665, -1.4840214252471924, 0.17091673612594604, -0.6041314005851746, 0.2437347024679184, -2.843064308166504, 2.7884361743927, -0.19355438649654388, 1.7954522371292114, -1.315782904624939, -1.0653737783432007, -1.0438053607940674, -1.7319742441177368, 0.12950393557548523, 0.6186688542366028, 1.0081405639648438, 1.8796988725662231, 0.03855448216199875, -1.4180113077163696, -2.544893980026245, -0.7239822149276733, -2.049328088760376, -2.8913991451263428, 3.0875461101531982, 0.8776184320449829, 3.552765130996704, 0.46349120140075684, -1.7290412187576294, -2.7576076984405518, 0.09937751293182373, 0.8177562355995178, -3.183929920196533, 1.6718189716339111, -2.6765012741088867, -3.0541956424713135, 3.146401882171631, -1.558281660079956, 0.11479654908180237, -3.434440851211548, -1.2019779682159424, -0.08119107037782669, 1.5799140930175781, 1.7055293321609497, -0.965046226978302, 0.6625899076461792, -2.0680348873138428, 1.8814966678619385, 2.497943878173828, -1.3132137060165405, -0.1529696136713028, -2.528925657272339, 1.8953516483306885, -1.220320224761963, -1.3456414937973022, 0.8198553323745728, 0.9580578804016113, -0.766946017742157, 3.8880491256713867, -0.26801836490631104, -1.9372488260269165, -0.9800150394439697, 1.6645509004592896, -1.621227502822876, -0.3851141929626465, 0.5691565275192261, 1.1196833848953247, -0.7874764204025269, -0.6377078294754028, 2.6834261417388916, -0.3572690486907959, 3.989518404006958, 3.4726204872131348, -1.5569629669189453, -0.8333522081375122, -2.060059070587158, 1.8209389448165894, -0.46594756841659546, 1.6539859771728516, -2.0514345169067383, 0.10370337218046188, 1.7163609266281128, -0.7864106893539429, -2.8305442333221436, -0.32353508472442627, 2.2218809127807617, -2.857546806335449, 3.1416776180267334, 1.74860417842865, 1.1996829509735107, -2.074697256088257, -0.7731634378433228, 0.5474370121955872, -3.3433754444122314, -1.8067268133163452, -2.0316433906555176, 0.9346501231193542, -0.15169310569763184, 0.63565593957901, -0.14971177279949188, -2.4284253120422363, -3.768573045730591, 2.5229921340942383, 0.16350829601287842, -3.405479907989502, -0.6084149479866028, -3.6326348781585693, 0.8189612030982971, -0.43328967690467834, 1.0508086681365967, 1.1303189992904663, 0.8114511370658875, 2.437001943588257, -0.09738662093877792, 3.4986956119537354, -1.6200902462005615, -0.3345978856086731, -1.561907172203064, 1.7684578895568848, 0.4350656569004059, -1.4948333501815796, -4.496615409851074, 0.8196244835853577, -0.5029028058052063, -0.7109497785568237, -0.49629154801368713, 0.553249180316925, -1.5120055675506592, 1.0468800067901611, -2.8404853343963623, -0.504245936870575, 0.9899663925170898, -3.051947832107544, 2.104153633117676, 0.5883322358131409, -2.552335023880005, 2.2126357555389404, 1.491952896118164, 3.2369418144226074, 1.827311635017395, 3.2106025218963623, -0.15247991681098938, 2.415010690689087, 2.9034273624420166, 3.4917402267456055, -1.2476131916046143, 0.8684611320495605, 2.5464441776275635, 2.201537847518921, -0.5568050146102905, -0.589732825756073, -0.3788229525089264, 0.0613369345664978, 0.3121238052845001, 0.11706514656543732, -1.4639674425125122, -1.8603800535202026, 1.3730392456054688, 0.022625114768743515, 1.4489473104476929, 0.9113878011703491, 0.9121313095092773, 1.7845033407211304, 1.6507844924926758, -0.7687185406684875, 0.9337760210037231, -0.875974714756012, -1.8760792016983032, -1.3009233474731445, 2.231595277786255, -2.0287139415740967, -0.6598399877548218, -0.8288301825523376, 2.9399750232696533, 0.7303081750869751, -2.2219812870025635, 0.5343230366706848, 1.0715734958648682, 2.402102470397949, -4.433779716491699, -2.193864107131958, 2.2761175632476807, -0.17442311346530914, 0.9682493209838867, 1.9718847274780273, 0.5837634801864624, 0.011038466356694698, -0.8675856590270996, -1.6080825328826904, -1.4551663398742676, 2.3240668773651123, 0.0757443979382515, 1.2409201860427856, -1.2372233867645264, -3.5028297901153564, 0.5509371757507324, 1.610313057899475, 3.600748062133789, 3.059380054473877, -2.7575302124023438, -0.7797229886054993, 0.4478912949562073, 3.389324188232422, -1.174078345298767, -2.242234468460083, -1.7332960367202759, 0.44699332118034363, -0.80967116355896, 1.7206270694732666, 1.4647643566131592, 1.4949501752853394, -0.8926811218261719, 0.9807851314544678, -2.2182562351226807, 0.5445992350578308, 0.40828004479408264, 0.13600413501262665, 1.044386863708496, -0.7970998883247375, -1.0340086221694946, 0.7464067935943604, 1.8643457889556885, 0.3256039023399353, 0.5548712611198425, 2.415097951889038, -2.372485399246216, 0.9682551026344299, -2.4342782497406006, 1.2272593975067139, -0.0803702175617218, -1.378517985343933, -1.0153088569641113, 1.911783218383789, -0.429904580116272, -0.03901063650846481, 0.11982414126396179, -1.5838735103607178, -0.7805300354957581, 0.3167373538017273, -5.256614685058594, 3.2283592224121094, 2.5111331939697266, -1.4519113302230835, -1.1925857067108154, 2.3169901371002197, 2.2873780727386475, 1.5974385738372803, -0.8275900483131409, 0.9240221977233887, 0.6036628484725952, -1.66788649559021, -1.929276704788208, 3.844733715057373, 1.0785657167434692, -2.5942211151123047, -3.3136327266693115, -1.6844580173492432, 0.7512235045433044, 4.683455944061279, -0.4173317849636078, 0.7147151231765747, -2.142235279083252, 1.2440943717956543, -1.0126358270645142, 0.034593041986227036, 1.9306585788726807, -4.444581031799316, -2.052420139312744, 2.0987491607666016, -1.882609486579895, 1.1991058588027954, -0.8911911845207214, 1.869270920753479, 0.4193097949028015, -2.8480401039123535, 1.273897409439087, 0.015158242546021938, -1.675512671470642, -3.0335116386413574, -0.7510258555412292, 0.9871913194656372, -1.4812060594558716, 1.3187955617904663, 0.07813544571399689, 1.087679386138916, -1.239986777305603, -2.2613120079040527, 2.295642375946045, 1.9626461267471313, 2.3720946311950684, -0.47242724895477295, 2.48113751411438, 1.2389791011810303, -1.1016842126846313, 0.33139318227767944, -3.0179970264434814, -1.3640269041061401, -1.111121654510498, 1.280372142791748, 1.6750613451004028, -2.1629414558410645, -0.9857407808303833, 1.1828409433364868, -3.711091995239258, -3.242985248565674, -1.0436714887619019, -0.34415310621261597, -2.515568733215332, 0.517536997795105, 1.8051875829696655, -0.3886939287185669, 0.4993996322154999, 0.3267984092235565, -2.831218719482422, 2.325183868408203, -2.493931531906128, 1.1194087266921997, -0.29127728939056396, -2.193248987197876, 3.0809431076049805, 0.9509249925613403, 1.6100585460662842, 0.6203168034553528, 0.11514042317867279, 0.6655176281929016, 0.6315016150474548, 3.3033199310302734, -2.7578585147857666, -1.3727692365646362, 0.47124096751213074, -1.568411946296692, 2.3459932804107666, -3.286759614944458, 1.1852017641067505, 1.3422858715057373, -2.3955159187316895, -1.789595127105713, 0.6011936664581299, 1.056384801864624, 1.1498889923095703, -1.5004634857177734, 1.16756272315979, 3.105429172515869, -0.17501385509967804, -0.14933793246746063, 2.5483694076538086, 0.8615145683288574, 1.6876670122146606, 0.5134483575820923, -2.5971877574920654, -3.046739101409912, 0.1374513804912567, -2.7972536087036133, 0.9349873661994934, 0.515704333782196, 2.529127836227417, -2.4587490558624268, 3.8437728881835938, -0.2547647953033447, -0.8236842155456543, 0.976439356803894, -5.62108039855957, 0.7106724381446838, -0.09261423349380493, -3.7039310932159424, 0.6603437066078186, 1.9668023586273193, 2.3091752529144287, -1.028257966041565, 1.7095897197723389, -1.8808733224868774, 2.189023971557617, -0.04116911441087723, 0.8471060991287231, -3.2936463356018066, -3.11248517036438, 0.8087319731712341, -0.9589046835899353, 0.1926809549331665, 2.334707021713257, -2.0614991188049316, -2.931725263595581, 1.1612260341644287, -0.724202036857605, -0.041153836995363235, 3.317732095718384, 0.7187873721122742, 0.33201876282691956, -0.02640708163380623, -3.494537115097046, 0.37233230471611023, -1.953096866607666, 2.0625128746032715, -1.7636396884918213, -0.7648437023162842, -0.9685900211334229, -0.13881373405456543, -2.058821201324463, -0.5990070104598999, 1.3510241508483887, -0.3145005404949188, -0.08745851367712021, -0.04136093705892563, -0.340108722448349, 1.0997934341430664, 0.4975596070289612, 3.3654561042785645, 1.6726841926574707, -1.8130919933319092, 2.176027536392212, 0.1165289655327797, -1.5672504901885986, -0.8993650078773499, 1.4548494815826416, 0.7751302123069763, 3.1814818382263184, 1.3707635402679443, 0.5763702392578125, -0.17949937283992767, -5.398594379425049, -1.179758906364441, -0.007682941388338804, -1.732004165649414, 1.2538197040557861, -0.7504690885543823, -0.8726385831832886, -1.2120798826217651, 1.5983041524887085, 1.5066065788269043, 1.4324584007263184, 0.8119679093360901, -3.4285762310028076, 0.04306288808584213, -1.8023500442504883, 0.34781116247177124, -4.068665504455566, 2.849588394165039, -0.939285159111023, -0.8144155144691467, -0.8320783972740173, -0.02103951945900917, -0.861079752445221, -3.3507115840911865, 1.0747675895690918, 0.1962452530860901, -2.3854525089263916, 0.3179831802845001, -2.555030584335327, -0.7437794804573059, 0.17434418201446533, -0.975404679775238, -5.203242301940918, 0.469068318605423, -2.040269136428833, -1.7101356983184814, -0.1495371162891388, 0.7463397979736328, 2.3228600025177, 2.782810688018799, -1.4175373315811157, 0.48444944620132446, -2.8871235847473145, 0.828216552734375, 1.3126373291015625, -0.677251398563385], [0.8208054900169373, 1.485982894897461, -1.1674739122390747, 2.3562965393066406, -0.3713163733482361, -2.4351656436920166, 3.3854482173919678, 0.033073317259550095, 0.009608077816665173, 1.9489611387252808, -0.6389588117599487, -3.0691747665405273, -3.0945966243743896, 2.5470049381256104, 1.7966115474700928, 0.2649471163749695, 0.9558171033859253, -0.14279493689537048, 1.7788437604904175, 2.948087215423584, -2.649857997894287, 0.7560095191001892, -0.001152777229435742, 2.5299465656280518, -1.6137348413467407, 2.978904962539673, 2.702880620956421, 3.3273634910583496, 3.2269368171691895, 0.1953684538602829, -1.104988694190979, -0.5127103924751282, -0.9981159567832947, 0.16737931966781616, -0.40950846672058105, -0.976094126701355, 1.5874533653259277, -4.209072113037109, 0.11747458577156067, 0.8915889263153076, -0.5799463987350464, 3.304800271987915, 1.7548469305038452, -3.694504976272583, 1.6493909358978271, 1.9897890090942383, 1.9374492168426514, 3.4804890155792236, -1.9902980327606201, -1.8335809707641602, -0.37349483370780945, 4.445351600646973, 1.3040157556533813, 0.05591234564781189, -1.4348870515823364, 1.7022314071655273, -1.2891520261764526, 1.5125266313552856, 0.6263504028320312, 2.497922420501709, 0.19157235324382782, -0.6086392998695374, 0.48324689269065857, 0.9104258418083191, -0.10282061994075775, 0.33093562722206116, -3.387901544570923, 0.40125560760498047, 0.9865270256996155, -1.8225680589675903, 2.2207114696502686, 0.8215728402137756, 0.09223364293575287, 0.15142598748207092, 0.7307179570198059, 0.6631417274475098, -3.241604804992676, 3.5243356227874756, 4.767755031585693, -0.9066267013549805, -1.1364219188690186, -0.5752515196800232, -3.34733510017395, 0.4450075030326843, -1.3801360130310059, -1.1695818901062012, -2.5204546451568604, 3.9198553562164307, -1.4635485410690308, 3.6573703289031982, 0.9545671939849854, 1.2726976871490479, 1.257305383682251, -2.304743528366089, -0.28337907791137695, 0.8977203369140625, 0.35440170764923096, -0.6053231954574585, 0.3463748097419739, -1.5466140508651733, 3.149671792984009, -1.954872965812683, 1.9278554916381836, -2.039550542831421, -0.7557600736618042, 3.268893003463745, 1.4186663627624512, 0.3690465986728668, -0.9338318705558777, 1.2767424583435059, -2.4185545444488525, -0.01205313391983509, 0.4998975992202759, 3.3010525703430176, -1.9679774045944214, 0.7492455840110779, -0.525363564491272, 0.17508311569690704, 2.0002243518829346, -2.6166067123413086, 1.2321271896362305, -5.033235549926758, -1.8058617115020752, 1.2416057586669922, 3.1541383266448975, -2.160029649734497, 0.7683442831039429, -0.5449807643890381, 0.38065603375434875, 1.59287428855896, 1.031472086906433, 1.0502630472183228, -0.3061525523662567, -0.42536237835884094, 1.3785256147384644, -0.9631399512290955, 1.220123052597046, -2.090592861175537, 0.3963819146156311, -0.557272732257843, -1.7571868896484375, 1.10723876953125, 5.583907127380371, -0.08999264240264893, 0.07214491814374924, -3.3880228996276855, -0.4297258257865906, -3.0912587642669678, -1.6925357580184937, 1.1114073991775513, 2.1423919200897217, -2.9967429637908936, 2.823660373687744, 0.7116397619247437, 3.1367475986480713, 0.7647468447685242, 4.6880598068237305, 0.4192289710044861, 1.4579941034317017, -3.60426664352417, -4.047604084014893, 0.21631692349910736, -3.8444156646728516, 0.7895642518997192, -0.1177448108792305, 1.1870967149734497, 0.6325429677963257, 2.343898057937622, 0.5540331602096558, -3.6561286449432373, -0.6182858347892761, -1.451439380645752, 1.9542826414108276, -0.9873417019844055, -0.7997341752052307, -0.23258118331432343, 2.182663917541504, 1.6246697902679443, 0.0930204838514328, -1.7218828201293945, -0.08523547649383545, 2.247385025024414, 2.4964475631713867, -1.3506094217300415, -0.7245239019393921, -5.05641508102417, -1.5724064111709595, 0.6235754489898682, -1.9344245195388794, -1.0386520624160767, 0.25611764192581177, 0.5906004309654236, -0.18969988822937012, 0.8838008046150208, -0.5846073627471924, 1.806871771812439, -2.091413974761963, 2.1056406497955322, -0.5566352009773254, 2.8846395015716553, 2.8301143646240234, 1.0943074226379395, -1.2970997095108032, -2.0562870502471924, 2.3050074577331543, -3.0351085662841797, 0.5146418809890747, -1.3445218801498413, -3.6102278232574463, -0.25845927000045776, 1.2698148488998413, 1.0174164772033691, -3.7066400051116943, -2.720475673675537, 1.1465562582015991, 2.3048152923583984, -1.0086969137191772, 3.9658138751983643, 0.5813481211662292, 2.7528798580169678, 3.847522735595703, 1.5455347299575806, -1.9989782571792603, 3.1838998794555664, -3.130863666534424, 1.0962222814559937, -3.5951080322265625, -2.138068914413452, -0.9582982659339905, -4.998193264007568, -1.900356411933899, -3.0547454357147217, -1.2268487215042114, -0.100985087454319, 3.4400999546051025, 1.2960326671600342, 2.818236827850342, -0.38980937004089355, 1.896357774734497, -0.5901569128036499, 2.8815324306488037, -3.2974326610565186, -1.8397456407546997, 0.7191040515899658, -1.1740297079086304, -2.9723756313323975, 3.4128317832946777, -0.4762519598007202, 0.4344136416912079, -0.1353996992111206, -2.841233968734741, 1.525270938873291, 3.020263910293579, 0.6638076901435852, 2.2499403953552246, -0.4712691605091095, 3.3861005306243896, 0.9697146415710449, 0.44133642315864563, -2.751038074493408, -0.4268285632133484, -0.2836838662624359, 1.8058799505233765, 0.10483682155609131, 3.376025676727295, 1.897978663444519, -2.61651611328125, -3.700101137161255, 2.3500006198883057, -0.35995471477508545, -1.4010897874832153, 0.660312831401825, 0.49973151087760925, -0.5278663039207458, -1.8593515157699585, 1.513439655303955, -1.4207435846328735, -0.6189054250717163, -0.08076906949281693, -1.5819735527038574, -2.4853475093841553, 0.15713214874267578, -0.2786020338535309, -0.7813909649848938, 0.7326823472976685, 0.8322405219078064, -0.7170578837394714, -2.2041172981262207, -1.8954542875289917, 4.193966388702393, -3.692443609237671, 2.0259315967559814, 3.3263819217681885, 2.0856382846832275, -1.124604344367981, -2.2440977096557617, 0.020439838990569115, -0.8281952142715454, 0.663192093372345, -0.6030436754226685, -3.884538173675537, -3.319730758666992, -0.9091230034828186, -0.8996971845626831, 1.7501219511032104, -0.4279142916202545, -1.0159084796905518, 2.738570213317871, -4.342142581939697, -2.525862216949463, -0.3620997667312622, -2.8423073291778564, 0.8770952820777893, 0.7138965129852295, -2.873899459838867, -1.7030683755874634, 0.21424421668052673, -4.275601387023926, 0.8458556532859802, -1.3144621849060059, -2.1940877437591553, -1.3458441495895386, 1.4388940334320068, 1.4754706621170044, 0.2716469168663025, 1.2483867406845093, 0.0796956941485405, -1.943525791168213, -3.8842031955718994, -1.0064043998718262, -0.08483260124921799, -0.7760007381439209, 2.8985280990600586, -0.18076497316360474, 0.20149531960487366, 4.490838050842285, -0.29969972372055054, -1.7376385927200317, 2.837630033493042, 0.7108457684516907, 1.8288204669952393, -0.9396599531173706, 2.628383159637451, 2.1037118434906006, 1.6090933084487915, -4.043141841888428, 2.253439426422119, -2.3716437816619873, 2.972543239593506, 0.8465273380279541, -0.13707703351974487, 0.534602165222168, -3.5212290287017822, -2.4056003093719482, 3.6211795806884766, -0.9118343591690063, 4.193663597106934, -0.2650353014469147, 1.9185694456100464, -2.5859062671661377, 5.090565204620361, -1.9166315793991089, -4.394428253173828, 2.2991559505462646, 3.191787004470825, -0.2724186182022095, -0.8911412358283997, -2.1087634563446045, 0.004039913881570101, 2.468252182006836, 2.48724102973938, -1.0752580165863037, 0.5268924236297607, -0.2864961326122284, -0.35628488659858704, 2.898610830307007, -0.39036646485328674, 2.4026544094085693, -0.6587333679199219, -0.776698887348175, -0.4294188618659973, 4.28283166885376, -1.7615725994110107, 2.875401496887207, -0.49961957335472107, -0.29594504833221436, -3.9146625995635986, -0.02659853734076023, 1.3637077808380127, 1.4895362854003906, -2.005882501602173, 2.5339667797088623, -0.4476955831050873, -1.5300135612487793, -0.10864574462175369, 4.526445388793945, -1.0514453649520874, 1.2519469261169434, -2.0066726207733154, 0.7007366418838501, 0.8668246269226074, -4.3587751388549805, -1.2168874740600586, 1.300392985343933, -1.4927173852920532, 2.44346284866333, 0.4157271683216095, 0.6495949625968933, -1.4651052951812744, -0.2335040271282196, -2.081023931503296, -1.4274046421051025, 2.4162044525146484, 0.10197698324918747, 5.4697160720825195, -1.707194447517395, 0.3831828534603119, 2.9055938720703125, -1.0710877180099487, -1.2740470170974731, -0.19542010128498077, 0.2775363624095917, 1.278738021850586, -0.014417937025427818, 1.4467201232910156, -1.3791829347610474, -1.4602341651916504, 1.6622216701507568, -0.3780463635921478, 1.9833579063415527, -2.0933361053466797, 0.5680724382400513, 2.6226110458374023, -0.8353018164634705, -1.9462944269180298, -0.7001994252204895, -0.7573192715644836, 0.8389744758605957, -0.9468300938606262, -0.6286401748657227, -1.2035080194473267, 0.018734559416770935, 0.3009074628353119, -2.048624038696289, 0.3849978446960449, 0.9984314441680908, 0.5700390338897705, -1.1644340753555298, -1.830276608467102, -1.670745611190796, 1.558127760887146, 2.164802074432373, -1.9364506006240845, 2.5524938106536865, 0.5706832408905029, 1.5623561143875122, 2.0284039974212646, 2.9068000316619873, 2.205264091491699, 1.1783387660980225, -0.795508086681366, -0.9841580390930176, 3.2270731925964355, -1.7918463945388794, 1.691291332244873, -0.2293768972158432, 1.1118781566619873, 0.683792233467102, 0.20511923730373383, 2.3332371711730957, 1.1151890754699707, 1.3464921712875366, 0.1272284835577011, -1.5578744411468506, -1.6259652376174927, -0.662404477596283, 0.2394157350063324, 0.039104510098695755, 0.17328445613384247, -1.5405625104904175, 0.5469279289245605, 1.7274969816207886, 0.09864287078380585, -1.255017876625061, 2.1693522930145264, 0.7435703873634338, -0.43770113587379456, -1.1622005701065063, -0.28940680623054504, -2.882338762283325, 3.210411548614502, 4.634507179260254, 2.2049620151519775, 2.716139316558838, 2.2179601192474365, 0.34555673599243164, -1.6244087219238281, -1.1731373071670532, -1.3052908182144165, -2.6945741176605225, 0.18911102414131165, -1.6988903284072876, 1.9268344640731812, -0.06696903705596924, -1.0855591297149658, 0.34864458441734314, 1.245802402496338, -1.60818350315094, -2.2019710540771484, 2.639075994491577, -4.450867176055908, 1.862620234489441, 1.345075011253357, 0.3516567647457123, -0.14540782570838928, 0.09860780835151672, 1.5656919479370117, -2.5322422981262207, -3.0711710453033447, -1.365978717803955, -0.31359198689460754, 1.0146448612213135, 0.7181377410888672, -1.491482138633728, -2.0484347343444824, 0.9800313115119934, 3.1770081520080566, 4.685002326965332, -2.220017194747925, 2.5000083446502686, -2.3333191871643066, -0.1703723520040512, -0.5523743033409119, -1.6092404127120972, 0.7896935939788818, -0.22881852090358734, -0.893528401851654, -1.982417106628418, -0.6402393579483032, 2.935288190841675, 1.9461491107940674, 1.9394422769546509, -3.1463046073913574, 1.8498873710632324, -1.1592864990234375, -1.243506669998169, -0.5423545837402344, -3.2482988834381104, 2.9198713302612305, 0.13920648396015167, -2.8749406337738037, 0.9667440056800842, -2.441603660583496, -0.2712671458721161, -1.6534264087677002, 5.895667552947998, -1.4941058158874512, 0.20802417397499084, -3.606337308883667, 0.9912742376327515, -0.37132522463798523, -0.5265970826148987, 0.489042729139328, 0.13904350996017456, -0.6920621395111084, -1.1166096925735474, -4.288817405700684, -3.1283814907073975, -4.547039031982422, 1.2945122718811035, 1.8997281789779663, -1.360489845275879, 0.6505191922187805, -2.326310873031616, -0.6900070905685425, -1.8687753677368164, -0.7521630525588989, 5.060125350952148, 0.8360613584518433, 0.4619145095348358, 0.08920670300722122, -3.6856627464294434, 1.233862280845642, 1.1098520755767822, 3.0048131942749023, 0.33661597967147827, -4.321749687194824, -2.6012728214263916, 4.125213146209717, 1.1414369344711304, -2.5704102516174316, 3.9852380752563477, -0.8557943105697632, -1.545677661895752, -2.878034830093384, -2.152592658996582, -0.7772335410118103, -1.614037036895752, -5.366296291351318, -2.4102783203125, 5.606921672821045, 0.4096701145172119, -1.314614176750183, 0.6266101002693176, 3.3947701454162598, -1.8975121974945068, -1.2606480121612549, -2.1727054119110107, 2.500763416290283, 3.416146755218506, 0.015236429870128632, 0.5755997896194458, -1.4686636924743652, 0.5074066519737244, -2.450923204421997, 1.2079404592514038, 3.1759769916534424, 1.457024335861206, 1.7612624168395996, -1.1589263677597046, -2.506514072418213, 3.274458885192871, -1.9288917779922485, 1.227644443511963, 1.899394154548645, -0.5169441103935242, 0.45286107063293457, -1.0637354850769043, 2.149165391921997, 1.3481425046920776, 0.8139723539352417, 4.053201198577881, -0.9968770742416382, -0.13314607739448547, -2.376969337463379, 1.8709059953689575, 3.792562484741211, -0.08899720013141632, 0.5604704022407532, 0.9130247235298157, 0.18199314177036285, 1.0237177610397339, -0.29168498516082764, 3.01328706741333, -0.13102197647094727, 0.37336739897727966, -1.5416773557662964, -0.07234467566013336, -4.297425746917725, 5.597070693969727, 0.8456544280052185, 1.8650248050689697, 1.4427670240402222, -0.19177527725696564, -0.5098352432250977, -1.1287115812301636, 2.343985080718994, -0.8122983574867249, -1.8484106063842773, 0.5618869662284851, -2.6777079105377197, -1.6838419437408447, 1.729632019996643, 0.9324756264686584, 1.092840552330017, 2.623436450958252, -2.2491862773895264, -2.4026246070861816, 2.6530110836029053, -0.12503287196159363, 1.7017316818237305, 0.2607573866844177, -0.3943191170692444, -0.6180847883224487, 0.6147396564483643, -2.056436061859131, 1.5386855602264404, 1.4947752952575684, 0.8647419810295105, -3.985931634902954, -1.0110158920288086, 1.9120665788650513, 0.2543999254703522, 1.468683123588562, -0.47210583090782166, -2.7534050941467285, -1.2364851236343384, -0.5626029968261719, 0.48267605900764465, 0.19867335259914398, 1.272780418395996, 0.08231603354215622, -0.17605088651180267, 0.12234610319137573, -5.053099155426025, -0.6738533973693848, -0.7700792551040649, 2.015533447265625, -0.685701847076416, 2.0987064838409424, 1.3719487190246582, -2.5708703994750977, 1.160171389579773, 1.2650396823883057, 1.000761866569519, -0.9281056523323059, 3.0508975982666016, -3.297076463699341, 2.524658203125, 0.7482661604881287, 0.41786304116249084, 1.582026720046997, -0.4327351748943329, -3.788393497467041, -1.4030640125274658, 0.3449638783931732, -1.0422885417938232, -0.5744433403015137, 0.35368314385414124, -2.6108591556549072, 1.905194640159607, -0.6706509590148926, -0.16283084452152252, -0.0063119144178926945, 1.6233738660812378, -4.273477077484131, 2.4253363609313965, 2.430211305618286, -2.7201855182647705, 4.081953525543213, 1.000767469406128, 0.23229889571666718, -0.812349259853363, -1.741100788116455, 0.7099238038063049, -2.2436327934265137, -0.7419487237930298, -3.1104202270507812, -1.7329628467559814, 1.9951781034469604, 1.2714091539382935, -0.5340914726257324, 0.2593890130519867, 3.4424445629119873, -0.46898046135902405, 0.7935852408409119, 0.34871986508369446, 1.2083375453948975, 1.3662536144256592, -5.913821220397949, -1.2836933135986328, 5.1623711585998535, -0.35476765036582947, -3.419886589050293, 0.9511688947677612, -0.034730214625597, -0.5295001864433289, -1.92654550075531, 3.443141460418701, -1.2760556936264038, -0.48619529604911804, -2.248924970626831, 0.030441652983427048, 1.1841102838516235, 2.8916001319885254, 2.1740450859069824, -2.6611173152923584, -5.023282527923584, 1.2385276556015015, -3.4463112354278564, 2.889178514480591, 0.24337786436080933, -0.7239816784858704, -1.6385866403579712, 0.2056984156370163, -2.2987802028656006, -0.9147773385047913, -0.4202564060688019, -3.8068978786468506, 0.7169033288955688, 0.9006331562995911, -3.3891913890838623, 3.808079242706299, 0.14200666546821594, 1.6243153810501099, -0.977640688419342, 1.7066644430160522, 0.5184857845306396, 5.901230335235596, -0.7382708191871643, 0.22453747689723969, -4.307531833648682, -1.3178071975708008, -4.3771162033081055, 0.8090757131576538, -0.20565436780452728, -0.5961493849754333, 0.29116091132164, 0.2438770979642868, 0.1129913330078125, 0.254698783159256, 0.09755589067935944, 0.36875662207603455, -0.6049580574035645, -0.2796943783760071, -0.7760155200958252, -2.6998391151428223, -1.9789621829986572, -1.9913653135299683, 0.7490496039390564, -0.32289233803749084, 1.13572096824646, 1.1057958602905273, 0.6999987959861755, 3.2063052654266357, 3.1325814723968506, -1.3949122428894043, -0.05021539703011513, -0.6032409071922302, 0.7303917407989502, -1.4365609884262085, 2.0615878105163574, -0.07381801307201385, 3.4312942028045654, 0.008636511862277985, 1.929583191871643, 0.334615021944046, -0.03833339735865593, 0.7608461380004883, 0.3906073272228241, -3.1371660232543945, -0.2668958306312561, -0.7733771800994873, -1.0322185754776, -0.4114893078804016, 1.1908820867538452, 1.1543959379196167, 0.7006641030311584, 1.9180289506912231, 4.521454811096191, 0.8628684878349304, 1.9337879419326782, 2.104323387145996, -0.7840303778648376, -0.22982080280780792, 2.557715892791748, -1.4228230714797974, -1.19547438621521, -0.967224657535553, 0.7294256687164307, 2.9491918087005615, 4.209010124206543, 3.3228893280029297, -0.05163665860891342, -0.6763051152229309, -1.9813653230667114, -3.8779807090759277, -0.35768964886665344, -3.321521520614624, 0.25037866830825806, 0.823290228843689, -0.581277072429657, 3.6145341396331787, 0.7646591663360596, -1.0084768533706665, 0.9662933945655823, -3.669553756713867, -1.0586541891098022, 0.036220382899045944, -1.7345919609069824, 0.18846623599529266, 1.287258267402649, -0.47566401958465576, -1.3952327966690063, 0.39797866344451904, 0.8105846047401428, 2.92756724357605, -3.312342882156372, 5.193816184997559, 0.6332179307937622, -3.03776478767395, 1.4484387636184692, 2.063307762145996, -1.7227661609649658, -0.5335474610328674, -1.931503415107727, -0.858509361743927, 0.9128767848014832, 1.1352362632751465, -0.03593108803033829, -1.6729737520217896, -0.5527230501174927, 0.480768084526062, -2.2004141807556152, -2.2302188873291016, 1.4357006549835205, 1.169496774673462, 2.146406412124634, -3.699079990386963, 4.002008438110352, 3.0771749019622803, -2.685314178466797, 0.1884763091802597, -0.9468103647232056, -1.2898672819137573, 0.2438676655292511, -3.5064127445220947, -0.9932069778442383, 2.853053569793701, 0.02358408086001873, -1.3038896322250366, -0.5653679370880127, -2.170175552368164, 4.405187606811523, -1.4267336130142212, 0.7710304856300354, 1.02073073387146, 0.9072826504707336, -1.010474443435669, -2.4093427658081055, 1.6027027368545532, 0.48615145683288574, -2.638646125793457, 1.658495545387268, 2.1911420822143555, 3.2886672019958496, -1.136380910873413, -3.475527763366699, -1.2279397249221802, 0.2142164707183838, 1.6772286891937256, 1.6932082176208496, 0.031072543933987617, -2.100372314453125, 1.7727935314178467, 2.0246059894561768, 2.795383930206299, -0.47282958030700684, 0.29307490587234497, 0.3593885898590088, 4.95263147354126, -4.511855125427246, 0.5415971875190735, -5.2484588623046875, -1.1570487022399902, -1.5896803140640259, 1.41410493850708, 2.1023924350738525, -2.0717780590057373, 3.124586343765259, 3.07593035697937, -0.07962918281555176, -1.3538010120391846, -0.2248135209083557, -3.202021837234497, 1.9074370861053467, -4.952643871307373, 1.178293228149414, 2.0123348236083984, 1.1785999536514282, 0.23649394512176514, -1.1868056058883667, -1.698125958442688, -0.4133884310722351, -1.4797688722610474, -1.5668914318084717, 3.767526865005493, 0.5143098831176758, 2.1692256927490234, 1.4665011167526245, 3.3242907524108887, -0.983708918094635, -2.254531145095825, -1.1560083627700806, 2.648815155029297, -2.601513147354126, -2.916989803314209, -0.5960361361503601, -0.9481489062309265, -1.5748095512390137, 2.4483349323272705, 1.0652066469192505, 0.010039866901934147, 3.5716419219970703, 1.8769503831863403, 0.2520236074924469, -0.04941389709711075, -0.8617636561393738, 2.580475330352783, -1.2538235187530518, -1.5627702474594116, 1.765364170074463, 1.6484240293502808, -2.3280587196350098, 3.452775001525879, -0.9203770756721497, 3.186570882797241, -3.543825149536133, -1.6324796676635742, -4.12746000289917, -1.9350550174713135, 0.0860019102692604, 0.9448962807655334, 1.6130443811416626, 0.3102128207683563, 0.7614718675613403, -1.6057649850845337, 1.9086416959762573, -0.0161453690379858, -1.356256365776062, -0.4460832178592682, -2.968778610229492, -2.286322593688965, -1.7578392028808594, -3.0587334632873535, 3.1062545776367188, -1.904186487197876, 1.9650177955627441, 3.1935741901397705, -1.325181484222412, -1.2606403827667236, 3.3087265491485596, 4.039201259613037, 0.4113241136074066, 2.6724627017974854, -2.0939855575561523, 1.9701515436172485, -3.5034258365631104, -2.560197591781616, 2.1534953117370605, -4.337438583374023, 2.3683664798736572, -1.785839557647705, -0.746609091758728, 0.7815219759941101, -0.5044118762016296, -2.007798194885254, -2.6096463203430176, 2.2046122550964355, 2.8143088817596436, -0.7921352982521057, -1.0526924133300781, -3.919583559036255, 2.038846731185913, 0.6195494532585144, 1.1482924222946167, 1.4828112125396729, -2.063844680786133, 0.2751641571521759, -0.9697672724723816, 2.675729751586914, -0.15736210346221924, 2.50386118888855, 1.9917316436767578, 1.3170207738876343, -1.5662018060684204, -0.4843362271785736, 2.3604023456573486, 2.09352445602417, 1.310862421989441, -1.5721032619476318, 0.4978844225406647, 0.1711750030517578, 0.5255776047706604, 0.21109405159950256, 1.7384965419769287, -0.8813506960868835, 1.4546250104904175, 0.5259778499603271, -0.9229434132575989, -1.6283574104309082, 2.5483670234680176, -0.6676025986671448, 0.6136820912361145, 4.399621963500977, -0.46396973729133606, 2.7823245525360107, 0.19744616746902466, 0.6656218767166138, -0.1994263082742691, 1.8744173049926758, -1.5371475219726562, -0.7485542297363281, 2.047172784805298, 5.200779914855957, -0.39513731002807617, -3.462207555770874, 0.4577587842941284, -0.9571805000305176, 1.910805583000183, 1.5561723709106445, -5.187231063842773, 1.2707841396331787, -1.7415727376937866, 1.1471619606018066, 0.29234251379966736, 1.96240234375, -0.502646267414093, 1.8793001174926758, 1.7853046655654907, 1.8822963237762451, 0.7510861158370972, -0.7273896336555481, -0.06559879332780838, 0.46527230739593506, -0.19545675814151764, -0.4296906888484955, -1.6728363037109375, 1.3833328485488892, 2.563997507095337, -2.915753126144409, 0.7167550325393677, 1.5791659355163574, 0.16164088249206543, -0.22148717939853668, 2.0517008304595947, 1.2843029499053955, -1.1445136070251465, -3.5207669734954834, -2.779691457748413, -2.9814488887786865, 1.2289081811904907, -0.0016823834739625454, 0.3958386778831482, 0.343375563621521, 0.23174187541007996, -1.164923906326294, -1.6026958227157593, 3.221301555633545, 1.632130742073059, 0.4199801981449127, 2.323094606399536, -2.4912612438201904, 1.7853291034698486, 0.052631083875894547, -2.8166067600250244, -0.6880027651786804, 1.3898974657058716, 1.2679553031921387, 1.5028634071350098, -2.7283096313476562, -0.3837651014328003, 0.4610983431339264, -0.38567987084388733, 1.0632668733596802, -2.162853240966797, -1.4904887676239014, 1.9734861850738525, 1.0326932668685913, 0.3680283725261688, -0.2297665923833847, 1.9985384941101074, -1.7999178171157837, 2.6043684482574463, -1.9074397087097168, -0.03344164043664932, 2.3451151847839355, 0.6630027294158936, -1.328844666481018, 1.4150640964508057, 5.452709674835205, 2.0526952743530273, -1.8451284170150757, -1.955356478691101, -2.0689759254455566, -0.35779869556427, -1.0584139823913574, -6.641152858734131, -0.6987521052360535, -1.0837918519973755, -1.1242103576660156, 1.2200822830200195, 1.9068673849105835, 2.8034210205078125, -0.7757505774497986, -1.841874599456787, 1.7051502466201782, 0.2653059959411621, 2.7749483585357666, -0.2492751032114029, -0.4153074622154236, 0.22359731793403625, 0.09275408089160919, 1.754629373550415, 2.9871137142181396, -2.550076723098755, 1.444356083869934, 0.5123148560523987, 1.4653798341751099, 3.044196367263794, -1.6539918184280396, -4.500839710235596, 0.9714558720588684, 0.8270797729492188, 0.2831050157546997, -1.0403257608413696, 0.11078779399394989, -2.844188690185547, 0.9604288935661316, 0.779970645904541, 0.4013776183128357, 2.7561213970184326, -1.5817891359329224, -2.086524486541748, -0.17196029424667358, 2.8002302646636963, -0.6751830577850342, -3.032109022140503, -2.057559013366699, 1.2531867027282715, -3.5331928730010986, -0.6042126417160034, 1.0639467239379883, 4.336789131164551, 0.8006736040115356, 0.09144929051399231, -1.496241807937622, -2.8812384605407715, -1.5717134475708008, -2.298948049545288, 0.4920525550842285, -3.240774393081665, -0.3890782594680786, -0.5659043192863464, -0.39823445677757263, 0.1332269161939621, 1.6776182651519775, -2.987545967102051, -2.988633394241333, 1.206692099571228, 1.8400988578796387, 0.9340807795524597, -1.304082989692688, 0.3462028205394745, -1.9234535694122314, -0.32644182443618774, -3.0215985774993896, 0.9373925924301147, -1.0489104986190796, 1.4474718570709229, -2.336024045944214, -0.49294552206993103, 0.6306353807449341, -1.6620463132858276, -1.4104584455490112, 0.4290265738964081, 0.6269640326499939, 0.21349066495895386, 0.3988206684589386, -0.1946863830089569, -1.3869198560714722, -0.6301116943359375, -0.5973270535469055, 0.5687021017074585, -1.8277233839035034, 1.3896162509918213, 0.08336882293224335, -0.10624773800373077, 2.943556308746338, 2.6585967540740967, 1.9004058837890625, -0.7190269827842712, 1.7381802797317505, 2.2831263542175293, -0.41071534156799316, -1.4257957935333252, -2.0689845085144043, -4.316508769989014, 0.3487439453601837, 0.7422134280204773, -0.06316764652729034, -0.5219526290893555, 3.2863082885742188, -2.0627682209014893, 1.0079327821731567, 0.597408652305603, -0.38630905747413635, -0.3864623010158539, 0.23051044344902039, 1.4383304119110107, 0.20027242600917816, 0.2762020528316498, -3.086749792098999, -2.76733136177063, 0.3351251184940338, 1.0183789730072021, -0.011307465843856335, 1.0500379800796509, -1.9319170713424683, 1.8356963396072388, 1.357273817062378, 3.376919984817505, 0.5934815406799316, 1.1283314228057861, -1.1782251596450806, -0.5977501273155212, -1.766008734703064, -2.25758957862854, 4.060823917388916, -1.4214178323745728, 3.250178575515747, 2.4766972064971924, 1.9099736213684082, -0.36324766278266907, 1.7827332019805908, 2.633310556411743, -3.820521116256714, 0.8935083150863647, -2.2020256519317627, -2.571168899536133, 1.4736002683639526, 4.098513126373291, 3.5150868892669678, -2.8404033184051514, -3.340374708175659, 1.117559790611267, 2.313181161880493, -3.3114216327667236, 1.3624145984649658, -2.0978052616119385, -0.5916585922241211, -0.8641067743301392, -0.42415520548820496, 3.7221627235412598, -0.7342109680175781, 0.8199198246002197, 1.9224549531936646, -0.1412641406059265, 2.165066719055176, -0.8519411087036133, -1.2236381769180298, 2.430715799331665, -1.18205726146698, 1.0293571949005127, -2.0564422607421875, 0.4020669162273407, 1.590407133102417, -1.9048943519592285, 1.4640944004058838, 2.776172637939453, -1.073262095451355, 0.4915509521961212, -3.1322996616363525, 1.8799877166748047, 4.863394737243652, -0.5501077771186829, -0.3124082088470459, -3.818925380706787, 4.061407566070557, 4.423552513122559, -0.8978762030601501, -0.9836332201957703, 0.3346216082572937, 1.5650314092636108, 0.16734878718852997, -0.8460046052932739, 1.0648388862609863, 1.8660959005355835, 2.107194185256958, -1.4781978130340576, 2.6702544689178467, 0.3655170798301697, 2.2895641326904297, 5.634376525878906, -1.4534437656402588, 0.33012092113494873, 1.4853627681732178, -2.1778407096862793, -3.4340362548828125, 3.389226198196411, -2.7655646800994873, 0.4270246922969818, -0.25492140650749207, -0.9741711020469666, 1.1906483173370361, -0.8756232261657715, 0.7143182158470154, 0.8883019089698792, -5.200746536254883, 0.013180595822632313, -0.8856637477874756, -2.153514862060547, -0.2704028785228729, -2.4496195316314697, 1.6874064207077026, 2.986159563064575, 0.9441795349121094, 0.6729028224945068, 0.9579904079437256, -2.7066574096679688, 3.076720714569092, -0.4653294086456299, -6.2667670249938965, -0.2553556263446808, -1.2701045274734497, 0.42274177074432373, 2.3488621711730957, -1.4466619491577148, 0.15885649621486664, 1.0399850606918335, -0.07046715170145035, -1.6073898077011108, -3.983321189880371, -1.2706923484802246, 1.6698179244995117, 1.125386357307434, 0.6006441712379456, 0.8867988586425781, -2.1111643314361572, 3.163700819015503, 2.3707292079925537, 0.6431095600128174, -1.0656967163085938, -0.2602088749408722, -2.0076351165771484, 0.9783147573471069, 1.8675423860549927, -1.9413138628005981, -0.6463962197303772, 1.750540852546692, 3.1430695056915283, 1.2248122692108154, -1.6871815919876099, 1.3613810539245605, -0.5292273759841919, 1.704993724822998, 2.2330853939056396, 5.734604835510254, -2.63883376121521, -3.6275510787963867, -1.7551733255386353, -2.150529384613037, -3.939335584640503, -2.443305730819702, -2.603865385055542, 1.9074747562408447, -0.908276379108429, -2.8260414600372314, 0.7149026989936829, -4.045986652374268, 0.5602176785469055, -1.5049753189086914, 1.1704864501953125, -2.835800886154175, 2.714219331741333, 1.4042912721633911, -1.8277857303619385, 0.37483739852905273, -3.0201778411865234, -2.798086404800415, -1.2651549577713013, 1.9012868404388428, 5.448891639709473, -1.4191641807556152, 0.7295227646827698, -0.492383748292923, -3.428588628768921, -3.3181636333465576, 1.1503303050994873, 0.9135470986366272, -1.317742109298706, -0.9418251514434814, -0.1009046882390976, -0.4413451850414276, -1.5642164945602417, -0.30939754843711853, 2.3376426696777344, -1.0498064756393433, -0.5713521242141724, 0.9509369134902954, 2.213437080383301, 0.3448783755302429, -1.2681787014007568, -1.5079846382141113, -3.4304373264312744, -3.7512028217315674, 1.5223928689956665, 2.7390992641448975, 0.988291323184967, -1.9246751070022583, -2.133653163909912, -1.8948978185653687, -0.4087989628314972, -1.2221181392669678, -1.5135741233825684, 1.4547913074493408, -1.5394139289855957, -1.3924458026885986, -0.7396057844161987, -1.7511341571807861, 3.416020154953003, 1.3953129053115845, 1.10354483127594, -0.7597717046737671, -2.8788230419158936, -0.035461846739053726, 1.516940951347351, -0.8476712703704834, 2.8774006366729736, 1.5324333906173706, 1.0053850412368774, -1.147665023803711, -1.872072696685791, -3.635058641433716, 3.314650297164917, 1.6064531803131104, 3.404176712036133, -0.8236781358718872, 1.9070035219192505, -0.388004869222641, -0.38121530413627625, 2.737436532974243, 1.9276660680770874, 0.4810612201690674, 0.8074643015861511, -2.6972548961639404, 3.168386459350586, 2.6780214309692383, -2.6883137226104736, -2.0880367755889893, 3.444458246231079, 2.169811248779297, 0.8557961583137512, -2.3809924125671387, -0.7925889492034912, -1.4265823364257812, 2.4475560188293457, -2.0253777503967285, -1.1985892057418823, 1.5183889865875244, -2.0818963050842285, -5.037276744842529, -1.5354511737823486, -1.4317951202392578, -3.1442227363586426, -1.782933235168457, 2.7663519382476807, 2.735905885696411, 2.328920602798462, -2.602809190750122, 3.2382972240448, -0.7644627094268799, -1.5046221017837524, -3.3907968997955322, -0.9545027613639832, -0.9120956063270569, 0.2078457474708557, -0.8580292463302612, -1.1461561918258667, -0.4508645236492157, -0.11776689440011978, 3.1561439037323, -0.8138713240623474, -3.4598281383514404, -0.12584541738033295, 0.23481445014476776, -1.5988214015960693, -0.6311424970626831, 1.0211247205734253, 1.4299850463867188, 1.1098425388336182, -1.060490369796753, -0.3341635763645172, -0.027805034071207047, -0.02594851516187191, -2.8310558795928955, 0.041853345930576324, -4.387540817260742, -0.5985747575759888, -1.7351062297821045, 0.07293722033500671, -3.2367429733276367, 5.353058338165283, 1.3255064487457275, -2.8730876445770264, -2.480414628982544, -1.8750308752059937, 1.3658320903778076, 0.12814214825630188, 2.1207494735717773, 3.9886186122894287, -0.17272920906543732, -1.7521178722381592, 4.432264804840088, -0.11738479137420654, 0.6845928430557251, -0.9150119423866272, 1.2008694410324097, -0.578309953212738, 5.8058319091796875, 3.580695867538452, 1.7170064449310303, 0.8565990924835205, 0.9779075384140015, 1.4257012605667114, -2.676154375076294, 0.3747130036354065, -1.397386908531189, 0.6496702432632446, 0.6438401341438293, -3.4240572452545166, 0.8528202772140503, 1.4472867250442505, 5.826412200927734, 0.29537779092788696, 1.6121309995651245, -2.673729658126831, 0.3012167811393738, -1.4956388473510742, -1.3902769088745117, 4.897520542144775, -0.5645079016685486, -0.7729863524436951, 6.6137309074401855, 2.9324662685394287, 5.567356109619141, -1.6060484647750854, -0.9599683284759521, -0.5628358721733093, 0.5834738612174988, 0.6999306082725525, 0.6131184697151184, 4.323384761810303, -2.970719814300537, -0.20790033042430878, -0.8554284572601318, -1.9743832349777222, -1.7947057485580444, 0.5919585227966309, 3.2164134979248047, 1.0473463535308838, 1.9099828004837036, 1.8592326641082764, 0.6882199645042419, -0.9591273665428162, -0.9051870107650757, -1.8000234365463257, 1.2902392148971558, -0.9252557158470154, -4.359439373016357, -2.8475587368011475, 0.12662479281425476, 0.8770329356193542, 1.5990535020828247, -0.06295087188482285, -0.2234140783548355, -0.8815596699714661, 2.1753013134002686, -1.149670124053955, 3.579681396484375, -0.1039937287569046, 0.5721879005432129, 0.927685558795929, -1.8433947563171387, 0.23975400626659393, -1.2148693799972534, 0.6394066214561462, 3.3860223293304443, -0.09251132607460022, 1.5591031312942505, -4.8077168464660645, -0.03736032545566559, -1.0067055225372314, -0.8208637833595276, -5.897048473358154, -1.152867078781128, -1.5221071243286133, 2.7556943893432617, -1.2247843742370605, 1.9084405899047852, -0.08451788127422333, -1.893136978149414, 1.2932095527648926, -3.814978837966919, -3.582836866378784, -4.097428798675537, -1.1447083950042725, 0.2950662672519684, -2.3872077465057373, 2.079087495803833, -1.7234872579574585, 1.3683074712753296, 1.965307354927063, -2.967816114425659, -2.1422977447509766, -0.3378700017929077, 0.07412842661142349, 1.7128093242645264, 0.9091756343841553, 0.9143849611282349, 3.400981903076172, 0.2870022654533386, 2.115255355834961, 1.203143835067749, 2.3471839427948, 3.851567029953003, -0.5891727805137634, -0.32413995265960693, -0.2817370593547821, 1.907407283782959, -0.0025845731142908335, -0.5294560790061951, 2.381263017654419, 2.4921486377716064, -0.18228253722190857, -2.2875213623046875, -0.13459372520446777, 0.8203269839286804, -1.9701694250106812, 0.5383040904998779, -1.3276476860046387, -0.9805245995521545, 0.4762120842933655, -2.593809127807617, -1.713037371635437, 2.355243444442749, -3.6144015789031982, -1.3942948579788208, -0.19790126383304596, 1.4440499544143677, -0.8957102298736572, -1.06422758102417, -1.5748870372772217, -0.9825806617736816, 1.2738407850265503, -1.381819486618042, -1.7565110921859741, 1.150032639503479, 1.7731726169586182, 1.0473538637161255, 1.0754830837249756, 0.7880573868751526, 0.41997233033180237, -0.23979410529136658, -0.0038818025495857, -4.622768402099609, -1.1950883865356445, 6.772830009460449, -0.7739112377166748, -0.6349379420280457, -0.5496405363082886, 3.21671724319458, 4.529236793518066, -3.05381441116333, -0.06564393639564514, -1.4945414066314697, -0.04374200850725174, -1.637009859085083, 1.2286837100982666, -1.5123414993286133, -0.5801254510879517, -0.27956056594848633, -0.40364012122154236, 0.7358008027076721, -1.1512455940246582, 0.15171529352664948, -0.06680337339639664, -1.3415391445159912, -1.975582480430603, -1.5746092796325684, -1.5493580102920532, -0.9292075634002686, 2.3650565147399902, -0.22544416785240173, 0.9033680558204651, 2.7248055934906006, 0.5313891172409058, -1.2239950895309448, 3.0652244091033936, -3.112598419189453, -1.1655826568603516, 0.3426567316055298, -0.43498215079307556, -0.6054113507270813, 1.0402657985687256, 1.02685546875, 1.8379781246185303, 1.142290711402893, -5.498292446136475, -1.6209437847137451, -3.9473342895507812, -0.6664806008338928, -0.6499231457710266, 0.9906331300735474, -1.2212343215942383, 0.5526634454727173, 0.10780664533376694, 0.7324598431587219, -1.800815463066101, -0.6235808730125427, 0.32724976539611816, 0.29822754859924316, -0.7733608484268188, 1.61370050907135, 0.9373789429664612, -0.1653113216161728, -1.4760901927947998, -0.7929192185401917, 0.8196182250976562, 3.5255675315856934, -2.143784284591675, -0.7479240298271179, -0.029643189162015915, 5.444549560546875, -1.6739681959152222, -0.39007294178009033, -1.0602564811706543, -4.355772018432617, -0.8749299049377441, 2.0169312953948975, -3.8385822772979736, -0.7524886727333069, -1.1245396137237549, 1.4385937452316284, -2.4926793575286865, -2.4028396606445312, -1.2973887920379639, -1.1850005388259888, -0.10440120100975037, 2.5827736854553223, 0.15707312524318695, -2.587278127670288, 0.41868361830711365, 1.1355923414230347, 1.4654160737991333, 4.386928081512451, 2.8649449348449707, 1.515413522720337, 1.0949764251708984, -3.1210498809814453, 2.193453788757324, -1.5506292581558228, 0.12658527493476868, 2.818264961242676, 3.9984493255615234, -2.439462900161743, -2.715759754180908, 0.24521403014659882, 1.3788875341415405, -0.6509278416633606, 2.107333183288574, -0.8934831023216248, -0.8173350691795349, 1.667346477508545, -1.7444992065429688, -1.6300119161605835, -0.28450173139572144, 2.082035541534424, 2.9324026107788086, -2.3111231327056885, 0.41324588656425476, 0.9444272518157959, -3.9293336868286133, -0.5393556356430054, 3.054920196533203, -2.2332191467285156, 2.2888035774230957, -5.141615867614746, 2.0122411251068115, -2.724320650100708, -4.115920066833496, -0.16712485253810883, -3.9396536350250244, 1.677404522895813, -1.4203994274139404, 0.43682804703712463, -2.2784342765808105, -2.10337233543396, -1.6841787099838257, 0.24870696663856506, -2.2448923587799072, 3.427536964416504, -1.27932870388031, 0.4027705192565918, 2.1552176475524902, 3.957441568374634, 0.251160591840744, -5.528078556060791, 2.6026134490966797, -2.198352098464966, 0.3917846083641052, -1.7112557888031006, 0.17209547758102417, -0.6102668642997742, 2.894733190536499, -1.00322425365448, -0.22863587737083435, -4.527499675750732, 0.7276379466056824, 1.18447744846344, -1.1965670585632324, -1.4205517768859863, 0.8544148206710815, -1.4731804132461548, 3.465144395828247, -3.5997183322906494, 1.1324739456176758, 2.0035715103149414, -0.3491728603839874, 1.9747891426086426, 2.48186993598938, -4.326330184936523, -1.5454870462417603, 2.8301172256469727, -1.2437795400619507, 0.5598584413528442, -3.5418713092803955, 0.2462940514087677, 0.08942839503288269, 2.4105207920074463, -2.260300397872925, -0.5414140224456787, 2.1638970375061035, -2.381053924560547, 3.256620407104492, -2.513784646987915, -1.3372795581817627, 1.7255760431289673, 0.39054980874061584, -2.0139529705047607, -0.5505139231681824, 0.09438744932413101, -1.6750589609146118, -3.59321928024292, 1.0281463861465454, 0.23603424429893494, 0.9548700451850891, 1.169005274772644, 0.9642980098724365, 2.2438511848449707, -0.35049358010292053, -1.1087281703948975, -0.21100659668445587, 0.13657303154468536, -0.16352784633636475, -0.5568654537200928, -1.0078647136688232, 1.09999680519104, -1.3106813430786133, -0.48381325602531433, 0.02949790097773075, -1.1709256172180176, 1.0452553033828735, -0.8114388585090637, -0.8969480395317078, -0.8646292090415955, 0.07678382098674774, -1.0438226461410522, -2.9416897296905518, 3.0498557090759277, 2.3501110076904297, 1.8868452310562134, -0.8830174803733826, 3.0048108100891113, 2.4358439445495605, -3.380199670791626, -2.2415382862091064, -1.071083903312683, 1.9591962099075317, -0.36377081274986267, 1.3589726686477661, -0.5725034475326538, 1.7318642139434814, -1.3818390369415283, -1.2325161695480347, -2.3449010848999023, 1.4142385721206665, 0.9368605613708496, 2.9685134887695312, 1.5342148542404175, -2.2178642749786377, 2.473727226257324, 0.28052404522895813, -1.6976008415222168, 4.0336737632751465, -0.8220781683921814, 0.18734802305698395, -0.8632181286811829, 4.179580211639404, -0.8311086893081665, 2.7084150314331055, 1.3198624849319458, -0.06859348714351654, 0.23316021263599396, 0.22208471596240997, 2.052628755569458, 2.3608791828155518, 0.12260275334119797, 0.6497810482978821, -0.15933679044246674, 1.355016827583313, 4.685457706451416, 0.21090173721313477, -0.63290935754776, 2.5190505981445312, -1.664023756980896, -0.4161338806152344, -1.9578503370285034, 0.6324215531349182, 3.463395357131958, -3.5823469161987305, 1.5582355260849, 2.8128182888031006, 0.7346333861351013, 0.31841540336608887, -0.8744408488273621, 3.4712634086608887, 0.6415289044380188, 0.8531042337417603, -3.7568342685699463, -0.9481786489486694, -0.813606858253479, -2.172945261001587, -0.7307519316673279, -2.6722512245178223, -1.4584063291549683, -1.2486116886138916, -0.6180440187454224, 1.3178520202636719, 0.3124677240848541, 1.3643513917922974, -1.2152488231658936, 2.760347366333008, 0.5197736620903015, -1.3361599445343018, -4.110218048095703, -0.49053338170051575, -1.439136266708374, 0.32303306460380554, -1.2064827680587769, 0.4248603880405426, 3.7303271293640137, -0.3637304902076721, 1.0700187683105469, -0.18706956505775452, -0.6563061475753784, 1.2823991775512695, 0.5763631463050842, -0.8324190378189087, 0.17612400650978088, -0.12087825685739517, 1.409859299659729, -0.05693753436207771, 2.690857410430908, -0.7123631238937378, 0.77955561876297, 0.01617000810801983, 3.0331947803497314, -4.0669403076171875, 0.6294606328010559, -2.6890869140625, -0.34864529967308044, 1.398250699043274, -0.3807055354118347, 1.994111180305481, 0.6200864911079407, 2.4107158184051514, 1.8303974866867065, -0.6049615740776062, -1.4827544689178467, 0.39922773838043213, -5.3870368003845215, -0.6745365858078003, 0.6478664875030518, -1.717769742012024, -0.7695151567459106, -1.3641371726989746, 1.565210223197937, -1.179098129272461, 2.326237201690674, -0.3163873255252838, 1.0130031108856201, 0.25580960512161255, -0.5370407700538635, 3.244809865951538, -1.0408620834350586, -1.81144118309021, -0.5810533165931702, 1.6974698305130005, 4.765541076660156, -2.502685308456421, -2.896463632583618, 1.5234569311141968, 0.3478996157646179, 0.23437856137752533, -1.5749046802520752, -2.632046699523926, 1.4813817739486694, -3.7838990688323975, 0.8095443844795227, 3.9642891883850098, -0.9985222816467285, 2.2435989379882812, 0.24613066017627716, 0.8975090980529785, 0.19892297685146332, -0.4042457640171051, -2.8732402324676514, -2.1884639263153076, -2.1214599609375, 3.106846332550049, -2.0367772579193115, 0.9678217768669128, 0.028842302039265633, 0.599297285079956, -5.814870357513428, 1.201507568359375, 3.842714786529541, 1.008013129234314, -1.3470511436462402, 0.19777923822402954, 0.1772051751613617, -3.0850484371185303, 2.8156514167785645, -0.5281120538711548, -1.0177563428878784, -0.7289393544197083, 1.647355318069458, -0.19049662351608276, -2.9250590801239014, -3.2896292209625244, 0.9885858297348022, 1.339025855064392, 0.30122894048690796, 0.8106754422187805, -0.3866733908653259, 0.9563462138175964, -1.5243955850601196, 0.6403137445449829, 1.1367050409317017, 3.5063560009002686, -4.468609809875488, -2.79256534576416, -2.4001779556274414, 2.389827251434326, 2.913741111755371, 2.6240856647491455, -1.4625623226165771, 0.7069383859634399, 2.158463478088379, 4.113163948059082, -0.5211775302886963, -0.40569359064102173, -2.7529237270355225, 1.042675256729126, 1.8612674474716187, 1.1468268632888794, -0.16690659523010254, 1.1888903379440308, -4.501358509063721, -0.3075026273727417, -3.775466203689575, 1.1042790412902832, -2.274876832962036, -3.093494176864624, -1.0857528448104858, 0.11407441645860672, -4.780210971832275, -4.296687126159668, -1.3779079914093018, -0.26182669401168823, -0.5554296970367432, 1.4082496166229248, -1.6654777526855469, -0.8640620112419128, -1.4283262491226196, 0.6343605518341064, 0.002573765814304352, -1.3833147287368774, -3.504570484161377, 0.48555949330329895, 0.39443621039390564, -2.187715530395508, 0.6056734323501587, -0.9674614071846008, 4.856010913848877, 2.9686877727508545, 0.45535412430763245, 2.2317166328430176, 0.9972611665725708, -0.19825424253940582, 0.42508989572525024, -2.6600353717803955, -2.015657663345337, 4.917187213897705, 0.9203116297721863, -0.595613420009613, 2.9711928367614746, 2.2722079753875732, 0.6028997898101807, -1.3456335067749023, -0.36657071113586426, 0.4277781546115875, -2.9818179607391357, -5.094021320343018, 1.6486828327178955, -4.298459529876709, -2.6501879692077637, 1.267755389213562, -3.812474489212036, -2.372225046157837, 0.9913545846939087, -0.5045484900474548, -2.4107351303100586, 2.124346971511841, -2.7633960247039795, 0.9276198148727417, -1.5849276781082153, -0.9465722441673279, -1.1882357597351074, -1.1049755811691284, -3.6491217613220215, -3.0872066020965576, -1.8798730373382568, -3.585710048675537, -2.3490068912506104, -0.4678472876548767, 1.1377384662628174, -0.6383145451545715, -1.222294807434082, 1.307371973991394, -1.6842389106750488, -0.6374592185020447, 1.7186594009399414, 0.0578182227909565, 1.0586594343185425, -2.3584370613098145, -1.7918497323989868, -0.34152302145957947, -0.3602524995803833, 1.8928877115249634, -1.909665822982788, 1.1777300834655762, 0.5754088163375854, 0.2028842717409134, -1.0612469911575317, 0.4992866814136505, -3.518683671951294, 3.1209728717803955, 2.7544467449188232, 2.4749627113342285, -1.5311124324798584, 0.631929874420166, -3.0938024520874023, -2.031575918197632, -0.5830240845680237, 0.25244060158729553, 0.2885739207267761, 0.8343134522438049, -0.2608431577682495, -0.4636484980583191, -0.899621844291687, -0.36654502153396606, -4.4300947189331055, -0.09859539568424225, 0.41037970781326294, -3.215177297592163, 1.155342698097229, -3.7050743103027344, -1.601621150970459, -0.40848082304000854, 4.515256881713867, -1.550144910812378, 1.950884461402893, 1.1459860801696777, -1.2595924139022827, -0.17524926364421844, 1.426161289215088, 1.5607410669326782, -2.188422918319702, -0.6174107193946838, 1.2579206228256226, 1.9599668979644775, 0.678922176361084, -0.019765134900808334, 1.7562828063964844, -0.3869888186454773, 0.17389197647571564, 0.16832560300827026, -0.48708605766296387, -3.2894668579101562, 0.08001671731472015, -1.4006688594818115, -1.866398572921753, -0.8563452363014221, -1.0883504152297974, -0.42992478609085083, 1.5233508348464966, 1.7845596075057983, -1.0994120836257935, -0.663792610168457, -1.0868228673934937, 1.413974404335022, -3.7162516117095947, -0.4993729293346405, -5.2912821769714355, -0.32281768321990967, 0.5734444260597229, -0.5320405960083008, -4.106888771057129, 2.281337261199951, 3.339588165283203, 0.007092613261193037, -1.2294018268585205, 2.0456700325012207, -0.08135324716567993, 0.5662485361099243, -3.8098702430725098, -5.0111541748046875, 0.07661332935094833, -2.200108528137207, 0.1996113806962967, -0.3601481020450592, 3.3409266471862793, 1.772929072380066, 0.7370206117630005, -2.097578287124634, 0.4824092984199524, -0.19324195384979248, -3.478109359741211, -3.9724624156951904, 2.287740707397461, 1.1146975755691528, -2.036604642868042, 1.571306824684143, 0.5285933017730713, -0.7793406844139099, -0.0722072571516037, 0.3038838505744934, 0.6372779607772827, 2.668254852294922, 1.6465517282485962, 1.298026442527771, -0.8217771649360657, 0.7039022445678711, -0.28103452920913696, -2.443756580352783, 0.14838799834251404, 0.7898268699645996, -0.5897057056427002, -1.7778468132019043, -0.4054970145225525, 3.0250959396362305, -0.33532217144966125, -1.3238084316253662, -2.531447410583496, 0.5405368208885193, -0.480521023273468, -0.8986310362815857, 1.0164294242858887, 0.31982436776161194, 0.9929845333099365, 2.5215072631835938, -0.28465402126312256, 0.3486051857471466, -1.7812741994857788, -0.7303594946861267, -0.9443317651748657, -0.41825222969055176, 1.4880880117416382, -2.357889175415039, -0.35406434535980225, 1.2319655418395996, -1.7881338596343994, 0.5745676159858704, -2.170854091644287, -1.56862211227417, 1.1910302639007568, 1.441939115524292, -0.854457676410675, -0.31813913583755493, 4.620700836181641, 1.5653226375579834, -1.717911720275879, 0.37502700090408325, -4.121086120605469, 2.0143444538116455, -0.47090280055999756, -0.8220165371894836, -1.356361746788025, 2.809858798980713, 0.2929977476596832, -1.7534234523773193, 2.1969387531280518, 1.6723854541778564, 0.6028305888175964, -2.316981554031372, -0.8029851913452148, 2.454730987548828, -2.6854677200317383, 0.6680652499198914, -3.4980971813201904, 0.4812532663345337, -1.4953585863113403, 3.3663156032562256, -3.4327826499938965, 2.284590244293213, -3.139572858810425, -2.1062731742858887, -0.6961934566497803, -1.0281201601028442, 1.2428690195083618, 1.9395071268081665, 0.2714408040046692, -0.049928586930036545, 4.715252876281738, -0.38285335898399353, 0.3444240987300873, 1.463923454284668, 3.975299835205078, -0.7980923652648926, -0.3657541573047638, -0.12062328308820724, -0.5478442907333374, -0.5631844401359558, 1.80625581741333, 2.3877675533294678, -3.2284388542175293, -0.7746323943138123, 0.02392261102795601, -1.9398192167282104, -3.0085911750793457, -2.111830949783325, -2.0921168327331543, 3.8346750736236572, -1.1385377645492554, -0.4783541262149811, -6.29000186920166, 0.6260601282119751, -2.157830238342285, 1.158487319946289, -1.826812744140625, -0.35688310861587524, 0.5102072954177856, 4.1775689125061035, -0.6010383367538452, -2.9263625144958496, -0.16050373017787933, 0.45560845732688904, 1.677247405052185, 0.53658127784729, 2.001314640045166, -1.099963665008545, -0.035316865891218185, 1.265370488166809, 1.1928613185882568, -2.9733684062957764, 1.7820826768875122, 0.9564377665519714, 1.8628942966461182, 1.5746345520019531, 1.2328603267669678, -2.123990297317505, -2.94769287109375, -2.607996702194214, -2.8135812282562256, -0.8229959011077881, 0.18613964319229126, 2.3461716175079346, 2.6669812202453613, 1.19096839427948, 2.9893851280212402, -0.762067437171936, 2.597050189971924, -2.982177972793579, -0.4358726441860199, 2.6458544731140137, 1.8374501466751099, 0.052853599190711975, 1.4081621170043945, -0.42341482639312744, -1.5587267875671387, -0.2051115185022354, 0.32554563879966736, 1.422653317451477, -0.8860060572624207, -2.5168066024780273, -1.3109853267669678, 1.6765190362930298, 2.935326337814331, -1.7829750776290894, 2.9680967330932617, -0.4392869174480438, 1.478453278541565, 2.005282402038574, -3.255197763442993, 0.08320312947034836, 4.188411235809326, 1.4884111881256104, -1.3752280473709106, 0.08121158927679062, -1.8481693267822266, 0.2850230038166046, -1.2477914094924927, 2.0636398792266846, 1.195548415184021, -1.9338667392730713, -1.9183931350708008, -1.6422079801559448, -2.260756254196167, 0.6979603171348572, -2.4271037578582764, -0.9997190237045288, -0.7945302724838257, -0.07999294251203537, 1.1242921352386475, 2.854301691055298, -5.917073726654053, 0.1758490353822708, 0.06177940592169762, 1.5964133739471436, -0.21539680659770966, -1.3414416313171387, -1.0294740200042725, -0.0027561942115426064, 1.2654863595962524, -0.10081038624048233, 0.5829219818115234, 2.452394485473633, -1.1005066633224487, 1.6347908973693848, -1.0652426481246948, 1.201483964920044, 0.02610447257757187, -3.4202983379364014, -0.6216427087783813, 0.17131385207176208, -0.4412999153137207, -1.5349254608154297, -3.0276145935058594, 0.1508573442697525, -2.5873525142669678, 0.6488679051399231, -1.051896572113037, -2.057241678237915, 1.347946286201477, 0.7952913045883179, 1.001291036605835, 2.6446568965911865, -2.3923721313476562, 0.8850821852684021, -1.6734609603881836, -0.9799702167510986, -1.940746545791626, 1.1681519746780396, 1.8089141845703125, -1.400590181350708, -0.13293670117855072, -1.680342674255371, -0.41396209597587585, -3.602586030960083, 0.029293401166796684, 0.10339106619358063, 1.6856880187988281, 0.40900149941444397, -0.741647481918335, 0.7374677658081055, 1.6350417137145996, 1.8408775329589844, 1.2583680152893066, 1.842990517616272, 2.5738866329193115, 0.6034682393074036, -0.28728047013282776, -2.064633846282959, -2.4805068969726562, 0.13759435713291168, 1.2004003524780273, 0.15416225790977478, -0.7456479668617249, 1.9681735038757324, -1.9032402038574219, 1.4140931367874146, 0.35818594694137573, -1.8314387798309326, 1.0257564783096313, -0.8875169157981873, -4.366974353790283, 0.43965426087379456, 0.2871721684932709, -0.40528202056884766, 2.9457848072052, -1.3828636407852173, -3.0869503021240234, 0.32558777928352356, -0.4357532560825348, -0.5095328092575073, 3.800703287124634, 1.6969373226165771, 1.8641200065612793, 1.3526391983032227, 0.6912965774536133, -0.2925882935523987, 1.3920226097106934, -0.7893946766853333, -1.2632570266723633, -2.6747546195983887, 0.19889673590660095, -1.9054183959960938, 1.2416150569915771, 0.6263231039047241, -2.9782538414001465, 1.3049790859222412, -1.3315025568008423, 0.642676055431366, 2.279470205307007, 0.5267164707183838, 1.4835379123687744, 0.6204538345336914, 1.6627416610717773, 0.09528741985559464, 2.0179386138916016, -0.2622431218624115, 0.4298485219478607, 1.8905551433563232, 1.0671628713607788, -3.0905601978302, -2.0198473930358887, 0.516626238822937, -3.080024242401123, 2.4258689880371094, 4.718626976013184, 2.0914316177368164, -2.7985589504241943, 0.4854644238948822, 2.352607488632202, -1.2310724258422852, 1.25852632522583, -4.496858596801758, 2.720602512359619, -0.927178680896759, -0.03195904195308685, 3.970667839050293, 0.5347771048545837, 3.096341371536255, 1.1384564638137817, -0.10035401582717896, 1.2072181701660156, -1.647850513458252, 0.5729377269744873, 1.3465439081192017, -0.06249849870800972, -0.008204626850783825, 1.2577892541885376, -3.772132396697998, -0.6533708572387695, 1.7287087440490723, 0.6380996704101562, 0.29996269941329956, -4.057003974914551, -3.7609262466430664, 0.3909628987312317, 0.7593188285827637, -0.08618700504302979, -2.6951193809509277, 1.207348346710205, 0.35057127475738525, 3.1107895374298096, 2.4361300468444824, -0.24090398848056793, 2.073612689971924, 2.348906993865967, 5.002715587615967, -2.9011342525482178, -2.3501741886138916, -4.012967109680176, -3.7861220836639404, -3.9786264896392822, 0.19762632250785828, -0.2635367214679718, -0.4434105455875397, 1.534616231918335, 1.2010178565979004, -1.9208788871765137, 1.1201200485229492, 0.2684212028980255, -1.655569076538086, -0.446696013212204, 0.5726381540298462, 0.3051663339138031, 0.14875216782093048, 0.817836344242096, 0.06485893577337265, 1.5015329122543335, 0.9700455069541931, 1.3108680248260498, -3.15486741065979, 0.05025269091129303, 1.6998076438903809, 2.068016290664673, -1.0539758205413818, -1.2888544797897339, -1.5526994466781616, -0.4565100073814392, 1.7746853828430176, 1.4054731130599976, 1.2499676942825317, 0.522062361240387, 1.3809616565704346, -0.7687206864356995, 2.049715518951416, 1.2021512985229492, 2.3706395626068115, 2.276747465133667, 1.106401801109314, 0.461820125579834, 0.6263973116874695, 0.1671525537967682, -1.1577714681625366, 1.7665481567382812, -4.965237140655518, -1.27492356300354, -1.5429065227508545, -2.071725606918335, 2.0140016078948975, -1.5963459014892578, 0.2561095356941223, 2.199390172958374, -2.617433786392212, -1.7096630334854126, -0.1154671460390091, -5.757704257965088, -0.009131832048296928, 2.1784963607788086, -1.3720043897628784, -1.703059434890747, -2.6907830238342285, -3.135667324066162, 1.1298744678497314, 3.1872730255126953, -0.36172381043434143, -3.0351617336273193, 1.4980753660202026, -0.12214796245098114, 3.663358688354492, -1.9174484014511108, 3.2543063163757324, -1.3745142221450806, 5.074276447296143, -0.8626661896705627, 2.6616017818450928, 2.3959109783172607, -1.0063505172729492, -0.12460410594940186, 0.9355727434158325, 1.5635600090026855, -1.108073353767395, -0.44937336444854736, 4.3711419105529785, 1.0295888185501099, 1.2162379026412964, -4.3667378425598145, 0.6164045929908752, -0.3298511803150177, 0.8385425209999084, -2.5622243881225586, 0.08084969967603683, -1.768389105796814, -0.2030183970928192, 1.142565131187439, 3.2278764247894287, 0.9368388056755066, -1.3143950700759888, 1.3536933660507202, -2.837581157684326, 0.8374128341674805, 0.41882240772247314, -1.8678014278411865, -3.8632869720458984, 3.20361065864563, 3.134748935699463, -0.6168975234031677, -0.7855218648910522, 0.31237441301345825, -0.642906665802002, 1.2341686487197876, 2.6285083293914795, 0.2604154050350189, 2.9746618270874023, -0.30761170387268066, 2.8677566051483154, -1.253394603729248, -0.3747357428073883, 1.0830594301223755, 1.5024025440216064, 1.9758861064910889, -0.01960514485836029, 1.6420701742172241, -1.7632713317871094, -1.2719852924346924, -0.5213958621025085, -0.09868992120027542, 0.9459770917892456, 2.8352856636047363, -0.9671052098274231, 1.3011479377746582, 0.6978338956832886, 0.9064886569976807, -4.1194305419921875, -0.07373356819152832, 0.05688811093568802, -0.13889454305171967, -1.001694679260254, -3.8160884380340576, 3.581385850906372, 0.35751235485076904, -3.730339527130127, 2.142972469329834, -1.0639352798461914, -0.09699130058288574, 0.2898239195346832, 3.392120122909546, -1.0063880681991577, -2.1803529262542725, 6.2596659660339355, 1.2274469137191772, -1.1417689323425293, -2.3621275424957275, -2.9661052227020264, 0.44432172179222107, 0.6972556710243225, -1.0630271434783936, -0.38163816928863525, -0.06782685220241547, -2.923490047454834, 0.03550758585333824, -0.17445528507232666, -3.38364315032959, 0.3499281108379364, 0.6040528416633606, -0.8074885606765747, -0.2908070683479309, 1.5950313806533813, 1.2160475254058838, 1.2756938934326172, -1.0266304016113281, 2.703230381011963, 0.9433382749557495, 3.4711267948150635, 2.447714328765869, -2.2776317596435547, -0.42818859219551086, 1.179463267326355, 0.8398968577384949, 1.9613641500473022, -2.9315078258514404, 0.7743397355079651, -1.9331387281417847, -0.7324655652046204, 1.9645267724990845, 0.4001424312591553, -2.098897695541382, -4.230462551116943, -1.430524468421936, -1.321459174156189, 1.0061100721359253, 4.646758556365967, -0.036266591399908066, -1.6818962097167969, -1.3056144714355469, -1.6046289205551147, 0.035910751670598984, 0.8471916913986206, 2.5555386543273926, 1.53019118309021, -0.49355873465538025, -1.8336639404296875, -6.894435405731201, 4.955181121826172, -0.8014724850654602, 2.856588125228882, -2.4292993545532227, 0.9523308873176575, -2.0152909755706787, -1.9996176958084106, -3.915071725845337, 1.3346806764602661, -2.0443506240844727, 2.1374897956848145, 1.1950324773788452, -0.999312698841095, 2.1726977825164795, 0.5082406401634216, -1.4940757751464844, -2.6565284729003906, 0.6004005074501038, -0.08629540354013443, -4.185638427734375, -0.3101397156715393, 3.8210575580596924, -2.216043472290039, 2.011949062347412, 0.487140417098999, 2.3051531314849854, -0.49803823232650757, -5.097838878631592, -4.440002918243408, -1.8624188899993896, 2.3009390830993652, 2.4784207344055176, 1.173556923866272, 0.6179744601249695, -1.9755032062530518, 0.6079497337341309, -1.2354655265808105, 1.2057312726974487, -0.9442827105522156, -1.5500049591064453, 1.9598467350006104, -2.234220027923584, -2.1612977981567383, -0.3887019157409668, 1.8667186498641968, -2.062553644180298, -2.964048385620117, 3.6171233654022217, -1.7674767971038818, 0.9928616285324097, -0.22632429003715515, -1.198360800743103, -0.183974951505661, 0.5137730836868286, -0.6311485171318054, -0.368144154548645, -0.4866635203361511, -2.2035977840423584, 1.1914151906967163, -0.011368407867848873, 1.1106408834457397, 3.574979782104492, -2.539360761642456, 0.4805450737476349, 1.5191740989685059, -0.8202537894248962, 5.189503192901611, -0.03471246734261513, -0.5941720008850098, -0.1348724514245987, 0.13637273013591766, -3.516693592071533, 0.12761858105659485, 0.491578608751297, -0.08096811920404434, -5.351711273193359, -4.11475944519043, 1.4983413219451904, 1.9481605291366577, -2.0702121257781982, 1.7094759941101074, -1.7276296615600586, 1.4182039499282837, -0.9428887963294983, 0.901864230632782, -1.3265581130981445, -0.5093585252761841, -2.505156993865967, 0.45858877897262573, 4.7910990715026855, 0.5347108244895935, 4.328646183013916, 0.10368341952562332, 0.19310900568962097, 1.1786783933639526, 0.04132629930973053, 4.3901872634887695, 1.3764642477035522, 0.04400482028722763, -1.841245174407959, -1.9309378862380981, 2.476776123046875, 0.4077291786670685, 4.5956220626831055, 1.5817286968231201, -1.6720693111419678, -0.4611068665981293, 0.3092147707939148, 2.333270311355591, -0.08652971684932709, -0.1765032261610031, 1.3492411375045776, -1.7046962976455688, -2.1160786151885986, -2.891996145248413, 0.371408611536026, 1.2799307107925415, 1.18386971950531, 0.7637516856193542, -2.8085508346557617, -1.4768953323364258, -3.6570732593536377, -1.5837702751159668, 1.8062092065811157, 3.2954609394073486, -1.3089956045150757, 0.07315654307603836, -2.2930305004119873, -2.522664785385132, 4.523180961608887, 0.42139795422554016, -0.2955050468444824, 1.1949329376220703, -1.0683040618896484, -2.278245687484741, 2.890568733215332, -3.8945512771606445, 0.06893076002597809, -1.3340704441070557, 0.94449782371521, -0.31615912914276123, 3.9707131385803223, -2.9857404232025146, 1.471394658088684, 0.36387985944747925, -1.5243525505065918, 4.670437335968018, -2.4956812858581543, -1.8336241245269775, 0.2332633137702942, -1.307181477546692, 0.7098504304885864, -0.9235106706619263, -0.9882604479789734, -0.014148903079330921, 1.2239211797714233, -2.1705145835876465, -0.7695932388305664, -3.44699764251709, 2.243008852005005, 2.5931766033172607, -1.777323842048645, 2.605138063430786, -0.7904174327850342, 1.8559701442718506, -0.41504865884780884, -1.0177562236785889, -0.8293296694755554, -2.0299501419067383, -1.895272970199585, -1.8495497703552246, 1.204755425453186, -0.27613013982772827, 0.6328141093254089, 0.2787608206272125, 0.5205624103546143, 0.15457487106323242, 3.1301581859588623, -0.8080813884735107, 0.5135006904602051, 1.9135781526565552, 2.265838146209717, 1.483712077140808, 3.4486184120178223, 2.673830032348633, 2.12103533744812, 0.7630415558815002, -2.2856290340423584, -2.7644600868225098, -1.3917300701141357, -0.9521849751472473, 3.1157679557800293, 1.529546856880188, -1.9710822105407715, 2.6855170726776123, -0.09011654555797577, -1.1445988416671753, -1.0183359384536743, 3.652719259262085, -1.6153626441955566, -1.7452447414398193, -2.9396402835845947, 0.0025555468164384365, 0.9431723952293396, -4.715844631195068, 2.7650532722473145, -0.8710580468177795, 0.7883391976356506, -2.547962188720703, -1.6159378290176392, 1.6573727130889893, -0.07568534463644028, -0.645537257194519, 2.8196210861206055, 4.063419342041016, -2.3655755519866943, -0.1886727511882782, 2.1919920444488525, 1.7316075563430786, -1.467733383178711, 2.3853378295898438, 2.124234914779663, 3.2792227268218994, 0.3889274597167969, -2.7677507400512695, 3.5572681427001953, -3.048642158508301, 0.0384732186794281, -0.7299830913543701, -0.7603499293327332, 2.584031820297241, -2.273392915725708, -3.0786373615264893, -1.029449701309204, -1.8483662605285645, -0.6183693408966064, 0.1004812940955162, -0.4287112057209015, 0.6004160046577454, 0.3234231472015381, -0.6900845766067505, 0.715742826461792, 2.634253740310669, -0.998198390007019, -1.1900290250778198, 0.25210699439048767, -3.3981118202209473, -0.2584056556224823, -0.04641904681921005, -1.7148187160491943, -4.179916858673096, 0.9971409440040588, 1.247438907623291, -2.526675224304199, 2.7285122871398926, -3.2299695014953613, 3.306898593902588, 1.5855565071105957, -1.3139361143112183, -2.802095413208008, 3.2200324535369873, -1.301500678062439, -3.298208713531494, 3.496892213821411, 0.7093592286109924, 0.1052783876657486, 1.5402286052703857, 0.17668481171131134, 0.11699582636356354, -3.885389566421509, -1.263850450515747, -1.5795129537582397, -0.24634486436843872, -0.3352842330932617, 0.9214820265769958, 0.47548237442970276, -1.5499011278152466, -2.5886342525482178, 1.7843608856201172, 1.045348048210144, 1.4001309871673584, 2.0805792808532715, -1.8816492557525635, -2.2869296073913574, 1.6839548349380493, -0.03265802189707756, 0.7868272662162781, -2.120785713195801, -1.3783446550369263, -3.1851565837860107, -1.4676964282989502, -0.06657600402832031, -2.0152947902679443, -0.5369149446487427, 2.2385659217834473, -1.4780458211898804, -0.9605181217193604, -4.366288185119629, 0.4320865571498871, 3.0833308696746826, -2.3504416942596436, -1.3249337673187256, 1.5647978782653809, 0.5974088311195374, 1.1153191328048706, -0.7682762145996094, 2.320265769958496, -0.27651792764663696, -1.917099952697754, 0.7242797017097473, 3.0740129947662354, -2.3861300945281982, -1.949814796447754, 4.145205020904541, 3.3339107036590576, -0.5032446384429932, 2.602282762527466, -1.6790810823440552, -3.734609603881836, -0.6760417222976685, 1.4416791200637817, -1.1713790893554688, 1.0967835187911987, -3.1237852573394775, 0.534701406955719, -1.42021644115448, 2.7896852493286133, 3.109009265899658, -1.6800028085708618, 0.23633678257465363, -1.3501583337783813, 0.9879172444343567, 0.3160017728805542, 0.34885963797569275, 0.3267895579338074, -1.417169213294983, 1.8447213172912598, -2.510188102722168, -3.1823782920837402, -5.806685924530029, -1.292447566986084, 2.3727779388427734, -2.271127939224243, -0.30192050337791443, -1.6500511169433594, 0.4374987781047821, 2.2198972702026367, 0.7237943410873413, 0.22434893250465393, -0.9036474227905273, 1.2074421644210815, 2.7890093326568604, 0.8114315271377563, -1.8286974430084229, 0.5165867209434509, 0.6102924942970276, -0.9004557132720947, -0.6585365533828735, -3.266343355178833, -0.158772274851799, -6.139590740203857, -0.39891165494918823, 0.9521306753158569, -0.35757648944854736, 3.0210511684417725, -1.3861905336380005, 3.7815754413604736, 1.3258695602416992, -2.551746368408203, -4.42661714553833, 0.7807908058166504, -2.4624013900756836, -2.5908823013305664, 1.1392756700515747, 1.2311875820159912, -0.5433053374290466, -0.9618757963180542, 0.9348487854003906, 2.9938220977783203, -2.7731807231903076, 2.213594675064087, -0.13961155712604523, 3.5004031658172607, -2.1262357234954834, 1.475606918334961, -1.913047432899475, 2.8410942554473877, 0.7447766065597534, -2.1906607151031494, 2.4519240856170654, 3.5955309867858887, -1.294283151626587, 0.09799569100141525, -2.3559792041778564, 0.7636982798576355, 0.7865471243858337, 2.2100167274475098, 1.0425245761871338, 0.9695172905921936, -1.3428056240081787, -2.5353918075561523, 2.168511152267456, 3.1287472248077393, 0.3462100327014923, -0.7941758632659912, 2.1437859535217285, -2.4730031490325928, 3.7447540760040283, 0.7118620276451111, 2.176072597503662, 2.0782363414764404, 3.3292958736419678, 0.5884924530982971, -3.029442548751831, -0.5206108689308167, -0.8952801823616028, 3.027559757232666, 0.21722730994224548, 1.6122859716415405, -1.393974781036377, -1.1401140689849854, 2.882132053375244, 1.1776643991470337, -1.5849394798278809, 2.2318756580352783, -0.7040975093841553, 0.147115096449852, 0.745559573173523, 2.7906296253204346, -3.971214771270752, -5.036129474639893, -2.0177900791168213, 2.1263065338134766, -0.6752117276191711, 1.1103955507278442, -1.4591397047042847, 0.5877026319503784, 3.2324955463409424, -0.3806931972503662, 2.528660774230957, 1.2100982666015625, -4.052248001098633, 2.8600900173187256, -0.6342774033546448, -0.7261576056480408, -1.0068938732147217, 1.5603725910186768, -0.837984561920166, 3.871131658554077, -0.027594931423664093, 0.14582449197769165, 0.8959382176399231, -0.726747989654541, 0.053315311670303345, 1.0838687419891357, 2.1656851768493652, 3.7923851013183594, 1.8425251245498657, 0.09639545530080795, -1.342900037765503, 0.6729087233543396, 2.1235299110412598, -2.1838502883911133, 2.216179370880127, -0.18410052359104156, -0.0513419434428215, -0.8376509547233582, -1.3624393939971924, 0.12771067023277283, 0.022497758269309998, -1.8954423666000366, 1.3879327774047852, -3.3910484313964844, 3.6605722904205322, -1.620689034461975, -0.8535394668579102, 2.356997013092041, 1.9741590023040771, 2.16784405708313, 3.2190475463867188, -0.5426556468009949, -1.2128561735153198, 0.11527281254529953, 3.4614622592926025, -1.6193612813949585, -0.2338121384382248, -1.1045581102371216, 1.1582528352737427, 1.5305285453796387, 2.4424889087677, -5.060257434844971, 1.0738686323165894, 2.453099489212036, 1.7755763530731201, -0.7567861676216125, -0.44496971368789673, 0.3476656377315521, -0.8292574286460876, -0.9582211971282959, 0.42166459560394287, -2.5253353118896484, 1.4184156656265259, 2.1680822372436523, 1.8979640007019043, -0.0742296651005745, 1.818350911140442, -1.871268391609192, 2.0834546089172363, 2.0639874935150146, 2.706385612487793, -1.5850876569747925, -0.7021918296813965, 2.6924867630004883, -1.29845130443573, -1.8723372220993042, 0.5872575640678406, -0.8033924698829651, 0.07237524539232254, 3.579402208328247, 1.4602956771850586, -1.5470179319381714, 1.0335243940353394, -1.4195945262908936, 2.9368667602539062, 0.35301077365875244, -0.8754358291625977, 1.089678406715393, 1.1672478914260864, -5.059511661529541, -1.5063116550445557, -1.9237934350967407, -1.724471926689148, 0.8059437870979309, -4.327749729156494, 2.4368178844451904, 2.827303886413574, 3.8925204277038574, -0.30624669790267944, 1.8849889039993286, -0.6554818749427795, -2.5247082710266113, 4.471401214599609, 0.06598208844661713, 0.39316806197166443, -0.43975192308425903, 2.422797203063965, -3.586827039718628, 2.1073131561279297, 1.0023517608642578, -2.0476841926574707, -2.409003734588623, -3.299290418624878, 1.000917911529541, 0.07649949193000793, 0.20502588152885437, -1.7963509559631348, 0.5705537796020508, -0.1711062490940094, -1.720715045928955, 5.379554271697998, 1.7896391153335571, 1.5462346076965332, -0.14361527562141418, -1.8905435800552368, 2.2802681922912598, -1.5843263864517212, 2.739492416381836, 0.5251774191856384, 0.8140082955360413, -1.2623910903930664, 0.4561823606491089, 2.423278570175171, -2.797569990158081, 0.8573533296585083, -1.0969762802124023, 0.8860142827033997, 1.7373369932174683, 1.5034925937652588, 1.2663449048995972, 2.907071590423584, -3.3573594093322754, 0.07367099076509476, -0.03709835186600685, -1.0542685985565186, -0.7240859866142273, -3.682295799255371, 0.8158869743347168, 1.6655974388122559, -0.2729058265686035, -3.020965814590454, 2.9584991931915283, -0.5743786692619324, -3.1158251762390137, 0.08415151387453079, -5.685263633728027, -1.3812158107757568, -2.223371744155884, 0.8809908628463745, -2.196794271469116, 2.8683321475982666, 0.7069526314735413, -1.2180532217025757, -0.6703500151634216, 2.602381706237793, -2.615370988845825, -0.16625334322452545, 0.08050429821014404, -1.451948881149292, -0.3595522344112396, 0.26462268829345703, -3.695688486099243, 0.5106348991394043, 3.1849794387817383, 0.70430588722229, -1.7584186792373657, 1.3073997497558594, 1.7043036222457886, -0.3175257742404938, -5.065123558044434, 0.19289469718933105, 2.8262805938720703, -1.7598270177841187, 0.9144824147224426, -0.9845048189163208, 1.1208996772766113, 0.02865336835384369, -0.4433472156524658, -1.0139672756195068, -1.0910413265228271, 4.567977428436279, 0.9968569278717041, 3.5022387504577637, 0.7098689675331116, -1.6587425470352173, -4.958041667938232, -1.4943604469299316, 3.9520630836486816, 0.18746718764305115, 0.42355239391326904, -0.017461271956562996, -0.16655217111110687, -0.7031517624855042, -4.543663501739502, -0.9548167586326599, 1.3633970022201538, -1.1485406160354614, -3.437896966934204, 4.982316493988037, 1.1703213453292847, -1.434325098991394, -1.4175816774368286, -0.5187342166900635, 4.216108798980713, -1.8719727993011475, -0.7412961721420288, -1.9054548740386963, -0.21914398670196533, -5.326845645904541, 0.32753345370292664, -1.0347942113876343, -2.6202266216278076, 1.3154338598251343, -1.9506486654281616, -0.8609406352043152, 0.4974417984485626, 0.35180237889289856, 0.36879754066467285, -1.4208379983901978, 2.6916308403015137, 3.169640064239502, -0.4017689824104309, 2.2196972370147705, -0.5003133416175842, -1.1304975748062134, -1.0119520425796509, 4.45181131362915, 2.2207181453704834, -2.1058757305145264, 0.9351421594619751, -0.9978599548339844, -1.1616437435150146, 1.162148118019104, 1.3319591283798218, -2.6022045612335205, 1.2227411270141602, -1.1070820093154907, -1.3721975088119507, 1.814760446548462, -0.5564291477203369, 2.3795793056488037, -2.682220697402954, 0.34132084250450134, 5.082377910614014, 0.6167668104171753, -0.7155380845069885, 1.627408742904663, -0.5895686745643616, 1.508071780204773, -3.4679605960845947, -1.449454426765442, 0.17969274520874023, 1.0013151168823242, -0.9240005016326904, -1.0298024415969849, 0.07232008129358292, -0.897826611995697, -1.1684644222259521, 1.6687997579574585, -1.9314770698547363, 3.322248697280884, 3.1667563915252686, -1.5064657926559448, 3.4396843910217285, 1.0077754259109497, 0.048197198659181595, 0.22702409327030182, 2.2608165740966797, 2.2552757263183594, 0.6619845032691956, -2.070817708969116, -3.1621949672698975, -0.19058938324451447, -0.2774314880371094, 3.250206232070923, -3.696150541305542, -1.8211989402770996, -1.2135990858078003, 1.4295636415481567, 0.34587207436561584, 0.06731685996055603, 2.7203526496887207, -1.648515224456787, 1.7255148887634277, -2.444768190383911, 3.0864028930664062, -0.9725954532623291, -1.1953083276748657, 0.6923305988311768, 2.312438488006592, -1.0806677341461182, -0.7392382621765137, -1.6888309717178345, -3.082125663757324, -4.003242015838623, 4.832826137542725, 1.7026574611663818, 1.416196584701538, -0.7464068531990051, -2.053340435028076, -2.758871555328369, -0.5347651243209839, 3.341578722000122, -2.2796735763549805, -2.708329439163208, 3.037358522415161, 1.6042572259902954, -2.777399778366089, -1.2935242652893066, -1.2422257661819458, 0.5281170010566711, -0.9330423474311829, -4.704370498657227, 0.9012117981910706, -0.03555284067988396, 1.2079285383224487, 0.2590649425983429, -0.9090575575828552, -0.8663958311080933, -1.2843127250671387, 1.5540030002593994, 1.7884330749511719, 0.4557645618915558, 0.49610763788223267, 1.830433964729309, 1.4769281148910522, 0.2815164625644684, -0.406846821308136, 1.343523621559143, -1.7493669986724854, 0.5340200066566467, -0.917762041091919, -1.1435599327087402, 3.0939056873321533, 3.5592124462127686, -0.9137920141220093, -2.1455135345458984, -1.9776610136032104, 1.3990811109542847, -0.5008164048194885, 0.9380267262458801, -1.734557867050171, -3.069446086883545, -1.0843110084533691, 2.859161138534546, 0.6738560795783997, -3.220050573348999, 3.1267056465148926, 3.7588772773742676, -4.272541046142578, -1.791102409362793, -0.6556500196456909, 0.6065930128097534, 0.07765892148017883, -1.026185393333435, 1.5269123315811157, -1.2794382572174072, -3.5582938194274902, -0.7366971969604492, -3.0832407474517822, 0.5013938546180725, 2.138079881668091, 1.1492761373519897, 0.819952666759491, 1.8937702178955078, 3.381044864654541, 1.300020694732666, 0.1673455685377121, 2.804947853088379, 0.6700536012649536, -0.6369941234588623, -2.097731113433838, -4.871641159057617, -0.7586735486984253, -1.1187609434127808, 0.006464815698564053, 0.33072757720947266, 3.3783621788024902, -3.0098578929901123, -1.0810872316360474, 0.7549048066139221, -1.8064905405044556, -0.5673866868019104, 1.722847819328308, 1.2399884462356567, -1.6468098163604736, -3.5574581623077393, 2.0557701587677, 2.530133008956909, 3.6717042922973633, -0.5007811784744263, -1.496519684791565, -0.1419408619403839, -0.03750365227460861, 3.9413883686065674, -1.9329420328140259, 1.2974494695663452, -0.08076165616512299, 0.22224634885787964, -1.5097110271453857, 2.0583860874176025, 1.9686204195022583, -2.4510440826416016, -1.3712960481643677, -1.8092879056930542, -0.848670244216919, 0.7104343771934509, -4.528425216674805, -2.9993224143981934, 2.1371347904205322, 3.8594744205474854, -0.805022120475769, 0.3515152633190155, 4.1267290115356445, -0.34281739592552185, -3.623931646347046, 1.0649681091308594, 1.348475694656372, 2.1140713691711426, -1.5108058452606201, 3.4152445793151855, -2.113905668258667, -0.1331566721200943, 4.587788105010986, -1.3713551759719849, -0.822822630405426, 1.5483125448226929, 4.532137393951416, 2.1327481269836426, -0.8943312764167786, -0.5248626470565796, -0.7275001406669617, 2.7981364727020264, -0.001368670491501689, 0.43814682960510254, 4.2814788818359375, 0.372323215007782, 0.5781649351119995, 4.074293613433838, -6.176571369171143, -0.05682285502552986, -2.6860339641571045, -1.7394005060195923, -0.8258008360862732, 0.24298717081546783, 2.2795565128326416, -1.6983660459518433, 1.261098861694336, 2.3731014728546143, -0.09504073858261108, 0.38529959321022034, -1.8044114112854004, 1.0834324359893799, -4.366330623626709, 1.8714885711669922, 0.6573238372802734, 2.151367664337158, -0.7038151025772095, -3.6068053245544434, -1.9932224750518799, -1.7036151885986328, -3.652148723602295, 0.8923254013061523, 1.1636168956756592, 2.5523841381073, -3.271063804626465, 0.07205817848443985, 0.23486405611038208, 0.2636677324771881, 2.3997957706451416, 3.4854729175567627, -2.115011215209961, -3.919288396835327, 2.09464430809021, -0.4447515308856964, -2.6336357593536377, -0.41480112075805664, -0.7434902191162109, 2.2962496280670166, -2.0814192295074463, 4.846863269805908, 1.1022140979766846, -1.1724516153335571, -1.6482654809951782, -1.183345079421997, 0.1896992176771164, 0.025459444150328636, -0.8041415214538574, -0.6333186626434326, -0.08121549338102341, 1.0447053909301758, 2.2014150619506836, -2.3161284923553467, -1.8483377695083618, -1.417832374572754, 0.6251328587532043, 1.9767279624938965, 0.7444854378700256, -1.5648552179336548, 1.2870529890060425, 1.5309909582138062, -0.4046211242675781, 1.1708710193634033, 3.0551812648773193, -3.245182991027832, 0.9556838870048523, -3.9776647090911865, -1.291589617729187, 1.2304673194885254, -3.376413345336914, 2.00714111328125, 2.6524622440338135, -2.770437717437744, -2.3165910243988037, -0.9378385543823242, 3.2586138248443604, -3.5834083557128906, -0.5064879655838013, 1.5564216375350952, -0.09399769455194473, -0.6171299815177917, 2.0403218269348145, -1.0624803304672241, -0.7300890684127808, 0.575118899345398, 2.3146088123321533, 0.38051024079322815, -1.9039684534072876, 2.426823139190674, -1.3191086053848267, -0.17490190267562866, 1.9138891696929932, 4.480225563049316, -3.4967966079711914, -1.1078805923461914, -1.416259527206421, -2.870424270629883, 1.0916465520858765, -0.5127965211868286, 1.6788859367370605, 1.6333867311477661, -0.7627860903739929, -1.9532335996627808, 1.6215672492980957, 0.6146373152732849, 0.20523032546043396, 0.9679089784622192, -3.6893019676208496, 3.4973981380462646, 4.299756050109863, 0.8526506423950195, 0.3495519459247589, 0.34430980682373047, -2.3033509254455566, 3.9112730026245117, -0.3047809600830078, 2.0507993698120117, 1.8836920261383057, -2.5395667552948, -1.2101532220840454, -1.833763837814331, -2.261317253112793, -0.04971791058778763, 1.5005375146865845, 2.024033546447754, 0.004622675012797117, 0.33026638627052307, 2.135890245437622, -2.3485047817230225, -0.5152009725570679, -1.2973905801773071, -3.7648916244506836, -5.036280155181885, 0.3921647369861603, -0.5557175278663635, 2.498096227645874, -3.7870755195617676, 0.32393014430999756, 0.9221126437187195, 2.3044943809509277, -1.9797922372817993, -0.31447646021842957, -0.14412586390972137, -0.9063475131988525, 1.5584797859191895, -4.099221706390381, 1.6077368259429932, -3.2995052337646484, -0.28354591131210327, -2.3724584579467773, -0.5693644881248474, 1.0234323740005493, -0.8721064329147339, 0.09077970683574677, -0.4243832230567932, -2.619621753692627, 0.9004879593849182, 0.8668388724327087, -4.887149333953857, -1.0261976718902588, 0.8775002956390381, 0.37165898084640503, 1.3446398973464966, -3.5634756088256836, -0.6349033117294312, -1.1572916507720947, -1.073944091796875, 4.119697093963623, 0.38327521085739136, -1.098044753074646, 1.0935055017471313, 0.5405016541481018, 3.3146414756774902, 2.8717870712280273, 0.37272217869758606, -0.2425616830587387, 1.5658918619155884, -0.2720629572868347, -1.1539371013641357, -2.421205520629883, 0.48466551303863525, -1.1171940565109253, 0.18027843534946442, -1.809012532234192, 0.1825893670320511, 3.3734028339385986, -2.5068914890289307, -0.39639464020729065, 0.07451308518648148, 0.34955132007598877, 0.28994083404541016, -1.8231052160263062, 1.099174976348877, -2.680095911026001, -2.0873401165008545, -0.09265647828578949, 2.4190263748168945, -0.7006011009216309, 3.9457366466522217, -0.16303299367427826, 2.9686808586120605, -2.5726308822631836, -0.6054105162620544, -1.057421326637268, 0.8329061269760132, 4.3243584632873535, 0.5371791124343872, 0.8015140295028687, 0.6309153437614441, 2.4460060596466064, -0.6175411343574524, -0.0168409813195467, -0.24712421000003815, 1.4982911348342896, -3.60044002532959, -1.5361425876617432, -1.1430648565292358, -1.0042592287063599, 0.8618021607398987, -0.6654427647590637, 2.07432222366333, 0.22522667050361633, 3.077493906021118, -1.2754583358764648, 0.5321353673934937, 2.3768622875213623, -0.6112984418869019, -1.874078392982483, -1.2766541242599487, -1.5054576396942139, 0.583990216255188, -3.3788645267486572, -2.5345818996429443, 0.012666966766119003, -0.05226860195398331, -0.8121558427810669, -1.6214094161987305, 2.9504692554473877, -1.1300320625305176, 0.5358031392097473, -2.1290531158447266, -1.0502195358276367, -2.1098859310150146, 1.4752000570297241, 0.22316144406795502, -1.121804118156433, -1.0310951471328735, 1.1528255939483643, 0.6566575169563293, 1.1097387075424194, -0.670005202293396, -0.20605352520942688, -3.068488121032715, -0.07945801317691803, -2.2746775150299072, -0.0063946922309696674, 0.41553354263305664, -0.5713494420051575, 0.9932127594947815, 1.1328622102737427, -2.5651333332061768, -0.5871843695640564, -0.3949137330055237, -3.030672311782837, -1.8514150381088257, 3.4118049144744873, 0.71796053647995, 0.767882764339447, -0.8974969387054443, -2.410022735595703, -0.0020191974472254515, -3.186617851257324, -0.36982017755508423, 0.08704125881195068, 0.262050062417984, -1.073748230934143, -1.193001627922058, 0.08905531466007233, -0.4040736258029938, 2.3126423358917236, 1.3699852228164673, -1.3771926164627075, -1.5610707998275757, 0.663284957408905, -2.6543705463409424, 0.44123899936676025, -2.081228017807007, -2.947740316390991, -1.617189884185791, 4.2865071296691895, 0.5686128735542297, -0.7856840491294861, -0.786168098449707, 2.0502419471740723, -0.23029501736164093, 1.427802324295044, -0.5854418873786926, 1.3826680183410645, -1.7815289497375488, 0.9553418755531311, 0.7160957455635071, 2.351806879043579, -1.493114948272705, 1.1082065105438232, 1.2475059032440186, 1.4404019117355347, 1.8071998357772827, -1.1345659494400024, -4.399155616760254, 0.5054798126220703, 0.4397369623184204, 1.79428231716156, 0.6071046590805054, -0.5590771436691284, -1.9758957624435425, 1.395017385482788, -0.7792337536811829, -0.8293711543083191, 2.874354600906372, -2.9785315990448, -1.661085605621338, -2.1994073390960693, 1.376482605934143, -1.145052194595337, -2.013892889022827, 2.2158164978027344, -0.9250255823135376, -1.634075403213501, 0.9500201344490051, 0.8804540634155273, -1.0359485149383545, -2.1032586097717285, 2.2928287982940674, -1.013094425201416, 0.42726951837539673, 2.3816471099853516, -0.8357980847358704, 0.44542840123176575, -2.0423758029937744, -0.5790660977363586]], "dtype": "float32"}], "e": {"__nd__": [[9.039360156748444e-05, 5.8711098063213285e-06, 7.410605758195743e-05, 7.735820872767363e-06, 5.580817287409445e-06, 0.00011460641690064222, 5.853821221535327e-06, 5.990794306853786e-06, 0.00024958589347079396, 3.0354910904861754e-06, 0.00028783807647414505, 0.00029422316583804786, 0.00027072077500633895, 3.651172846730333e-06, 0.00018267365521751344, 0.00029684617766179144, 0.00017396868497598916, 1.0494578418729361e-05, 0.0002607313508633524, 1.897970946629357e-06, 2.765809404081665e-05, 1.1922883459192235e-05, 0.0006654383614659309, 2.0070423488505185e-05, 9.635140486352611e-06, 0.00011186580377398059, 2.064433374471264e-06, 9.811558811634313e-06, 1.8156951000491972e-06, 5.5313328630290926e-05, 3.0635064831585623e-06, 4.283324869902572e-06, 1.7833346646511927e-05, 7.956947229104117e-06, 0.00016441896150354296, 0.0011139578418806195, 0.00024225017114076763, 0.0003485303604975343, 1.1225369007661357e-06, 1.943660572578665e-05, 0.0002131601795554161, 0.00012059662549290806, 8.955338853411376e-05, 1.5896051763775176e-06, 0.00021035797544755042, 0.0034470311366021633, 1.0459120858286042e-05, 6.111755737947533e-06, 7.13059853296727e-05, 5.15928040840663e-05, 4.3528467358555645e-05, 0.0002012064214795828, 3.4849986150220502e-06, 0.0002592049422673881, 3.231368054912309e-06, 1.1855991033371538e-06, 0.0004883966757915914, 0.00030117115238681436, 0.00010040491179097444, 0.00015886836627032608, 0.00048396000056527555, 0.00018386395822744817, 1.538736978545785e-05, 0.00021920005383435637, 5.372000941861188e-06, 7.919752533780411e-05, 9.849480875345762e-07, 5.8705496485345066e-05, 6.055441917851567e-05, 2.8049385946360417e-05, 2.4452692741760984e-06, 1.001616283247131e-06, 0.0001723089226288721, 1.0085979738505557e-05, 0.00019586608686950058, 2.5229279344785027e-05, 2.1907655536779203e-05, 4.8032677568699e-07, 0.000157659305841662, 4.4241882278583944e-05, 1.1552172509254888e-05, 6.651574949501082e-06, 1.1391391581128119e-06, 4.204049218969885e-06, 0.00048212011461146176, 6.725704588461667e-05, 0.0006714051123708487, 0.0001575817004777491, 1.348461773886811e-05, 7.775281846988946e-05, 9.683441248853342e-07, 0.00037683043046854436, 2.4236605895566754e-05, 3.573075809981674e-05, 5.538613550015725e-05, 4.864540824200958e-05, 4.9512265832163393e-05, 3.1737476092530414e-05, 8.466457984468434e-06, 1.6830408640089445e-05, 1.2409905139065813e-06, 6.481335731223226e-05, 1.4933656302673626e-06, 0.00012631311255972832, 1.0177541298617143e-05, 1.2029025128867943e-05, 6.068686957405589e-07, 3.481302337604575e-05, 5.657661859004293e-06, 5.985210373182781e-05, 3.336165173095651e-05, 2.574524660303723e-05, 0.00011916148650925606, 1.255888128071092e-05, 1.3279609447636176e-05, 8.91765648702858e-06, 1.9946619431721047e-05, 1.4901303984515835e-05, 0.00012672498996835202, 0.00010734073293861002, 1.5467625416931696e-05, 7.87002318247687e-06, 1.415114547853591e-05, 0.00010158427903661504, 1.928039637277834e-05, 1.9400131350266747e-05, 2.7465350740385475e-06, 0.0002991024812217802, 1.2956480532011483e-05, 3.4189277357654646e-05, 0.00027212282293476164, 8.972113505478774e-07, 1.8071779777528718e-05, 0.00021254141756799072, 2.761576297416468e-06, 6.6575457822182216e-06, 1.9519282432156615e-05, 2.4262526494567282e-05, 1.757533391355537e-05, 2.9621873181895353e-05, 4.086102308065165e-06, 6.345522251649527e-06, 1.7114605725510046e-05, 1.3686994861927815e-05, 0.000403074111090973, 1.9979972421424463e-05, 7.778314466122538e-05, 0.0005434263730421662, 2.9497854484361596e-05, 3.620260031311773e-05, 1.1876059033966158e-05, 3.7163070373935625e-05, 6.505983037641272e-06, 3.826142710749991e-05, 0.00023936363868415356, 5.271706868370529e-06, 9.677141497377306e-05, 0.00018486498447600752, 5.548037734115496e-06, 0.00047401359188370407, 3.3622996852500364e-05, 0.00012614126899279654, 0.00012404941662680358, 0.00012261512165423483, 5.531995338969864e-05, 7.576711504952982e-05, 8.03974035079591e-05, 6.324494734144537e-06, 0.0004487674741540104, 1.2190801498945802e-05, 3.960265530622564e-05, 8.520185474480968e-06, 0.000809606397524476, 0.00042347825365141034, 6.836521060904488e-05, 0.0002788000856526196, 0.0025450335815548897, 5.319283445714973e-05, 0.000444978621089831, 8.781639917287976e-05, 1.712201537884539e-06, 1.8337797882850282e-05, 0.0009826666209846735, 3.910736268153414e-05, 0.00037116234307177365, 2.0497822333709337e-05, 1.9515415260684676e-05, 0.0002533170045353472, 0.00045306587708182633, 3.917135109077208e-05, 2.9724185424129246e-06, 2.545185679991846e-06, 0.00028394925175234675, 5.4776781325927004e-05, 1.5545354472124018e-05, 0.000119394862849731, 0.000128225379739888, 0.00035454551107250154, 5.4429769079433754e-05, 0.0007846729131415486, 0.0013702205615118146, 2.677076554391533e-05, 2.2118752895039506e-05, 0.0005972810904495418, 0.0002969913766719401, 3.8241309084696695e-05, 2.875665359169943e-07, 4.467941107577644e-05, 5.042875272920355e-05, 0.0001060304930433631, 1.367614822811447e-05, 2.39151904679602e-05, 0.00011935482325498015, 9.299800694861915e-06, 1.0578654837445356e-05, 0.00010574576299404725, 1.8237566109746695e-05, 1.6825284546939656e-05, 3.357089008204639e-05, 9.32849980017636e-06, 1.399108259647619e-05, 1.2151399459980894e-06, 5.035217782278778e-06, 0.00023255555424839258, 4.549240202322835e-06, 6.31324655842036e-05, 1.8611402992974035e-05, 5.214995780988829e-06, 3.466025736997835e-05, 1.2287373465369456e-05, 4.449667176231742e-05, 7.506740803364664e-05, 2.6481049644644372e-05, 3.143923095194623e-05, 1.9805209376499988e-05, 6.17088153376244e-06, 1.9766854109093401e-07, 8.748925210966263e-06, 4.9267600843450055e-06, 0.00024002128338906914, 1.1964606528636068e-05, 6.741848210367607e-06, 7.806861685821787e-05, 0.00031927975942380726, 1.0222091759715113e-06, 0.00021243584342300892, 0.000599800783675164, 5.860019427927909e-06, 3.38085646944819e-06, 1.1978404472756665e-05, 0.0015912407543510199, 0.0010717180557549, 2.153146306227427e-05, 6.790759471186902e-06, 0.0002359644859097898, 5.001498357160017e-05, 0.00014444629778154194, 2.490370206942316e-05, 2.4514702090527862e-06, 8.942527347244322e-05, 7.012599780864548e-07, 4.984324186807498e-05, 0.00014713227574247867, 0.00022309139603748918, 0.00010194136848440394, 2.1016427126596682e-05, 6.789703911636025e-05, 3.5582393138611224e-06, 0.000607146299444139, 5.13144550495781e-05, 1.0931792530755047e-05, 0.00025004707276821136, 4.058470494783251e-06, 3.834143353742547e-05, 6.542838491441216e-06, 6.965528314140101e-07, 3.1569015845889226e-05, 6.193188164616004e-05, 5.633924956782721e-06, 0.00025891061522997916, 3.817553442786448e-05, 0.0001764260232448578, 1.4322077731776517e-05, 0.00038965922431088984, 1.2925675946462434e-05, 3.7391760997707024e-05, 6.873763777548447e-05, 1.21110815598513e-05, 1.241496192960767e-05, 0.0001243710285052657, 1.0718079465732444e-05, 0.00032889487920328975, 4.4666961912298575e-05, 5.3122357712709345e-06, 0.00011233602708671242, 0.00031522277276962996, 0.00012010255159111694, 0.0001744022301863879, 7.515226752730086e-05, 6.846349424449727e-05, 0.000737812602892518, 9.482635505264625e-06, 6.9055877247592434e-06, 5.482335473061539e-05, 7.087955964379944e-06, 1.7588372429599985e-05, 7.006285613897489e-06, 1.9287836039438844e-05, 1.2111830983485561e-05, 4.850845652981661e-05, 2.6441710360813886e-05, 0.0001008475519483909, 1.1468277989479247e-05, 2.903887434513308e-05, 0.0014154408127069473, 0.00020880564989056438, 3.3741953302524053e-06, 8.151528732014413e-07, 8.035724931687582e-06, 0.00017129004118032753, 1.4963271496526431e-05, 4.5161186790210195e-06, 8.419257937930524e-05, 7.430671757902019e-06, 0.00012726949353236705, 4.409455414133845e-06, 4.111608359380625e-05, 0.0002639518352225423, 0.002655451651662588, 2.2203163098311052e-05, 4.4217900722287595e-05, 0.0023375186137855053, 0.00027550902450457215, 0.0003113104321528226, 9.098779628402553e-06, 0.00011716259177774191, 0.004410138353705406, 2.489567850716412e-05, 1.1964880286541302e-05, 5.134172533871606e-05, 2.8520154955913313e-05, 0.00011762856593122706, 4.1730170778464526e-05, 0.000721450662240386, 0.00021193764405325055, 0.0010800925083458424, 7.904881931608543e-05, 0.00024043922894634306, 8.648089533380698e-06, 0.00023360195336863399, 0.0009343771380372345, 4.7415953304152936e-05, 8.517648893757723e-06, 3.224060492357239e-06, 7.853728311602026e-05, 8.597515989094973e-05, 0.00021037031547166407, 3.7449313822435215e-05, 2.6880582026933553e-06, 2.2703186914441176e-05, 6.557685878760822e-07, 0.0005439846427179873, 3.919482333003543e-05, 2.702974234125577e-05, 0.00016401312313973904, 7.408489636873128e-06, 1.7319560356554575e-05, 0.0001859722688095644, 6.9408220042532776e-06, 8.930683543439955e-05, 2.7554569896892644e-05, 0.00022529784473590553, 0.0025872811675071716, 4.939333666698076e-05, 0.0007911717402748764, 3.534269126248546e-05, 6.275371561059728e-05, 0.00010260263661621138, 4.492520474741468e-06, 0.00030435045482590795, 1.3962174307380337e-05, 0.00010458106407895684, 1.8066955362883164e-06, 4.2113624658668414e-05, 0.00010203401325270534, 6.473946268670261e-05, 0.00217259768396616, 0.0007423839415423572, 0.00016127647540997714, 1.3167525139579084e-05, 1.877337126643397e-05, 1.817403062887024e-05, 3.1932981983118225e-06, 2.3978540411917493e-05, 0.000457097397884354, 0.0002844391274265945, 0.00038638527621515095, 1.5236224498949014e-05, 0.0012475057737901807, 0.0007538376958109438, 4.014679871033877e-05, 6.238034256966785e-05, 0.00011201378219993785, 0.00013107126869726926, 0.00011907529551535845, 4.31591433880385e-05, 8.684657586854883e-06, 3.6955232644686475e-05, 6.266668788157403e-05, 1.4852481626803637e-06, 0.0018016319954767823, 6.412423681467772e-05, 0.00013566236884798855, 1.7497501175967045e-05, 3.191154610249214e-05, 5.288829925120808e-06, 1.2227945262566209e-05, 5.132254500495037e-06, 0.00023892825993243605, 4.613481507931283e-07, 5.4437747166957706e-05, 0.00018664737581275403, 0.00020732633129227906, 5.708100616175216e-06, 6.488091003120644e-06, 7.070431456668302e-05, 2.0848527128691785e-05, 1.0032035788754001e-05, 0.00011130457278341055, 7.282196747837588e-05, 1.519230045232689e-05, 3.730937805812573e-06, 0.0007343211909756064, 0.00019268013420514762, 4.5267031964613125e-05, 3.6361641832627356e-05, 5.1175170483475085e-06, 8.791233085503336e-06, 5.574902388616465e-05, 5.521218554349616e-06, 0.0004400159523356706, 0.0011767999967560172, 0.0018080607987940311, 4.691785579780117e-05, 2.538251646910794e-05, 3.021647353307344e-05, 9.963373486243654e-06, 7.541589639004087e-06, 7.503328379243612e-05, 9.139933536062017e-05, 4.548812739813002e-06, 9.677957859821618e-05, 2.2470037947641686e-05, 0.0001996579230763018, 8.566753422201145e-06, 0.00015045468171592802, 3.779725739150308e-05, 1.968459173440351e-06, 1.6137646525749005e-05, 1.080887159332633e-06, 8.193589019356295e-05, 2.65622184087988e-05, 2.038085767708253e-06, 5.899700227018911e-06, 7.94566858530743e-07, 8.871240424923599e-05, 0.000373097718693316, 0.00016129440336953849, 0.00020413921447470784, 0.0002179165167035535, 4.13813249906525e-05, 5.2055795094929636e-05, 2.513050958441454e-06, 0.0006087386864237487, 0.0002654638374224305, 2.7160649551660754e-05, 1.3074708476779051e-05, 0.00014066167932469398, 6.943658718228107e-06, 0.0003335538785904646, 4.862328933086246e-05, 4.045837613375625e-06, 0.0005534405936487019, 8.356804755749181e-05, 0.00011632537643890828, 0.0009005286847241223, 1.805596002668608e-05, 4.4084117689635605e-06, 0.00036363894469104707, 1.8484317934053252e-06, 0.00023833499290049076, 6.719541488564573e-06, 0.0005167554481886327, 4.961452759744134e-06, 1.7161956975542125e-06, 0.0003436021215748042, 5.367802077671513e-05, 8.173065725713968e-05, 1.223813887918368e-05, 4.748127651055256e-07, 0.0001616260560695082, 6.100544396758778e-06, 3.4046475775539875e-05, 8.045195158956631e-07, 2.464862700435333e-05, 8.234786037064623e-06, 0.000134348520077765, 1.3965019206807483e-05, 0.0005974714877083898, 1.222247192345094e-06, 0.0006972013507038355, 0.0004336893616709858, 0.0009422367438673973, 0.0003800485283136368, 2.8989828933845274e-05, 4.762487151310779e-05, 1.0803613804455381e-05, 8.010015153558925e-05, 1.1837782949442044e-05, 1.5109384548850358e-05, 4.0116035961546004e-05, 1.571465531924332e-06, 0.0015489477664232254, 4.8865535063669086e-05, 0.0009000393329188228, 5.9083990890940186e-06, 0.0004774362896569073, 0.00048813348985277116, 1.4033334991836455e-06, 4.208962127449922e-05, 9.567944653099403e-05, 8.781840006122366e-05, 0.0002474068314768374, 0.0001762231986504048, 2.3072982003213838e-05, 0.000449023034889251, 0.00012790919572580606, 8.471123146591708e-05, 0.0007057491457089782, 3.914611170330318e-06, 9.66039442573674e-05, 2.66147999354871e-05, 0.00016097494517453015, 2.884990863094572e-05, 0.0010139140067622066, 5.8847188483923674e-05, 0.00017894001211971045, 5.17293628945481e-05, 1.2879873793281149e-05, 3.19584651151672e-05, 1.6412135664722882e-05, 2.8060279873898253e-05, 5.9084257372887805e-05, 3.862375524477102e-05, 0.0011233447585254908, 1.5025598258944228e-06, 8.221048483392224e-05, 2.9365066438913345e-05, 3.7103000067872927e-05, 0.001890062354505062, 0.0001095674087991938, 2.6837336918106303e-05, 3.2199845009017736e-05, 1.454637913411716e-05, 1.681897174421465e-06, 0.001626811339519918, 0.0007728777709417045, 1.0278607987856958e-05, 3.111138539679814e-06, 8.985840395325795e-06, 2.2593043468077667e-05, 0.0005939707043580711, 0.00032706110505387187, 7.427379750879481e-05, 6.536949513247237e-05, 5.006643732485827e-06, 2.835892519215122e-05, 0.00012962993059772998, 1.5479634384973906e-05, 2.5711977286846377e-05, 0.0005851365276612341, 0.00019261185661889613, 5.689914178219624e-05, 2.5229348921129713e-06, 5.335462265065871e-05, 7.518617621826706e-07, 1.7918815728990012e-06, 6.632257736782776e-06, 0.00015105018974281847, 4.225919838063419e-06, 4.236188033246435e-05, 0.0002523127768654376, 7.706266842433251e-06, 0.00016993361350614578, 1.2816438356821891e-05, 6.015870894771069e-06, 6.857859261799604e-05, 1.0036899766419083e-05, 3.952852057409473e-05, 0.0036906502209603786, 2.482552190485876e-06, 0.003435726510360837, 2.0271263565518893e-06, 1.1685824574669823e-07, 4.749812069348991e-05, 4.3772128265118226e-05, 2.45317532971967e-05, 0.00038498829235322773, 8.144247840391472e-05, 5.7439447118667886e-06, 3.4587952541187406e-05, 0.0004124505794607103, 0.00015108202933333814, 3.890279185725376e-05, 0.00017340420163236558, 0.0003537556913215667, 2.698004436751944e-06, 1.2021168913634028e-06, 8.366251859115437e-05, 0.0006923082983121276, 0.0001677164837019518, 9.017919364850968e-05, 1.1854834156110883e-05, 1.001139708023402e-06, 0.0001335819688392803, 1.6496893294970505e-05, 2.5944376830011606e-05, 1.0885733900067862e-05, 4.976474201612291e-07, 1.7934493371285498e-05, 0.0004869794356636703, 3.585418016882613e-05, 5.558150223805569e-05, 8.338130282936618e-05, 2.296045931871049e-05, 5.475960278999992e-06, 0.0002670966787263751, 6.672747986158356e-05, 0.0004196721129119396, 4.836293010157533e-05, 2.062133717117831e-05, 0.00012122997577534989, 5.849081503583875e-07, 1.1702764822985046e-05, 0.00026767558301799, 7.915265450719744e-05, 4.61447962152306e-05, 6.532823704219481e-07, 2.9775745133520104e-05, 2.190035411331337e-05, 2.909403519879561e-05, 0.0001366858632536605, 5.2435170800890774e-05, 0.00027472517103888094, 1.5940888715704205e-06, 2.1084368199808523e-05, 0.00016978282656054944, 8.973429430625401e-06, 4.566591087495908e-05, 8.089513721643016e-05, 0.0018419993575662374, 7.818537415005267e-05, 3.139737600577064e-06, 3.2962302611849736e-06, 3.992226629634388e-05, 0.00018077345157507807, 2.2055624867789447e-05, 0.00043758985702879727, 4.946100671077147e-05, 6.927036793058505e-06, 0.0011161532020196319, 4.5093667722539976e-05, 0.00023845460964366794, 1.399882330588298e-05, 4.006711606052704e-05, 8.635182166472077e-05, 1.8058384739561006e-05, 2.089711961161811e-05, 0.00015366385923698545, 1.5400228221551515e-05, 9.091296305996366e-06, 1.6381665773224086e-05, 0.0001467476540710777, 0.0003619141352828592, 2.6828813133761287e-05, 4.3429307083897584e-07, 9.401004717801698e-06, 3.352705607539974e-05, 9.641118958825245e-05, 0.0006121254991739988, 1.1083263871114468e-06, 1.4129860801403993e-06, 4.1076513298321515e-05, 0.0002979807904921472, 0.0003727999283000827, 1.3401262322076946e-07, 0.01724683865904808, 2.3731752662570216e-05, 4.605743015417829e-05, 0.0003476829151622951, 0.0008333055884577334, 2.3938287995406426e-05, 1.5899284335318953e-05, 0.0002999024582095444, 0.0006659924401901662, 3.876436676364392e-05, 0.0005026658182032406, 3.152685530949384e-05, 8.027477633731905e-06, 5.347213573259069e-06, 1.1938072930206545e-05, 1.5589901522616856e-05, 0.0006611209246329963, 9.943148324964568e-05, 4.685111707658507e-05, 0.00014287515659816563, 0.007434412371367216, 1.96143319044495e-05, 0.00045547776971943676, 1.0489018222870072e-06, 4.4436305870476644e-06, 1.810335561458487e-05, 1.2473386959754862e-05, 1.5763767805765383e-05, 4.621793414116837e-05, 5.880525804968784e-06, 1.3425804354483262e-05, 3.564214011930744e-06, 4.866043309448287e-05, 0.00044093633187003434, 3.79466509912163e-05, 0.00018923269817605615, 6.272785867622588e-06, 1.685035567788873e-05, 7.426243246300146e-05, 2.0925456283293897e-06, 1.4865229786664713e-05, 2.9447935503412737e-06, 1.9434599380474538e-05, 0.00010076395119540393, 3.026510967174545e-05, 6.8080339588050265e-06, 0.0005974730593152344, 4.7264307795558125e-05, 2.2601578166359104e-05, 4.638447717297822e-05, 8.20904733700445e-06, 3.09708048007451e-05, 0.00011833849566755816, 1.5398887853734777e-06, 5.014767521061003e-05, 1.3517435036192182e-05, 9.91433444141876e-06, 3.8223421142902225e-05, 0.00014947052113711834, 0.0035969004966318607, 6.937551370356232e-05, 0.00023248011711984873, 1.9335066099301912e-05, 0.005566813983023167, 0.0009226926486007869, 1.961246380233206e-05, 5.053031782153994e-05, 0.00011652902321657166, 9.692392268334515e-06, 0.0001052788647939451, 6.984449282754213e-05, 4.904527304461226e-05, 9.381160634802654e-05, 5.707500167773105e-05, 2.1664936866727658e-06, 0.00033063176670111716, 1.6390795281040482e-06, 0.0001774644770193845, 2.824238436005544e-05, 0.0007655699737370014, 5.2566749218385667e-05, 2.9000619178987108e-05, 0.00014012752217240632, 1.5238001651596278e-05, 0.0001344600023003295, 5.4574047680944204e-06, 9.500160558673088e-06, 3.0907678592484444e-05, 1.0337241292290855e-05, 2.0161547581665218e-05, 2.2405862182495184e-05, 4.0709514905756805e-06, 3.31709929923818e-06, 0.00012800974946003407, 8.324070950038731e-05, 0.00020954254432581365, 2.759853123279754e-05, 0.0002983954909723252, 1.0789031875901856e-05, 0.00013123861572239548, 0.0011233361437916756, 0.00010297026165062562, 1.6750322174630128e-05, 1.689302735030651e-05, 0.00039030189509503543, 0.00019539723871275783, 2.917582605732605e-05, 2.3869650249253027e-05, 3.528507659211755e-05, 1.9526240066625178e-05, 1.1247404472669587e-05, 3.8111655157990754e-05, 0.000398246047552675, 0.0001380675967084244, 6.216991096152924e-06, 4.387505669001257e-06, 0.0012132195988669991, 3.957382796215825e-05, 2.505780321371276e-05, 0.0010307702468708158, 1.2142331797804218e-06, 0.00015796591469552368, 6.429086170101073e-06, 0.0002904639404732734, 9.79940978140803e-06, 7.741338777123019e-05, 0.0006501121679320931, 1.3485345334629528e-05, 2.216800112364581e-06, 7.545950938947499e-05, 0.00029015992186032236, 0.00033112079836428165, 1.259171767742373e-05, 6.005594968883088e-06, 0.00011091706255683675, 6.856477739347611e-06, 3.716729906955152e-06, 0.0005251063266769052, 5.993800368742086e-05, 8.277616871055216e-06, 0.0032027943525463343, 4.586425347952172e-05, 0.001286124810576439, 0.0010912521975114942, 0.00016627914737910032, 8.432409231318161e-05, 3.2487707812833833e-06, 8.053972123889253e-06, 1.4764175375603372e-06, 4.579071173793636e-05, 6.281836704147281e-06, 0.00010127565474249423, 1.709829484752845e-05, 7.82384886406362e-05, 1.84204491233686e-05, 0.00018784712301567197, 6.348814622469945e-06, 4.237535449647112e-06, 0.00033337887725792825, 2.7678790502250195e-05, 0.0002878729428630322, 1.865104422904551e-05, 5.89974551985506e-05, 0.00046804011799395084, 6.607320301554864e-06, 9.858718840405345e-06, 3.3195312425959855e-05, 0.00013944397505838424, 0.00027579080779105425, 8.095046359812841e-05, 0.0008648538496345282, 4.699547389463987e-06, 2.2986474505160004e-05, 0.0002018170343944803, 0.0007150064338929951, 0.0005018198280595243, 0.00022211519535630941, 4.532991806627251e-05, 0.0002509686746634543, 4.55919471278321e-05, 6.311850302154198e-05, 0.00021877586550544947, 2.6310126486350782e-05, 1.6401001630583778e-05, 0.00011419787915656343, 8.179645192285534e-06, 9.761725959833711e-05, 0.00031899730674922466, 1.5039053096188582e-06, 0.001340458169579506, 2.001981920329854e-05, 4.729321153718047e-05, 6.73979302519001e-05, 3.202171137672849e-05, 9.622138350096066e-06, 5.872999736311613e-06, 0.00010601924441289157, 3.368627221789211e-05, 3.070454113185406e-05, 3.1821335142012686e-05, 4.7706387704238296e-05, 7.272802122315625e-06, 3.3005367185978685e-06, 4.691554295277456e-06, 0.0002702271449379623, 1.118463660532143e-05, 8.426834341435097e-08, 2.777841109491419e-05, 0.00040449804509989917, 1.0797423783515114e-05, 0.00044015920138917863, 0.0032044320832937956, 8.81829510035459e-06, 1.4312856364995241e-05, 2.307606700924225e-05, 1.3974528201288194e-06, 1.188259102491429e-05, 2.4419190594926476e-05, 8.788616833044216e-05, 6.635525642195717e-05, 9.441401743970346e-06, 6.20558494119905e-05, 5.817129203933291e-05, 8.498570241499692e-05, 8.728369721211493e-06, 0.00043753403588198125, 3.817772085312754e-05, 0.00015680058277212083, 0.00014673735131509602, 6.097692221374018e-06, 0.00026340706972405314, 4.771118710777955e-06, 0.0001607745507499203, 6.319969543255866e-05, 0.00024273773306049407, 4.8607387725496665e-05, 7.941055628180038e-06, 0.0005793296732008457, 2.2144092781672953e-06, 0.0001374213898088783, 8.818965579848737e-05, 0.0024000476114451885, 8.852169230522122e-06, 0.00013355270493775606, 3.11295225401409e-05, 5.131657235324383e-05, 4.530803198576905e-05, 5.487284852279117e-06, 5.685501673724502e-05, 9.372269232699182e-06, 5.776742909802124e-05, 1.2432462881406536e-07, 9.960767056327313e-05, 0.00013943089288659394, 0.00016030846745707095, 6.914602181495866e-06, 7.249906502693193e-06, 0.003948827274143696, 0.006158717442303896, 6.3074253375816625e-06, 0.0001652772625675425, 4.9280773964710534e-05, 0.00030695582972839475, 2.9599435947602615e-05, 7.868178727221675e-06, 0.0011336030438542366, 1.0103697604790796e-05, 7.859488505346235e-06, 1.585813151905313e-05, 0.00012048164353473112, 2.6878673452301882e-05, 5.101606802782044e-06, 0.0004940464277751744, 0.0003624215896707028, 8.413775503868237e-05, 3.0049707220314303e-06, 0.0010448630200698972, 6.956475772312842e-06, 2.756201138254255e-05, 2.6632710614649113e-06, 3.1647143714508275e-06, 7.228589674923569e-05, 4.9366783059667796e-05, 0.00033801532117649913, 1.1915486538782716e-05, 1.6965746851838048e-07, 0.0015378670068457723, 0.00011538749095052481, 2.41500106312742e-06, 2.103313454426825e-05, 5.5932025134097785e-05, 1.2151719602115918e-05, 1.5664596503484063e-05, 8.161200639733579e-06, 1.962836176971905e-05, 7.965725671965629e-06, 0.0001315548870479688, 0.0007234885124489665, 3.308095256215893e-05, 3.495116743579274e-06, 4.1453963603999e-06, 5.9552176026045345e-06, 0.001200372353196144, 0.00022798383724875748, 0.000297988677630201, 8.208269719034433e-05, 1.9605717170634307e-05, 8.966417954070494e-05, 2.119971213687677e-06, 3.3647476811893284e-05, 6.590929842786863e-05, 9.198956831824034e-06, 5.408483048086055e-06, 2.2755566533305682e-05, 0.00012018976121908054, 5.081025665276684e-05, 0.00027421311824582517, 2.4588450742157875e-06, 8.590217476012185e-06, 2.7411861083237454e-05, 0.0005066830781288445, 3.6755602650373476e-06, 2.880951569750323e-06, 1.2569398677442223e-05, 0.0003853547968901694, 0.00022873068519402295, 1.1673787412291858e-05, 0.00018511501548346132, 8.88217255123891e-05, 1.0711104550864547e-05, 3.819141784333624e-05, 0.00012491870438680053, 0.00020056705398019403, 7.509301940444857e-05, 4.280855137039907e-06, 7.540978549513966e-05, 0.00016109133139252663, 8.58146358950762e-06, 9.486418275628239e-05, 2.1444473532028496e-05, 0.007782649714499712, 0.0016588550060987473, 9.293446510127978e-07, 0.00016939466877374798, 0.00013301144645083696, 4.898842962575145e-05, 8.661562560519087e-07, 4.078478468727553e-06, 5.609335130429827e-05, 2.7554558528208872e-06, 8.288247386190051e-07, 0.00011688932863762602, 4.889947740593925e-05, 8.203566721931566e-06, 0.00024281155492644757, 2.079193654935807e-05, 6.0364291130099446e-06, 0.0002758694754447788, 3.869853117066668e-06, 0.00044510519364848733, 8.877638720150571e-06, 0.0018686625408008695, 8.86422349140048e-05, 0.0011205064365640283, 0.00018959060253109783, 6.348914780573978e-07, 1.88125795830274e-05, 0.000558649015147239, 2.528689401515294e-05, 5.102798240841366e-06, 4.582355472848576e-07, 1.6849606254254468e-05, 9.20557722565718e-05, 4.2389428926981054e-06, 9.077000868273899e-06, 0.0007112740422599018, 1.428061182195961e-06, 6.767289505660301e-06, 9.298367513110861e-05, 7.547710265498608e-05, 7.46746372897178e-05, 3.431318327784538e-05, 0.0006795413210056722, 6.97564537404105e-05, 9.31938920984976e-05, 5.627695645671338e-05, 0.00017600294086150825, 6.0180507716722786e-05, 0.0002805288240779191, 0.00010715901589719579, 2.520576526876539e-05, 0.0003218508791178465, 1.989318207051838e-06, 9.49376990320161e-05, 0.00021796362125314772, 1.4567931430065073e-05, 0.0012572580017149448, 3.973802904511103e-06, 7.433901191689074e-05, 1.8437061953591183e-05, 1.6798223441583104e-05, 2.7443849830888212e-05, 7.199816991487751e-06, 0.0006135908770374954, 3.31819974235259e-05, 4.467149494757905e-07, 0.0002601324813440442, 4.8301717470167205e-05, 4.821038601221517e-06, 1.388500436405593e-06, 9.021397090691607e-06, 3.668408317025751e-05, 5.052655978943221e-05, 0.00017813584418036044, 2.3872229576227255e-05, 9.537595906294882e-05, 3.9142796595115215e-06, 5.868615062354365e-06, 2.7476642571855336e-05, 1.059511851053685e-05, 1.0104068678629119e-05, 4.998213626095094e-05, 4.727212967736705e-07, 1.948473072843626e-05, 0.00034620746737346053, 0.0004570959717966616, 9.491145647189114e-06, 1.5868823538767174e-05, 5.242321276455186e-05, 3.1958063573256368e-06, 2.8839713195338845e-05, 2.4589502572780475e-05, 6.340656454995042e-06, 0.0004657796525862068, 8.193306712200865e-05, 1.0803572877193801e-05, 4.396213626023382e-06, 4.064293534611352e-06, 0.0026063129771500826, 1.0418151532576303e-06, 1.7993856090470217e-05, 3.1957263217918808e-06, 0.0001529594010207802, 5.350721949071158e-06, 3.2792574984341627e-06, 0.001013477216474712, 1.4014642147230916e-05, 5.644534212478902e-06, 2.6182671717833728e-05, 7.852393173379824e-05, 2.5150651708827354e-05, 8.873116894392297e-05, 3.390252459212206e-05, 1.528884058643598e-05, 5.430123201222159e-05, 6.653981927229324e-06, 0.00010628024756442755, 0.0007499406929127872, 0.0008797008777037263, 4.01773868361488e-05, 3.589236439438537e-05, 3.8511367165483534e-05, 3.6580302548827603e-05, 2.3945289285620674e-05, 1.4304917385743465e-05, 0.00010602868860587478, 3.885467322106706e-06, 0.00011345319944666699, 8.383388376387302e-06, 2.862559813365806e-05, 4.12094232160598e-06, 0.00019419849559199065, 3.6862129491055384e-05, 8.776461618253961e-05, 5.14676321472507e-05, 0.00011322772479616106, 0.0004615927755367011, 7.710880163358524e-05, 5.656762027683726e-07, 8.858757792040706e-05, 1.011921312965569e-06, 7.248467682074988e-06, 0.000351787923136726, 0.0006269795121625066, 0.0024743021931499243, 1.9139211872243322e-05, 5.9792937463498674e-06, 4.4420680751500186e-06, 3.4994583984371275e-05, 0.0007903727237135172, 3.3829949188657338e-06, 7.62969193601748e-06, 3.073347397730686e-05, 0.0002033137861872092, 6.674279262597338e-08, 1.6149588191183284e-05, 5.481352491187863e-05, 6.787307938793674e-05, 0.000124622747534886, 1.6349727957276627e-05, 0.0038931008893996477, 5.803519343317021e-06, 3.4797954867826775e-05, 4.104887557332404e-05, 1.4576702596968971e-05, 1.1903662198164966e-06, 2.002431938308291e-05, 0.0007862315396778286, 2.0660680092987604e-05, 0.00015414037625305355, 0.0020389112178236246, 0.0002326011162949726, 7.872129936004058e-05, 0.00017107102030422539, 0.00010303308226866648, 1.9420875105424784e-05, 4.637050551536959e-06, 3.731198739842512e-05, 6.786604353692383e-05, 1.4552711945725605e-05, 0.0002456927904859185, 1.5108865227375645e-05, 2.239160494355019e-06, 7.396417004201794e-06, 0.0002714642614591867, 3.176241989422124e-06, 6.079169543227181e-05, 3.641706143753254e-06, 3.3327345590805635e-05, 1.4368392839969601e-05, 1.174005137727363e-05, 3.527668013703078e-05, 0.00011863179679494351, 6.452299567172304e-05, 2.8044245482305996e-05, 2.6001769128924934e-06, 5.027992756367894e-06, 0.00011102292046416551, 0.00042332903831265867, 4.8193851398536935e-05, 0.00019458534370642155, 4.729079591925256e-05, 3.422920542561769e-07, 3.06804781757819e-06, 0.0004027949180454016, 7.60011316742748e-05, 0.019845793023705482, 6.268962897593156e-05, 0.0007084647077135742, 1.003710849545314e-06, 0.00038147333543747663, 2.4735849365242757e-05, 2.5278648536186665e-05, 0.000612775853369385, 2.7826920359075302e-06, 2.4921124349930324e-05, 9.194972517434508e-05, 3.248030770919286e-05, 0.00012164659710833803, 1.492609953857027e-06, 3.326246951473877e-05, 1.7588957916814252e-06, 0.0003002602607011795, 3.90951845474774e-06, 0.0005388660356402397, 2.10315702133812e-05, 8.671878276800271e-06, 0.0003793757932726294, 1.1371082109690178e-05, 0.00021346556604839861, 1.848622014222201e-05, 7.990010635694489e-05, 0.00016435656289104372, 0.00010223361459793523, 4.205026016279589e-06, 0.00035207706969231367, 0.00021264322276692837, 9.020738616527524e-06, 1.0810557796503417e-05, 0.000505748437717557, 7.617753453814657e-06, 0.00048500014236196876, 0.0002813930914271623, 0.001728408387862146, 1.8136726794182323e-05, 5.3995867347111925e-05, 1.4646226190961897e-05, 0.0020076148211956024, 0.00023197704285848886, 3.674389881780371e-05, 6.972216215217486e-05, 2.532870348659344e-05, 0.00019539003551471978, 7.890605047577992e-05, 8.876000356394798e-05, 3.8367623346857727e-05, 0.0014709916431456804, 1.659976624068804e-05, 6.344712164718658e-05, 0.00032941196695901453, 8.213677210733294e-05, 1.474544478696771e-05, 0.0004603801644407213, 6.956073048058897e-05, 0.00019556762708816677, 1.8093964172294363e-05, 6.509030208690092e-05, 0.00013104447862133384, 9.145987860392779e-06, 8.151220754371025e-06, 2.2470053409051616e-06, 8.693396375747398e-05, 2.3344819055637345e-06, 2.0581295757438056e-05, 1.2175089068477973e-05, 5.505293756868923e-06, 2.1010175714764046e-06, 4.873569560004398e-05, 0.0010032207937911153, 2.563118187026703e-06, 2.7268504709354602e-05, 6.373149972205283e-06, 1.760751001711469e-05, 7.095102046150714e-05, 3.206027258784161e-06, 2.2326735233946238e-06, 0.00029298706795088947, 5.9502549447643105e-06, 0.000136753122205846, 4.3729651224566624e-05, 2.575786129455082e-05, 0.0002537469263188541, 2.3636341666133376e-06, 0.00018599590111989528, 1.2247365702933166e-05, 2.4897517505451106e-06, 0.04694608226418495, 0.00424521928653121, 0.0015127573860809207, 0.0013609587913379073, 4.6943355300754774e-06, 2.8852181799265963e-07, 5.1092563808197156e-05, 0.00021529532386921346, 7.722711416136008e-06, 8.65401525516063e-05, 0.00010826172365341336, 0.0006354712531901896, 0.00028489489341154695, 2.309400588274002e-05, 1.7531921912450343e-05, 0.0005767685943283141, 7.598246156703681e-05, 0.00012895460531581193, 7.125850061129313e-07, 5.8432721061763e-07, 1.772798782440077e-06, 0.000482230941997841, 1.6256193703156896e-05, 2.5349912903038785e-05, 7.336572707572486e-06, 2.7353591576684266e-05, 7.773398465360515e-06, 0.00036838254891335964, 0.002877343213185668, 6.672633389825933e-06, 5.460273314383812e-06, 5.287784006213769e-06, 0.00011638631258392707, 2.24309551413171e-05, 2.7332541776559083e-06, 0.00030731476726941764, 1.0488715673773186e-07, 6.55567055218853e-05, 0.00025733228540048003, 4.793128027813509e-06, 0.0011590858921408653, 4.642777639674023e-05, 2.094534102070611e-05, 2.1561583707807586e-05, 1.3484774171956815e-05, 2.136427974619437e-05, 1.5165323020482901e-05, 6.385789106388984e-07, 0.0003388371260371059, 0.00012962256732862443, 7.846550033718813e-06, 0.00031055856379680336, 0.0004714788228739053, 6.391685019480065e-05, 2.6520141545915976e-05, 1.983981064768159e-06, 1.8433311197441071e-06, 5.187420538277365e-05, 4.46339890913805e-06, 2.496401430107653e-05, 0.0005824424442835152, 3.0870724003762007e-06, 5.308549225446768e-05, 4.846601950703189e-05, 3.953390478272922e-05, 7.10829908712185e-06, 0.00014059343084227294, 3.316286893095821e-05, 9.445778204053568e-08, 9.511215466773137e-06, 4.824055849894648e-06, 3.986163210356608e-05, 1.1269089554843958e-06, 4.171854743617587e-05, 0.00014688348164781928, 1.7255350712730433e-06, 2.0053967091371305e-05, 4.8122863518074155e-05, 8.26632822281681e-06, 0.0007272078655660152, 0.000420887372456491, 0.00014933670172467828, 3.5595171539171133e-06, 1.0195829418080393e-05, 7.470699347322807e-05, 1.8064763935399242e-05, 1.4451771676249336e-05, 0.000212309489143081, 1.6732068615965545e-05, 1.4052197911951225e-06, 3.7543231883319095e-05, 4.1584640712244436e-05, 4.032411379739642e-05, 4.022497159894556e-05, 1.925245214806637e-06, 5.740842243540101e-05, 6.337295053526759e-05, 8.19331944512669e-06, 3.1041383863339433e-06, 4.9784166549216025e-06, 3.1863553886068985e-05, 6.838423723820597e-05, 8.448087100987323e-06, 8.114155207294971e-05, 6.0938491515116766e-05, 8.06110165285645e-06, 2.4634326109662652e-05, 1.7570586351212114e-05, 5.3115950322535355e-06, 0.00016431565745733678, 5.790766590507701e-05, 5.696046173397917e-06, 0.0001392805716022849, 6.727151685481658e-06, 5.3556625061901286e-05, 4.482424174057087e-06, 6.179475894896314e-05, 5.1609538786578923e-05, 0.00026647187769412994, 1.2399535080476198e-05, 0.0001238770055351779, 6.938367005204782e-05, 1.4317625755211338e-05, 0.0005249865353107452, 0.0006898257997818291, 1.4294661923486274e-05, 5.777423666586401e-06, 5.3350926464190707e-05, 5.422226240625605e-05, 1.4210434528649785e-05, 1.0327612471883185e-06, 6.729728193022311e-05, 9.940783456841018e-06, 5.977687396807596e-05, 3.1188781576929614e-05, 1.8156128135160543e-05, 5.193479955778457e-05, 2.0945455617038533e-05, 0.0012735167983919382, 0.00016373493417631835, 2.85102469206322e-05, 5.2443137974478304e-06, 0.00033135092235170305, 1.676652937021572e-05, 1.7661248421063647e-05, 3.191350697306916e-05, 6.5875758537004e-07, 8.009256271179765e-05, 6.000449138809927e-05, 0.00021651270799338818, 4.4987518776906654e-05, 3.616353296820307e-06, 3.154327714582905e-05, 1.4696905736855115e-07, 8.575937681598589e-05, 0.0009247990092262626, 4.694170002039755e-06, 3.7854797483305447e-06, 2.2070864361012354e-05, 8.081039413809776e-05, 3.9728779483994e-06, 8.201447485589597e-07, 4.833717321162112e-05, 1.0611818652250804e-05, 0.00010668612958397716, 0.00025464745704084635, 2.5736675524967723e-05, 1.7443119304516586e-06, 8.566143151256256e-06, 0.00021322000247891992, 1.779302692739293e-05, 2.524079718568828e-05, 0.0010380431776866317, 1.9769566279137507e-05, 0.00015373718633782119, 7.17853254172951e-07, 8.075319783529267e-05, 7.544588697783183e-06, 1.3476184903993271e-05, 4.427251042216085e-06, 0.0009880902944132686, 1.5608893590979278e-05, 1.0775323062262032e-05, 1.221077582158614e-05, 1.4343743714562152e-05, 0.0001290422078454867, 3.0262979635153897e-05, 5.527066969079897e-05, 7.891481459409988e-07, 0.0003521670587360859, 1.8806989828590304e-05, 1.8253796270073508e-06, 2.503968516975874e-06, 1.3847189620719291e-05, 0.001305400743149221, 8.862676622811705e-06, 2.291281816724222e-05, 0.0019073563162237406, 0.005643763113766909, 0.00017571957141626626, 7.47181911719963e-05, 2.6391302526462823e-05, 0.00012284598778933287, 8.254253771156073e-05, 3.147623033328273e-07, 6.288134045462357e-06, 0.0008888799929991364, 0.00027207419043406844, 3.144067886751145e-05, 9.886124462354928e-05, 0.00013337488053366542, 1.007951777864946e-05, 4.805749631486833e-05, 2.906034239913424e-07, 0.00137561303563416, 0.006533198524266481, 0.0006917509599588811, 3.432213816267904e-06, 1.046181296260329e-06, 4.004363654530607e-05, 0.00011788355914177373, 5.616432463284582e-05, 6.213353481143713e-05, 1.5183104551397264e-06, 4.990622528566746e-06, 6.577654858119786e-05, 2.718720452321577e-06, 0.00037835066905245185, 3.96897394239204e-06, 7.728244781901594e-06, 4.9140249757329e-05, 6.28316456641187e-06, 4.4978347432333976e-05, 3.979473694926128e-05, 8.097643876681104e-05, 3.173745790263638e-05, 4.398304099595407e-06, 0.00323872035369277, 0.0003437643754296005, 4.383291889098473e-05, 7.123413979570614e-06, 1.488335237809224e-05, 0.0024325873237103224, 3.7043234897282673e-06, 0.000395252020098269, 5.993665126879932e-06, 5.826074630022049e-05, 0.0001244820305146277, 4.267059193807654e-05, 6.084760116209509e-06, 0.00018844266014639288, 6.636607849941356e-06, 4.6348261093953624e-05, 0.00021280204236973077, 0.00039294641464948654, 0.0003640309441834688, 5.4559304771828465e-06, 1.1783202353399247e-05, 9.685347322374582e-05, 1.0933382327493746e-05, 0.00023962721752468497, 1.3012190720473882e-05, 1.2682196938840207e-05, 1.2059174878231715e-05, 3.168922194163315e-05, 6.38666051600012e-06, 8.054802310653031e-05, 2.3544736905023456e-05, 0.000932103197555989, 5.656752819049871e-06, 6.1146570260461885e-06, 0.00021468588965944946, 3.1827989005250856e-05, 4.291480945539661e-05, 9.63178445090307e-06, 9.376063098898157e-05, 6.193212902871892e-05, 4.00612998419092e-06, 0.004302274901419878, 0.0014944336144253612, 2.4131628379109316e-05, 0.00012566466466523707, 6.274358383961953e-06, 0.0003425386385060847, 5.4115557759359945e-06, 0.000724468962289393, 8.555499334761407e-06, 1.3991234482091386e-05, 4.037305916426703e-05, 6.512480467790738e-05, 0.00012459098070394248, 0.0019493296276777983, 3.42594631774773e-07, 2.710374246817082e-05, 3.9775420646037674e-07, 0.00019626939320005476, 3.5575114452512935e-05, 0.000833926082123071, 1.528952816443052e-05, 3.7191952287685126e-05, 3.018589495695778e-06, 1.4880649814585922e-06, 3.5299597129778704e-06, 0.00011516798986122012, 2.038310412899591e-06, 0.00011379118222976103, 0.0010958741186186671, 2.3843949747970328e-05, 1.2341067304078024e-05, 1.457779944757931e-05, 0.00031256640795618296, 6.711126388836419e-06, 1.8140730389859527e-05, 7.341390642068291e-07, 3.4533597954578e-06, 2.4711349055905885e-07, 2.1667881355824647e-06, 1.0426163498777896e-05, 2.246137000838644e-07, 2.9963939596200362e-05, 2.1757694412372075e-05, 3.6802084650844336e-05, 7.145888048398774e-06, 0.003926691133528948, 0.00026986628654412925, 1.476530269428622e-05, 5.565398168982938e-06, 0.0001967008865904063, 3.1512448913417757e-05, 0.00014676450518891215, 0.0007601940305903554, 7.68452537158737e-06, 0.0002704304934013635, 7.245473534567282e-05, 1.1733368410205003e-06, 5.826781489304267e-06, 3.265835402999073e-05, 4.665130290959496e-06, 6.038386572981835e-07, 5.107278047944419e-05, 0.00011898581578861922, 6.635741556237917e-06, 9.544854947307613e-06, 1.4332962564367335e-05, 3.0002881885593524e-06, 1.619373324501794e-05, 6.566482170455856e-06, 1.8061842865790823e-06, 3.281087992945686e-05, 0.0004847831733059138, 2.9378685212577693e-05, 1.0443472092447337e-05, 7.192905468400568e-05, 2.419190195723786e-06, 0.000625402492005378, 1.633539613976609e-05, 6.164660590002313e-06, 5.993916147417622e-06, 3.915228262485471e-06, 0.00011961710697505623, 9.758722626429517e-06, 3.538915188983083e-05, 3.30654984281864e-05, 7.856950105633587e-05, 0.0001270479115191847, 5.201179374125786e-05, 0.00048608321230858564, 4.8749250709079206e-05, 4.591332981362939e-05, 3.0252645956352353e-05, 9.094844426726922e-05, 0.0002509485057089478, 0.0007281686994247139, 6.552420018124394e-06, 5.387454621086363e-06, 4.953558800480096e-06, 3.1112813303479925e-05, 0.00010706351895350963, 7.248188194353133e-05, 1.9079695903201355e-06, 8.44943497213535e-06, 0.0011500841937959194, 3.652332816272974e-05, 1.103413615055615e-05, 0.00018321422976441681, 2.7248161131865345e-05, 6.596580533368979e-06, 4.513275416684337e-05, 3.2172856663237326e-06, 1.5880745195318013e-05, 0.00013511728320736438, 1.5381436242023483e-05, 0.00018598027236294, 5.780179890280124e-06, 3.967753400502261e-06, 2.5291128622484393e-05, 1.913827145472169e-05, 9.997511369874701e-05, 2.9607347187265987e-06, 0.001637229579500854, 0.0025934050790965557, 1.7078027667594142e-05, 0.0024030834902077913, 6.260022200876847e-05, 3.1066654628375545e-05, 0.0023365579545497894, 0.0001068852943717502, 1.3767188647761941e-05, 0.0016438245074823499, 6.944568440303556e-07, 1.2954395742781344e-06, 5.293215963320108e-06, 1.4227066458261106e-06, 1.1120921044494025e-05, 0.0002760017814580351, 0.00014659563021268696, 4.701363195636077e-06, 1.4612245422540582e-06, 8.509572217008099e-05, 6.092125204304466e-06, 0.0005471808835864067, 1.1484621609270107e-05, 5.557249824050814e-05, 9.548513844492845e-06, 4.3514908611541614e-05, 0.0004216198285575956, 0.00016531375877093524, 3.824618033831939e-05, 0.0005956101813353598, 8.944888395490125e-05, 1.8335443883188418e-06, 0.0003801596467383206, 1.3968029861644027e-06, 4.7322162572527304e-05, 4.6462268073810264e-05, 4.3086409277748317e-05, 0.0008652537944726646, 9.117708395933732e-05, 2.723778561630752e-05, 1.795754360500723e-05, 0.00026461720699444413, 9.51491529121995e-06, 0.0034863899927586317, 0.000640114420093596, 7.275668031070381e-05, 8.721694939595181e-06, 0.0001580524112796411, 1.0492773071746342e-05, 9.917506395140663e-05, 0.00016214711649809033, 3.0317818527691998e-05, 2.1612308046314865e-05, 0.0002718638861551881, 0.00035563981509767473, 8.400555088883266e-06, 1.1033475857402664e-05, 0.0005036837537772954, 1.4476480600933428e-06, 0.00047403640928678215, 1.010246342048049e-05, 8.48403124109609e-06, 4.635845107259229e-05, 7.249238478834741e-06, 1.823372622311581e-05, 3.7953403079882264e-05, 1.804710154829081e-05, 0.00010950072464765981, 0.00047073527821339667, 1.478622039030597e-06, 1.5183300092758145e-05, 2.7101478394797596e-07, 0.0001158215309260413, 7.53869289837894e-06, 6.810713966842741e-05, 2.736652641033288e-05, 0.0005493120406754315, 1.5216137398965657e-05, 0.00021646104869432747, 0.007241684943437576, 3.312349508632906e-05, 0.00023874003090895712, 0.00034551694989204407, 1.7186748664244078e-05, 2.2353813164954772e-06, 2.467283957230393e-05, 0.00021849146287422627, 0.00012870288628619164, 1.5615883967257105e-05, 0.00019465900550130755, 0.00015346419240813702, 2.2108173652668484e-05, 0.00019273882207926363, 1.6624884665361606e-05, 4.4806962250731885e-05, 0.0006250567967072129, 1.3181746908230707e-05, 0.0007932338630780578, 0.00011312803690088913, 0.0006303568952716887, 5.205908109928714e-06, 3.3915493986569345e-05, 0.0003951827820856124, 3.766910595004447e-05, 0.00032911865855567157, 1.5669397726014722e-06, 6.222540832823142e-05, 0.0001823870261432603, 3.576998642529361e-05, 2.872772529372014e-05, 6.451151421060786e-05, 1.021334082906833e-05, 0.00029889182769693434, 0.0034992743749171495, 0.0007039031479507685, 0.00010550239676376805, 1.1949038707825821e-05, 8.170276851160452e-05, 3.779923281399533e-05, 0.00019762379815801978, 0.0007318295538425446, 0.0012355046346783638, 5.878344381926581e-05, 1.997947265408584e-06, 3.609851410146803e-05, 5.178672290639952e-05, 0.0005904403515160084, 2.1268238924676552e-05, 0.00044473021989688277, 0.00010182546247961, 0.0007561538368463516, 0.00019788167264778167, 7.693922088947147e-05, 0.0002969195193145424, 0.00011037895455956459, 5.322206561686471e-05, 3.7192073705227813e-06, 0.0006061235908418894, 7.199925676104613e-06, 6.263015620788792e-06, 4.679089761339128e-05, 0.0013534071622416377, 0.00010066960385302082, 1.38653410886036e-06, 1.138385687227128e-05, 1.3539279279939365e-05, 0.0001843285863287747, 0.00013848993694409728, 3.4194467843917664e-06, 1.6944170056376606e-05, 0.0003234057512599975, 2.924050932051614e-05, 7.075999747030437e-06, 2.8994540116400458e-05, 3.9458454921259545e-06, 0.002288153627887368, 5.2320949180284515e-05, 1.084517043636879e-05, 1.8930606529465877e-05, 1.2293258805584628e-05, 0.0011472707847133279, 2.3251825496117817e-06, 5.7076711527770385e-05, 5.640740710077807e-05, 4.907601123704808e-06, 0.00017101105186156929, 0.0002500802220311016, 9.658718227001373e-06, 8.242546556402885e-08, 0.00014507623563986272, 1.9472636267892085e-05, 4.429048203746788e-05, 1.2567912563099526e-05, 3.698404543683864e-05, 1.0394552191428374e-05, 1.1085123787779594e-06, 3.673511673696339e-05, 1.1061843906645663e-05, 1.2051887097186409e-05, 3.831727372016758e-05, 2.8076296075596474e-05, 0.0002511855273041874, 0.08695755153894424, 1.6202251572394744e-05, 1.1167204320372548e-05, 0.00010208790627075359, 0.0009858300909399986, 6.962761108297855e-05, 9.628924090065993e-06, 0.0001914080639835447, 5.221391984377988e-05, 8.131743925332557e-06, 2.0328739992692135e-05, 7.919967902125791e-05, 0.0006102623301558197, 0.0011562019353732467, 1.6262010831269436e-05, 2.3839036202844e-06, 2.8969921913812868e-05, 0.00305181578733027, 0.00014912307960912585, 0.0011951777851209044, 3.540234683896415e-05, 6.029829091858119e-05, 0.00012543615594040602, 0.0007961945957504213, 0.0002597382990643382, 0.0015413026558235288, 0.0003652936138678342, 0.0011840729275718331, 1.3433977073873393e-05, 0.0008414913900196552, 8.876257197698578e-05, 3.0160063033690676e-05, 9.29145153349964e-06, 4.4458472984842956e-05, 0.00043135820305906236, 1.2138706551922951e-05, 0.0005374926258809865, 1.2310961210459936e-05, 3.024650322913658e-05, 0.00016806302301120013, 3.6020043125972734e-07, 4.232769788359292e-05, 2.104311715811491e-05, 4.612944849213818e-06, 1.6133575400090194e-06, 7.442504283972085e-05, 6.155831215437502e-06, 4.826380973099731e-05, 3.340037210364244e-06, 0.0009525530622340739, 1.1540787454578094e-05, 0.0001768788497429341, 0.0002837649662978947, 1.0651234561009915e-06, 1.3295867574925069e-05, 3.322392512927763e-05, 1.9099605196970515e-05, 0.00018602743512019515, 4.0901588249653287e-07, 3.036723683180753e-05, 0.0004354176053311676, 1.2007363693555817e-05, 2.262123416585382e-06, 1.1662839369819267e-06, 1.6176147710211808e-06, 9.251826122635975e-05, 4.5222827793622855e-06, 0.00015461797011084855, 7.665771590836812e-06, 1.7147833204944618e-05, 4.902353248326108e-05, 7.272629432009126e-07, 9.827848771237768e-06, 1.7340722479275428e-05, 8.942360000219196e-05, 1.1306775377306622e-05, 0.00011045653809560463, 2.3876742488937452e-05, 6.083876996854087e-06, 2.4276598196593113e-05, 0.0011860643280670047, 0.00013123889220878482, 9.41325124586001e-05, 4.998167423764244e-05, 2.5743100195541047e-05, 0.00020682605099864304, 1.5448991689481772e-05, 7.49285709389369e-06, 3.092906354140723e-06, 1.3646715160575695e-05, 0.00027937220875173807, 1.809580862754956e-05, 2.6931718821288086e-05, 9.594672883395106e-05, 1.5201429960143287e-05, 3.0923838494345546e-05, 6.42826898911153e-06, 4.538671873888234e-06, 0.0004216081870254129, 0.00010814610868692398, 0.0024246093817055225, 2.5679442842374556e-05, 1.2548880476970226e-05, 1.5052333765197545e-05, 7.643297976756003e-06, 2.596661659026722e-07, 7.484687375836074e-05, 0.0007270142086781561, 9.558640158502385e-05, 1.0228893643215997e-06, 6.965054490137845e-05, 2.9963299311930314e-05, 3.188777918694541e-05, 4.1791041439864784e-05, 1.2012634442726267e-06, 5.6338826652790885e-06, 8.28664815344382e-06, 2.6120307666133158e-05, 1.594476202626538e-06, 3.879151336150244e-05, 3.1357262741948944e-06, 0.0009131694096140563, 8.790770516498014e-05, 3.2680454751243815e-06, 0.00017102767014876008, 0.00021706384723074734, 5.21675565323676e-06, 2.6275394702679478e-05, 5.51126618120179e-07, 2.0769490220118314e-05, 7.729256321908906e-05, 2.793651901811245e-06, 0.0001750143856042996, 6.28781253908528e-06, 2.340217906748876e-05, 2.4055239009612706e-06, 3.7679881643271074e-05, 0.00024129632220137864, 5.668102403433295e-06, 2.082132596115116e-05, 0.00016191654140129685, 0.0004475709283724427, 0.0005216668942011893, 5.077473360870499e-06, 2.272922301926883e-06, 0.00010314112296327949, 7.801268111506943e-06, 0.0004726840415969491, 0.00012803169374819845, 2.1041805666754954e-05, 0.00012490528752095997, 1.227577740792185e-05, 0.0016588835278525949, 3.1919578304950846e-06, 4.941689985571429e-05, 9.27397413761355e-05, 0.0003241047670599073, 4.341348812886281e-06, 0.001201439998112619, 4.175647973170271e-06, 0.00022841397731099278, 0.00011953524517593905, 8.887002877600025e-06, 0.00019825140770990402, 5.908833190915175e-05, 2.7638612664304674e-05, 0.0003562697093002498, 6.141347694210708e-05, 0.00011246849317103624, 8.377366611966863e-05, 0.00015765138959977776, 3.481973544694483e-05, 5.71950158700929e-06, 2.10728649108205e-06, 0.0008799366769380867, 5.421800324256765e-06, 0.0009437747648917139, 5.512738061952405e-05, 4.710423672804609e-05, 0.0006286751595325768, 4.544741386780515e-06, 0.00019536397303454578, 7.322376768570393e-05, 2.5839608497335576e-05, 1.743419488775544e-05, 2.3506847810494946e-06, 2.1027637558290735e-05, 1.1603947314142715e-06, 1.4477650438493583e-06, 4.7362632358272094e-06, 5.1013639676966704e-06, 1.576528688929102e-06, 4.2804499571502674e-06, 0.0005280468612909317, 0.001069317921064794, 9.802935528568923e-05, 0.004318726249039173, 8.05984927865211e-06, 1.6141697415150702e-05, 3.0039867851883173e-05, 1.8762346371659078e-05, 8.825392797007225e-06, 2.574317477410659e-06, 2.3107513698050752e-05, 2.3612460609001573e-06, 0.0008227468351833522, 1.345077089354163e-05, 1.2096171303710435e-05, 1.5601294762745965e-06, 1.610467370483093e-05, 6.298004882410169e-05, 5.0195132644148543e-05, 1.863086254161317e-05, 1.9236154003010597e-06, 0.00022077369794715196, 7.688770892855246e-06, 3.6764015476364875e-06, 1.1631304914772045e-05, 3.033329085155856e-05, 1.50299158008238e-07, 6.780528201488778e-05, 0.00017572662909515202, 2.4546991426177556e-06, 2.6092875486938283e-05, 0.0005943736177869141, 0.0005040658870711923, 1.3787952411803417e-05, 6.438526725105476e-06, 2.9221675958979176e-06, 9.843865700531751e-05, 3.224752072128467e-05, 0.00022106568212620914, 0.00012663840607274324, 0.00012922151654493064, 1.5161911505856551e-05, 0.00017101388948503882, 5.352885779075223e-08, 0.00015880529826972634, 0.0005872888723388314, 1.4680607591799344e-06, 2.083943400066346e-06, 0.00015628553228452802, 4.200360308459494e-07, 7.284589628397953e-06, 1.0608668162603863e-05, 2.0214231710724562e-07, 1.0255977031192742e-05, 9.109505481319502e-05, 0.00021580666361842304, 0.001032576197758317, 0.0013842396438121796, 5.387479541241191e-05, 3.903099695889978e-06, 6.554450374096632e-05, 3.2307941637554904e-06, 7.034389909676975e-06, 0.00013636508083436638, 8.641691238153726e-06, 1.3515043974621221e-05, 0.00022130015713628381, 1.5780431112943916e-06, 1.5884548702160828e-05, 0.00042605865746736526, 1.1140594324388076e-05, 1.8300335113963229e-06, 2.734476038313005e-05, 2.9493621696019545e-05, 8.221384632633999e-06, 5.525236701942049e-05, 2.066299657599302e-06, 4.0389616628999647e-07, 3.951233156840317e-05, 5.989747660350986e-05, 1.3365118320507463e-05, 3.9031041524140164e-05, 3.982188900408801e-06, 4.529932630248368e-05, 0.00015453077503480017, 2.407061583653558e-05, 2.388819666521158e-05, 5.0949798605870456e-05, 5.436860192276072e-06, 1.2346197763690725e-05, 8.616348736723012e-07, 0.0002717107709031552, 2.9765902581857517e-05, 1.9137034541927278e-05, 5.2832333494734485e-06, 2.4244624000857584e-05, 0.0007055058376863599, 0.0009320792159996927, 3.826638567261398e-05, 0.0009898383868858218, 5.901522854401264e-06, 0.00011967037426074967, 9.017326374305412e-05, 1.8040655049844645e-05, 6.268264405662194e-05, 7.497500291719916e-07, 2.5974457457778044e-05, 5.447512194223236e-06, 0.0002726311213336885, 0.00029348177486099303, 5.585858161794022e-06, 7.981152884894982e-05, 0.0003076473076362163, 1.010844516713405e-05, 2.7317303192830877e-06, 0.00012181267811683938, 4.287039701011963e-05, 1.8376536900177598e-05, 6.412181392079219e-06, 6.892441160744056e-05, 7.138069486245513e-05, 0.0003149154072161764, 2.8108064725529402e-05, 3.1944687179930042e-06, 4.851636731473263e-06, 7.80226571350795e-07, 5.469379630085314e-06, 0.00010026519885286689, 5.5554239224875346e-05, 0.00021701586956623942, 1.680118657532148e-05, 6.90259548719041e-05, 4.8916583182290196e-05, 8.822469681035727e-06, 6.243278676265618e-06, 1.4640447261626832e-05, 9.483876056037843e-06, 0.00022970382997300476, 5.80972118768841e-05, 5.6291657529072836e-05, 4.174612331553362e-05, 2.8151773676654557e-06, 7.376204303000122e-05, 0.000629617425147444, 1.5837299542909022e-06, 8.644008744340681e-07, 9.793769095267635e-06, 6.562633188877953e-06, 1.1640609045571182e-05, 0.0018116345163434744, 2.0641018636524677e-05, 6.898487299622502e-06, 0.00013963256787974387, 3.407397525734268e-05, 2.886742731789127e-05, 5.061360207037069e-05, 5.82041175221093e-05, 8.722144002604182e-07, 4.243981311446987e-05, 7.215562072815374e-05, 7.605129667354049e-06, 0.00040593231096863747, 6.260657391976565e-05, 2.3511929612141103e-05, 6.200181087478995e-05, 0.00015592343697790056, 0.00040700577665120363, 8.011620229808614e-05, 4.485248791752383e-05, 0.004726848565042019, 9.894704817270394e-06, 3.0302420782390982e-05, 0.00011718959285644814, 0.00015562832413706928, 1.3757824490312487e-05, 0.0005600869772024453, 1.7832659068517387e-05, 5.2210184549039695e-06, 3.4627933928277344e-05, 4.599178282660432e-05, 0.00021578821178991348, 2.7007004973711446e-05, 2.0579976990120485e-05, 8.366858310182579e-06, 3.665712029032875e-06, 0.0003595960442908108, 5.1266040827613324e-05, 0.0010321339359506965, 5.065929872216657e-05, 4.695485131378518e-06, 1.533914837636985e-05, 0.0002026203292189166, 6.87974679749459e-05, 0.0016447853995487094, 2.631630741234403e-05, 3.2991127227433026e-05, 3.5430075513431802e-06, 2.45195951720234e-06, 0.00013132199819665402, 0.0004260888381395489, 8.956265446613543e-06, 3.4283552849956322e-06, 0.00015814219659660012, 0.00013508196570910513, 5.311945278663188e-05, 0.00017805903917178512, 0.0013521724613383412, 2.4270053472719155e-05, 8.973858712124638e-06, 7.626541901117889e-06, 0.000561976688914001, 2.919654070865363e-05, 8.77521961228922e-06, 0.0008904394344426692, 1.2979130588064436e-06, 0.0002873186895158142, 1.2752029761031736e-05, 5.18686392751988e-05, 1.9870935830113012e-06, 2.2636086214333773e-06, 2.0905321434838697e-05, 0.0012399378465488553, 0.0019070998532697558, 0.00038920159568078816, 5.240670725470409e-05, 5.413900453277165e-06, 9.862069418886676e-05, 0.00010756907431641594, 0.0017087634187191725, 1.6000118193915114e-05, 1.4644804650743026e-05, 7.541366358054802e-05, 3.6031512991030468e-06, 7.37676236894913e-05, 0.00041124800918623805, 3.9092457882361487e-05, 0.00019934891315642744, 0.0003841874422505498, 6.716675125062466e-05, 3.5597593523561954e-05, 4.6108914830256253e-05, 1.234894443769008e-05, 0.0008372222655452788, 5.4288117098622024e-05, 0.000305931200273335, 0.00010246057354379445, 2.275042606925126e-05, 7.607380211993586e-06, 1.3725742064707447e-05, 6.28276836778241e-07, 2.9580949558294378e-05, 2.4914117602747865e-05, 2.9542847187258303e-05, 3.526369982864708e-05, 0.0008546769386157393, 1.299310679314658e-05, 7.126980199245736e-05, 0.0004935914766974747, 1.5170019651122857e-05, 1.7729142200550996e-05, 7.358133871093742e-07, 4.710908615379594e-05, 2.3488075385103002e-05, 6.495205889223143e-05, 9.870999929262325e-05, 0.0004642845597118139, 0.00015082245226949453, 1.7833546735346317e-05, 0.00010802552424138412, 6.811491402913816e-06, 7.795164492563345e-06, 4.7288998757721856e-05, 0.007451192010194063, 0.00047068073763512075, 5.404385956353508e-05, 3.0528342449542833e-06, 0.009067857638001442, 2.5667468435131013e-06, 2.441673495923169e-05, 2.8360758733469993e-05, 0.0001047688492690213, 8.329540833074134e-06, 7.571173682663357e-06, 2.424692411295837e-06, 1.914010317705106e-05, 5.544894520426169e-06, 6.747360203007702e-06, 1.1741999514924828e-06, 0.0004640846746042371, 2.530446408854914e-06, 4.00825883843936e-05, 3.02843218378257e-05, 4.366897701402195e-05, 2.6158031687373295e-05, 0.006458076182752848, 4.095919939572923e-05, 4.086567884087344e-08, 0.00018445271416567266, 7.301883215404814e-06, 5.9328276620362885e-06, 4.595207792590372e-05, 0.0017449065344408154, 3.478984217508696e-05, 4.293119491194375e-05, 8.388467904296704e-06, 2.0919519556628074e-06, 1.3269514965941198e-05, 0.00011555132368812338, 6.203536031534895e-05, 4.433600224729162e-06, 3.6487144825514406e-06, 1.89846869034227e-05, 0.0005447158473543823, 9.149905963568017e-06, 9.437942935619503e-06, 4.271677244105376e-05, 0.0023962054401636124, 4.4469419663073495e-05, 4.501406692725141e-06, 5.5670179790467955e-06, 2.1700578145100735e-05, 0.00018072105012834072, 0.0005966433091089129, 2.342329935345333e-05, 0.0007155445055104792, 2.1311956516001374e-05, 8.109147165669128e-05, 0.00011733902647392824, 0.0010210138279944658, 1.7560916603542864e-05, 0.00025431241374462843, 0.0001622535492060706, 1.3792234312859364e-05, 0.0004251064092386514, 0.0006569025572389364, 3.739642124855891e-05, 7.230738447105978e-06, 6.10880961176008e-05, 4.4637836253969e-05, 6.204141755006276e-06, 8.753995643928647e-05, 2.314823359483853e-05, 0.00010847040539374575, 9.127367775363382e-06, 0.0001569930464029312, 0.001959293382242322, 2.0562231384246843e-06, 7.611412229380221e-07, 4.560647357720882e-05, 0.00020794951706193388, 0.0001408253301633522, 7.193897181423381e-05, 7.358918810496107e-05, 0.00014110634219832718, 0.0001122768662753515, 0.0015617209719493985, 6.063834462111117e-06, 3.119104803772643e-05, 5.62746072318987e-06, 6.584585207747295e-05, 3.484186891000718e-05, 0.00013192780897952616, 2.0712799596367404e-05, 0.0001013488945318386, 8.579372661188245e-07, 6.714187293255236e-06, 0.00016331704682670534, 2.077367389574647e-05, 0.00026136261294595897, 1.4222880054148845e-05, 1.731453085085377e-05, 0.0006698461365886033, 0.0004593453777488321, 3.803517029155046e-05, 3.0541690648533404e-05, 6.921788008185104e-05, 5.6930730352178216e-05, 5.663225692842389e-07, 7.89787809480913e-05, 1.1186311894562095e-06, 0.00022964246454648674, 1.6500236597494222e-05, 3.135985764401994e-07, 0.00011454063496785238, 4.192568303551525e-05, 3.034021756320726e-05, 9.061169112101197e-05, 5.294674565448076e-07, 4.7213188736350276e-06, 7.111374998203246e-06, 0.00013004646461922675, 2.344154609090765e-06, 2.2735068341717124e-05, 1.5882540537859313e-05, 0.0008601454319432378, 2.2917072328709764e-06, 2.153809145966079e-05, 7.919004565337673e-05, 2.3623890228918754e-05, 0.0009570056572556496, 2.131695691787172e-05, 9.484778274782002e-06, 7.776365237077698e-05, 1.384882580168778e-05, 4.835760137211764e-06, 8.964488370111212e-05, 3.54662915924564e-06, 1.70184484886704e-05, 9.277081517211627e-06, 8.832527237245813e-05, 9.626730843592668e-07, 1.7048414520104416e-05, 7.826638466212898e-06, 7.058212503352479e-08, 1.41090758916107e-05, 5.457499355543405e-06, 0.001361739356070757, 4.569484190142248e-06, 2.720822158153169e-05, 0.00011205649207113311, 2.0752283944602823e-06, 0.0004279141139704734, 0.00013092177687212825, 3.630020364653319e-05, 1.6699692423571832e-05, 2.20788388105575e-05, 5.29374519828707e-05, 1.2284911463211756e-05, 0.00011528026516316459, 0.00018510886002331972, 0.0001109526856453158, 1.8971972167491913e-05, 2.5726576495799236e-05, 2.9631906727445312e-05, 0.0008543550502508879, 0.001469378941692412, 4.3230877054156736e-05, 1.566172250022646e-05, 3.3963967780437088e-06, 3.4484753996366635e-05, 0.0020521667320281267, 2.961490099551156e-05, 3.404394419703749e-06, 9.81467110250378e-06, 6.430053326766938e-05, 3.422804002184421e-05, 0.00011610335786826909, 0.00013355635746847838, 1.015905672829831e-05, 0.00017659066361375153, 7.302487574634142e-06, 1.2638859516300727e-05, 0.00014171050861477852, 9.136953849520069e-06, 2.0923367628711276e-05, 0.0003011044464074075, 4.6659835788886994e-05, 4.5283144572749734e-05, 5.323264122125693e-05, 0.0018236328614875674, 0.00015277144848369062, 7.888145773904398e-05, 9.053463145392016e-05, 6.562775524798781e-05, 0.00013496003521140665, 9.0716865088325e-05, 0.00011517803795868531, 1.3148212019586936e-05, 4.421893390826881e-05, 3.5158700484316796e-05, 0.00039640782051719725, 0.001141754095442593, 1.020458421407966e-05, 2.8587543056346476e-05, 3.266248313593678e-05, 3.849306085612625e-05, 4.856609621128882e-07, 1.2713118167084758e-06, 4.666688710130984e-06, 1.691522447799798e-05, 2.0886056518065743e-05, 0.00021078875579405576, 3.6007826565764844e-05, 3.0000685001141392e-05, 1.6964559108600952e-05, 2.0041337847942486e-05, 0.0002556992694735527, 2.53811613220023e-05, 2.316528025403386e-06, 4.047225047543179e-06, 3.563220889191143e-05, 5.418896398623474e-05, 5.7482789088680875e-06, 1.8539532220529509e-06, 1.1043489394069184e-06, 1.9537365005817264e-06, 0.0004604595305863768, 1.417627754563e-05, 2.386694359302055e-05, 1.670054189162329e-05, 1.5989572830221732e-06, 2.5488225219305605e-06, 3.317324626550544e-06, 9.487241641181754e-07, 1.6558702554902993e-05, 3.1771940030012047e-06, 5.1857354264939204e-05, 3.490865128696896e-05, 0.0005609244690276682, 3.752966676984215e-06, 0.00014499970711767673, 9.681451729193213e-07, 7.021959390840493e-06, 5.7775640016188845e-05, 5.35825893166475e-05, 1.2988531352675636e-06, 1.8841175915440544e-05, 0.00026875842013396323, 0.00013643671991303563, 0.0001471206924179569, 3.634471067925915e-05, 0.00015568213711958379, 9.307346044806764e-05, 1.631943268876057e-05, 0.0002777989720925689, 3.0411776606342755e-05, 4.280886423657648e-05, 1.3515070804714924e-06, 0.0001486233959440142, 4.619461378752021e-06, 4.7140194510575384e-05, 4.9292142648482695e-05, 1.3739120731770527e-05, 5.5995733418967575e-05, 0.00024198605387937278, 2.341375193282147e-06, 1.622134368517436e-05, 3.8807429518783465e-06, 0.00034426929778419435, 6.862107693450525e-05, 0.00018128633382730186, 3.0448377401626203e-06, 2.4268047127407044e-05, 5.6103948736563325e-05, 1.5309653917938704e-06, 0.00023140903795138001, 4.7128637561399955e-06, 3.537108932505362e-05, 0.00010412361734779552, 0.0007118535577319562, 6.6438801695767324e-06, 1.6412766854045913e-05, 3.542518970789388e-05, 3.2190573620027862e-06, 2.6592848371365108e-05, 1.633411670809437e-06, 0.00012774261995218694, 0.00036879716208204627, 7.483914032491157e-06, 0.0005795096512883902, 7.663597352802753e-05, 1.3383797522692475e-05, 0.0001413610443705693, 3.1241557735484093e-05, 7.985865522641689e-05, 1.2948026778758503e-05, 6.755004869773984e-05, 2.4171497443603585e-06, 0.000312963267788291, 7.289316999958828e-05, 5.563428931054659e-05, 2.7259806302026846e-05, 1.0392695912742056e-05, 0.0009251011651940644, 0.0004395325086079538, 1.2179394616396166e-05, 6.474078691098839e-05, 1.0026139534602407e-05, 2.999534444825258e-05, 2.8237242077011615e-05, 9.284121915698051e-06, 1.0956568075926043e-05, 6.788857717765495e-06, 5.954293101240182e-06, 1.9589008388720686e-06, 2.58269501500763e-05, 8.666756912134588e-06, 6.06442081334535e-05, 4.3106620069011115e-06, 3.667806231533177e-05, 8.514757610100787e-06, 2.3180708012660034e-05, 2.8652717446675524e-05, 0.00023301718465518206, 0.00019186593999620527, 0.00041238617268390954, 4.265638199285604e-05, 0.00014268264931160957, 0.00020169893105048686, 1.3596729331766255e-05, 1.0432379895064514e-05, 1.15583861770574e-06, 2.759489325399045e-05, 1.5389756526928977e-06, 6.261844191612909e-06, 5.908961611567065e-06, 2.626172545205918e-06, 6.500740710180253e-05, 4.1141058318316936e-05, 0.00014151672075968236, 8.655371493659914e-05, 0.0006917585269547999, 2.895546685977024e-06, 2.3152972062234767e-05, 0.0004064316162839532, 1.3841390682500787e-05, 0.0008121466962620616, 0.0002036640071310103, 2.2874057322042063e-05, 0.0006853654631413519, 2.3897957362351008e-05, 0.0003914418339263648, 4.699894634541124e-05, 1.3287592082633637e-05, 0.00015914818504825234, 0.0001421303313691169, 9.900436998577788e-05, 8.937133679864928e-05, 0.0001664849987719208, 3.0302735467557795e-05, 2.5552693841746077e-05, 3.6749461287399754e-05, 5.797625362902181e-06, 4.192839696770534e-05, 0.0005728445830754936, 0.0002542862494010478, 1.6829276319185738e-06, 1.842017263697926e-05, 7.247270787047455e-06, 9.65486719906039e-07, 3.824358827841934e-06, 1.1477630323497579e-05, 6.325225967884762e-06, 4.465747224458028e-06, 0.00022133428137749434, 2.5430936148040928e-05, 4.013064972241409e-05, 3.329991886857897e-05, 9.94345173239708e-05, 6.036131890141405e-05, 1.2133349628129508e-05, 5.5800679547246546e-05, 1.5541310858679935e-05, 5.2534753194777295e-05, 1.6736279576434754e-05, 0.000447316822828725, 9.838313417276368e-05, 4.47886031906819e-06, 1.0540142284298781e-05, 2.624891021696385e-05, 6.875410235807067e-06, 3.443516106926836e-05, 8.375192555831745e-05, 0.00011496271326905116, 9.349289030069485e-05, 7.987399612829904e-07, 4.042560249217786e-06, 0.00014323517098091543, 1.3163920812075958e-05, 3.1054803457664093e-06, 0.00020751022384501994, 1.848305510065984e-05, 0.001550930319353938, 3.94551498175133e-05, 1.602562224434223e-05, 3.817863034782931e-05, 0.0005172063247300684, 4.415989849348989e-07, 6.524624950543512e-06, 7.847985398257151e-05, 3.819126050075283e-06, 1.2443134437489789e-05, 3.1305316952057183e-06, 2.1769041268271394e-05, 5.4429292504210025e-06, 5.4774832278781105e-06, 3.346607627463527e-06, 9.657699411036447e-05, 0.0001984412519959733, 4.597687075147405e-05, 1.1872137292812113e-05, 4.9589092668611556e-05, 3.1098823001229903e-06, 2.2085334421717562e-05, 0.00014862050011288375, 2.5783627279452048e-05, 3.591034692362882e-05, 5.172450255486183e-05, 1.7071189404305187e-06, 8.384121611015871e-05, 0.00011911823821719736, 9.020308061735705e-05, 0.0010241544805467129, 4.624613154646795e-07, 0.000934327021241188, 2.070052141789347e-05, 2.2347330741467886e-05, 0.009093092754483223, 2.2744852685718797e-05, 3.4713523291429738e-06, 3.971847763750702e-05, 0.0013112352462485433, 8.282265480374917e-05, 0.0019074968295171857, 3.145438677165657e-05, 7.81281960371416e-06, 7.94959305494558e-06, 1.15033935799147e-06, 0.001010907581076026, 3.0049419365241192e-05, 1.691274155746214e-05, 2.4385572032770142e-05, 0.00032341244514100254, 2.88708019979822e-06, 0.0001852339191827923, 0.0004394860879983753, 5.985335883451626e-05, 1.6820079054014059e-06, 1.648455872782506e-05, 1.4051061953068711e-05, 1.4412426025955938e-06, 0.00014587266196031123, 1.6367695934604853e-05, 4.231049388181418e-05, 1.539102527203795e-06, 4.868887481279671e-05, 2.0864916223217733e-05, 3.0619343306170776e-05, 5.927450911258347e-05, 2.432591099932324e-05, 6.401971768354997e-05, 1.6890211327336146e-06, 0.00017443094111513346, 0.0009623434161767364, 6.059334918973036e-05, 4.364290361991152e-05, 0.0005044065765105188, 2.471798325132113e-05, 1.7746458979672752e-05, 7.227606693049893e-05, 1.664763840381056e-05, 1.5708341379649937e-05, 3.8556911022169515e-05, 1.896214234875515e-05, 7.197825470939279e-05, 5.007492291042581e-05, 1.2698988030024339e-05, 0.00022778451966587454, 4.780671588378027e-05, 0.004196264781057835, 4.239008433160052e-08, 2.3738025447528344e-06, 9.053953363036271e-06, 5.170612894289661e-06, 4.37762173532974e-05, 0.0008121071732603014, 0.0005573239759542048, 1.6509558918187395e-05, 0.0002386727137491107, 0.001160163665190339, 0.00017209051293320954, 3.112197737209499e-05, 0.0003517061413731426, 8.410691407334525e-06, 1.185054406960262e-05, 3.3637295473454287e-06, 1.8707936760620214e-05, 0.00040005979826673865, 0.00034406784106977284, 5.2638633860624395e-06, 1.4669336451333947e-05, 1.2322005204623565e-05, 2.0145398593740538e-05, 7.487762104574358e-06, 3.3059288853110047e-06, 0.00014792769798077643, 7.364726479863748e-05, 8.690415415912867e-05, 6.50360161671415e-05, 1.8632559658726677e-05, 0.0021064598113298416, 3.5244815080659464e-05, 3.609794657677412e-05, 7.333695248235017e-05, 3.319150710012764e-05, 0.00038632392534054816, 9.362448736283113e-07, 1.457333655707771e-05, 2.3814247469999827e-05, 1.895201307888783e-06, 8.488855200994294e-06, 7.980033274179732e-07, 0.00026678043650463223, 4.458024704945274e-05, 0.00013770602527074516, 0.00045705761294811964, 5.85053967370186e-05, 3.018126699316781e-05, 0.0007839780300855637, 1.5645678104192484e-06, 0.0023963823914527893, 2.031557414738927e-06, 3.873741297866218e-05, 7.105105032678694e-05, 0.00012877065455541015, 9.159381988865789e-06, 2.4456601749989204e-05, 5.3233656217344105e-05, 9.992413652071264e-06, 9.920043339661788e-06, 7.794471457600594e-05, 0.0001214750373037532, 3.010873570019612e-06, 0.00023341402993537486, 6.958024459891021e-05, 1.3561576679421705e-06, 0.00021043565357103944, 9.06100103748031e-06, 0.0004367699730210006, 6.34353345958516e-05, 0.00010862322233151644, 0.0006589335389435291, 4.207399979350157e-05, 9.636139293434098e-05, 4.22046741732629e-06, 8.450684617855586e-06, 0.0028802782762795687, 5.127243184688268e-06, 3.7718884414061904e-05, 4.918223112326814e-06, 1.537605385237839e-05, 6.32943629170768e-05, 0.0001971050660358742, 9.834418960963376e-06, 0.00017049667076207697, 4.274883758625947e-05, 0.002369231777265668, 7.532149197686522e-07, 5.826201686431887e-06, 7.354778063017875e-05, 4.855764359490422e-07, 4.966589403920807e-05, 0.00018650013953447342, 0.001255640061572194, 3.725401029441855e-06, 1.952161937879282e-06, 6.318265604932094e-06, 8.369200804736465e-05, 0.00014862183888908476, 9.40719764912501e-05, 2.4829689209582284e-05, 2.2708634787704796e-05, 1.3042766113358084e-05, 2.782123601718922e-06, 6.179688170959707e-06, 7.66338757784979e-07, 4.386279033496976e-05, 0.0003075007989536971, 8.201010132324882e-06, 9.177187166642398e-06, 4.348854417912662e-06, 9.224412497133017e-05, 3.5861921787727624e-06, 0.0009139271569438279, 6.808595571783371e-07, 0.0001928295096149668, 9.693494575913064e-06, 8.644276931590866e-06, 0.0011802432127296925, 3.0009377951500937e-06, 9.459025750402361e-05, 7.166605064412579e-05, 0.0003394084342289716, 9.604225488146767e-05, 4.248755067237653e-05, 1.6311953459080542e-06, 3.2615556847304106e-05, 0.0015697716735303402, 0.00024803230189718306, 4.901023658021586e-06, 0.0003739242965821177, 1.374847943225177e-05, 1.1427481695136521e-05, 1.2234553651069291e-05, 0.00012798227544408292, 1.5115471796889324e-05, 6.255482730921358e-05, 3.208237103535794e-05, 1.5439098206115887e-05, 2.377247074036859e-05, 4.296069164411165e-06, 0.00012746750144287944, 9.583155588188674e-06, 0.0001158049053628929, 5.155230155651225e-06, 1.586720100021921e-05, 6.805325369896309e-07, 8.020650420803577e-05, 0.0006956526776775718, 3.805297819781117e-05, 6.189104897202924e-05, 6.24360791334766e-06, 7.317788004002068e-06, 8.982165309134871e-05, 3.179268787789624e-06, 3.0406563382712193e-06, 0.0004106750129722059, 5.906093429075554e-05, 0.00029124823049642146, 1.9934796000598e-05, 1.9640347090899013e-06, 4.388713568914682e-05, 5.011512621422298e-05, 0.0004059172060806304, 8.529658543920959e-07, 5.632209649775177e-06, 0.0022946656681597233, 5.916071677347645e-05, 6.642399239353836e-05, 0.00012747652363032103, 1.3800605302094482e-05, 1.5727335267001763e-05, 1.572370638314169e-05, 2.6952129701385275e-06, 1.7033264157362282e-05, 3.1009337817522464e-06, 1.1119988812424708e-05, 5.9400572354206815e-06, 7.244148491736269e-06, 0.000199650734430179, 0.00021403058781288564, 0.0002461164549458772, 0.0015480340225622058, 9.683131793281063e-05, 3.748232757061487e-06, 5.285643783281557e-05, 1.54933786689071e-05, 5.357006375561468e-05, 9.130661965173203e-06, 3.600408308557235e-05, 4.381085545901442e-06, 3.1697027225163765e-06, 0.00035478343488648534, 1.547049032524228e-05, 6.012051017023623e-06, 2.340821265534032e-05, 0.00011730103869922459, 4.329756848164834e-05, 5.391656486608554e-06, 1.3817023500450887e-05, 8.616656486992724e-06, 1.1007172133759013e-06, 5.106195999360352e-07, 5.34908167537651e-07, 0.00044995936332270503, 6.440351626224583e-06, 7.014681955297419e-07, 0.0003032041131518781, 4.941440056427382e-05, 8.217701179091819e-06, 2.8249487513676286e-06, 5.269917528494261e-05, 2.3345834051724523e-05, 9.648550985730253e-06, 8.395686018047854e-05, 3.520776954246685e-05, 1.7961488993023522e-05, 5.583895472227596e-06, 4.240856287651695e-05, 3.678946814034134e-05, 8.664794222568162e-06, 0.00014688103692606091, 3.0165403586579487e-05, 2.937367071353947e-06, 6.623273657169193e-05, 4.843944407184608e-05, 7.978401845321059e-05, 3.098161869274918e-07, 3.189457856933586e-05, 0.00017692298570182174, 5.292743935569888e-06, 0.00017732047126628458, 0.00021097181888762861, 9.452749509364367e-05, 0.005248068366199732, 0.0001232609065482393, 2.3032791887089843e-06, 5.7019169616978616e-05, 4.102696038899012e-05, 0.0009403778240084648, 0.00012340264220256358, 0.0002718803007155657, 1.1682127478707116e-05, 0.0001292884408030659, 4.078066012880299e-06, 3.932877461920725e-06, 6.721031240886077e-05, 1.5455685797860497e-06, 5.213355052546831e-06, 9.529490853310563e-06, 2.275528822792694e-05, 2.2110951249487698e-05, 5.331919965101406e-06, 2.077720819215756e-05, 7.720909343333915e-05, 0.00011041336983907968, 1.651492311793845e-05, 2.0981120542273857e-05, 3.3553322282386944e-05, 7.040845275696483e-07, 3.5437230962998e-06, 2.309470255568158e-05, 0.0004037423641420901, 0.00023361969215329736, 8.584652277932037e-06, 1.082600010704482e-05, 5.74986006540712e-06, 1.9956974938395433e-05, 7.67077726777643e-05, 3.5686859973793617e-06, 5.0970764277735725e-05, 8.108939800877124e-05, 0.0009653374436311424, 0.00011646348866634071, 0.00010236044909106568, 1.7889999071485363e-05, 8.418218385486398e-06, 2.6044101559818955e-06, 0.00024271577422041446, 9.240569488611072e-05, 0.0006432925583794713, 0.00040450989035889506, 1.76676367118489e-05, 8.933917342801578e-06, 1.0179260243603494e-05, 0.00028585619293153286, 0.0014486819272860885, 0.0046536740846931934, 0.0002548482734709978, 0.00035550561733543873, 0.0005662845796905458, 0.0004211211344227195, 0.0002392542373854667, 3.666092743515037e-05, 0.00047262466978281736, 9.256800694856793e-05, 0.00011162608279846609, 3.336695226607844e-05, 0.0001574459602124989, 2.414071605016943e-05, 8.743919011067192e-07, 6.744248821632937e-05, 3.134900543955155e-05, 9.272590250475332e-05, 1.892138016046374e-06, 0.0005474088829942048, 1.906115357996896e-05, 8.4241571585153e-07, 0.0006829078774899244, 4.262398931587086e-07, 1.5821578926988877e-05, 0.00014628168719355017, 4.9493752158014104e-05, 1.0093174751091283e-05, 0.0002578583371359855, 0.0005309233092702925, 0.0001449161791242659, 2.6765879738377407e-05, 0.00010391495015937835, 0.0014058415545150638, 4.332270327722654e-05, 2.079907289953553e-06, 3.420592520342325e-06, 1.4531675333273597e-05, 0.0001672785438131541, 3.462461393155536e-07, 0.0002257868618471548, 4.822872597287642e-06, 3.937747533200309e-05, 1.215590145875467e-06, 0.0004287357733119279, 4.568001168081537e-05, 0.00026275371783412993, 0.011819843202829361, 0.0003251855378039181, 0.0005574738024733961, 0.00024671474238857627, 8.90482115210034e-05, 8.352639611075574e-07, 2.5104538508458063e-05, 8.009791599761229e-06, 1.3015676813665777e-05, 8.342133241967531e-07, 6.14306700299494e-05, 3.1392013625008985e-05, 9.57505289989058e-06, 1.8634529624250717e-05, 6.817889516241848e-05, 7.567500119876058e-07, 6.7003065851167776e-06, 6.346461759676458e-06, 2.557761581556406e-05, 5.755171059718123e-06, 1.106743366108276e-05, 7.111868762876838e-05, 0.001139933941885829, 4.126043131691404e-05, 1.6335938198608346e-05, 9.60808974923566e-05, 2.0296114598750137e-05, 1.5608580724801868e-05, 8.729710430088744e-07, 1.6527443222003058e-05, 3.2148083846550435e-05, 1.8978438447447843e-06, 1.661999340285547e-05, 3.565225597412791e-06, 0.00022491824347525835, 0.00041446200339123607, 0.005888571031391621, 6.848630437161773e-05, 0.0009038819116540253, 2.925342641901807e-06, 0.00011889576126122847, 0.00023203575983643532, 0.00026323244674131274, 3.807203029282391e-05, 2.8455381198000396e-06, 2.016867438214831e-05, 1.9461294868960977e-06, 0.0010394331766292453, 9.939845040207729e-05, 0.0001286492624785751, 3.870633008773439e-06, 0.0002055938821285963, 6.393802323145792e-06, 4.755987538374029e-05, 3.5308308724779636e-05, 3.0203169444575906e-05, 9.114807471632957e-05, 7.6246901699050795e-06, 4.81699771626154e-06, 0.00015300026279874146, 0.00015371947665698826, 0.0003219370555598289, 0.00016495211457367986, 0.003029961371794343, 0.008629762567579746, 0.0001020930940285325, 5.059494651504792e-05, 3.958271463488927e-06, 0.0001120362285291776, 1.8549006199464202e-05, 2.9919817734480603e-06, 6.881339504616335e-05, 1.744538349157665e-05, 1.9556919141905382e-05, 5.370660073822364e-05, 3.6261715194996214e-06, 2.3909246010589413e-05, 0.00016537506598979235, 2.2782993255532347e-05, 1.1617417840170674e-05, 3.8019461499061435e-06, 5.046141450293362e-05, 9.905330080073327e-05, 0.0014055767096579075, 3.1197414500638843e-06, 1.999616506509483e-05, 0.0004350121016614139, 2.99125576930237e-06, 0.0006444090395234525, 4.0901932152337395e-06, 0.0002757192123681307, 0.00011770903074648231, 5.2222327212803066e-05, 3.079424277530052e-05, 1.569148821545241e-06, 0.0005689800018444657, 1.4247226317820605e-05, 5.317508566804463e-06, 0.0001159269959316589, 5.232301919022575e-05, 0.00012564937060233206, 1.0641855396897881e-06, 5.163249170436757e-06, 1.5920559235382825e-05, 5.355776920623612e-06, 0.0002889993484131992, 1.7923734048963524e-05, 7.309157808776945e-05, 8.773984882282093e-05, 0.00011740886111510918, 6.246662087505683e-05, 3.34977958118543e-05, 6.76755053063971e-06, 4.0495469875168055e-05, 0.00015321174578275532, 0.00013568451686296612, 0.00011834119504783303, 4.275585524737835e-05, 7.55396977183409e-05, 4.407724645716371e-06, 2.8058675525244325e-05, 7.692537474213168e-05, 8.240602255682461e-06, 0.00011004954285454005, 2.5243452910217457e-05, 5.837972821609583e-06, 0.003027002327144146, 9.410651728103403e-06, 6.207070327945985e-06, 0.00017076748190447688, 2.4884931917767972e-05, 0.00017058149387594312, 5.251424681773642e-06, 2.3661556042497978e-05, 0.00017791542632039636, 0.00022629548038821667, 8.269411773653701e-05, 1.529616565676406e-05, 6.644586392212659e-06, 6.082298932597041e-05, 9.585504449205473e-05, 2.3005416096566478e-06, 0.0005554078961722553, 3.1280785606213612e-06, 0.00013992884487379342, 9.07727826415794e-06, 6.6266288740735035e-06, 3.600199124775827e-05, 6.574206508958014e-06, 0.0001490172289777547, 1.1944998732360546e-05, 3.739325620699674e-05, 0.0007847303058952093, 9.332963600172661e-06, 1.3157155080989469e-05, 0.00029296550201252103, 4.9660931836115196e-05, 4.7704874305054545e-05, 2.6585757950670086e-05, 0.00019124617392662913, 0.00013051113637629896, 0.00030594796407967806, 2.1396498084413906e-07, 6.257096538320184e-05, 0.0032955193892121315, 3.1349779874290107e-06, 8.213863475248218e-05, 0.0001630113838473335, 0.00047026266111060977, 4.8499377953703515e-06, 5.552663787966594e-05, 3.8265290640993044e-05, 3.334533403176465e-06, 9.999785106629133e-07, 2.4703225790290162e-05, 0.00028192487661726773, 0.00010554203618085012, 6.139211473055184e-05, 1.052308471116703e-05, 0.00012834498193114996, 0.00011357830226188526, 0.0001097407512133941, 4.339251609053463e-05, 9.51633683143882e-06, 2.8222801120136864e-05, 1.1874686833834858e-06, 0.0018014954403042793, 0.00037373427767306566, 0.0005652362015098333, 2.175484951294493e-05, 7.02476900187321e-05, 0.0004941638908348978, 1.1801129403465893e-05, 1.0104092325491365e-05, 7.205159818113316e-06, 1.8544475324233645e-06, 1.3028245575696928e-06, 0.0006309988093562424, 2.1893231405556435e-06, 1.5237666957546026e-05, 0.00017784653755370528, 6.912912795087323e-05, 6.161084456834942e-05, 0.00023443278041668236, 0.0003396135289222002, 2.7046837203670293e-05, 1.892934847091965e-06, 8.766372957325075e-06, 0.0001287885825149715, 0.00022374559193849564, 3.7682098081859294e-06, 7.804438064340502e-05, 5.733963553211652e-05, 1.891507963591721e-05, 1.0564695912762545e-05, 1.9041597624891438e-05, 1.1874361689478974e-06, 3.5617569665191695e-05, 2.1932763047516346e-05, 1.9337829144205898e-05, 5.1908500608988106e-05, 0.0032199756242334843, 0.0003764968132600188, 4.717280717159156e-06, 8.877724030753598e-05, 1.7270647731493227e-05, 5.730581688112579e-05, 0.00015422205615323037, 2.2688816898153163e-05, 1.531843008706346e-05, 1.1472441201476613e-06, 4.175938738626428e-05, 3.277818177593872e-05, 9.982220944948494e-06, 0.0017311744159087539, 4.460596392164007e-05, 5.7242094044340774e-05, 5.949366823188029e-05, 0.0017884245608001947, 9.164110088022426e-05, 1.1020265446859412e-05, 4.1345319914398715e-06, 4.362311756267445e-06, 0.00028170584118925035, 0.00016732991207391024, 0.00010251422645524144, 0.0005837180651724339, 0.0025914397556334734, 1.8313656937607448e-06, 5.0525795813882723e-05, 4.251855330039689e-07, 1.3024109648540616e-05, 0.00025158675271086395, 0.000724061275832355, 6.675465556327254e-05, 0.00026884395629167557, 1.250236982741626e-05, 3.405945244594477e-05, 0.0002396834024693817, 2.316701466043014e-05, 2.6542862542555667e-05, 0.0004978538490831852, 0.0015612232964485884, 3.7453983168234117e-06, 7.039680349407718e-05, 4.2108695197384804e-05, 1.1435184205765836e-05, 0.0007303279126062989, 0.00014111207565292716, 1.507880642748205e-05, 0.00013877954916097224, 7.164628641476156e-06, 0.00010526923142606393, 6.173348083393648e-05, 5.1320854254299775e-05, 7.064371311571449e-05, 2.7052368750446476e-05, 7.495092722820118e-05, 0.0002666473446879536, 2.8288802695897175e-06, 0.0016223269049078226, 2.9661703592864797e-05, 1.7892145478981547e-05, 4.633074422599748e-05, 2.8505157388281077e-05, 5.74873183722957e-06, 7.003416249062866e-05, 1.3007145753363147e-05, 4.315965270507149e-06, 0.003358917310833931, 4.692971924669109e-05, 0.0006311133038252592, 0.00010804313933476806, 3.4155793400714174e-05, 0.00011260044266236946, 2.1983591068419628e-05, 0.00015131062536966056, 9.65518211160088e-07, 5.082389179733582e-05, 4.830501893593464e-06, 3.5568285966292024e-05, 2.4161990950233303e-06, 1.2133105883549433e-05, 0.0004295362741686404, 0.0001308670180151239, 3.7855836581002222e-06, 3.3937894841074012e-06, 6.946123176021501e-05, 6.915508038218832e-06, 0.00029109729803167284, 8.552004146622494e-06, 1.6367746866308153e-05, 2.328325308553758e-06, 0.0006101374747231603, 0.0030753130558878183, 2.305912130395882e-05, 6.0539881815202534e-05, 9.228296221408527e-06, 0.00016656181833241135, 1.0860668226087e-05, 4.461336175154429e-06, 0.0005826595006510615, 0.00034707458689808846, 0.00010447522072354332, 8.51469740155153e-05, 4.574195190798491e-05, 3.8448208215413615e-05, 0.00013855872384738177, 0.0004610794421751052, 0.00016439586761407554, 0.0009092435357160866, 4.510214785113931e-05, 0.0009014693787321448, 2.267702711833408e-06, 4.969163455825765e-06, 4.781752795679495e-05, 6.265606771194143e-06, 7.382593139482196e-06, 1.3825647329213098e-05, 0.0009512245305813849, 0.003396848915144801, 2.877439101212076e-06, 1.3135415883880341e-06, 4.435249502421357e-05, 9.75391740212217e-05, 1.167212576547172e-05, 1.6287420294247568e-05, 1.5608044122927822e-05, 5.158117801329354e-06, 0.00022978257038630545, 0.00025144367828033864, 2.5765892132767476e-05, 4.444428213901119e-06, 3.7747915484942496e-05, 1.243558131136524e-06, 1.8168236692872597e-06, 2.5206336431438103e-05, 1.0358960935263894e-05, 0.00025257820379920304, 2.31647882174002e-05, 4.6454460971290246e-06, 2.654960371728521e-05, 5.0054692110279575e-05, 0.0008107361500151455, 7.886673120083287e-05, 0.0008680291357450187, 1.677901991570252e-06, 0.0018474499229341745, 2.2543912564287893e-05, 1.9519666238920763e-05, 4.680750407715095e-06, 0.000107287269202061, 3.983718852396123e-05, 0.0001460197236156091, 1.885291112557752e-06, 1.1634485417744145e-05, 5.459957174025476e-05, 7.825291504559573e-06, 3.3183830510097323e-06, 9.813986707740696e-07, 5.354959739634069e-06, 6.661427960352739e-07, 0.00018542267207521945, 0.0005619088187813759, 2.320054363735835e-06, 6.091735940572107e-06, 4.534010713541647e-06, 1.2432390576577745e-05, 3.3230739973078016e-06, 1.0554154869168997e-05, 1.7002515733111068e-06, 0.00013419323659036309, 6.439247499656631e-07, 3.074221240240149e-05, 1.4591709259548225e-05, 9.473603313381318e-06, 7.707024451519828e-06, 0.00015897942648734897, 1.860545637555333e-07, 4.093889583600685e-05, 5.776609214080963e-06, 2.4337772629223764e-05, 1.117486954171909e-05, 5.023503490519943e-06, 7.779240695526823e-05, 5.133675949764438e-05, 5.842503014719114e-06, 4.533323954092339e-05, 0.00023374415468424559, 4.924784298054874e-05, 2.449851081109955e-06, 9.073858564079273e-06, 0.00012110859825043008, 3.70738489436917e-05, 5.931138730375096e-05, 0.00014971758355386555, 0.00045062959543429315, 6.515699351439252e-05, 6.467170169344172e-05, 3.681069938465953e-05, 0.0009468038333579898, 0.001239351462572813, 5.419330773293041e-05, 1.34200272441376e-05, 3.5136097721988335e-05, 5.39133134225267e-06, 7.500312494812533e-05, 7.572313734272029e-06, 1.825807089517184e-06, 5.026266535423929e-06, 8.191891538444906e-05, 2.4954581022029743e-05, 1.744475775922183e-05, 6.290380952123087e-06, 3.388983532204293e-05, 8.196132512239274e-06, 4.288836498744786e-05, 1.975788109120913e-05, 4.6127926907502115e-05, 2.105640305671841e-06, 0.0005876432405784726, 2.978871088998858e-05, 0.00021770394232589751, 9.697818313725293e-06, 1.2457340744731482e-05, 1.2728944966511335e-05, 6.396235676220385e-06, 4.114851253689267e-05, 6.711131572956219e-05, 9.90697808447294e-05, 0.00023683950712438673, 3.757121885428205e-05, 8.755416274652816e-06, 2.837121201082482e-06, 1.752628304529935e-05, 4.656927103496855e-06, 2.006284830713412e-06, 0.0007925296667963266, 8.694730786373839e-05, 0.0012619949411600828, 5.7464942074147984e-05, 6.41502356302226e-06, 2.2934934804652585e-06, 3.992734127677977e-05, 8.189517393475398e-05, 1.497435391684121e-06, 0.0001923858217196539, 2.4872622361726826e-06, 1.7048688505383325e-06, 0.0008405746193602681, 7.609536169184139e-06, 4.054775490658358e-05, 1.1656081824185094e-06, 1.0866730917769019e-05, 3.3331161830574274e-05, 0.0001754927943693474, 0.00019898176833521575, 1.3772000784229022e-05, 7.012460991973057e-05, 4.570620603772113e-06, 0.00023726567451376468, 0.00043949560495093465, 9.722765753394924e-06, 3.1022547773318365e-05, 2.8827889764215797e-06, 0.00024057587143033743, 1.0669226867321413e-05, 9.412535291630775e-06, 8.206726488424465e-05, 9.423029405297711e-05, 1.678923399595078e-05, 0.001764694694429636, 2.7651100026560016e-05, 5.20924550073687e-06, 1.3567386304202955e-05, 0.00019099262135569006, 7.145310974010499e-06, 2.4595652575953864e-05, 6.386940367519855e-05, 0.00011076017108280212, 1.644805888645351e-05, 1.9105495084659196e-05, 0.0005290643312036991, 2.529014636820648e-05, 0.0019531568977981806, 0.0011647996725514531, 7.619577445439063e-06, 1.5710536899860017e-05, 4.607220489560859e-06, 0.00022332381922751665, 2.2685733711114153e-05, 0.0001889854174805805, 4.647127752832603e-06, 4.0100436308421195e-05, 0.000201148766791448, 1.64655975822825e-05, 2.132168674506829e-06, 2.6157835236517712e-05, 0.0003334739594720304, 2.0753650460392237e-06, 0.0008366128895431757, 0.00020774015865754336, 0.0001199850084958598, 4.5402707655739505e-06, 1.6685171431163326e-05, 6.249714351724833e-05, 1.2767380894729286e-06, 5.935534318268765e-06, 4.7400158109667245e-06, 9.205019159708172e-05, 3.106217263848521e-05, 6.82610843796283e-05, 3.11237781716045e-05, 3.1875690638116794e-06, 8.345278388333099e-07, 0.00045064324513077736, 4.2571802623569965e-05, 1.199858843392576e-06, 1.96734290511813e-05, 9.560443459122325e-07, 8.199391595553607e-05, 2.3438833522959612e-05, 0.00010338837455492467, 0.00011194446415174752, 8.138044358929619e-05, 0.0004135117051191628, 3.279569500591606e-05, 0.0011955713853240013, 7.153441856644349e-06, 2.587005110399332e-05, 7.5819975791091565e-06, 0.00021190577535890043, 5.5854467063909397e-05, 8.107994290185161e-06, 4.0295452663485776e-07, 8.204831829061732e-05, 2.1862679204787128e-05, 1.775618875399232e-05, 2.200769813498482e-05, 6.286144343903288e-05, 7.969950274855364e-06, 0.00010298299457645044, 2.111077492372715e-06, 2.1833335267729126e-05, 9.728552686283365e-05, 1.7087054402509239e-06, 0.000296437821816653, 6.510595994768664e-05, 2.8160884539829567e-06, 0.00033040513517335057, 0.00016071567370090634, 0.0009202318615280092, 0.00022475153673440218, 0.0008963100262917578, 3.103774361079559e-05, 0.00040451731183566153, 0.0006592555437237024, 0.0011872845934703946, 1.0381970241724048e-05, 8.615473780082539e-05, 0.0004613366618286818, 0.0003267585998401046, 2.071542985504493e-05, 2.0044424672960304e-05, 2.475087603670545e-05, 3.843701051664539e-05, 4.93929474032484e-05, 4.06398467021063e-05, 8.362157132069115e-06, 5.625466201308882e-06, 0.00014269693929236382, 3.6977475247113034e-05, 0.00015395050286315382, 8.993360097520053e-05, 9.000049612950534e-05, 0.00021533333347178996, 0.00018838135292753577, 1.675950079516042e-05, 9.196976316161454e-05, 1.5054984942253213e-05, 5.537840479519218e-06, 9.843000043474603e-06, 0.00033672922290861607, 4.753921984956833e-06, 1.8687294868868776e-05, 1.578174124006182e-05, 0.0006837954861111939, 7.503780216211453e-05, 3.9184751585708e-06, 6.168291292851791e-05, 0.00010555766493780538, 0.0003993292630184442, 4.2908689579235215e-07, 4.030215222883271e-06, 0.00035205992753617465, 3.03640917991288e-05, 9.519555169390514e-05, 0.00025971076684072614, 6.480918818851933e-05, 3.6551504308590665e-05, 1.5181813068920746e-05, 7.239856131491251e-06, 8.436078132945113e-06, 0.0003693522303365171, 3.8994799979263917e-05, 0.00012503629841376096, 1.0490399290574715e-05, 1.088558178707899e-06, 6.271628080867231e-05, 0.00018090949743054807, 0.0013240254484117031, 0.0007705186726525426, 2.2936710593057796e-06, 1.6576082998653874e-05, 5.6575448979856446e-05, 0.0010717071127146482, 1.1174177416251041e-05, 3.839911641989602e-06, 6.387786470440915e-06, 5.652467007166706e-05, 1.6087018593680114e-05, 0.00020200872677378356, 0.00015640491619706154, 0.00016119809879455715, 1.4805559658270795e-05, 9.639641211833805e-05, 3.933098923880607e-06, 6.232004670891911e-05, 5.4378229833673686e-05, 4.1416857129661366e-05, 0.00010272656072629616, 1.629053076612763e-05, 1.2854259693995118e-05, 7.625558646395802e-05, 0.0002332310687052086, 5.006327774026431e-05, 6.296349602052942e-05, 0.00040455261478200555, 3.3709629860823043e-06, 9.519609739072621e-05, 3.1689667139289668e-06, 0.0001233398070326075, 3.335853398311883e-05, 9.108115591516253e-06, 1.3096892871544696e-05, 0.00024456155369989574, 2.351831062696874e-05, 3.476715937722474e-05, 4.075212564202957e-05, 7.417264896503184e-06, 1.6562711607548408e-05, 4.962135062669404e-05, 1.8844843907572795e-07, 0.0009123677155002952, 0.00044533066102303565, 8.463582162221428e-06, 1.0969274626404513e-05, 0.0003667476412374526, 0.0003560467157512903, 0.00017859532090369612, 1.5801324479980394e-05, 9.107706864597276e-05, 6.611176650039852e-05, 6.819576356065227e-06, 5.250940375844948e-06, 0.001689888071268797, 0.0001062983283190988, 2.7005696665582946e-06, 1.3152820201867144e-06, 6.707497050228994e-06, 7.662377174710855e-05, 0.003909402526915073, 2.3815868189558387e-05, 7.387681398540735e-05, 4.243755483912537e-06, 0.00012543381308205426, 1.313194752583513e-05, 3.74226801795885e-05, 0.000249221600824967, 4.2447712189641607e-07, 4.642549811251229e-06, 0.0002948400506284088, 5.501794930751203e-06, 0.00011991578503511846, 1.4827634913672227e-05, 0.00023438259086105973, 5.498133032233454e-05, 2.0951890746800927e-06, 0.00012922838504891843, 3.670239675557241e-05, 6.767766990378732e-06, 1.7404996697223396e-06, 1.705866088741459e-05, 9.701593080535531e-05, 8.219240044127218e-06, 0.00013516272883862257, 3.908814687747508e-05, 0.0001072715240297839, 1.0461450074217282e-05, 3.7673501083190786e-06, 0.00035900139482691884, 0.00025732244830578566, 0.0003875242837239057, 2.2539214114658535e-05, 0.000432171014836058, 0.00012479381985031068, 1.2013123523502145e-05, 5.035394860897213e-05, 1.7677133428151137e-06, 9.241063708032016e-06, 1.190028342534788e-05, 0.000130067826830782, 0.00019301062275189906, 4.156786872044904e-06, 1.3489924640452955e-05, 0.00011798114428529516, 8.839028282636718e-07, 1.4115644262346905e-06, 1.2730648450087756e-05, 2.5624034606153145e-05, 2.9215523227321682e-06, 6.0656140703940764e-05, 0.0002198337169829756, 2.4507762645953335e-05, 5.956591485301033e-05, 5.0123115215683356e-05, 2.1307309907570016e-06, 0.00036976500996388495, 2.985455466841813e-06, 0.00011072975757997483, 2.7015386876882985e-05, 4.032694960187655e-06, 0.0007873138529248536, 9.356055670650676e-05, 0.00018086346972268075, 6.722200487274677e-05, 4.056169927935116e-05, 7.033021392999217e-05, 6.797809328418225e-05, 0.000983388046734035, 2.292918225066387e-06, 9.160627087112516e-06, 5.7912013289751485e-05, 7.532838480983628e-06, 0.00037754024378955364, 1.35110701648955e-06, 0.00011826000263681635, 0.00013837532605975866, 3.2942150482995203e-06, 6.038096216798294e-06, 6.59487268421799e-05, 0.00010396649304311723, 0.00011415679182391614, 8.062473170866724e-06, 0.00011619229917414486, 0.0008068300085142255, 3.0346160201588646e-05, 3.1135416065808386e-05, 0.00046222569653764367, 8.555833483114839e-05, 0.0001954590406967327, 6.040864536771551e-05, 2.6925699785351753e-06, 1.7176288338305312e-06, 4.1476840124232695e-05, 2.2043445824238006e-06, 9.208123810822144e-05, 6.054508048691787e-05, 0.0004534167528618127, 3.092360884693335e-06, 0.0016882651252672076, 2.8020014724461362e-05, 1.586316284374334e-05, 9.597840835340321e-05, 1.3089018580103584e-07, 7.357875438174233e-05, 3.295258284197189e-05, 8.902550803213671e-07, 6.996726733632386e-05, 0.0002583941677585244, 0.0003638927300926298, 1.292839260713663e-05, 0.0001997913495870307, 5.511355084308889e-06, 0.0003226950648240745, 3.469219518592581e-05, 8.433440962107852e-05, 1.3418342632576241e-06, 1.6083337186501012e-06, 8.115945820463821e-05, 1.3856842997483909e-05, 4.38320312241558e-05, 0.00037330316263251007, 4.6005907279322855e-06, 1.926988943523611e-06, 0.00011545835150172934, 1.752243042574264e-05, 3.469272633083165e-05, 0.000997663359157741, 7.41782714612782e-05, 5.038546078139916e-05, 3.520812242641114e-05, 1.0976227713399567e-06, 5.2458173740888014e-05, 5.127340045874007e-06, 0.00028434739215299487, 6.196868980623549e-06, 1.682456786511466e-05, 1.3723282791033853e-05, 3.146482049487531e-05, 4.612927568814484e-06, 1.98593879758846e-05, 0.00013958978524897248, 2.639522972458508e-05, 3.312291664769873e-05, 3.468554132268764e-05, 2.572787707322277e-05, 0.00010857891902560368, 5.9456411690916866e-05, 0.0010464303195476532, 0.00019255235383752733, 5.8978735069104005e-06, 0.00031852832762524486, 4.061806248500943e-05, 7.541592367488192e-06, 1.4706930414831731e-05, 0.00015486183110624552, 7.847766391932964e-05, 0.0008705852669663727, 0.00014237257710192353, 6.433180533349514e-05, 3.0210347176762298e-05, 1.635051916082375e-07, 1.1110882041975856e-05, 3.587357423384674e-05, 6.396044227585662e-06, 0.00012665965186897665, 1.706816146906931e-05, 1.5105293641681783e-05, 1.0757508789538406e-05, 0.00017874996410682797, 0.0001630880869925022, 0.0001514328469056636, 8.14225131762214e-05, 1.1724641808541492e-06, 3.774098877329379e-05, 5.961569513601717e-06, 5.1187482313252985e-05, 6.181762728374451e-07, 0.0006247004494071007, 1.4131392163108103e-05, 1.6010877516237088e-05, 1.5730562154203653e-05, 3.5397610190557316e-05, 1.5280906154657714e-05, 1.267405991711712e-06, 0.00010589536395855248, 4.3988537072436884e-05, 3.3275334772042697e-06, 4.9683210818329826e-05, 2.8085078156436794e-06, 1.718272324069403e-05, 4.30356158176437e-05, 1.3630080502480268e-05, 1.987796309776968e-07, 5.7786324759945273e-05, 4.699305463873316e-06, 6.537456556543475e-06, 3.1129213311942294e-05, 7.625047874171287e-05, 0.00036890673800371587, 0.0005843467661179602, 8.759567208471708e-06, 5.8682016970124096e-05, 2.0148813746345695e-06, 8.275631989818066e-05, 0.00013433292042464018, 1.83647389349062e-05], [7.732704398222268e-05, 0.00015038838319014758, 1.0588456461846363e-05, 0.000359076278982684, 2.347466943319887e-05, 2.980445060529746e-06, 0.0010049433913081884, 3.517415461828932e-05, 3.435839244048111e-05, 0.00023893687466625124, 1.7962378478841856e-05, 1.581009428264224e-06, 1.5413238543260377e-06, 0.0004345205379649997, 0.00020517224038485438, 4.4353222619974986e-05, 8.850467565935105e-05, 2.950156522274483e-05, 0.00020155898528173566, 0.0006489304360002279, 2.404592123639304e-06, 7.247544272104278e-05, 3.399065099074505e-05, 0.00042717118049040437, 6.776789177820319e-06, 0.0006692403112538159, 0.0005078158574178815, 0.000948234461247921, 0.0008576319669373333, 4.137209907639772e-05, 1.1271185940131545e-05, 2.0379467969178222e-05, 1.2542493095679674e-05, 4.023018482257612e-05, 2.2595026166527532e-05, 1.2821765267290175e-05, 0.0001664494484430179, 5.056887744103733e-07, 3.8271780795184895e-05, 8.299888577312231e-05, 1.9054283256991766e-05, 0.0009270787704735994, 0.00019677977252285928, 8.4597394334196e-07, 0.00017708489031065255, 0.0002488940372131765, 0.00023620203137397766, 0.0011051399633288383, 4.650339633371914e-06, 5.439337655843701e-06, 2.3423584934789687e-05, 0.002900359220802784, 0.00012536808208096772, 3.598674447857775e-05, 8.103947038762271e-06, 0.00018669376731850207, 9.375372428621631e-06, 0.00015443369920831174, 6.366205343510956e-05, 0.0004137081268709153, 4.121534584555775e-05, 1.8515329429646954e-05, 5.517363388207741e-05, 8.457715011900291e-05, 3.070475941058248e-05, 4.737875497085042e-05, 1.149511149378668e-06, 5.0830363761633635e-05, 9.126481018029153e-05, 5.499571670952719e-06, 0.00031354714883491397, 7.73864085203968e-05, 3.7317855458240956e-05, 3.959347304771654e-05, 7.066541002131999e-05, 6.604788359254599e-05, 1.330604618487996e-06, 0.0011546746827661991, 0.004003776703029871, 1.3744126590609085e-05, 1.092240654543275e-05, 1.914395033963956e-05, 1.197101482830476e-06, 5.310365668265149e-05, 8.560017704439815e-06, 1.0566159289737698e-05, 2.73678438134084e-06, 0.0017148718470707536, 7.874972652643919e-06, 0.0013189729070290923, 8.839412475936115e-05, 0.0001215026350109838, 0.00011964674922637641, 3.395648491277825e-06, 2.5632451070123352e-05, 8.350935240741819e-05, 4.850369441555813e-05, 1.857682946138084e-05, 4.811591861653142e-05, 7.2472653300792444e-06, 0.0007938623311929405, 4.818030902242754e-06, 0.00023394680465571582, 4.426847681315849e-06, 1.5982241166057065e-05, 0.0008943805005401373, 0.00014059798559173942, 4.9219252105103806e-05, 1.3375255548453424e-05, 0.00012199508637422696, 3.0303669973363867e-06, 3.362215284141712e-05, 5.610000516753644e-05, 0.0009236108744516969, 4.755305326398229e-06, 7.19868767191656e-05, 2.0123226931900717e-05, 4.054130840813741e-05, 0.00025150494184345007, 2.4858920824044617e-06, 0.00011667187209241092, 2.2179600023264356e-07, 5.592221441474976e-06, 0.00011778301268350333, 0.0007974160835146904, 3.9243809624167625e-06, 7.337494753301144e-05, 1.9732311557163484e-05, 4.9793990910984576e-05, 0.0001673542137723416, 9.546029468765482e-05, 9.727104043122381e-05, 2.5055309379240498e-05, 2.2239632016862743e-05, 0.0001350660459138453, 1.298894130741246e-05, 0.00011527969763847068, 4.206560788588831e-06, 5.0583235861267895e-05, 1.9491248167469166e-05, 5.871155281056417e-06, 0.0001029740524245426, 0.009055661037564278, 3.110117540927604e-05, 3.657566776382737e-05, 1.1493716556287836e-06, 2.2142801753943786e-05, 1.5464771649931208e-06, 6.2632707340526395e-06, 0.00010340420703869313, 0.0002899272949434817, 1.6997739749058383e-06, 0.0005730073899030685, 6.933002441655844e-05, 0.0007836683071218431, 7.311146327992901e-05, 0.0036970782093703747, 5.175221667741425e-05, 0.00014623755123466253, 9.258636168851808e-07, 5.943031169408641e-07, 4.224792428431101e-05, 7.282015985765611e-07, 7.494859892176464e-05, 3.0249917472247034e-05, 0.00011153461673529819, 6.405750900739804e-05, 0.0003546517691574991, 5.9220717957941815e-05, 8.790703418526391e-07, 1.8337579604121856e-05, 7.970911610755138e-06, 0.00024021176795940846, 1.2678359780693427e-05, 1.5294664990506135e-05, 2.6968165911966935e-05, 0.0003018415591213852, 0.0001727608178043738, 3.734723213710822e-05, 6.082133040763438e-06, 3.124948125332594e-05, 0.00032202311558648944, 0.0004130984016228467, 8.816534318611957e-06, 1.648934448894579e-05, 2.1671399963452131e-07, 7.062731583573623e-06, 6.348564056679606e-05, 4.917566457152134e-06, 1.2044236427755095e-05, 4.396333315526135e-05, 6.142634083516896e-05, 2.8149746867711656e-05, 8.235499262809753e-05, 1.8965678464155644e-05, 0.00020728819072246552, 4.203108346700901e-06, 0.00027946551563218236, 1.9503679141053e-05, 0.0006090362439863384, 0.000576717546209693, 0.00010165103594772518, 9.301154932472855e-06, 4.353374151833123e-06, 0.00034112390130758286, 1.635796252230648e-06, 5.6933284213300794e-05, 8.8703691289993e-06, 9.203607760355226e-07, 2.627923277032096e-05, 0.00012115287245251238, 9.412792132934555e-05, 8.35770094909094e-07, 2.2406422885978827e-06, 0.00010710338392527774, 0.0003410583594813943, 1.2410480849212036e-05, 0.0017955239163711667, 6.0860624216729775e-05, 0.0005338517366908491, 0.0015952105168253183, 0.00015961633471306413, 4.610148153005866e-06, 0.0008215050911530852, 1.4864260720059974e-06, 0.00010184586426476017, 9.343821716356615e-07, 4.0115164665621705e-06, 1.305198202317115e-05, 2.2970601776250987e-07, 5.08798484588624e-06, 1.6039876982176793e-06, 9.978069101634901e-06, 3.076116990996525e-05, 0.0010613937629386783, 0.0001243712322320789, 0.0005699080647900701, 2.3044540284899995e-05, 0.00022669286408927292, 1.886071913759224e-05, 0.0006071468815207481, 1.2583553825606941e-06, 5.405909178080037e-06, 6.9849455030635e-05, 1.0519267561903689e-05, 1.741701794344408e-06, 0.001032842556014657, 2.1136182112968527e-05, 5.254405186860822e-05, 2.97205460810801e-05, 1.985765265999362e-06, 0.00015641444770153612, 0.0006974997813813388, 6.609188858419657e-05, 0.00032284704502671957, 2.124176171491854e-05, 0.0010055991588160396, 8.97432619240135e-05, 5.290906483423896e-05, 2.1731989363615867e-06, 2.2207048459677026e-05, 2.5624640329624526e-05, 0.00020708270312752575, 3.779115650104359e-05, 0.0009955187560990453, 0.00022706060553900898, 2.4861171823431505e-06, 8.412529837187321e-07, 0.0003568226529750973, 2.374290124862455e-05, 8.382518899452407e-06, 6.586130621144548e-05, 5.6090688303811476e-05, 2.0072926417924464e-05, 5.300953944242792e-06, 0.00015457476547453552, 8.21937919681659e-06, 1.8326220015296713e-05, 3.138936881441623e-05, 6.995483545324532e-06, 2.8345714326860616e-06, 3.9820046367822215e-05, 2.575519283709582e-05, 1.557780706207268e-05, 7.080436625983566e-05, 7.821635517757386e-05, 1.6612915715086274e-05, 3.7551228615484433e-06, 5.11298821948003e-06, 0.002255675382912159, 8.477196047351754e-07, 0.0002580542641226202, 0.000947304128203541, 0.00027393107302486897, 1.1052248737541959e-05, 3.607952976381057e-06, 3.473258038866334e-05, 1.486549990659114e-05, 6.605120870517567e-05, 1.8619224647409283e-05, 6.995626904426899e-07, 1.2306069265832775e-06, 1.3709859558730386e-05, 1.3839697203366086e-05, 0.0001958521897904575, 2.218295048805885e-05, 1.2321303984208498e-05, 0.0005262668710201979, 4.426816815339407e-07, 2.7220248739467934e-06, 2.3692025933996774e-05, 1.983634774660459e-06, 8.180461009033024e-05, 6.948666123207659e-05, 1.9219471596443327e-06, 6.197648417582968e-06, 4.2160445445915684e-05, 4.731404033009312e-07, 7.928856939543039e-05, 9.141057489614468e-06, 3.7929744394205045e-06, 8.85864756128285e-06, 0.00014347091200761497, 0.00014881574315950274, 4.46513804490678e-05, 0.00011858440848300233, 3.685288902488537e-05, 4.873013494943734e-06, 6.99797055858653e-07, 1.2438965313776862e-05, 3.126207593595609e-05, 1.5662002624594606e-05, 0.0006175539456307888, 2.8402389943948947e-05, 4.162635741522536e-05, 0.0030353327747434378, 2.5217508664354682e-05, 5.987055374134798e-06, 0.0005810682778246701, 6.927499634912238e-05, 0.00021188819664530456, 1.329753013123991e-05, 0.00047135967179201543, 0.0002789270074572414, 0.0001700906577752903, 5.969609446765389e-07, 0.00032397868926636875, 3.1759109333506785e-06, 0.0006649963324889541, 7.934184395708144e-05, 2.9670736694242805e-05, 5.808110654470511e-05, 1.0060273325507296e-06, 3.0698784030391835e-06, 0.0012720917584374547, 1.3672737622982822e-05, 0.002254992490634322, 2.610698720673099e-05, 0.00023178441915661097, 2.5633935365476646e-06, 0.0055292281322181225, 5.005847469874425e-06, 4.2013047618638666e-07, 0.0003391336358617991, 0.0008280100300908089, 2.5914940124494024e-05, 1.3958617273601703e-05, 4.130815341341076e-06, 3.4167613193858415e-05, 0.0004016136226709932, 0.00040931266266852617, 1.1611317859205883e-05, 5.7635039411252365e-05, 2.555267928983085e-05, 2.383019273111131e-05, 0.0006176049937494099, 2.303170731465798e-05, 0.0003761141560971737, 1.761066960170865e-05, 1.56510723172687e-05, 2.2149599317344837e-05, 0.002465302823111415, 5.845462055731332e-06, 0.00060343585209921, 2.0648005374823697e-05, 2.5312370780739002e-05, 6.78803019127372e-07, 3.313664274173789e-05, 0.00013307941844686866, 0.00015092371904756874, 4.578428161039483e-06, 0.0004288919735699892, 2.1748453946202062e-05, 7.368577826127876e-06, 3.052641841350123e-05, 0.003145360155031085, 1.189113208965864e-05, 0.00011900734534719959, 4.5748124648525845e-06, 6.85782142682001e-05, 8.09687189757824e-05, 4.353796612122096e-07, 1.0077959814225323e-05, 0.00012491471716202796, 7.648586688446812e-06, 0.0003917802532669157, 5.15713072672952e-05, 6.51591835776344e-05, 7.86272266850574e-06, 2.694328941288404e-05, 4.247006472724024e-06, 8.164811333699618e-06, 0.0003812451905105263, 3.7683232221752405e-05, 0.008078441955149174, 6.172128905745922e-06, 4.9919970479095355e-05, 0.000621932849753648, 1.1659841220534872e-05, 9.518062142888084e-06, 2.7989184673060663e-05, 4.4915126636624336e-05, 0.0001222387800225988, 3.354273576405831e-05, 0.00014459813246503472, 8.568180419388227e-06, 7.901116987341084e-06, 0.00017937165102921426, 2.3317214072449133e-05, 0.00024729850701987743, 4.195037035970017e-06, 6.005799878039397e-05, 0.000468646758235991, 1.4760231351829134e-05, 4.85954024043167e-06, 1.6895357475732453e-05, 1.5957341020111926e-05, 7.874483708292246e-05, 1.3202526133682113e-05, 1.8148684830521233e-05, 1.0213702807959635e-05, 3.4673401387408376e-05, 4.5977205445524305e-05, 4.386862201499753e-06, 5.001065801479854e-05, 9.235776087734848e-05, 6.017622217768803e-05, 1.0620692592056002e-05, 5.457341103465296e-06, 6.401246082532452e-06, 0.0001616390945855528, 0.0002964979794342071, 4.907612947135931e-06, 0.0004369121161289513, 6.021500303177163e-05, 0.00016232400957960635, 0.0002586930640973151, 0.0006226834957487881, 0.00030874088406562805, 0.00011056206858484074, 1.5359439203166403e-05, 1.2718787729681935e-05, 0.0008577489643357694, 5.671149665431585e-06, 0.00018466246547177434, 2.7054717065766454e-05, 0.00010345289774704725, 6.74259863444604e-05, 4.1777482692850754e-05, 0.0003508909430820495, 0.00010379599552834406, 0.00013080798089504242, 3.864690734189935e-05, 7.166116120060906e-06, 6.6944112404598854e-06, 1.7546137314639054e-05, 4.323515895521268e-05, 3.5386939998716116e-05, 4.046845424454659e-05, 7.2912557698145974e-06, 5.880143362446688e-05, 0.00019147078273817897, 3.75578019884415e-05, 9.700916962174233e-06, 0.0002978501725010574, 7.157949585234746e-05, 2.1966907297610305e-05, 1.0644440408213995e-05, 2.5478411771473475e-05, 1.905795556922385e-06, 0.0008435758645646274, 0.0035042979288846254, 0.0003086476353928447, 0.0005145936738699675, 0.0003126856463495642, 4.807657387573272e-05, 6.704839506710414e-06, 1.0528659004194196e-05, 9.225279427482747e-06, 2.2994365735939937e-06, 4.111402449780144e-05, 6.22359630142455e-06, 0.00023370806593447924, 3.182554428349249e-05, 1.1492322300910018e-05, 4.822525443159975e-05, 0.00011827834532596171, 6.814514108555159e-06, 3.7631907616741955e-06, 0.0004764268815051764, 3.9707546761746926e-07, 0.00021917237609159201, 0.0001306227350141853, 4.837073720409535e-05, 2.9424581953207962e-05, 3.755648504011333e-05, 0.00016286640311591327, 2.7047135517932475e-06, 1.5778564375068527e-06, 8.682066436449531e-06, 2.486960329406429e-05, 9.38673983910121e-05, 6.978199235163629e-05, 7.658040885871742e-06, 4.387693024909822e-06, 9.067390055861324e-05, 0.0008158629643730819, 0.0036857917439192533, 3.695888835864025e-06, 0.00041457198676653206, 3.2999889754137257e-06, 2.869910531444475e-05, 1.958695793291554e-05, 6.807315457990626e-06, 7.495829049730673e-05, 2.706982741074171e-05, 1.3925335224485025e-05, 4.687133241532138e-06, 1.7939391909749247e-05, 0.0006406776956282556, 0.00023826592951081693, 0.00023667326604481786, 1.4636505056841997e-06, 0.00021639937767758965, 1.067550419975305e-05, 9.813231372390874e-06, 1.9784201867878437e-05, 1.321727268077666e-06, 0.0006308761658146977, 3.911260500899516e-05, 1.919947180795134e-06, 8.947706373874098e-05, 2.961318614325137e-06, 2.594479883555323e-05, 6.513076186820399e-06, 0.012368470430374146, 7.637974704266526e-06, 4.1899020288838074e-05, 9.239484484169225e-07, 9.169909753836691e-05, 2.347446024941746e-05, 2.009841955441516e-05, 5.549433990381658e-05, 3.9106231270125136e-05, 1.7033400581567548e-05, 1.1140961760247592e-05, 4.6692849764440325e-07, 1.4901202121109236e-06, 3.6066680308977084e-07, 0.0001241822901647538, 0.00022745819296687841, 8.729852197575383e-06, 6.521943578263745e-05, 3.3231974612135673e-06, 1.7068441593437456e-05, 5.251233233138919e-06, 1.6039834008552134e-05, 0.005363455042243004, 7.851578266127035e-05, 5.4009113227948546e-05, 3.720506720128469e-05, 8.534874496035627e-07, 0.00011687448568409309, 0.00010324351023882627, 0.0006868057535029948, 4.764864934259094e-05, 4.5180192387306306e-07, 2.5243041363864904e-06, 0.002105801599100232, 0.00010655648657120764, 2.603425400593551e-06, 0.0018307415302842855, 1.4460834790952504e-05, 7.25405516277533e-06, 1.9140156837238465e-06, 3.953675332013518e-06, 1.5642706785001792e-05, 6.7747419052466284e-06, 1.589670972634849e-07, 3.0555511330021545e-06, 0.009266490116715431, 5.125988172949292e-05, 9.13966869120486e-06, 6.36785916867666e-05, 0.0010143552208319306, 5.10247673446429e-06, 9.646452781453263e-06, 3.874950380122755e-06, 0.0004148851439822465, 0.0010362721513956785, 3.455232217675075e-05, 6.051178206689656e-05, 7.834793905203696e-06, 5.652284744428471e-05, 2.9338486911001382e-06, 0.0001138838124461472, 0.0008150221547111869, 0.000146095801028423, 0.00019804626936092973, 1.0679349543352146e-05, 2.7752039386541583e-06, 0.0008993723895400763, 4.944849479215918e-06, 0.00011615003313636407, 0.00022738223196938634, 2.0293367924750783e-05, 5.3522351663559675e-05, 1.1745883057301398e-05, 0.00029189776978455484, 0.00013102403318043798, 7.680046110181138e-05, 0.001959489891305566, 1.2558041817101184e-05, 2.9787599487463012e-05, 3.1590423077432206e-06, 0.00022099593479651958, 0.001509903115220368, 3.1132152798818424e-05, 5.960316411801614e-05, 8.479724056087434e-05, 4.0822418668540195e-05, 9.472292731516063e-05, 2.5420433303224854e-05, 0.000692650384735316, 2.9850938517483883e-05, 4.9432379455538467e-05, 7.283131708391011e-06, 3.165491943946108e-05, 4.629262662092515e-07, 0.009175654500722885, 7.927261322038248e-05, 0.0002197000285377726, 0.00014402765373233706, 2.809138641168829e-05, 2.0438144929357804e-05, 1.1006947715941351e-05, 0.0003546826192177832, 1.5103702935448382e-05, 5.35926938027842e-06, 5.9687656175810844e-05, 2.3385482563753612e-06, 6.317959559964947e-06, 0.00019188002625014633, 8.646277274237946e-05, 0.00010150203161174431, 0.0004690337518695742, 3.589640300560859e-06, 3.0790270102443174e-06, 0.0004831124097108841, 3.0030256311874837e-05, 0.00018660050409380347, 4.41677839262411e-05, 2.2940850612940267e-05, 1.8341266695642844e-05, 6.29271671641618e-05, 4.3527256821107585e-06, 0.0001585268328199163, 0.00015171649283729494, 8.080026600509882e-05, 6.321090495475801e-07, 1.2381734450173099e-05, 0.00023028205032460392, 4.388788147480227e-05, 0.00014780907076783478, 2.1223995645414107e-05, 2.168060973417596e-06, 9.882377526082564e-06, 1.938763125508558e-05, 5.514214717550203e-05, 4.15090580645483e-05, 0.0001215126903844066, 3.694958286359906e-05, 2.8536598620121367e-05, 3.8458678318420425e-05, 2.17433793636701e-07, 1.734639772621449e-05, 1.575502028572373e-05, 0.00025538488989695907, 1.714208337944001e-05, 0.00027753436006605625, 0.00013418064918369055, 2.602227596071316e-06, 0.00010857157758437097, 0.00012057572166668251, 9.257324563805014e-05, 1.3452065104502253e-05, 0.0007191973854787648, 1.2588036497618305e-06, 0.00042491810745559633, 7.191640906967223e-05, 5.1681574404938146e-05, 0.00016554862668272108, 2.2076266759540886e-05, 7.701613640165306e-07, 8.36598701425828e-06, 4.804807758773677e-05, 1.200051701744087e-05, 1.915942812047433e-05, 4.846885349252261e-05, 2.5002209440572187e-06, 0.00022870498651172966, 1.7402038793079555e-05, 2.8916358132846653e-05, 3.381574060767889e-05, 0.00017253708210773766, 4.7414656023647694e-07, 0.00038474262692034245, 0.0003866228216793388, 2.2412925773096504e-06, 0.0020166474860161543, 9.257376223104075e-05, 4.29285537393298e-05, 1.5102933502930682e-05, 5.966363005427411e-06, 6.921115709701553e-05, 3.6096307667321526e-06, 1.6204508938244544e-05, 1.5171264067248558e-06, 6.015114649926545e-06, 0.00025023898342624307, 0.00012134618009440601, 1.9948358385590836e-05, 4.410738620208576e-05, 0.0010638852836564183, 2.1290432414389215e-05, 7.52505729906261e-05, 4.822888513444923e-05, 0.00011392903979867697, 0.00013341865269467235, 9.194332051265519e-08, 9.426689757674467e-06, 0.005940861534327269, 2.3866376068326645e-05, 1.1133257658002549e-06, 8.809423889033496e-05, 3.286827995907515e-05, 2.0040157323819585e-05, 4.956465090799611e-06, 0.0010646269656717777, 9.498962754150853e-06, 2.092705835821107e-05, 3.590578444345738e-06, 3.5081709938822314e-05, 0.00011120202543679625, 0.0006132902926765382, 0.0002992512017954141, 2.3776699435984483e-06, 2.2401455623821676e-07, 0.00011742101924028248, 1.0842917390618823e-06, 0.0006118069286458194, 4.3406802433310077e-05, 1.6498288459843025e-05, 6.610449418076314e-06, 4.1801686165854335e-05, 3.4159584174631163e-06, 1.36325588755426e-05, 2.2353477106662467e-05, 7.560410608675738e-07, 6.969591049710289e-05, 8.375295146834105e-05, 1.1480293551358045e-06, 0.0015335145872086287, 3.9222279156092554e-05, 0.00017269959789700806, 1.280195101571735e-05, 0.00018752322648651898, 5.7152552471961826e-05, 0.012437465600669384, 1.6264217265415937e-05, 4.2596657294780016e-05, 4.5827147232557763e-07, 9.110532118938863e-06, 4.27467114150204e-07, 7.642531272722408e-05, 2.770419632724952e-05, 1.874803456303198e-05, 4.5531262003351e-05, 4.342847751104273e-05, 3.8100584788480774e-05, 4.390100002638064e-05, 3.7517002056119964e-05, 4.920498395222239e-05, 1.8583614291856065e-05, 2.5727074898895808e-05, 1.5661771612940356e-05, 2.287361894559581e-06, 4.7033549890329596e-06, 4.645379249268444e-06, 7.19727759133093e-05, 2.4639379262225702e-05, 0.00010594914783723652, 0.00010282557923346758, 6.85276318108663e-05, 0.0008401189697906375, 0.0007804102497175336, 8.434462870354764e-06, 3.236323027522303e-05, 1.8615552107803524e-05, 7.064236706355587e-05, 8.09039283922175e-06, 0.0002674215065781027, 3.160831693094224e-05, 0.0010520884534344077, 3.432502853684127e-05, 0.00023435134789906442, 4.755339978146367e-05, 3.275006383773871e-05, 7.282682781806216e-05, 5.029197927797213e-05, 1.4770874940950307e-06, 2.6058460207423195e-05, 1.570314634591341e-05, 1.212197275890503e-05, 2.255031358799897e-05, 0.00011195761908311397, 0.00010794633999466896, 6.8573237513192e-05, 0.0002316591708222404, 0.0031297020614147186, 8.06490279501304e-05, 0.00023533881176263094, 0.00027909764321520925, 1.5536745195277035e-05, 2.7042709916713648e-05, 0.0004391996772028506, 8.202305252780207e-06, 1.029608665703563e-05, 1.2935993254359346e-05, 7.05741549609229e-05, 0.0006496476125903428, 0.002289865631610155, 0.00094400136731565, 3.231726441299543e-05, 1.730392250465229e-05, 4.692065886047203e-06, 7.04165131537593e-07, 2.3796741515980102e-05, 1.2284052672839607e-06, 4.3711748730856925e-05, 7.751942030154169e-05, 1.9028944734600373e-05, 0.0012636661995202303, 7.310505316127092e-05, 1.241321206180146e-05, 8.943674765760079e-05, 8.673475804243935e-07, 1.1805719623225741e-05, 3.528502566041425e-05, 6.005323484714609e-06, 4.108752546017058e-05, 0.00012328472803346813, 2.1148611267562956e-05, 8.431759852101095e-06, 5.0664068112382665e-05, 7.654071669094265e-05, 0.0006357500678859651, 1.2397322279866785e-06, 0.006130640394985676, 6.410075729945675e-05, 1.6314570530084893e-06, 0.00014484685380011797, 0.00026788184186443686, 6.0767629292968195e-06, 1.9959212295361795e-05, 4.931951934850076e-06, 1.4421626474359073e-05, 8.478469680994749e-05, 0.00010589780868031085, 3.282883335486986e-05, 6.3869988480291795e-06, 1.9580129446694627e-05, 5.503703869180754e-05, 3.7690542740165256e-06, 3.65837627214205e-06, 0.0001430134871043265, 0.00010958878556266427, 0.0002910935436375439, 8.421124562119076e-07, 0.0018617025343701243, 0.0007383464835584164, 2.3208281163533684e-06, 4.108793655177578e-05, 1.320278624916682e-05, 9.368668543174863e-06, 4.342806641943753e-05, 1.0210438858848647e-06, 1.2604215044120792e-05, 0.0005900998949073255, 3.4841959859477356e-05, 9.238214261131361e-06, 1.933410021592863e-05, 3.884766101691639e-06, 0.0027861776761710644, 8.170291948772501e-06, 7.357231515925378e-05, 9.44404091569595e-05, 8.43117231852375e-05, 1.2388441064103972e-05, 3.0584110390918795e-06, 0.00016900715127121657, 5.5334119679173455e-05, 2.4317037059518043e-06, 0.000178704533027485, 0.0003044114855583757, 0.0009122421615757048, 1.0922854926320724e-05, 1.0530708323130966e-06, 9.967188816517591e-06, 4.215927765471861e-05, 0.0001820837933337316, 0.00018501677550375462, 3.5103850677842274e-05, 4.16562352256733e-06, 0.00020034318731632084, 0.00025771241053007543, 0.0005570317152887583, 2.1208641555858776e-05, 4.561849345918745e-05, 4.874618025496602e-05, 0.004816822707653046, 3.735823383976822e-07, 5.848881119163707e-05, 1.788477135278299e-07, 1.069942027243087e-05, 6.941778337932192e-06, 0.0001399581233272329, 0.00027855922235175967, 4.286455805413425e-06, 0.0007741956505924463, 0.0007374281412921846, 3.142516652587801e-05, 8.788440027274191e-06, 2.7178461095900275e-05, 1.3843302895111265e-06, 0.00022921842173673213, 2.404109409326338e-07, 0.00011055703362217173, 0.00025456931325607, 0.00011059094686061144, 4.310901931603439e-05, 1.0385729183326475e-05, 6.228355232451577e-06, 2.250752913823817e-05, 7.748268217255827e-06, 7.101789833541261e-06, 0.0014725710498169065, 5.691438855137676e-05, 0.00029781245393678546, 0.00014748690591659397, 0.0009453252423554659, 1.2724501175398473e-05, 3.5705054415302584e-06, 1.071055703505408e-05, 0.0004810895479749888, 2.523697503420408e-06, 1.8408887854093337e-06, 1.8750157323665917e-05, 1.3185125681047793e-05, 7.045779057079926e-06, 0.0003936937137041241, 9.873553790384904e-05, 3.437323175603524e-05, 0.0012106107315048575, 0.00022233575873542577, 4.3783711589640006e-05, 3.238917997805402e-05, 1.4374770216818433e-05, 0.00044931023148819804, 9.712510291137733e-06, 7.131117854441982e-06, 0.0001988602743949741, 0.00017691374523565173, 3.3173939755215542e-06, 0.0010749326320365071, 1.3556433259509504e-05, 0.0008237023139372468, 9.835499668042758e-07, 6.65094285068335e-06, 5.486899681272916e-07, 4.914466899208492e-06, 3.708602525875904e-05, 8.754339069128036e-05, 0.00017076403310056776, 4.6407036279561e-05, 7.287241169251502e-05, 6.831015070929425e-06, 0.0002294947043992579, 3.3484844607301056e-05, 8.766887731326278e-06, 2.1783549527754076e-05, 1.7479779899076675e-06, 3.4587792470119894e-06, 5.867326308361953e-06, 1.5976037275322597e-06, 0.0007601325633004308, 5.068534846941475e-06, 0.00024280438083223999, 0.000829491182230413, 9.04359512787778e-06, 9.646526450524107e-06, 0.0009307258878834546, 0.0019322483567520976, 5.1344733947189525e-05, 0.0004926017136313021, 4.192313554085558e-06, 0.00024405407020822167, 1.0240981964670937e-06, 2.6301495381630957e-06, 0.0002931644266936928, 4.447689718745096e-07, 0.00036343655665405095, 5.705318017135141e-06, 1.6129166397149675e-05, 7.434825965901837e-05, 2.0549290638882667e-05, 4.5696656343352515e-06, 2.5032552457560087e-06, 0.0003085397183895111, 0.0005676739383488894, 1.541133133287076e-05, 1.1876312782987952e-05, 6.754708579137514e-07, 0.000261408684309572, 6.323056004475802e-05, 0.00010728948836913332, 0.00014991215721238405, 4.320596872275928e-06, 4.480870484258048e-05, 1.2903144124720711e-05, 0.0004942137165926397, 2.907492671511136e-05, 0.00041617234819568694, 0.00024937803391367197, 0.00012700914521701634, 7.106689281499712e-06, 2.096599928336218e-05, 0.0003605535894166678, 0.00027609989047050476, 0.0001262293808395043, 7.064872988848947e-06, 5.5987176892813295e-05, 4.0383176383329555e-05, 5.7559311244403943e-05, 4.2027844756376e-05, 0.00019358850840944797, 1.4095950973569416e-05, 0.00014574569649994373, 5.7582350564189255e-05, 1.3521686923922971e-05, 6.678416411887156e-06, 0.0004351128009147942, 1.745516601658892e-05, 6.286065035965294e-05, 0.0027707137633115053, 2.1397381715360098e-05, 0.0005498045356944203, 4.145814818912186e-05, 6.621189822908491e-05, 2.787727862596512e-05, 0.0002217732835561037, 7.316197752516018e-06, 1.6097823390737176e-05, 0.00026359426556155086, 0.006173481233417988, 2.292208773724269e-05, 1.0671917607396608e-06, 5.3785130148753524e-05, 1.3066579413134605e-05, 0.0002299918414792046, 0.00016132333257701248, 1.9014035501641047e-07, 0.0001212703573401086, 5.963547664578073e-06, 0.00010716827091528103, 4.558509317575954e-05, 0.00024217016471084207, 2.0585604943335056e-05, 0.00022285879822447896, 0.00020286545623093843, 0.00022352753148879856, 7.211949559859931e-05, 1.6442158084828407e-05, 3.1869181839283556e-05, 5.419076842372306e-05, 2.7988158763037063e-05, 2.2143580281408504e-05, 6.387876965163741e-06, 0.00013571690942626446, 0.00044196724775247276, 1.8431667285767617e-06, 6.968557136133313e-05, 0.00016507570398971438, 3.9999988075578585e-05, 2.7269015845376998e-05, 0.00026479054940864444, 0.00012292091560084373, 1.083438291971106e-05, 1.006492311717011e-06, 2.1118130462127738e-06, 1.7259702644878416e-06, 0.00011629690561676398, 3.397265390958637e-05, 5.0555765483295545e-05, 4.797182555194013e-05, 4.29046485805884e-05, 1.0615491191856563e-05, 6.852013029856607e-06, 0.0008528126054443419, 0.00017405459948349744, 5.1791106670862064e-05, 0.0003473499964457005, 2.8178581032989314e-06, 0.00020287040388211608, 3.586885577533394e-05, 2.035276111200801e-06, 1.710268588794861e-05, 0.0001366107608191669, 0.00012092779070371762, 0.00015294855984393507, 2.2231579350773245e-06, 2.3184249585028738e-05, 5.396505002863705e-05, 2.3139900804380886e-05, 9.854419477051124e-05, 3.913315595127642e-06, 7.665651537536178e-06, 0.0002448692684993148, 9.55769355641678e-05, 4.916916441288777e-05, 2.7044176022172906e-05, 0.00025108130648732185, 5.6255598792631645e-06, 0.00046017495333217084, 5.052072538092034e-06, 3.291066241217777e-05, 0.000355083670001477, 6.60387086099945e-05, 9.01052771951072e-06, 0.0001400924229528755, 0.007942218333482742, 0.0002650539972819388, 5.376888566388516e-06, 4.815701686311513e-06, 4.298483872844372e-06, 2.3794145818101242e-05, 1.1808555427705869e-05, 4.442670231696866e-08, 1.6919828340178356e-05, 1.1512650416989345e-05, 1.105660430766875e-05, 0.00011527499736985192, 0.00022908787650521845, 0.000561526685487479, 1.5665920727769844e-05, 5.394412255554926e-06, 0.00018723949324339628, 4.436914241523482e-05, 0.0005457639927044511, 2.6521696781856008e-05, 2.246437725261785e-05, 4.2556628613965586e-05, 3.7337282265070826e-05, 0.00019673697534017265, 0.0006747565930709243, 2.65690414380515e-06, 0.00014425670087803155, 5.680095637217164e-05, 0.00014732162526343018, 0.0007143940310925245, 6.509394552267622e-06, 3.777202550736547e-07, 8.989965863293037e-05, 7.781374006299302e-05, 4.516593980952166e-05, 1.2024094758089632e-05, 3.8016722101019695e-05, 1.979906528504216e-06, 8.891378092812374e-05, 7.42330084904097e-05, 5.0836566515499726e-05, 0.0005355850444175303, 6.996773663558997e-06, 4.223709765938111e-06, 2.8653568733716384e-05, 0.0005597378476522863, 1.732334931148216e-05, 1.640710252104327e-06, 4.347840331320185e-06, 0.00011915498180314898, 9.940631571225822e-07, 1.8597471353132278e-05, 9.861121361609548e-05, 0.002601978601887822, 7.578587246825919e-05, 3.7288598832674325e-05, 7.621677468705457e-06, 1.9078936475125374e-06, 7.067627393553266e-06, 3.4153849810536485e-06, 5.5661617807345465e-05, 1.331710109298001e-06, 2.306139504071325e-05, 1.9323731976328418e-05, 2.2851203539175913e-05, 3.887942511937581e-05, 0.00018215473392046988, 1.7154789020423777e-06, 1.713614551590581e-06, 0.00011374173482181504, 0.0002142914745490998, 8.66016634972766e-05, 9.236428923031781e-06, 4.810764585272409e-05, 4.971813723386731e-06, 2.455207686580252e-05, 1.6580457895543077e-06, 8.688894740771502e-05, 1.1921312761842273e-05, 0.0001447068789275363, 3.2910750178416492e-06, 2.0786272216355428e-05, 6.393543299054727e-05, 6.457175459217979e-06, 8.304353286803234e-06, 5.2261755627114326e-05, 6.370113260345533e-05, 4.212868952890858e-05, 5.070674524176866e-05, 2.800972833938431e-05, 8.5021447375766e-06, 1.8121998436981812e-05, 1.8725968402577564e-05, 6.009582648403011e-05, 5.471292752190493e-06, 0.00013657235831487924, 3.698850196087733e-05, 3.05997091345489e-05, 0.0006459968280978501, 0.0004858184838667512, 0.00022761239961255342, 1.6580233932472765e-05, 0.00019352728850208223, 0.00033374078338965774, 2.2567774067283608e-05, 8.177958079613745e-06, 4.29844703830895e-06, 4.541759892617847e-07, 4.82300492876675e-05, 7.148243457777426e-05, 3.194675446138717e-05, 2.0191982912365347e-05, 0.0009100927854888141, 4.325250301917549e-06, 9.323946142103523e-05, 6.184596713865176e-05, 2.3125345251173712e-05, 2.3121801859815605e-05, 4.285184695618227e-05, 0.0001433900761185214, 4.1575483919586986e-05, 4.4855234591523185e-05, 1.5534658359683817e-06, 2.138077206836897e-06, 4.757766510010697e-05, 9.421856520930305e-05, 3.364723306731321e-05, 9.724915435072035e-05, 4.92991239298135e-06, 0.00021335012570489198, 0.00013222593406680971, 0.0009964094497263432, 6.160356861073524e-05, 0.00010516912152525038, 1.0475227099959739e-05, 1.8718046703725122e-05, 5.819588295707945e-06, 3.5596019642980536e-06, 0.0019744837190955877, 8.21383946458809e-06, 0.0008777983020991087, 0.0004050196148455143, 0.00022980058565735817, 2.366484477533959e-05, 0.00020234445401001722, 0.00047368797822855413, 7.458111781488697e-07, 8.315834566019475e-05, 3.7629854432452703e-06, 2.6014511149696773e-06, 0.00014853765605948865, 0.002050320617854595, 0.001144044566899538, 1.9874153167620534e-06, 1.2054628086843877e-06, 0.0001040423521772027, 0.0003439235733821988, 1.2408748943926184e-06, 0.0001329074293607846, 4.176330548943952e-06, 1.883241748146247e-05, 1.4341128007799853e-05, 2.2266494852374308e-05, 0.0014072617050260305, 1.6330383004969917e-05, 7.725859177298844e-05, 0.00023268676886800677, 2.9546761652454734e-05, 0.00029657644336111844, 1.4516663213726133e-05, 1.0010155165218748e-05, 0.0003868179046548903, 1.0435161129862536e-05, 9.525861969450489e-05, 4.352698397269705e-06, 5.0871618441306055e-05, 0.00016694182704668492, 5.064948254585033e-06, 0.00014713237760588527, 0.0005464325658977032, 1.1634515431069303e-05, 5.563370723393746e-05, 1.4842930795566645e-06, 0.00022301208809949458, 0.004405605606734753, 1.9631403120001778e-05, 2.4899060008465312e-05, 7.470022183042602e-07, 0.001975636463612318, 0.0028378181159496307, 1.3864922038919758e-05, 1.272546433028765e-05, 4.755371264764108e-05, 0.0001627588499104604, 4.022895882371813e-05, 1.4603097952203825e-05, 9.869923087535426e-05, 0.00021993546397425234, 0.00027990000671707094, 7.760450898786075e-06, 0.0004915151512250304, 4.9045840569306165e-05, 0.00033589627128094435, 0.009524424560368061, 7.954950888233725e-06, 4.734017056762241e-05, 0.00015029514906927943, 3.855102477245964e-06, 1.0976834801113e-06, 0.0010087472619488835, 2.1418577489384916e-06, 5.215724013396539e-05, 2.6372370484750718e-05, 1.2846445315517485e-05, 0.00011193144746357575, 1.4176916920405347e-05, 6.951596878934652e-05, 8.272651757579297e-05, 1.8758780129246588e-07, 3.448135976213962e-05, 1.4035284948477056e-05, 3.9500309867435135e-06, 2.5967230612877756e-05, 2.9376758448051987e-06, 0.00018394645303487778, 0.000674113049171865, 8.748066466068849e-05, 6.669574213447049e-05, 8.869723387761042e-05, 2.2718190848536324e-06, 0.0007380112074315548, 2.136830698873382e-05, 6.460078338932362e-08, 2.63609217654448e-05, 9.555660653859377e-06, 5.193433025851846e-05, 0.00035641665454022586, 8.009083103388548e-06, 3.988876778748818e-05, 9.627640974940732e-05, 3.1714407668914646e-05, 6.819925147283357e-06, 6.337613172036072e-07, 9.550045433570631e-06, 0.0001807394000934437, 0.00010485984239494428, 6.204639794304967e-05, 8.260226604761556e-05, 4.1209095797967166e-06, 0.0008050779579207301, 0.00036429628380574286, 6.473796383943409e-05, 1.1722869203367736e-05, 2.623329419293441e-05, 4.5704109652433544e-06, 9.05183915165253e-05, 0.00022025383077561855, 4.883804194832919e-06, 1.78292793862056e-05, 0.00019593424804043025, 0.0007886383100412786, 0.00011582153820199892, 6.2968952079245355e-06, 0.00013277013204060495, 2.0045625205966644e-05, 0.00018721018568612635, 0.00031745104934088886, 0.010528520680963993, 2.431247594358865e-06, 9.045544402397354e-07, 5.882988716621185e-06, 3.961841230193386e-06, 6.622598220928921e-07, 2.9562825147877447e-06, 2.5177682800858747e-06, 0.00022922706557437778, 1.3721471987082623e-05, 2.0161642169114202e-06, 6.955661228857934e-05, 5.952651349616644e-07, 5.958810288575478e-05, 7.555403044534614e-06, 0.0001096972991945222, 1.9965832507296e-06, 0.0005136065883561969, 0.00013859133468940854, 5.470951691677328e-06, 4.950509901391342e-05, 1.6604031998213031e-06, 2.07332141144434e-06, 9.603074431652203e-06, 0.00022781299776397645, 0.007911953143775463, 8.232371328631416e-06, 7.058100163703784e-05, 2.0797951947315596e-05, 1.1036795513064135e-06, 1.2325369880272774e-06, 0.00010750835645012558, 8.484155114274472e-05, 9.11112510948442e-06, 1.3268770089780446e-05, 3.076364373555407e-05, 2.1887004550080746e-05, 7.120812369976193e-06, 2.4974136977107264e-05, 0.00035244019818492234, 1.191063620353816e-05, 1.921874536492396e-05, 8.807380800135434e-05, 0.0003112745762336999, 4.8043970309663564e-05, 9.574081559549086e-06, 7.532700692536309e-06, 1.1016410326192272e-06, 7.993434678610356e-07, 0.00015596492448821664, 0.0005265453946776688, 9.14259726414457e-05, 4.965744210494449e-06, 4.029269803140778e-06, 5.115834028401878e-06, 2.2611064196098596e-05, 1.0025382835010532e-05, 7.490714324376313e-06, 0.0001457699399907142, 7.2996349445020314e-06, 8.455292118014768e-06, 1.6242520359810442e-05, 5.9067992879136e-06, 0.0010361409513279796, 0.00013735257380176336, 0.00010259437840431929, 1.5918254575808533e-05, 1.9125075141346315e-06, 3.284424019511789e-05, 0.0001551169261801988, 1.457877988286782e-05, 0.0006046434282325208, 0.00015753878687974066, 9.300221427110955e-05, 1.0800293239299208e-05, 5.233946467342321e-06, 8.977888796835032e-07, 0.0009362556738778949, 0.0001696421968517825, 0.001023941789753735, 1.4932799786038231e-05, 0.00022911906125955284, 2.308616240043193e-05, 2.3243441319209523e-05, 0.0005256705917418003, 0.00023390249407384545, 5.505317312781699e-05, 7.630226173205301e-05, 2.2932806587050436e-06, 0.000808859127573669, 0.000495347601827234, 2.3138773030950688e-06, 4.217326932121068e-06, 0.0010660297702997923, 0.0002979869022965431, 8.008066652109846e-05, 3.1463589493796462e-06, 1.5404340956592932e-05, 8.17152795207221e-06, 0.0003933871630579233, 4.490035280468874e-06, 1.026406607707031e-05, 0.00015534170961473137, 4.243303010298405e-06, 2.2090148377174046e-07, 7.328619176405482e-06, 8.129042726068292e-06, 1.4667008372271084e-06, 5.7219235714001115e-06, 0.0005410924786701798, 0.0005248666275292635, 0.00034937955206260085, 2.5204290068359114e-06, 0.0008674305863678455, 1.5843757864786312e-05, 7.558072411484318e-06, 1.146187742051552e-06, 1.3101614968036301e-05, 1.366916603728896e-05, 4.18915442423895e-05, 1.4428552276513074e-05, 1.0816601388796698e-05, 2.1679643396055326e-05, 3.0249249903135933e-05, 0.0007990169688127935, 1.5079963304742705e-05, 1.0697341394916293e-06, 3.000586548296269e-05, 4.303667810745537e-05, 6.878612111904658e-06, 1.8103328329743817e-05, 9.447761840419844e-05, 0.0001421984052285552, 0.00010324252070859075, 1.1784061825892422e-05, 2.4363222109968774e-05, 3.3096690458478406e-05, 3.315819049021229e-05, 2.006079739658162e-06, 3.5484346881275997e-05, 4.2303406644350616e-07, 1.8702618035604246e-05, 6.0022357502020895e-06, 3.660465881694108e-05, 1.3370895430853125e-06, 0.007188922725617886, 0.0001280914875678718, 1.9235080799262505e-06, 2.848588565029786e-06, 5.218486421654234e-06, 0.00013336242409422994, 3.868223575409502e-05, 0.0002837199717760086, 0.0018369408790022135, 2.863154441001825e-05, 5.900991709495429e-06, 0.002862650202587247, 3.0260809580795467e-05, 6.747998850187287e-05, 1.3629361092171166e-05, 0.00011308137618470937, 1.9085489839199e-05, 0.011305789463222027, 0.001221621292643249, 0.00018947265925817192, 8.014499326236546e-05, 9.048153879120946e-05, 0.0001415905571775511, 2.342184188819374e-06, 4.9498939915793017e-05, 8.413616342295427e-06, 6.516408757306635e-05, 6.478527939179912e-05, 1.1086921176683973e-06, 7.98427063273266e-05, 0.00014468008885160089, 0.01154087670147419, 4.572366742650047e-05, 0.00017060812388081104, 2.347870122321183e-06, 4.5991429942660034e-05, 7.626274509675568e-06, 8.473650268570054e-06, 0.00455854507163167, 1.9350734874024056e-05, 1.5709285435150377e-05, 0.025361035019159317, 0.0006388722686097026, 0.00890701450407505, 6.829079211456701e-06, 1.3030202353547793e-05, 1.938311652338598e-05, 6.099013626226224e-05, 6.852296064607799e-05, 6.282523099798709e-05, 0.0025673334021121264, 1.7445880757804844e-06, 2.764204327831976e-05, 1.4466126231127419e-05, 4.724940936284838e-06, 5.65495702176122e-06, 6.150981789687648e-05, 0.0008486541919410229, 9.698774374555796e-05, 0.00022980269568506628, 0.00021843117428943515, 6.772519554942846e-05, 1.3041165402682964e-05, 1.3763928109256085e-05, 5.624965524475556e-06, 0.0001236527896253392, 1.3490457604348194e-05, 4.3509055558388354e-07, 1.973245389308431e-06, 3.8623584259767085e-05, 8.17995096440427e-05, 0.0001683915324974805, 3.1953681173035875e-05, 2.721652163018007e-05, 1.4093005120230373e-05, 0.0002996273688040674, 1.0778659088828135e-05, 0.0012203826336190104, 3.066876161028631e-05, 6.030567237758078e-05, 8.604960021330044e-05, 5.3862181630393025e-06, 4.3249787267996e-05, 1.0098317943629809e-05, 6.449868669733405e-05, 0.0010055204620584846, 3.102294067502953e-05, 0.00016179682279471308, 2.7790426315732475e-07, 3.278194708400406e-05, 1.2435220014594961e-05, 1.4974885743868072e-05, 9.349847118755861e-08, 1.0744254723249469e-05, 7.4270678851462435e-06, 0.0005353564047254622, 9.998688256018795e-06, 0.00022944856027606875, 3.127191303065047e-05, 5.124850304127904e-06, 0.00012402061838656664, 7.499561434087809e-07, 9.459187708671379e-07, 5.654176788993936e-07, 1.0832272892002948e-05, 4.570942837744951e-05, 3.1268639304471435e-06, 0.0002721424971241504, 6.07238234806573e-06, 0.00013369295629672706, 0.00024287469568662345, 1.7496612372269738e-06, 3.994588496425422e-06, 2.4273089366033673e-05, 3.6648289096774533e-05, 0.00018867908511310816, 8.447147411061451e-05, 8.491266635246575e-05, 0.0010206757578998804, 4.5342309022089466e-05, 0.0002821654488798231, 0.00011333886504871771, 0.00035581900738179684, 0.0016016751760616899, 1.887928920041304e-05, 2.4608658350189216e-05, 2.5674575226730667e-05, 0.00022921159688849002, 3.394201849005185e-05, 2.004104135266971e-05, 0.0003681539965327829, 0.0004113263275939971, 2.8359319912851788e-05, 3.4546353617770365e-06, 2.9744509447482415e-05, 7.729005301371217e-05, 4.744892976304982e-06, 5.829651854583062e-05, 9.021319783641957e-06, 1.2765084647980984e-05, 5.478685852722265e-05, 2.543215259720455e-06, 6.136170668469276e-06, 0.00035869833664037287, 9.165274263978063e-07, 8.439671546511818e-06, 2.7919824788114056e-05, 0.0001442125503672287, 1.3894986295781564e-05, 1.1740104127966333e-05, 7.045233360258862e-06, 1.2738865734718274e-05, 0.0001216416058014147, 8.545619493816048e-06, 5.875124315934954e-06, 0.0001074763567885384, 0.0002004191483138129, 9.698847134131938e-05, 9.975541615858674e-05, 7.483574881916866e-05, 5.179069921723567e-05, 2.677434531506151e-05, 3.3898017136380076e-05, 3.343623120599659e-07, 1.030006205837708e-05, 0.029734648764133453, 1.5694762623752467e-05, 1.803474879125133e-05, 1.9640578102553263e-05, 0.0008489119936712086, 0.0031541523057967424, 1.6054817706390168e-06, 3.1867744837654755e-05, 7.634648682142142e-06, 3.257340722484514e-05, 6.620880867558299e-06, 0.0001162708067568019, 7.499953426304273e-06, 1.9050870832870714e-05, 2.5730516426847316e-05, 2.272801248182077e-05, 7.102551171556115e-05, 1.0761690646177158e-05, 3.9604929042980075e-05, 3.1830815714783967e-05, 8.896866347640753e-06, 4.719277967524249e-06, 7.047190138109727e-06, 7.2274065132660326e-06, 1.3437250345305074e-05, 0.00036223558709025383, 2.7161326215718873e-05, 8.398232603212819e-05, 0.000519072636961937, 5.789479109807871e-05, 1.0006583579524886e-05, 0.0007295754039660096, 1.5138255093916086e-06, 1.0608500815578736e-05, 4.793735206476413e-05, 2.2026717488188297e-05, 1.8575192370917648e-05, 9.630344720790163e-05, 9.50206012930721e-05, 0.00021383749844972044, 0.00010664750152500346, 1.3930991826782702e-07, 6.7281116571393795e-06, 6.569837296410697e-07, 1.747476198943332e-05, 1.7766507880878635e-05, 9.164032235275954e-05, 1.0034246770374011e-05, 5.91396565141622e-05, 3.7903555494267493e-05, 7.078861381160095e-05, 5.620512183668325e-06, 1.8240736608277075e-05, 4.720444485428743e-05, 4.5854154450353235e-05, 1.570340282341931e-05, 0.00017087611195165664, 8.688776142662391e-05, 2.8844719054177403e-05, 7.776824531902093e-06, 1.5399253243231215e-05, 7.723529415670782e-05, 0.0011560979764908552, 3.9886544982437044e-06, 1.6107971532619558e-05, 3.303590710856952e-05, 0.007877673022449017, 6.3806505750108045e-06, 2.303846849827096e-05, 1.1786818504333496e-05, 4.366891062090872e-07, 1.4186749467626214e-05, 0.000255742110311985, 7.324618422899221e-07, 1.6034611689974554e-05, 1.1052964509872254e-05, 0.000143427838338539, 2.813864739437122e-06, 3.078365125475102e-06, 9.298466466134414e-06, 1.0404492968518753e-05, 3.065626515308395e-05, 0.00045034411596134305, 3.9817696233512834e-05, 2.5598794763936894e-06, 5.172400051378645e-05, 0.00010593551996862516, 0.000147326965816319, 0.002735764952376485, 0.0005971589125692844, 0.0001548801810713485, 0.00010171905887546018, 1.5010853076091735e-06, 0.00030511600198224187, 7.218224709504284e-06, 3.862205630866811e-05, 0.0005699241301044822, 0.001855088397860527, 2.967664840980433e-06, 2.251234036521055e-06, 4.3486576032591984e-05, 0.0001351149403490126, 1.7748667232808657e-05, 0.0002799389185383916, 1.3925966413808055e-05, 1.5027821063995361e-05, 0.00018029326747637242, 5.946120836597402e-06, 6.667376055702334e-06, 2.560369102866389e-05, 0.00027294596657156944, 0.0006388315814547241, 3.374054585947306e-06, 5.144350143382326e-05, 8.750233973842114e-05, 6.689169254059379e-07, 1.984362279472407e-05, 0.0007220963016152382, 3.6474166336120106e-06, 0.00033564091427251697, 1.9901450798442966e-07, 0.0002545454481150955, 2.232043698313646e-06, 5.550584774027811e-07, 2.8792455850634724e-05, 6.620492172260128e-07, 0.000182115807547234, 8.222208634833805e-06, 5.267106826067902e-05, 3.4861709536926355e-06, 4.153145255259005e-06, 6.3158322518575005e-06, 4.36387381341774e-05, 3.6050871585757704e-06, 0.0010481429053470492, 9.467923518968746e-06, 5.090742706670426e-05, 0.00029366978560574353, 0.0017805540701374412, 4.374594209366478e-05, 1.3522161168566527e-07, 0.0004593680496327579, 3.7768343190691667e-06, 5.035122376284562e-05, 6.147112799226306e-06, 4.0420367440674454e-05, 1.8485217879060656e-05, 0.0006152148125693202, 1.2478585631470196e-05, 2.707477324292995e-05, 3.6778328649234027e-07, 7.044809899525717e-05, 0.00011124285811092705, 1.0284842574037611e-05, 8.220956260629464e-06, 7.997012289706618e-05, 7.79948641138617e-06, 0.001088311430066824, 9.300842975790147e-07, 0.000105605686258059, 0.000252348167123273, 2.4000277335289866e-05, 0.00024518853751942515, 0.0004071200964972377, 4.497371719480725e-07, 7.255438049469376e-06, 0.0005767192342318594, 9.810553819988854e-06, 5.956670065643266e-05, 9.854735480985255e-07, 4.353356780484319e-05, 3.7213318137219176e-05, 0.0003790844639297575, 3.5499654131854186e-06, 1.9802817405434325e-05, 0.0002962297585327178, 3.1461652270081686e-06, 0.0008834712207317352, 2.755099785645143e-06, 8.934844117902685e-06, 0.00019110333232674748, 5.0289087084820494e-05, 4.541626822174294e-06, 1.962343230843544e-05, 3.7398316635517403e-05, 6.373694759531645e-06, 9.3614863772018e-07, 9.514334669802338e-05, 4.308920688345097e-05, 8.842089300742373e-05, 0.0001095349362003617, 8.925846486818045e-05, 0.0003208871348761022, 2.3968601453816518e-05, 1.1229116353206336e-05, 2.7556314307730645e-05, 3.9009737520245835e-05, 2.8896209187223576e-05, 1.9499188056215644e-05, 1.2420813618518878e-05, 0.00010223100980510935, 9.175683771900367e-06, 2.097696778946556e-05, 3.504861888359301e-05, 1.055197117239004e-05, 9.678515198174864e-05, 1.5116689610294998e-05, 1.3877796845918056e-05, 1.433363740943605e-05, 3.6745732359122485e-05, 1.1982121577602811e-05, 1.7959758906727075e-06, 0.0007184484857134521, 0.0003568620595615357, 0.000224546660319902, 1.4072475096327253e-05, 0.0006868041236884892, 0.0003888066567014903, 1.1583987316043931e-06, 3.617199126892956e-06, 1.1659885785775259e-05, 0.00024139496963471174, 2.3652468371437863e-05, 0.00013245076115708798, 1.9196631910745054e-05, 0.0001923088129842654, 8.545452146790922e-06, 9.921678611135576e-06, 3.2619891499052756e-06, 0.00013997682253830135, 8.684273780090734e-05, 0.0006623219233006239, 0.00015781968249939382, 3.703854417835828e-06, 0.0004038184997625649, 4.504952084971592e-05, 6.231626684893854e-06, 0.0019215972861275077, 1.4956711311242543e-05, 4.104160325368866e-05, 1.4353878214024007e-05, 0.0022234569769352674, 1.4822252524027135e-05, 0.0005106340977363288, 0.0001273705711355433, 3.1773888622410595e-05, 4.296554470784031e-05, 4.24923055106774e-05, 0.000265036360360682, 0.00036072556395083666, 3.846855179290287e-05, 6.517131259897724e-05, 2.9017570341238752e-05, 0.00013192783808335662, 0.0036874704528599977, 4.201976116746664e-05, 1.807137050491292e-05, 0.0004225420125294477, 6.4444193412782624e-06, 2.2445819922722876e-05, 4.803707270184532e-06, 6.404973100870848e-05, 0.0010864096693694592, 9.463823289479478e-07, 0.00016165651322808117, 0.0005668283556587994, 7.09426385583356e-05, 4.678926052292809e-05, 1.4193688912200741e-05, 0.0010949913412332535, 6.463572208303958e-05, 7.986538548721e-05, 7.948546567604353e-07, 1.318473368883133e-05, 1.5083952348504681e-05, 3.874021331284894e-06, 1.6386968127335422e-05, 2.3513439373346046e-06, 7.91557158663636e-06, 9.763261914486066e-06, 1.83420143002877e-05, 0.00012711476301774383, 4.651179915526882e-05, 0.00013316510012373328, 1.0094487151945941e-05, 0.0005378531641326845, 5.7226206990890205e-05, 8.944853107095696e-06, 5.582325002251309e-07, 2.0836470866925083e-05, 8.069584509939887e-06, 4.700581484939903e-05, 1.0183364793192595e-05, 5.204447734286077e-05, 0.0014187981141731143, 2.365342152188532e-05, 9.921180753735825e-05, 2.8223888875800185e-05, 1.7653466784395278e-05, 0.00012268712453078479, 6.055799167370424e-05, 1.48028429975966e-05, 4.058352715219371e-05, 3.015527909155935e-05, 0.0001393651618855074, 3.214641037629917e-05, 0.0005017468356527388, 1.6691092241671868e-05, 7.420220936182886e-05, 3.458459104876965e-05, 0.0007065776153467596, 5.829219276165531e-07, 6.386036693584174e-05, 2.3120887817640323e-06, 2.4012942958506756e-05, 0.0001377566804876551, 2.3255292035173625e-05, 0.0002499721304047853, 6.326452421490103e-05, 0.00037915841676294804, 0.00021222261420916766, 1.8583548808237538e-05, 7.7251697803149e-06, 5.0727390771498904e-05, 1.5570400080378022e-07, 1.7334550648229197e-05, 6.504665361717343e-05, 6.107200988481054e-06, 1.576390968693886e-05, 8.698069905221928e-06, 0.00016278796829283237, 1.046608667820692e-05, 0.00034844328183680773, 2.4800181563477963e-05, 9.371341730002314e-05, 4.394979259814136e-05, 1.9889610484824516e-05, 0.0008730983245186508, 1.2017647350148764e-05, 5.561106718232622e-06, 1.9033202988794073e-05, 0.0001858069299487397, 0.003994922153651714, 2.785849801512086e-06, 1.8790656213241164e-06, 0.00015613097639288753, 4.818934394279495e-05, 4.3017924326704815e-05, 7.0451087594847195e-06, 2.447804718030966e-06, 0.0001496980112278834, 7.736305747130245e-07, 7.646113954251632e-05, 0.0017927883891388774, 1.2537398106360342e-05, 0.00032080619712360203, 4.352645555627532e-05, 8.349171548616141e-05, 4.1519419028190896e-05, 2.271425000799354e-05, 1.9232145405112533e-06, 3.8143655274325283e-06, 4.078699930687435e-06, 0.0007605825085192919, 4.439141775947064e-06, 8.957355021266267e-05, 3.502564868540503e-05, 6.196288450155407e-05, 1.0150653650953245e-07, 0.00011315356096019968, 0.0015875592362135649, 9.324695565737784e-05, 8.84796190803172e-06, 4.147195795667358e-05, 4.062743028043769e-05, 1.5561111013084883e-06, 0.0005684365751221776, 2.006799513765145e-05, 1.2298556612222455e-05, 1.6416697690146975e-05, 0.00017672477406449616, 2.8127329642302357e-05, 1.8260939214087557e-06, 1.2682132819463732e-06, 9.145290096057579e-05, 0.00012983496708329767, 4.5991990191396326e-05, 7.65476725064218e-05, 2.311692151124589e-05, 8.855151827447116e-05, 7.410090802295599e-06, 6.455722177634016e-05, 0.00010605346324155107, 0.0011340995552018285, 3.90092452562385e-07, 2.0847999167017406e-06, 3.0865696771797957e-06, 0.00037132049328647554, 0.0006270206067711115, 0.00046933835255913436, 7.882743375375867e-06, 6.900484004290774e-05, 0.0002946245367638767, 0.0020805804524570704, 2.020764077315107e-05, 2.2681388145429082e-05, 2.169104845961556e-06, 9.653576125856489e-05, 0.00021887608454562724, 0.00010713236406445503, 2.879874227801338e-05, 0.0001117348438128829, 3.775243442305509e-07, 2.5021505280165002e-05, 7.801821197972458e-07, 0.00010266972822137177, 3.498594878692529e-06, 1.5430239272973267e-06, 1.1490095857880078e-05, 3.814187220996246e-05, 2.856543801499356e-07, 4.632683214822464e-07, 8.579110726714134e-06, 2.6190888092969544e-05, 1.9527204131009057e-05, 0.00013914101873524487, 6.43505609332351e-06, 1.4341770111059304e-05, 8.157289812515955e-06, 6.41740407445468e-05, 3.411755460547283e-05, 8.532851097697858e-06, 1.0229266536043724e-06, 5.530137423193082e-05, 5.048491220804863e-05, 3.81722111342242e-06, 6.235922774067149e-05, 1.2932930985698476e-05, 0.004373195115476847, 0.0006624374072998762, 5.365595279727131e-05, 0.00031701684929430485, 9.224974201060832e-05, 2.7909971322515048e-05, 5.2056420827284455e-05, 2.3802438136044657e-06, 4.533891569735715e-06, 0.004649084061384201, 8.541740680811927e-05, 1.875808447948657e-05, 0.0006640989449806511, 0.00033011671621352434, 6.2186511058826e-05, 8.860513844410889e-06, 2.3586335373693146e-05, 5.219655213295482e-05, 1.7253333908229251e-06, 2.087155337449076e-07, 0.0001769595401128754, 4.624479572612472e-07, 2.4037988168856828e-06, 0.0001209036199725233, 7.518366373915342e-07, 3.174065341227106e-06, 9.170646080747247e-05, 2.0546483938232996e-05, 3.054155513382284e-06, 0.0002847425057552755, 2.146507767974981e-06, 8.60439395182766e-05, 6.974848474783357e-06, 1.3205931281845551e-05, 1.0370886229793541e-05, 1.1271334187767934e-05, 8.852514952195634e-07, 1.5527564301009988e-06, 5.193278866499895e-06, 9.432048955204664e-07, 3.2486236705153715e-06, 2.1314572222763672e-05, 0.00010616311919875443, 1.7973954527406022e-05, 1.002361204882618e-05, 0.0001257895492017269, 6.3154520830721594e-06, 1.7989334082813002e-05, 0.00018978610751219094, 3.605539313866757e-05, 9.809120820136741e-05, 3.218132405891083e-06, 5.671131020790199e-06, 2.418458097963594e-05, 2.3735830836812966e-05, 0.00022590758453588933, 5.040838459535735e-06, 0.00011049478780478239, 6.050022420822643e-05, 4.1684215830173343e-05, 1.1775148777815048e-05, 5.606574268313125e-05, 1.0085913118018652e-06, 0.0007714031962677836, 0.0005346888792701066, 0.00040431771776638925, 7.3604851422715e-06, 6.40182479401119e-05, 1.5425483752551372e-06, 4.4622911445912905e-06, 1.8995729988091625e-05, 4.380197424325161e-05, 4.541362795862369e-05, 7.837865996407345e-05, 2.6216661353828385e-05, 2.1404255676316097e-05, 1.384074039378902e-05, 2.358694291615393e-05, 4.0540996337767865e-07, 3.083476622123271e-05, 5.129626879352145e-05, 1.3662380524692708e-06, 0.00010804858175106347, 8.370796535928093e-07, 6.859380391688319e-06, 2.261825829918962e-05, 0.0031103643123060465, 7.221721716632601e-06, 0.00023939687525853515, 0.00010704233136493713, 9.656640941102523e-06, 2.8559483325807378e-05, 0.00014165572065394372, 0.00016206206055358052, 3.8145219605212333e-06, 1.8353634004597552e-05, 0.00011972038919338956, 0.00024158108863048255, 6.709841545671225e-05, 3.336385634611361e-05, 0.00019706253078766167, 2.310963100171648e-05, 4.049304698128253e-05, 4.026827446068637e-05, 2.090842644975055e-05, 1.2684192824963247e-06, 3.6864719731966034e-05, 8.386048648389988e-06, 5.263728780846577e-06, 1.4452870345849078e-05, 1.1460288078524172e-05, 2.2138396161608398e-05, 0.00015611440176144242, 0.0002027143636951223, 1.1334216651448514e-05, 1.7521797417430207e-05, 1.1477807674964424e-05, 0.00013993984612170607, 8.277754659502534e-07, 2.0653098545153625e-05, 1.7135053553829493e-07, 2.4641218260512687e-05, 6.038149513187818e-05, 1.9989311113022268e-05, 5.600940653494035e-07, 0.00033314424217678607, 0.0009598974138498306, 3.4272074117325246e-05, 9.952625987352803e-06, 0.0002631984534673393, 3.1371037039207295e-05, 5.994855746394023e-05, 7.537971669080434e-07, 2.2674802835354058e-07, 3.673946775961667e-05, 3.7702063764299965e-06, 4.154800990363583e-05, 2.373831011937e-05, 0.0009611831046640873, 0.00020037034119013697, 7.111219747457653e-05, 4.17727869717055e-06, 5.512743882718496e-05, 2.8050215405528434e-05, 1.050355763254629e-06, 6.406806392078579e-07, 0.0003352843632455915, 0.00010374499106546864, 4.439908025233308e-06, 0.00016378345026168972, 5.773315206170082e-05, 1.5609779438818805e-05, 3.1659270462114364e-05, 4.6114255383145064e-05, 6.436154217226431e-05, 0.000490533304400742, 0.00017658282013144344, 0.0001246194588020444, 1.4961214219511021e-05, 6.879564898554236e-05, 2.5692619601613842e-05, 2.954949877675972e-06, 3.947336881537922e-05, 7.496828766306862e-05, 1.8869230189011432e-05, 5.751101980422391e-06, 2.2685846488457173e-05, 0.0007008782704360783, 2.433501140330918e-05, 9.056020644493401e-06, 2.706864279389265e-06, 5.842682367074303e-05, 2.1046142137493007e-05, 1.3854461030859966e-05, 9.403505828231573e-05, 4.685523163061589e-05, 9.185606177197769e-05, 0.0004235813394188881, 2.559979293437209e-05, 4.822335540666245e-05, 5.7314241530548315e-06, 1.639340007386636e-05, 1.3235551705292892e-05, 2.2398322471417487e-05, 0.00015070530935190618, 3.2198961434914963e-06, 2.3883167159510776e-05, 0.00011665301281027496, 5.692243121302454e-06, 6.0449354350566864e-05, 3.882130840793252e-06, 7.089509381330572e-06, 0.0001119742082664743, 0.0001439084589947015, 1.4480177014775109e-05, 2.4756773200351745e-05, 0.00345624890178442, 0.00016280626005027443, 6.106333785282914e-06, 4.951448499923572e-05, 5.521984007827996e-07, 0.00025508139515295625, 2.124954517057631e-05, 1.4957632629375439e-05, 8.765964594203979e-06, 0.0005651533138006926, 4.561497189570218e-05, 5.89329238209757e-06, 0.00030618117307312787, 0.00018120404274668545, 6.21822036919184e-05, 3.354345608386211e-06, 1.5245022950693965e-05, 0.00039621986798010767, 2.320471821803949e-06, 6.637387559749186e-05, 1.0295698302797973e-06, 5.50637450942304e-05, 7.628411822224734e-06, 0.0009858990088105202, 1.0990603414029465e-06, 0.00033422972774133086, 1.4735367130924715e-06, 4.141114914091304e-06, 1.6963174857664853e-05, 1.2171754860901274e-05, 0.00011793189332820475, 0.0002366886183153838, 4.464217636268586e-05, 3.237251439713873e-05, 0.003798992373049259, 2.3205397155834362e-05, 4.802214971277863e-05, 0.00014710721734445542, 0.0018126373179256916, 1.5319796148105524e-05, 2.3605603928444907e-05, 3.0162969778757542e-05, 1.96758901438443e-05, 1.937636079674121e-05, 0.0002071605558739975, 0.00037055645952932537, 1.3482391523211845e-06, 1.5683448509662412e-05, 3.485375418676995e-05, 4.8911088015302084e-06, 1.679753609096224e-06, 4.118163360544713e-06, 4.2001552174042445e-06, 0.0015748469159007072, 1.0899320841417648e-05, 2.1091796952532604e-05, 6.311709910278296e-08, 6.364357977872714e-05, 3.933021616830956e-06, 0.00010838889284059405, 5.47627723790356e-06, 2.381594094913453e-05, 5.668136873282492e-05, 0.002218989422544837, 1.865659942268394e-05, 1.8237152517031063e-06, 2.898372804338578e-05, 5.366959885577671e-05, 0.00018208719848189503, 5.819617217639461e-05, 0.0002517793036531657, 1.132796660385793e-05, 3.2849002309376374e-05, 0.00012061561574228108, 0.00011217942665098235, 1.739973527037364e-06, 0.00020221287559252232, 8.855962369125336e-05, 0.0002192324463976547, 0.00016432937991339713, 0.00011675743735395372, 4.068392627232242e-06, 1.785226686479291e-06, 2.507387989680865e-06, 2.0414433947735233e-06, 1.4942991583666299e-05, 4.0992039430420846e-05, 0.00035545899299904704, 0.0004899089108221233, 0.0001119672815548256, 0.0006762909470126033, 1.5881752915447578e-05, 0.00045681954361498356, 1.7247124333152897e-06, 2.2007110601407476e-05, 0.00047966730198822916, 0.00021372463379520923, 3.58768375008367e-05, 0.00013912883878219873, 2.2282985810306855e-05, 7.160010682127904e-06, 2.7719239369616844e-05, 4.7124070988502353e-05, 0.00014115966041572392, 1.4030481906956993e-05, 2.7467865493235877e-06, 9.172895261144731e-06, 0.00018195461598224938, 0.0006407020846381783, 5.721684374293545e-06, 0.0006620459607802331, 2.1932100935373455e-05, 0.0001492602750658989, 0.0002527803007978946, 1.3126401654517394e-06, 3.698237196658738e-05, 0.0022431793622672558, 0.00015075401461217552, 8.602133675594814e-06, 3.690879384521395e-05, 5.360562681744341e-06, 4.525265103438869e-05, 9.771274562808685e-06, 0.00026797084137797356, 0.00011248126975260675, 4.920309947920032e-06, 4.9970371946983505e-06, 6.586553809029283e-06, 3.54834764948464e-06, 6.838808621978387e-05, 3.0045700896152994e-06, 1.252240235771751e-05, 1.537446405563969e-05, 3.141373963444494e-05, 0.00010474516602698714, 0.0005908368621021509, 9.164476466594351e-08, 4.057236947119236e-05, 3.6198500311002135e-05, 0.0001679475390119478, 2.743560071394313e-05, 8.897734005586244e-06, 1.2155287549830973e-05, 3.39361940859817e-05, 0.00012062959285685793, 3.076654320466332e-05, 6.095648495829664e-05, 0.0003952951810788363, 1.1321817510179244e-05, 0.0001745182235026732, 1.1728193385351915e-05, 0.00011315089068375528, 3.4929886169265956e-05, 1.1128673804705613e-06, 1.827612504712306e-05, 4.038878614665009e-05, 2.188799589930568e-05, 7.332472705456894e-06, 1.6481009197377716e-06, 3.957096487283707e-05, 2.559688937253668e-06, 6.511182436952367e-05, 1.1885767889907584e-05, 4.349220489530126e-06, 0.00013099833449814469, 7.537906640209258e-05, 9.262224193662405e-05, 0.0004790931998286396, 3.110757234026096e-06, 8.246058860095218e-05, 6.383887921401765e-06, 1.2772163245244883e-05, 4.886575425189221e-06, 0.00010944151290459558, 0.00020771198614966124, 8.386708032048773e-06, 2.979383680212777e-05, 6.34010666544782e-06, 2.2494619770441204e-05, 9.274208991882915e-07, 3.504145206534304e-05, 3.7736557715106755e-05, 0.00018363063281867653, 5.122561924508773e-05, 1.620939110580366e-05, 7.114400068530813e-05, 0.00017456199566368014, 0.00021445841412059963, 0.0001197739620693028, 0.00021491202642209828, 0.000446359597845003, 6.222186493687332e-05, 2.5532644940540195e-05, 4.317188540881034e-06, 2.8483257210609736e-06, 3.904959885403514e-05, 0.0001130283490056172, 3.970195757574402e-05, 1.6144675100804307e-05, 0.00024357180518563837, 5.073333340988029e-06, 0.00013995646440889686, 4.8687594244256616e-05, 5.45100238014129e-06, 9.491623495705426e-05, 1.4009299775352702e-05, 4.3182447484468867e-07, 5.282013808027841e-05, 4.535001062322408e-05, 2.2690725018037483e-05, 0.0006474380497820675, 8.536700988770463e-06, 1.5531544477198622e-06, 4.7126057324931026e-05, 2.20097390410956e-05, 2.0444327674340457e-05, 0.001522245118394494, 0.00018570800602901727, 0.00021950132213532925, 0.00013161453534848988, 6.793387728976086e-05, 2.5397481294930913e-05, 0.00013690139167010784, 1.5453624655492604e-05, 9.621317985875066e-06, 2.3454649635823444e-06, 4.1518331272527575e-05, 5.062294803792611e-06, 0.00011778410407714546, 6.366031448123977e-05, 1.7314937394985463e-06, 0.00012548890663310885, 8.986610737338196e-06, 6.47099077468738e-05, 0.0003325228171888739, 5.7624900364317e-05, 0.00015002113650552928, 6.328777089947835e-05, 0.00017946495790965855, 3.7431989767355844e-05, 0.00025599985383450985, 2.617998507048469e-05, 5.23047310707625e-05, 0.00022538125631399453, 9.892887464957312e-05, 1.5475577583856648e-06, 4.514935426414013e-06, 5.704637442249805e-05, 1.5639490129615297e-06, 0.0003849476051982492, 0.003811832284554839, 0.00027552267420105636, 2.0723418856505305e-06, 5.5296117352554575e-05, 0.00035775406286120415, 9.936013157130219e-06, 0.00011979292321484536, 3.7922700357739814e-07, 0.0005168955540284514, 1.3464540643326472e-05, 3.2959491363726556e-05, 0.0018042605370283127, 5.809127105749212e-05, 0.0007526344270445406, 0.00010623937123455107, 3.07805894408375e-05, 0.00011380158684914932, 6.54949371892144e-06, 6.035091064404696e-05, 0.0001308147475356236, 3.196813850081526e-05, 3.375179585418664e-05, 0.0001197046585730277, 7.827874242138932e-07, 1.7705360733089037e-05, 0.00019170294399373233, 6.441444566007704e-05, 4.593378980644047e-05, 5.887429210815753e-07, 7.916087838566455e-07, 5.0309863581787795e-05, 7.27156875655055e-05, 3.121976260445081e-05, 2.2981832898949506e-06, 0.0001138164006988518, 4.831825935980305e-05, 0.0007635875954292715, 0.0003889179206453264, 2.6744644856080413e-05, 0.0002706566301640123, 0.0003564326325431466, 0.005064212251454592, 1.870309688456473e-06, 3.2448335787194083e-06, 6.152486093924381e-07, 7.719127097516321e-07, 6.367436071741395e-07, 4.1465616959612817e-05, 2.6146139134652913e-05, 2.184184813813772e-05, 0.00015788304153829813, 0.00011309816181892529, 4.984631232218817e-06, 0.00010430907423142344, 4.450757842278108e-05, 6.4991359067789745e-06, 2.177020360250026e-05, 6.0332833527354524e-05, 4.6173434384400025e-05, 3.948774974560365e-05, 7.709779310971498e-05, 3.631014624261297e-05, 0.00015274519682861865, 8.977295510703698e-05, 0.00012623007933143526, 1.451171101507498e-06, 3.578364703571424e-05, 0.00018624181393533945, 0.0002691461704671383, 1.1861079656227957e-05, 9.378161848871969e-06, 7.203297172964085e-06, 2.1557596483035013e-05, 0.00020072255574632436, 0.0001387552183587104, 0.00011877203360199928, 5.735733066103421e-05, 0.00013539547217078507, 1.5776438885950483e-05, 0.0002642653707880527, 0.00011322642239974812, 0.0003642636293079704, 0.000331618677591905, 0.00010288790508639067, 5.400401641963981e-05, 6.366503657773137e-05, 4.022106440970674e-05, 1.0691690476960503e-05, 0.00019909586990252137, 2.374023608808784e-07, 9.509722985967528e-06, 7.2741850090096705e-06, 4.286680450604763e-06, 0.0002549939672462642, 6.895661044836743e-06, 4.396297663333826e-05, 0.00030693269218318164, 2.4838368517521303e-06, 6.156911240395857e-06, 3.0318895369418897e-05, 1.0747833556479236e-07, 3.3720516512403265e-05, 0.0003005862236022949, 8.629908734292258e-06, 6.1977034420124255e-06, 2.3081706785887945e-06, 1.4793029095017118e-06, 0.00010533152089919895, 0.0008242808398790658, 2.370093352510594e-05, 1.635709281799791e-06, 0.00015221799549181014, 3.011701483046636e-05, 0.001326895086094737, 5.001760200684657e-06, 0.000881429121363908, 8.608275493315887e-06, 0.005439893342554569, 1.4361802641360555e-05, 0.0004872805729974061, 0.00037358637200668454, 1.2439635611372069e-05, 3.004313475685194e-05, 8.673097181599587e-05, 0.00016251954366452992, 1.1236472346354276e-05, 2.1711995941586792e-05, 0.0026929168961942196, 9.528068039799109e-05, 0.00011483268463052809, 4.3192662246838154e-07, 6.30320209893398e-05, 2.4468512492603622e-05, 7.871083653299138e-05, 2.6248242193105398e-06, 3.6895438825013116e-05, 5.805752152809873e-06, 2.7777321520261467e-05, 0.00010667676542652771, 0.0008584382012486458, 8.684084605192766e-05, 9.141671398538165e-06, 0.0001317533606197685, 1.9930321286665276e-06, 7.862196798669174e-05, 5.173117824597284e-05, 5.256350050331093e-06, 7.145883387238428e-07, 0.0008378582424484193, 0.0007821036269888282, 1.8363054550718516e-05, 1.5513589460169896e-05, 4.650745904655196e-05, 1.7891605239128694e-05, 0.0001169102979474701, 0.00047141866525635123, 4.415268267621286e-05, 0.0006664067041128874, 2.501877497707028e-05, 0.0005988402408547699, 9.716676686366554e-06, 2.3394537493004464e-05, 0.00010051406570710242, 0.00015287809947039932, 0.0002454576315358281, 3.336919326102361e-05, 0.00017579323321115226, 5.835540832777042e-06, 9.537706318951678e-06, 2.0203227904858068e-05, 3.083185220020823e-05, 8.76380581757985e-05, 0.0005797076737508178, 1.2937538485857658e-05, 0.00012500905722845346, 6.837944238213822e-05, 8.424480620305985e-05, 5.53113409296202e-07, 3.161098720738664e-05, 3.602187643991783e-05, 2.9616858228109777e-05, 1.2497686839196831e-05, 7.491244673474284e-07, 0.0012224644888192415, 4.8654808779247105e-05, 8.161956088770239e-07, 0.0002900956606026739, 1.1743536560970824e-05, 3.088426819886081e-05, 4.5470427721738815e-05, 0.0010116706835106015, 1.2439168131095357e-05, 3.845429546345258e-06, 0.017799118533730507, 0.00011612709204200655, 1.0864160685741808e-05, 3.2062778245744994e-06, 1.7526572264614515e-06, 5.306725142872892e-05, 6.833991210442036e-05, 1.1754205843317322e-05, 2.3233613319462165e-05, 3.1798255804460496e-05, 1.8289613308297703e-06, 3.525988358887844e-05, 2.8582167942658998e-05, 1.154416622739518e-06, 4.8287194658769295e-05, 6.225825200090185e-05, 1.5176523447735235e-05, 2.5442759579163976e-05, 0.00016771559603512287, 0.00011481082765385509, 0.00012186723324703053, 1.2189902008685749e-05, 0.0005079935071989894, 8.74071047292091e-05, 0.0010948417475447059, 0.0003934494452551007, 3.4889699236373417e-06, 2.217686596850399e-05, 0.00011068646563217044, 7.881750934757292e-05, 0.0002419188676867634, 1.8143557554139988e-06, 7.381618343060836e-05, 4.923893357045017e-06, 1.6358910215785727e-05, 0.00024268518609460443, 5.0773811381077394e-05, 4.171770797256613e-06, 4.949867502546113e-07, 8.13937822385924e-06, 9.077321010408923e-06, 9.306966967415065e-05, 0.0035474945325404406, 3.281782119302079e-05, 6.330265023279935e-06, 9.222293556376826e-06, 6.838779881945811e-06, 3.527410081005655e-05, 7.939457282191142e-05, 0.0004382444894872606, 0.00015718594659119844, 2.077353019558359e-05, 5.438886546471622e-06, 3.448616325840703e-08, 0.004829119425266981, 1.5268102288246155e-05, 0.0005921893171034753, 2.997980573127279e-06, 8.819666982162744e-05, 4.535554126050556e-06, 4.607201390172122e-06, 6.785253390262369e-07, 0.0001292720262426883, 4.405649178806925e-06, 0.00028850953094661236, 0.00011242325126659125, 1.2527491890068632e-05, 0.00029884828836657107, 5.657000656356104e-05, 7.638203896931373e-06, 2.388605707892566e-06, 6.20312785031274e-05, 3.121637928416021e-05, 5.176788704375213e-07, 2.4955608751042746e-05, 0.0015535467537119985, 3.710604460138711e-06, 0.000254471116932109, 5.5388871260220185e-05, 0.0003411735815461725, 2.0680681700468995e-05, 2.0792026589333545e-07, 4.0141293311535264e-07, 5.284718554321444e-06, 0.0003397389082238078, 0.00040571828139945865, 0.00011003463441738859, 6.313105404842645e-05, 4.719652224594029e-06, 6.250134174479172e-05, 9.892459274851717e-06, 0.00011363250087015331, 1.3236201084509958e-05, 7.222732165246271e-06, 0.00024155205755960196, 3.6437677408684976e-06, 3.919407390640117e-06, 2.307007525814697e-05, 0.00022007247025612742, 4.326178896008059e-06, 1.756265874064411e-06, 0.001266942359507084, 5.811051323689753e-06, 9.18447767617181e-05, 2.713743015192449e-05, 1.0266410754411481e-05, 2.831136589520611e-05, 5.688384408131242e-05, 1.8103219190379605e-05, 2.354925345571246e-05, 2.0917263100272976e-05, 3.757074182431097e-06, 0.00011201731831533834, 3.364518124726601e-05, 0.00010332497913623229, 0.0012146582594141364, 2.685528443180374e-06, 5.502476415131241e-05, 0.000155463712872006, 1.4984022527642082e-05, 0.006104255560785532, 3.286886203568429e-05, 1.8785141946864314e-05, 2.9736220312770456e-05, 3.900192677974701e-05, 1.0106004992849194e-06, 3.8661986764054745e-05, 5.563524246099405e-05, 3.1383118766825646e-05, 1.6130262281421892e-07, 5.557030817726627e-07, 0.00015225847891997546, 0.0002387456625001505, 4.29317333328072e-06, 0.00018805119907483459, 6.047280294296797e-06, 0.0001405329821864143, 1.3254663826955948e-05, 8.385612454731017e-05, 9.031154149852227e-06, 2.0447891074582003e-05, 2.77897265732463e-06, 5.382978997658938e-05, 0.004098340403288603, 5.808741843793541e-05, 0.0025808766949921846, 3.7747591704828665e-05, 4.12787267123349e-05, 0.00011059961980208755, 3.5465647670207545e-05, 0.00274469587020576, 0.00013478791515808553, 3.556076990207657e-05, 5.397808763518697e-06, 4.934741809847765e-06, 0.0004050515708513558, 5.116048487252556e-05, 0.0033706482499837875, 0.00016549929569009691, 6.3927782321115956e-06, 2.145872713299468e-05, 4.6360742999240756e-05, 0.0003509025555104017, 3.120906330877915e-05, 2.852369289030321e-05, 0.00013116806803736836, 6.187567123561166e-06, 4.100707883480936e-06, 1.8874791294365423e-06, 4.9335649237036705e-05, 0.00012238464842084795, 0.0001111752717406489, 7.303874008357525e-05, 2.051738420050242e-06, 7.770565389364492e-06, 8.782403710938524e-07, 6.982925697229803e-06, 0.00020715089340228587, 0.0009184607770293951, 9.191165190713946e-06, 3.6612687836168334e-05, 3.435655571593088e-06, 2.730742380663287e-06, 0.0031351090874522924, 5.18645865668077e-05, 2.5323510271846317e-05, 0.00011241206084378064, 1.1692343832692131e-05, 3.4868285183620173e-06, 0.0006126580992713571, 6.92592834639072e-07, 3.645829565357417e-05, 8.963563232100569e-06, 8.750851702643558e-05, 2.4805840439512394e-05, 0.0018043422605842352, 1.7185791421070462e-06, 0.00014821041258983314, 4.896560494671576e-05, 7.4104095801885705e-06, 0.00363249727524817, 2.8054305403202306e-06, 5.439103006210644e-06, 4.2969975766027346e-05, 9.207853508996777e-06, 6.920607847860083e-05, 1.3514018974092323e-05, 1.266671642952133e-05, 3.35517615894787e-05, 0.00011571837967494503, 3.883449153363472e-06, 1.576267823111266e-05, 1.083547772395832e-06, 0.00032061696401797235, 0.0004550534358713776, 5.754110134148505e-06, 0.00046052923426032066, 1.543782855151221e-05, 0.0002177196874981746, 2.2470192561740987e-05, 1.2298558431211859e-05, 1.4848645150777884e-05, 4.469551640795544e-06, 5.113915449328488e-06, 5.353168035071576e-06, 0.00011352166620781645, 2.5818935682764277e-05, 6.40748767182231e-05, 4.497015834203921e-05, 5.727136158384383e-05, 3.971834303229116e-05, 0.0007785214111208916, 1.51675285451347e-05, 5.6868349929573014e-05, 0.00023063039407134056, 0.00032802060013636947, 0.00015004727174527943, 0.0010704738087952137, 0.0004932757583446801, 0.0002838010841514915, 7.298689160961658e-05, 3.4611789487826172e-06, 2.144224936273531e-06, 8.461345714749768e-06, 1.3132017556927167e-05, 0.0007673985091969371, 0.00015708469436503947, 4.740563781524543e-06, 0.0004990744637325406, 3.109732278971933e-05, 1.0833459782588761e-05, 1.2291430721234065e-05, 0.001312852487899363, 6.765767011529533e-06, 5.941689778410364e-06, 1.799660481083265e-06, 3.411693251109682e-05, 8.739260374568403e-05, 3.046455390176561e-07, 0.0005403902614489198, 1.4241784811019897e-05, 7.485684182029217e-05, 2.6625282316672383e-06, 6.7618766479426995e-06, 0.0001785039930837229, 3.154934893245809e-05, 1.7844600733951665e-05, 0.0005706975352950394, 0.0019796148408204317, 3.1952417884895112e-06, 2.817867607518565e-05, 0.0003046703350264579, 0.000192259467439726, 7.842086233722512e-06, 0.0003696572093758732, 0.00028471057885326445, 0.0009036670671775937, 5.020756725571118e-05, 2.1371806724346243e-06, 0.0011933341156691313, 1.6138071714522084e-06, 3.536460644681938e-05, 1.639957190491259e-05, 1.5909054127405398e-05, 0.0004509110585786402, 3.503790367176407e-06, 1.5661195220673108e-06, 1.215558313560905e-05, 5.359507213142933e-06, 1.833604801504407e-05, 3.762691267183982e-05, 2.2165279006003402e-05, 6.203223892953247e-05, 4.702415753854439e-05, 1.7067117369151674e-05, 6.961507460800931e-05, 0.00047413495485670865, 1.2541459000203758e-05, 1.035230525303632e-05, 4.3787364120362327e-05, 1.1378340332157677e-06, 2.6280642487108707e-05, 3.248632492613979e-05, 6.125249910837738e-06, 5.206492801335116e-07, 9.223864617524669e-05, 0.00011847206769743934, 2.719812755458406e-06, 0.0005210003000684083, 1.3461771004585898e-06, 0.0009290261659771204, 0.00016613402112852782, 9.145867807092145e-06, 2.0650261376431445e-06, 0.0008517309906892478, 9.260310434910934e-06, 1.2573792673720163e-06, 0.001123417285270989, 6.917209248058498e-05, 3.7807847547810525e-05, 0.0001587716251378879, 4.060629362356849e-05, 3.825346357189119e-05, 6.989673124735418e-07, 9.61560999712674e-06, 7.0127175604284275e-06, 2.6599525881465524e-05, 2.4335935449926183e-05, 8.551743667339906e-05, 5.4746895330026746e-05, 7.223482498375233e-06, 2.5564102088537766e-06, 0.00020267408399377018, 9.679413051344454e-05, 0.00013801595196127892, 0.00027254875749349594, 5.184062501939479e-06, 3.456680360613973e-06, 0.00018331264436710626, 3.293645931989886e-05, 7.474374433513731e-05, 4.081451152160298e-06, 8.575365427532233e-06, 1.407875288350624e-06, 7.842376362532377e-06, 3.183805529261008e-05, 4.535536845651222e-06, 1.9892113414243795e-05, 0.0003191956493537873, 7.761630513414275e-06, 1.3023040992266033e-05, 4.3212088485233835e-07, 5.2421921282075346e-05, 0.0007429057150147855, 3.2439656934002414e-06, 9.045836122822948e-06, 0.00016272085485979915, 6.184598169056699e-05, 0.00010380949242971838, 1.5783452909090556e-05, 0.0003463687899056822, 2.580892396508716e-05, 5.003503247280605e-06, 7.021191413514316e-05, 0.0007360155577771366, 3.1302354273066157e-06, 4.842463113163831e-06, 0.0021483241580426693, 0.0009544630884192884, 2.057329038507305e-05, 0.0004592161567416042, 6.348110218823422e-06, 8.12717814824282e-07, 1.7308480892097577e-05, 0.00014387104602064937, 1.0547187230258714e-05, 0.00010190304601565003, 1.4969848507462302e-06, 5.808687274111435e-05, 8.223712939070538e-06, 0.0005538664408959448, 0.0007622293778695166, 6.342261713143671e-06, 4.310224539949559e-05, 8.820512448437512e-06, 9.139177564065903e-05, 4.667646499001421e-05, 4.823562630917877e-05, 4.718272612080909e-05, 8.248811354860663e-06, 0.00021528432262130082, 2.7650264655676438e-06, 1.4117922546574846e-06, 1.0234072078674217e-07, 9.3445260063163e-06, 0.0003650433791335672, 3.51173548551742e-06, 2.5161569283227436e-05, 6.535096872539725e-06, 5.270640758681111e-05, 0.00031329196644946933, 7.017784082563594e-05, 4.258862463757396e-05, 1.3785135706712026e-05, 0.00011382708180462942, 0.0005534921656362712, 7.660557457711548e-05, 5.465965841722209e-06, 5.704411887563765e-05, 6.264793773880228e-05, 1.382920345349703e-05, 1.7614134776522405e-05, 1.2980912060811534e-06, 2.9033955797785893e-05, 7.336176111039094e-08, 2.283573485328816e-05, 8.817901107249781e-05, 2.379943362029735e-05, 0.0006980490870773792, 8.508347491442692e-06, 0.001493404502980411, 0.0001281380100408569, 2.6524717213760596e-06, 4.0682226654098486e-07, 7.429391553159803e-05, 2.9003658710280433e-06, 2.5506697056698613e-06, 0.00010632643534336239, 0.00011656229617074132, 1.9765400793403387e-05, 1.3005371329199988e-05, 8.666820212965831e-05, 0.0006792983040213585, 2.1256073523545638e-06, 0.0003113236161880195, 2.9595630621770397e-05, 0.001127368537709117, 4.059267666889355e-06, 0.0001488360285293311, 5.023821358918212e-06, 0.0005830847658216953, 7.166589057305828e-05, 3.805995447692112e-06, 0.0003951092658098787, 0.0012398792896419764, 9.327389307145495e-06, 3.753350392798893e-05, 3.226051831006771e-06, 7.303484017029405e-05, 7.47228114050813e-05, 0.00031021170434542, 9.652121661929414e-05, 8.972555224318057e-05, 8.885605893738102e-06, 2.696208412089618e-06, 0.0002975997340399772, 0.0007774236728437245, 4.8107991460710764e-05, 1.537991556688212e-05, 0.0002903317508753389, 2.8697793368337443e-06, 0.0014394153840839863, 6.934543489478528e-05, 0.0002998585405293852, 0.0002719109470490366, 0.0009500685846433043, 6.1296988860704e-05, 1.6450910607090918e-06, 2.0219094949425198e-05, 1.3900962585466914e-05, 0.000702607212588191, 4.228640318615362e-05, 0.00017063456471078098, 8.442373655270785e-06, 1.0882154128921684e-05, 0.0006075110286474228, 0.00011048753367504105, 6.974766165512847e-06, 0.0003170672571286559, 1.682962647464592e-05, 3.942315743188374e-05, 7.17220245860517e-05, 0.0005543897277675569, 6.414804829546483e-07, 2.2115506226327852e-07, 4.5242336454975884e-06, 0.0002853010082617402, 1.7322852727374993e-05, 0.00010329963697586209, 7.90976901043905e-06, 6.124859646661207e-05, 0.0008624126203358173, 2.3255579435499385e-05, 0.00042662228224799037, 0.00011412981984904036, 5.915496217312466e-07, 0.000594266748521477, 1.804666499083396e-05, 1.6462427083752118e-05, 1.2432878065737896e-05, 0.00016200235404539853, 1.4720685612701345e-05, 0.0016333198873326182, 3.310364263597876e-05, 3.9372309402097017e-05, 8.336066093761474e-05, 1.645271186134778e-05, 3.5893404856324196e-05, 0.00010059544729301706, 0.0002967599139083177, 0.001509635359980166, 0.00021481204021256417, 3.747348819160834e-05, 8.884767339623068e-06, 6.669613503618166e-05, 0.00028450993704609573, 3.832004495052388e-06, 0.00031212932663038373, 2.8307809770922177e-05, 3.23267922794912e-05, 1.472559779358562e-05, 8.712849194125738e-06, 3.866554834530689e-05, 3.480412851786241e-05, 5.113049155625049e-06, 0.000136342627229169, 1.14589943223109e-06, 0.0013232029741629958, 6.729825599904871e-06, 1.449347928428324e-05, 0.0003593278815969825, 0.00024503408349119127, 0.0002974012750200927, 0.0008508925093337893, 1.9778246496571228e-05, 1.011866879707668e-05, 3.8187608879525214e-05, 0.0010843115160241723, 6.738767297065351e-06, 2.6934989364235662e-05, 1.1276040822849609e-05, 0.00010836347792064771, 0.00015723898832220584, 0.00039139887667261064, 2.1588289200735744e-07, 9.959449380403385e-05, 0.0003955739666707814, 0.00020090148609597236, 1.5965850252541713e-05, 2.1807818484376185e-05, 4.817806984647177e-05, 1.4849717445031274e-05, 1.305298792431131e-05, 5.1878418162232265e-05, 2.72345960183884e-06, 0.0001405627408530563, 0.0002974721137434244, 0.00022705727315042168, 3.159530751872808e-05, 0.0002096813841490075, 5.23815788255888e-06, 0.0002733335713855922, 0.00026806400273926556, 0.0005095988744869828, 6.973732524784282e-06, 1.6861727999639697e-05, 0.0005025650025345385, 9.288592082157265e-06, 5.2325622164062224e-06, 6.122134072938934e-05, 1.5238815649354365e-05, 3.658409332274459e-05, 0.001220041885972023, 0.0001465745153836906, 7.24433903087629e-06, 9.565640357322991e-05, 8.22882884676801e-06, 0.000641689810436219, 4.8436275392305106e-05, 1.4179573554429226e-05, 0.00010118157661054283, 0.00010934261081274599, 2.1604395783469954e-07, 7.545313565060496e-06, 4.970124336978188e-06, 6.06640605838038e-06, 7.618633389938623e-05, 4.4909921825819765e-07, 0.00038918553036637604, 0.0005750989657826722, 0.0016686308663338423, 2.50529501499841e-05, 0.0002241302136098966, 1.766802233760245e-05, 2.725167860262445e-06, 0.0029769050888717175, 3.6350949812913314e-05, 5.04209274367895e-05, 2.192190549976658e-05, 0.0003837669501081109, 9.421519280294888e-07, 0.0002799333306029439, 9.272054012399167e-05, 4.390987214719644e-06, 3.059448090425576e-06, 1.2560198001665412e-06, 9.258768841391429e-05, 3.6735287721967325e-05, 4.177358277956955e-05, 5.645661076414399e-06, 6.020720684318803e-05, 2.8678050512098707e-05, 6.089239832363091e-06, 0.007381945848464966, 0.00020374666200950742, 0.00015972807887010276, 2.9477374482667074e-05, 5.138158485351596e-06, 0.00033278827322646976, 6.979043519095285e-06, 0.000526752439327538, 5.7536279200576246e-05, 7.680321868974715e-05, 9.629652595322113e-06, 5.370040889829397e-05, 0.0003839517303276807, 2.0743923414556775e-06, 8.020546374609694e-05, 1.1361858923919499e-05, 8.253748819697648e-05, 0.00019336416153237224, 0.00015304483531508595, 0.0001207332024932839, 0.0006228525890037417, 1.1851612953250878e-06, 3.663152892841026e-05, 3.27905363519676e-05, 1.185760811495129e-05, 1.649656769586727e-05, 8.563659434912552e-07, 7.69476464483887e-05, 0.00017997820395976305, 2.590231815702282e-05, 1.6590953464401537e-06, 0.000655722338706255, 1.9160666852258146e-05, 1.5089485714270268e-06, 3.7017463910160586e-05, 1.1555307111166258e-07, 8.550779966753908e-06, 3.6835115224675974e-06, 8.212390821427107e-05, 3.78272261514212e-06, 0.0005991850048303604, 6.900582229718566e-05, 1.0066218237625435e-05, 1.7407275663572364e-05, 0.00045926158782094717, 2.4889657197491033e-06, 2.881755972339306e-05, 3.688269862323068e-05, 7.966851626406424e-06, 2.3752458218950778e-05, 4.4338838051771745e-05, 8.449733286397532e-07, 5.670561222359538e-05, 0.000822392466943711, 6.882342859171331e-05, 5.863927526661428e-06, 0.00012579304166138172, 0.0001870810374384746, 2.4771963580860756e-05, 2.148349409480943e-07, 4.126988278585486e-05, 0.0005745107773691416, 5.85567477173754e-06, 8.492093911627308e-05, 1.2714377589873038e-05, 0.00010439042671350762, 3.501903120195493e-05, 2.1843230570084415e-05, 1.2345245522737969e-05, 1.1429490768932737e-05, 0.00327874394133687, 9.221245272783563e-05, 0.0011294397991150618, 6.920735904714093e-05, 6.478543582488783e-06, 2.3911672997201094e-07, 7.636030204594135e-06, 0.001771003007888794, 4.10464963351842e-05, 5.197644713916816e-05, 3.344081051182002e-05, 2.8808950446546078e-05, 1.684554990788456e-05, 3.618862933763012e-07, 1.3097502232994884e-05, 0.00013303806190378964, 1.0790840860863682e-05, 1.0934537613138673e-06, 0.0049619534984230995, 0.0001096791893360205, 8.108502697723452e-06, 8.245408935181331e-06, 2.0257073629181832e-05, 0.002306178444996476, 5.23446942679584e-06, 1.6215086361626163e-05, 5.062110176368151e-06, 2.7332987883710302e-05, 1.6536380087472935e-07, 4.7217836254276335e-05, 1.2090790733054746e-05, 2.476909685356077e-06, 0.00012680774671025574, 4.8384267756773625e-06, 1.4386606380867306e-05, 5.5962402257137e-05, 4.837778033106588e-05, 4.9206995754502714e-05, 8.218603397835977e-06, 0.0005021350807510316, 0.0008098737453110516, 2.2770578652853146e-05, 0.000313229305902496, 2.063368447124958e-05, 1.0987307177856565e-05, 1.2370149306661915e-05, 0.002919155405834317, 0.00031354924431070685, 4.142761099501513e-06, 8.669363160151988e-05, 1.2545704521471635e-05, 1.0650369404174853e-05, 0.00010878640750888735, 0.0001289206848014146, 2.521953319956083e-06, 0.00011558190453797579, 1.1247617294429801e-05, 8.628241630503908e-06, 0.00020892989414278418, 1.950769728864543e-05, 0.00036753463791683316, 2.3280188088392606e-06, 4.787335637956858e-05, 0.0054841432720422745, 6.305486022029072e-05, 1.6638181477901526e-05, 0.00017323464271612465, 1.8871816791943274e-05, 0.00015374725626315922, 1.0610698382151895e-06, 7.986749551491812e-06, 4.072862066095695e-05, 9.26244756556116e-05, 1.3507401490642224e-05, 1.2151295777584892e-05, 3.658207424450666e-05, 1.3865609616914298e-05, 1.0577973625913728e-05, 0.00018055546388495713, 4.93208199259243e-06, 0.0009433968225494027, 0.0008075416553765535, 7.544150321336929e-06, 0.0010609527816995978, 9.322479309048504e-05, 3.57101671397686e-05, 4.2702708014985546e-05, 0.00032637754338793457, 0.0003245741536375135, 6.597149331355467e-05, 4.290573997423053e-06, 1.4405763977265451e-06, 2.812471939250827e-05, 2.578535713837482e-05, 0.0008778225746937096, 8.44582984882436e-07, 5.5071063798095565e-06, 1.0111154551850632e-05, 0.00014213849499356002, 4.809173697140068e-05, 3.639950227807276e-05, 0.0005167663912288845, 6.545141786773456e-06, 0.0001910916471388191, 2.951962187580648e-06, 0.0007451914716511965, 1.2866703400504775e-05, 1.0297796507074963e-05, 6.800416304031387e-05, 0.0003436682454776019, 1.1548672773642465e-05, 1.624849028303288e-05, 6.286517873377306e-06, 1.5606659644618048e-06, 6.212611651790212e-07, 0.004272969905287027, 0.00018677332263905555, 0.00014025116979610175, 1.6132427845150232e-05, 4.366220764495665e-06, 2.156241635020706e-06, 1.9934923329856247e-05, 0.0009618100593797863, 3.481853354969644e-06, 2.268023763463134e-06, 0.0007095257751643658, 0.0001692700752755627, 2.1166581518627936e-06, 9.334469723398797e-06, 9.825808774621692e-06, 5.770565985585563e-05, 1.3385820238909218e-05, 3.081612192090688e-07, 8.380143117392436e-05, 3.284125341451727e-05, 0.00011388245184207335, 4.409309622133151e-05, 1.3710757230001036e-05, 1.4308337995316833e-05, 9.420852620678488e-06, 0.00016097375191748142, 0.0002035010838881135, 5.367798075894825e-05, 5.588779094978236e-05, 0.00021223035582806915, 0.00014903279952704906, 4.509424979914911e-05, 2.265524562972132e-05, 0.00013042024511378258, 5.917246653552866e-06, 5.804730608360842e-05, 1.3591929928225e-05, 1.0844720236491412e-05, 0.0007508035050705075, 0.0011956566013395786, 1.3645997569255996e-05, 3.981763256888371e-06, 4.709479071607348e-06, 0.0001378711313009262, 2.062330713670235e-05, 8.694407006260008e-05, 6.0055281210225075e-06, 1.5805804878255003e-06, 1.1506675036798697e-05, 0.0005937149981036782, 6.675934855593368e-05, 1.3595961263490608e-06, 0.0007758381543681026, 0.0014598887646570802, 4.7459059260290815e-07, 5.675370630342513e-06, 1.766505192790646e-05, 6.241659866645932e-05, 3.67779030057136e-05, 1.2195327144581825e-05, 0.00015667140542063862, 9.466885785514023e-06, 9.694217624200974e-07, 1.628983227419667e-05, 1.5589266695315018e-06, 5.6184006098192185e-05, 0.0002886798174586147, 0.00010739508434198797, 7.726112380623817e-05, 0.00022610703308600932, 0.0010005279909819365, 0.00012486822379287332, 4.0228827856481075e-05, 0.0005623847246170044, 6.650597788393497e-05, 1.7997703253058717e-05, 4.176640231889905e-06, 2.6069531600114715e-07, 1.593574597791303e-05, 1.1117021131212823e-05, 3.425056638661772e-05, 4.736889968626201e-05, 0.0009978475281968713, 1.6776272104834788e-06, 1.1543829714355525e-05, 7.239542173920199e-05, 5.588705789705273e-06, 1.929510835907422e-05, 0.00019058267935179174, 0.00011759267363231629, 6.556313564942684e-06, 9.70232122199377e-07, 0.0002658702724147588, 0.00042725083767436445, 0.0013380151940509677, 2.062403473246377e-05, 7.6195597102923784e-06, 2.952677277789917e-05, 3.27772468153853e-05, 0.0017521986737847328, 4.924861968902405e-06, 0.00012454757234081626, 3.1389601645059884e-05, 4.2499174014665186e-05, 7.519707651226781e-06, 0.0002665666688699275, 0.000243680682615377, 2.933493988166447e-06, 8.636023267172277e-06, 5.5730938584019896e-06, 1.4564222510671243e-05, 6.924650369910523e-05, 3.674430502087489e-07, 1.6953952126641525e-06, 0.0002884071145672351, 0.0016143904067575932, 1.5214001905405894e-05, 4.836389416595921e-05, 0.002108996035531163, 2.4153296180884354e-05, 9.078343623514229e-07, 9.871198562905192e-05, 0.0001310677034780383, 0.00028183156973682344, 7.511479452659842e-06, 0.0010353375691920519, 4.109628207515925e-06, 2.97872848022962e-05, 0.003344345837831497, 8.635513040644582e-06, 1.4945580915082246e-05, 0.00016006032819859684, 0.0031633146572858095, 0.0002871447359211743, 1.3914159353589639e-05, 2.0133309590164572e-05, 1.6440340914414264e-05, 0.0005585671169683337, 3.398331318749115e-05, 5.2740575483767316e-05, 0.0024619700852781534, 4.93807892780751e-05, 6.066720015951432e-05, 0.0020012592431157827, 7.069834850881307e-08, 3.2150095648830757e-05, 2.3191582840809133e-06, 5.976515694783302e-06, 1.4901135727996007e-05, 4.338984581409022e-05, 0.0003325515135657042, 6.226860477909213e-06, 0.00012010148930130526, 0.0003651615115813911, 3.094457133556716e-05, 5.002574835089035e-05, 5.600337317446247e-06, 0.00010055156599264592, 4.3210255284975574e-07, 0.00022112471924629062, 6.56647389405407e-05, 0.00029254131368361413, 1.6834379493957385e-05, 9.235160973730672e-07, 4.636759967979742e-06, 6.194260095071513e-06, 8.82575932337204e-07, 8.306004019686952e-05, 0.00010894631122937426, 0.00043686421122401953, 1.2919781511300243e-06, 3.6572495446307585e-05, 4.3038813601015136e-05, 4.429651380633004e-05, 0.0003750405157916248, 0.0011106616584584117, 4.105087555217324e-06, 6.756702646271151e-07, 0.00027640926418825984, 2.181257696065586e-05, 2.4439179924229393e-06, 2.247575503133703e-05, 1.6179548765649088e-05, 0.00033814943162724376, 4.245328000251902e-06, 0.004333373159170151, 0.00010245793964713812, 1.0535881301620975e-05, 6.546776603499893e-06, 1.0421731531096157e-05, 4.113821341888979e-05, 3.4907359804492444e-05, 1.5227405128825922e-05, 1.8063974493998103e-05, 3.137535895803012e-05, 9.673194290371612e-05, 0.0003075548156630248, 3.357208470333717e-06, 5.359660008252831e-06, 8.243342563218903e-06, 6.358458631439134e-05, 0.0002456643560435623, 7.16450231266208e-05, 7.116265351214679e-06, 0.000123259422252886, 0.000157311704242602, 2.2705726223648526e-05, 0.00010973949247272685, 0.0007222848362289369, 1.3258520539238816e-06, 8.849288860801607e-05, 6.373563223860401e-07, 9.352546840091236e-06, 0.00011647836799966171, 1.1627930689428467e-06, 0.0002532505604904145, 0.00048284733202308416, 2.131445853592595e-06, 3.3556559628777904e-06, 1.3321772712515667e-05, 0.0008852340979501605, 9.453783604840282e-07, 2.0506671717157587e-05, 0.00016136355407070369, 3.097686203545891e-05, 1.8358787201577798e-05, 0.0002617945719975978, 1.1760635061364155e-05, 1.63978329510428e-05, 6.0482689150376245e-05, 0.00034441493335179985, 4.978673314326443e-05, 5.069639883004129e-06, 0.0003853150992654264, 9.098683221964166e-06, 2.8569404094014317e-05, 0.0002307021350134164, 0.0030032906215637922, 1.0309097433491843e-06, 1.1238637853239197e-05, 8.256318324129097e-06, 1.9286378574179253e-06, 0.00010138091602129862, 2.03777126444038e-05, 0.00018238580378238112, 0.00017427334387321025, 1.5870344213908538e-05, 4.825935775443213e-06, 0.00017222564201802015, 6.292072794167325e-05, 4.17821247538086e-05, 8.958135731518269e-05, 8.503870390086377e-07, 0.0011239858577027917, 0.002507381606847048, 7.98291657702066e-05, 4.826903386856429e-05, 4.8016663640737534e-05, 3.4003805922111496e-06, 0.0017002171371132135, 2.5089697373914532e-05, 0.00026455195620656013, 0.00022383972827810794, 2.6849752430280205e-06, 1.0146056411031168e-05, 5.4383431233873125e-06, 3.5463574477034854e-06, 3.237933196942322e-05, 0.0001525932311778888, 0.0002575649123173207, 3.418753112782724e-05, 4.734705726150423e-05, 0.00028804840985685587, 3.2502550766366767e-06, 2.0328774553490803e-05, 9.298450095229782e-06, 7.884759725129697e-07, 2.2112173780897137e-07, 5.037036680732854e-05, 1.9521585272741504e-05, 0.0004137800424359739, 7.711770990681543e-07, 4.704800448962487e-05, 8.557138062315062e-05, 0.00034094892907887697, 4.699452802015003e-06, 2.484761535015423e-05, 2.9462327802320942e-05, 1.3747964658250567e-05, 0.00016169600712601095, 5.644048997055506e-07, 0.00016986009723041207, 1.2557500213006278e-06, 2.562817644502502e-05, 3.1733245577925118e-06, 1.9256984160165302e-05, 9.469589713262394e-05, 1.4226861821953207e-05, 3.726363866007887e-05, 2.226141805294901e-05, 2.478408305250923e-06, 8.374079334316775e-05, 8.096987585304305e-05, 2.5668362013675505e-07, 1.2195177987450734e-05, 8.183774480130523e-05, 4.934800017508678e-05, 0.00013056590978521854, 9.644113561080303e-07, 1.8035372704616748e-05, 1.0696820936573204e-05, 1.1626583727775142e-05, 0.002094217808917165, 4.992457979824394e-05, 1.1349725355103146e-05, 0.00010156955249840394, 5.8424768212717026e-05, 0.0009362474083900452, 0.0006012587109580636, 4.9400496209273115e-05, 2.6700348826125264e-05, 0.00016289895575027913, 2.5924158762791194e-05, 1.0732765076681972e-05, 3.0223441171983723e-06, 5.5251959565794095e-05, 1.1134453416161705e-05, 4.0752482163952664e-05, 5.574629085458582e-06, 4.084676766069606e-05, 0.000992911052890122, 2.7741566555050667e-06, 2.2893285859026946e-05, 3.6662386264652014e-05, 4.8269004764733836e-05, 4.5475746446754783e-05, 5.496618541656062e-06, 0.00010214703070232645, 2.3329705527430633e-06, 4.2202659642498475e-06, 3.101843685726635e-05, 0.00038232255610637367, 1.6888572645257227e-05, 0.0017598342383280396, 2.8910511900903657e-05, 0.000662432808894664, 2.5976505639846437e-06, 1.8575206922832876e-05, 1.1820283361885231e-05, 7.826843648217618e-05, 0.0025698344688862562, 5.823097308166325e-05, 7.584958802908659e-05, 6.395333184627816e-05, 0.00039277790347114205, 1.8351240214542486e-05, 3.346155790495686e-05, 2.657880395418033e-05, 0.00015225083916448057, 9.294133178627817e-07, 7.323553745663958e-06, 1.0850090802705381e-05, 1.2465677173167933e-05, 8.056307706283405e-05, 1.7492908227723092e-05, 0.0002708487445488572, 4.262602305971086e-05, 0.0007385820499621332, 9.504638910584617e-06, 5.793801028630696e-05, 0.0003665373951662332, 1.8466160327079706e-05, 5.223459538683528e-06, 9.493280231254175e-06, 7.551760063506663e-06, 6.102163752075285e-05, 1.1599463505262975e-06, 2.6983930183632765e-06, 3.44636537192855e-05, 3.229684807593003e-05, 1.5105855709407479e-05, 6.7249793573864736e-06, 0.0006504780612885952, 1.0992423085554037e-05, 5.815090480609797e-05, 4.047847141919192e-06, 1.1905717656190973e-05, 4.126181465835543e-06, 0.00014877547801006585, 4.253808219800703e-05, 1.1083241588494275e-05, 1.2135597899032291e-05, 0.00010777695570141077, 6.562100315932184e-05, 0.00010323180322302505, 1.7413278328604065e-05, 2.7693140509654768e-05, 1.5820953649381408e-06, 3.14305470965337e-05, 3.499292461128789e-06, 3.381293936399743e-05, 5.156132101546973e-05, 1.9218798115616664e-05, 9.187703108182177e-05, 0.00010564670083113015, 2.617199697851902e-06, 1.891686588351149e-05, 2.2927213649381883e-05, 1.6430692539870506e-06, 5.34319224243518e-06, 0.001031782478094101, 6.976963049964979e-05, 7.334109250223264e-05, 1.3870181646780111e-05, 3.056332161577302e-06, 3.3961212466238067e-05, 1.405819489264104e-06, 2.3509817765443586e-05, 3.712458783411421e-05, 4.422491474542767e-05, 1.1628861102508381e-05, 1.0321577974536922e-05, 3.7199435610091314e-05, 2.2718160835211165e-05, 0.00034373829839751124, 0.00013391744869295508, 8.585250725445803e-06, 7.143247330532176e-06, 6.605734233744442e-05, 2.3937657260830747e-06, 5.290391345624812e-05, 4.246139724273235e-06, 1.7851419897851883e-06, 6.753415618732106e-06, 0.0024743806570768356, 6.009046410326846e-05, 1.5511073797824793e-05, 1.5503568647545762e-05, 0.0002644045162014663, 2.7029889679397456e-05, 0.00014188836212269962, 1.8949856894323602e-05, 0.00013562670210376382, 5.729964414058486e-06, 8.846262790029868e-05, 6.96396455168724e-05, 0.00035746776848100126, 7.645546247658785e-06, 0.00010307375487172976, 0.0001184799984912388, 0.0001436874154023826, 0.0002073562063742429, 1.0942697372229304e-05, 4.1814905671344604e-07, 5.641404277412221e-05, 5.282450729282573e-05, 0.0002046949084615335, 6.24485473963432e-05, 1.9456108930171467e-05, 4.717799583886517e-06, 0.00013731198851019144, 1.561144927109126e-05, 1.4848028513370082e-05, 0.0006028044736012816, 1.731012844174984e-06, 6.463381851062877e-06, 3.7728509596490767e-06, 0.00013479038898367435, 1.082854942069389e-05, 4.541899670584826e-06, 0.0003120160836260766, 1.3493562619260047e-05, 6.640338142460678e-06, 8.799310307949781e-05, 8.207983046304435e-05, 1.2076842722308356e-05, 4.153617737756576e-06, 0.0003369946498423815, 1.2356025763438083e-05, 5.217000943957828e-05, 0.0003682954120449722, 1.4752908100490458e-05, 5.312601206242107e-05, 4.414358500071103e-06, 1.9071063434239477e-05]], "dtype": "float32"}}, {"a": [{"__nd__": [[-1000.0, 0.0, 1000.0]], "dtype": "float32"}], "e": {"__nd__": [[0.0, 0.0, 1.0]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(softmax_rows, json.loads(_TESTS), 'triton', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")